In [1]:
"""
USDINR HMM-State Strategy Backtest
====================================
Signal  : LightGBM CHALLENGER50 predicted ret_60s
Regimes : 4-state HMM (Quiet / Stressed / Trending / Mean-reverting)
Entry   : Every bar — no signal filtering (all signals traded)
Exit    : Fixed 60s hold
Sizing  : State-based position multiplier (derived from realized vol per state)
Costs   : 1 bps round-trip (conservative NSE USDINR futures estimate)

Outputs : results/strategy_backtest/
            metrics.json        — overall + per-state Sharpe, hit rate, PnL
            trade_log.csv       — every trade with entry/exit/state/PnL
            daily_pnl.csv       — daily PnL + Sharpe + drawdown
            state_metrics.csv   — per-HMM-state breakdown
            comparison.csv      — flat sizing vs HMM-state sizing

No plots — CSV/JSON outputs only.
"""

import sys
sys.path.insert(0, r"C:\projects\hedge_fund")

import json
import logging
import pathlib
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import lightgbm as lgb
from scipy.stats import spearmanr

from data.store.duckdb_client import DuckDBClient
from data.store.s3_client import S3Client
from data.processing.usdinr.features import USDINRFeatureBuilder
from data.processing.usdinr.constants import CHALLENGER50
from data.processing.usdinr.regimes import (
    add_soft_hmm_features,
    save_hmm_model,
    load_hmm_model,
    HMM_PROB_COLS,
)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)s  %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger(__name__)

# ---------------------------------------------------------------------------
# Config
# ---------------------------------------------------------------------------

OUT_DIR   = pathlib.Path(r"C:\projects\hedge_fund\results\strategy_backtest")
MODEL_DIR = pathlib.Path(r"C:\projects\hedge_fund\models")
OUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

HORIZONS   = [60, 300]
TRAIN_DAYS = 15
VAL_DAYS   = 5
COST_BPS   = 1.0        # round-trip transaction cost in bps
HOLD_SEC   = 60         # fixed exit after 60 seconds

# LightGBM params — same as lgbm_baseline
LGB_PARAMS = {
    "objective":             "regression",
    "metric":                "rmse",
    "verbosity":             -1,
    "boosting_type":         "gbdt",
    "num_leaves":            31,
    "learning_rate":         0.05,
    "feature_fraction":      0.8,
    "bagging_fraction":      0.8,
    "bagging_freq":          5,
    "min_child_samples":     100,
    "n_estimators":          1000,
    "early_stopping_rounds": 50,
}

# Per-state position size multiplier
# Derived from your experiment table:
#   State 4: RV=1.08 bps² (quietest)   → full size = 1.0
#   State 3: RV=2.59 bps²              → 0.65
#   State 1: RV=3.14 bps²              → 0.59
#   State 2: RV=5.62 bps² (stressed)   → 0.44
# Formula: multiplier = sqrt(RV_state4 / RV_state_k) — vol-normalizing
STATE_SIZE = {
    1: 0.59,
    2: 0.44,
    3: 0.65,
    4: 1.00,
}

# Per-state stop-loss distance in bps
# Derived from future_60s_return_std per state × 1.5
STATE_STOP_BPS = {
    1: 2.34,   # 1.5616 * 1.5
    2: 3.08,   # 2.0507 * 1.5
    3: 2.04,   # 1.3594 * 1.5
    4: 1.33,   # 0.8865 * 1.5
}

# ---------------------------------------------------------------------------
# Step 1 — Load raw data
# ---------------------------------------------------------------------------

log.info("=== Step 1: Loading raw data ===")

db = DuckDBClient()
s  = S3Client()

all_contracts = ["26SEPFUT", "26OCTFUT", "26NOVFUT"]
all_frames    = []

for contract in all_contracts:
    files = s.list_files(f"processed/1s/USDINR{contract}/")
    if not files:
        log.info("No files for USDINR%s — skipping", contract)
        continue
    for f in files:
        parts = f.split("/")
        sym   = parts[2]
        date  = parts[3].replace(".parquet", "")
        try:
            df_day           = db.read_parquet(f"processed/1s/{sym}/{date}.parquet")
            df_day["symbol"] = sym
            all_frames.append(df_day)
        except Exception as e:
            log.warning("Failed %s/%s: %s", sym, date, e)

db.close()

if not all_frames:
    raise RuntimeError("No data loaded")

df_raw = pd.concat(all_frames, ignore_index=True)
log.info("Raw: %d rows | symbols: %s", len(df_raw), df_raw["symbol"].unique().tolist())

# ---------------------------------------------------------------------------
# Step 2 — Build features
# ---------------------------------------------------------------------------

log.info("=== Step 2: Building features ===")

builder = USDINRFeatureBuilder(data=df_raw, auto_build=True)
df      = builder.dataset.copy()
log.info("Dataset: %d rows, %d cols", len(df), len(df.columns))

# ---------------------------------------------------------------------------
# Step 3 — Validate tick_count
# ---------------------------------------------------------------------------

if "tick_count" not in df.columns:
    raise RuntimeError("tick_count not found in builder output")
df["tick_count"] = df["tick_count"].fillna(0).astype(int)

# ---------------------------------------------------------------------------
# Step 4 — Split dates
# ---------------------------------------------------------------------------

log.info("=== Step 4: Splitting ===")

df        = df.sort_values(["symbol", "date", "ts_sec"]).reset_index(drop=True)
all_dates = sorted(df["date"].unique())
n_days    = len(all_dates)
log.info("Total days: %d  (%s → %s)", n_days, all_dates[0], all_dates[-1])

if n_days < TRAIN_DAYS + VAL_DAYS + 1:
    raise RuntimeError(f"Need at least {TRAIN_DAYS + VAL_DAYS + 1} days, got {n_days}")

train_dates = all_dates[:TRAIN_DAYS]
val_dates   = all_dates[TRAIN_DAYS : TRAIN_DAYS + VAL_DAYS]
test_dates  = all_dates[TRAIN_DAYS + VAL_DAYS:]
train_end   = str(train_dates[-1])

log.info("Train: %d  Val: %d  Test: %d days",
         len(train_dates), len(val_dates), len(test_dates))

mask_train = df["date"].isin(train_dates)
mask_val   = df["date"].isin(val_dates)
mask_test  = df["date"].isin(test_dates)

# ---------------------------------------------------------------------------
# Step 5 — Fit HMM
# ---------------------------------------------------------------------------

log.info("=== Step 5: Fitting HMM ===")

df, hmm_model = add_soft_hmm_features(
    df             = df,
    train_end_date = train_end,
    K              = 4,
    restarts       = 3,
    max_iter       = 70,
    tol            = 1e-6,
    seed           = 42,
)
save_hmm_model(hmm_model, MODEL_DIR / "hmm_usdinr_strategy.pkl")
log.info("State distribution:\n%s",
         df["state_filtered"].value_counts().sort_index().to_string())

# ---------------------------------------------------------------------------
# Step 6 — Targets + mid price
# ---------------------------------------------------------------------------

log.info("=== Step 6: Building targets ===")

if "mid" not in df.columns:
    df["mid"] = (df["bid_p1"] + df["ask_p1"]) / 2.0
df["log_mid"] = np.log(df["mid"])

g = df.groupby(["symbol", "date"], sort=False)
df["ret_60s"]  = (g["log_mid"].shift(-60)  - df["log_mid"]) * 1e4
df["ret_300s"] = (g["log_mid"].shift(-300) - df["log_mid"]) * 1e4

# Forward mid price (what you actually get at exit after HOLD_SEC)
df["mid_exit"] = g["mid"].shift(-HOLD_SEC)

log.info("ret_60s non-null: %.1f%%", df["ret_60s"].notna().mean() * 100)

# ---------------------------------------------------------------------------
# Step 7 — Train LightGBM on training data
# ---------------------------------------------------------------------------

log.info("=== Step 7: Training LightGBM ===")

features = CHALLENGER50
target   = "ret_60s"

valid_mask = df[features + [target]].notna().all(axis=1)

tr = df[mask_train & valid_mask]
va = df[mask_val   & valid_mask]

X_tr, y_tr = tr[features].values, tr[target].values
X_va, y_va = va[features].values, va[target].values

ds_tr = lgb.Dataset(X_tr, label=y_tr, feature_name=features)
ds_va = lgb.Dataset(X_va, label=y_va, feature_name=features, reference=ds_tr)

params = {k: v for k, v in LGB_PARAMS.items()
          if k not in ("n_estimators", "early_stopping_rounds")}

lgbm_model = lgb.train(
    params,
    ds_tr,
    num_boost_round     = LGB_PARAMS["n_estimators"],
    valid_sets          = [ds_va],
    callbacks           = [
        lgb.early_stopping(LGB_PARAMS["early_stopping_rounds"], verbose=False),
        lgb.log_evaluation(100),
    ],
)
log.info("LightGBM best iteration: %d", lgbm_model.best_iteration)

# ---------------------------------------------------------------------------
# Step 8 — Generate signals on val + test
# ---------------------------------------------------------------------------

log.info("=== Step 8: Generating signals ===")

eval_mask   = (mask_val | mask_test) & valid_mask & df["mid_exit"].notna()
df_eval     = df[eval_mask].copy().reset_index(drop=True)

df_eval["pred_ret_60s"] = lgbm_model.predict(df_eval[features].values)

ic_val, _  = spearmanr(
    df_eval.loc[df_eval["date"].isin(val_dates),  "ret_60s"],
    df_eval.loc[df_eval["date"].isin(val_dates),  "pred_ret_60s"],
)
ic_test, _ = spearmanr(
    df_eval.loc[df_eval["date"].isin(test_dates), "ret_60s"],
    df_eval.loc[df_eval["date"].isin(test_dates), "pred_ret_60s"],
)
log.info("Signal IC — val: %.4f  test: %.4f", ic_val, ic_test)

# ---------------------------------------------------------------------------
# Step 9 — Simulate trades
# ---------------------------------------------------------------------------

log.info("=== Step 9: Simulating trades ===")

# Direction: +1 (long) if predicted return > 0, -1 (short) if < 0
df_eval["direction"] = np.sign(df_eval["pred_ret_60s"]).astype(int)

# Actual PnL = direction × realized_ret_60s (in bps) − transaction cost
# This is a gross PnL assuming you can trade at mid
df_eval["gross_pnl_bps"] = df_eval["direction"] * df_eval["ret_60s"]
df_eval["net_pnl_bps"]   = df_eval["gross_pnl_bps"] - COST_BPS

# State from HMM (hard assignment — most probable state at bar time)
df_eval["state"] = df_eval["state_filtered"].astype(int)

# Flat sizing: every trade = 1 unit
df_eval["size_flat"]  = 1.0

# HMM-state sizing: scale by vol-normalized multiplier
df_eval["size_state"] = df_eval["state"].map(STATE_SIZE)

# Stop-loss distance for this state (bps)
df_eval["stop_bps"]   = df_eval["state"].map(STATE_STOP_BPS)

# Apply stop-loss: if gross move against us exceeds stop distance → cap loss
# Actual exit return (what market gave us)
df_eval["actual_ret_bps"] = df_eval["direction"] * df_eval["ret_60s"]

# Stop triggered if adverse move > stop_bps
df_eval["stopped_out"] = df_eval["actual_ret_bps"] < -df_eval["stop_bps"]

# PnL with stop: if stopped, loss = -stop_bps; else actual_ret
df_eval["pnl_with_stop"] = np.where(
    df_eval["stopped_out"],
    -df_eval["stop_bps"],
    df_eval["actual_ret_bps"],
)
df_eval["net_pnl_flat_stop"]  = df_eval["pnl_with_stop"] - COST_BPS
df_eval["net_pnl_state_stop"] = (df_eval["pnl_with_stop"] - COST_BPS) * df_eval["size_state"]

log.info("Total signals: %d", len(df_eval))
log.info("Stopped out: %.1f%%", df_eval["stopped_out"].mean() * 100)

# ---------------------------------------------------------------------------
# Step 10 — Compute metrics
# ---------------------------------------------------------------------------

log.info("=== Step 10: Computing metrics ===")


def compute_strategy_metrics(pnl_series: pd.Series, label: str) -> dict:
    """Compute Sharpe, hit rate, drawdown from a bps PnL series."""
    pnl  = pnl_series.dropna().values
    n    = len(pnl)
    if n == 0:
        return {}

    total_pnl  = float(pnl.sum())
    mean_pnl   = float(pnl.mean())
    std_pnl    = float(pnl.std()) if pnl.std() > 0 else 1e-9
    # Annualized Sharpe: ~23,400 trading seconds per day × 252 days
    sharpe     = mean_pnl / std_pnl * np.sqrt(23400 * 252)
    hit_rate   = float(np.mean(pnl > 0))
    # Max drawdown on cumulative PnL curve
    cum        = np.cumsum(pnl)
    running_max = np.maximum.accumulate(cum)
    drawdown   = cum - running_max
    max_dd     = float(drawdown.min())

    log.info(
        "%s | total=%.1fbps  mean=%.4fbps  sharpe=%.2f  hit=%.3f  maxDD=%.1fbps  n=%d",
        label, total_pnl, mean_pnl, sharpe, hit_rate, max_dd, n,
    )
    return {
        "total_pnl_bps":  round(total_pnl, 2),
        "mean_pnl_bps":   round(mean_pnl,  5),
        "std_pnl_bps":    round(std_pnl,   5),
        "sharpe":         round(sharpe,     3),
        "hit_rate":       round(hit_rate,   4),
        "max_drawdown_bps": round(max_dd,   2),
        "n_trades":       n,
    }


# Evaluate on test set only
test_mask = df_eval["date"].isin(test_dates)
te        = df_eval[test_mask].copy()

results = {}

# 1. Flat sizing, no stop
results["flat_no_stop"]    = compute_strategy_metrics(te["net_pnl_bps"],         "flat_no_stop   ")
# 2. Flat sizing + stop
results["flat_with_stop"]  = compute_strategy_metrics(te["net_pnl_flat_stop"],   "flat_with_stop ")
# 3. State sizing, no stop
results["state_no_stop"]   = compute_strategy_metrics(
    te["net_pnl_bps"] * te["size_state"],                                         "state_no_stop  ")
# 4. State sizing + stop (primary strategy)
results["state_with_stop"] = compute_strategy_metrics(te["net_pnl_state_stop"],  "state_with_stop")

# ---------------------------------------------------------------------------
# Step 11 — Per-state breakdown
# ---------------------------------------------------------------------------

log.info("=== Step 11: Per-state breakdown ===")

state_rows = []
for state in [1, 2, 3, 4]:
    s_mask = te["state"] == state
    if s_mask.sum() < 10:
        continue
    pnl  = te.loc[s_mask, "net_pnl_state_stop"]
    n    = int(s_mask.sum())
    m    = compute_strategy_metrics(pnl, f"state_{state}")
    m["state"]     = state
    m["n_bars"]    = n
    m["pct_bars"]  = round(n / len(te) * 100, 1)
    m["size_mult"] = STATE_SIZE[state]
    state_rows.append(m)

state_metrics = pd.DataFrame(state_rows)

# ---------------------------------------------------------------------------
# Step 12 — Daily PnL
# ---------------------------------------------------------------------------

log.info("=== Step 12: Daily PnL ===")

daily_rows = []
for day, grp in te.groupby("date"):
    for strat, col in [
        ("flat_no_stop",    "net_pnl_bps"),
        ("state_with_stop", "net_pnl_state_stop"),
    ]:
        pnl = grp[col].dropna()
        if len(pnl) == 0:
            continue
        daily_rows.append({
            "date":        str(day),
            "strategy":    strat,
            "total_pnl":   round(float(pnl.sum()),  2),
            "n_trades":    len(pnl),
            "hit_rate":    round(float((pnl > 0).mean()), 3),
            "mean_pnl":    round(float(pnl.mean()),  5),
        })

daily_pnl = pd.DataFrame(daily_rows)

# ---------------------------------------------------------------------------
# Step 13 — Comparison table
# ---------------------------------------------------------------------------

comparison_rows = []
for strat, m in results.items():
    comparison_rows.append({"strategy": strat, **m})
comparison = pd.DataFrame(comparison_rows)

# ---------------------------------------------------------------------------
# Step 14 — Trade log (val + test)
# ---------------------------------------------------------------------------

trade_log = df_eval[[
    "symbol", "date", "ts_sec",
    "pred_ret_60s", "ret_60s",
    "direction", "state",
    "size_flat", "size_state", "stop_bps",
    "stopped_out",
    "gross_pnl_bps", "net_pnl_bps",
    "net_pnl_flat_stop", "net_pnl_state_stop",
]].copy()
trade_log["split"] = np.where(
    trade_log["date"].isin(val_dates), "val", "test"
)

# ---------------------------------------------------------------------------
# Step 15 — Save outputs
# ---------------------------------------------------------------------------

log.info("=== Step 15: Saving outputs ===")

with open(OUT_DIR / "metrics.json", "w") as f:
    json.dump(
        {
            "split_dates": {
                "train": [str(d) for d in train_dates],
                "val":   [str(d) for d in val_dates],
                "test":  [str(d) for d in test_dates],
            },
            "config": {
                "hold_sec":   HOLD_SEC,
                "cost_bps":   COST_BPS,
                "state_size": STATE_SIZE,
                "state_stop": STATE_STOP_BPS,
                "feature_set": "CHALLENGER50",
                "signal":      "ret_60s",
            },
            "signal_ic": {"val": round(ic_val, 4), "test": round(ic_test, 4)},
            "results":   results,
        },
        f,
        indent=2,
    )
log.info("Saved metrics.json")

comparison.to_csv(OUT_DIR / "comparison.csv", index=False)
log.info("Saved comparison.csv")

state_metrics.to_csv(OUT_DIR / "state_metrics.csv", index=False)
log.info("Saved state_metrics.csv")

daily_pnl.to_csv(OUT_DIR / "daily_pnl.csv", index=False)
log.info("Saved daily_pnl.csv")

trade_log.to_csv(OUT_DIR / "trade_log.csv", index=False)
log.info("Saved trade_log.csv (%d trades)", len(trade_log))

# ---------------------------------------------------------------------------
# Step 16 — Final summary
# ---------------------------------------------------------------------------

print("\n" + "=" * 70)
print("STRATEGY BACKTEST — FINAL SUMMARY (TEST SET)")
print("=" * 70)
print(f"\nSignal IC (test): {ic_test:.4f}")
print(f"Transaction cost: {COST_BPS} bps round-trip")
print(f"Hold time: {HOLD_SEC}s fixed")
print()
print(comparison.to_string(index=False))

print("\n--- Per-State Breakdown (state_with_stop strategy) ---")
if not state_metrics.empty:
    print(state_metrics[[
        "state", "pct_bars", "size_mult", "n_trades",
        "total_pnl_bps", "sharpe", "hit_rate", "max_drawdown_bps"
    ]].to_string(index=False))

print("\n--- Daily PnL Summary (test days) ---")
print(
    daily_pnl.groupby("strategy")[["total_pnl", "hit_rate"]]
    .agg(["mean", "std", "min", "max"])
    .round(3)
    .to_string()
)
print("=" * 70)

23:26:47  INFO  === Step 1: Loading raw data ===


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

... (11 outputs left out)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

23:28:46  INFO  Raw: 472806 rows | symbols: ['USDINR26SEPFUT', 'USDINR26OCTFUT', 'USDINR26NOVFUT']
23:28:46  INFO  === Step 2: Building features ===
23:28:53  INFO  preprocess: 881382 rows after regularization
23:28:53  INFO  calculate_features: building CORE31 ...
23:29:15  INFO  build_core31_from_regularized: built 881382 rows, 31 CORE31 features
23:29:16  INFO  calculate_features: building book geometry (ADDED6 + CONTROLS9) ...
23:29:21  INFO  _build_book_geometry_features: built 881382 rows
23:29:21  INFO  calculate_features: building shock recovery (CHALLENGERS 1-2) ...
23:29:23  INFO  _build_shock_recovery_features: bid_shocks=22175 ask_shocks=42052
23:29:24  INFO  _build_shock_recovery_features: built 881382 rows
23:29:24  INFO  calculate_features: building update clustering (CHALLENGER 3) ...
23:29:26  INFO  _build_update_clustering_features: obs=472806 book_change=326695 bid=149451 ask=233683
23:29:27  INFO  _build_update_clustering_features: built 881382 rows
23:29:27  INFO  


STRATEGY BACKTEST — FINAL SUMMARY (TEST SET)

Signal IC (test): 0.2291
Transaction cost: 1.0 bps round-trip
Hold time: 60s fixed

       strategy  total_pnl_bps  mean_pnl_bps  std_pnl_bps    sharpe  hit_rate  max_drawdown_bps  n_trades
   flat_no_stop     -161866.43      -0.81645      1.09397 -1812.310    0.1664        -161869.81    198256
 flat_with_stop     -157665.29      -0.79526      1.02766 -1879.185    0.1664        -157668.67    198256
  state_no_stop     -111881.94      -0.56433      0.73454 -1865.628    0.1664        -111884.14    198256
state_with_stop     -108961.80      -0.54960      0.68812 -1939.499    0.1664        -108964.51    198256

--- Per-State Breakdown (state_with_stop strategy) ---
 state  pct_bars  size_mult  n_trades  total_pnl_bps    sharpe  hit_rate  max_drawdown_bps
     1       2.6       0.59      5132       -1916.05 -1007.349    0.2525          -1914.08
     2      16.5       0.44     32790      -10551.26 -1480.369    0.2056         -10549.77
     3    

In [2]:
import pandas as pd
tl = pd.read_csv(r"C:\projects\hedge_fund\results\strategy_backtest\trade_log.csv")
print(tl["pred_ret_60s"].describe(percentiles=[.5,.75,.9,.95,.99]))

count    292488.000000
mean          0.003455
std           0.232540
min          -2.629626
50%           0.013218
75%           0.141983
90%           0.258664
95%           0.343629
99%           0.530039
max           1.595132
Name: pred_ret_60s, dtype: float64


In [3]:
"""
USDINR HMM-State Strategy Backtest
====================================
Signal  : LightGBM CHALLENGER50 predicted ret_60s
Regimes : 4-state HMM (Quiet / Stressed / Trending / Mean-reverting)
Entry   : Every bar — no signal filtering (all signals traded)
Exit    : Fixed 60s hold
Sizing  : State-based position multiplier (derived from realized vol per state)
Costs   : 1 bps round-trip (conservative NSE USDINR futures estimate)

Outputs : results/strategy_backtest/
            metrics.json        — overall + per-state Sharpe, hit rate, PnL
            trade_log.csv       — every trade with entry/exit/state/PnL
            daily_pnl.csv       — daily PnL + Sharpe + drawdown
            state_metrics.csv   — per-HMM-state breakdown
            comparison.csv      — flat sizing vs HMM-state sizing

No plots — CSV/JSON outputs only.
"""

import sys
sys.path.insert(0, r"C:\projects\hedge_fund")

import json
import logging
import pathlib
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import lightgbm as lgb
from scipy.stats import spearmanr

from data.store.duckdb_client import DuckDBClient
from data.store.s3_client import S3Client
from data.processing.usdinr.features import USDINRFeatureBuilder
from data.processing.usdinr.constants import CHALLENGER50
from data.processing.usdinr.regimes import (
    add_soft_hmm_features,
    save_hmm_model,
    load_hmm_model,
    HMM_PROB_COLS,
)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)s  %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger(__name__)

# ---------------------------------------------------------------------------
# Config
# ---------------------------------------------------------------------------

OUT_DIR   = pathlib.Path(r"C:\projects\hedge_fund\results\strategy_backtest")
MODEL_DIR = pathlib.Path(r"C:\projects\hedge_fund\models")
OUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

HORIZONS   = [60, 300]
TRAIN_DAYS = 15
VAL_DAYS   = 5
COST_BPS   = 1.0        # round-trip transaction cost in bps
HOLD_SEC   = 60         # fixed exit after 60 seconds

# Signal thresholds to sweep — only enter when |pred_ret_60s| > threshold
# We test multiple thresholds to find the break-even point
THRESHOLDS = [0.0, 0.10, 0.20, 0.30, 0.40, 0.50, 0.75, 1.00]

# LightGBM params — same as lgbm_baseline
LGB_PARAMS = {
    "objective":             "regression",
    "metric":                "rmse",
    "verbosity":             -1,
    "boosting_type":         "gbdt",
    "num_leaves":            31,
    "learning_rate":         0.05,
    "feature_fraction":      0.8,
    "bagging_fraction":      0.8,
    "bagging_freq":          5,
    "min_child_samples":     100,
    "n_estimators":          1000,
    "early_stopping_rounds": 50,
}

# Per-state position size multiplier
# Derived from your experiment table:
#   State 4: RV=1.08 bps² (quietest)   → full size = 1.0
#   State 3: RV=2.59 bps²              → 0.65
#   State 1: RV=3.14 bps²              → 0.59
#   State 2: RV=5.62 bps² (stressed)   → 0.44
# Formula: multiplier = sqrt(RV_state4 / RV_state_k) — vol-normalizing
STATE_SIZE = {
    1: 0.59,
    2: 0.44,
    3: 0.65,
    4: 1.00,
}

# Per-state stop-loss distance in bps
# Derived from future_60s_return_std per state × 1.5
STATE_STOP_BPS = {
    1: 2.34,   # 1.5616 * 1.5
    2: 3.08,   # 2.0507 * 1.5
    3: 2.04,   # 1.3594 * 1.5
    4: 1.33,   # 0.8865 * 1.5
}

# ---------------------------------------------------------------------------
# Step 1 — Load raw data
# ---------------------------------------------------------------------------

log.info("=== Step 1: Loading raw data ===")

db = DuckDBClient()
s  = S3Client()

all_contracts = ["26SEPFUT", "26OCTFUT", "26NOVFUT"]
all_frames    = []

for contract in all_contracts:
    files = s.list_files(f"processed/1s/USDINR{contract}/")
    if not files:
        log.info("No files for USDINR%s — skipping", contract)
        continue
    for f in files:
        parts = f.split("/")
        sym   = parts[2]
        date  = parts[3].replace(".parquet", "")
        try:
            df_day           = db.read_parquet(f"processed/1s/{sym}/{date}.parquet")
            df_day["symbol"] = sym
            all_frames.append(df_day)
        except Exception as e:
            log.warning("Failed %s/%s: %s", sym, date, e)

db.close()

if not all_frames:
    raise RuntimeError("No data loaded")

df_raw = pd.concat(all_frames, ignore_index=True)
log.info("Raw: %d rows | symbols: %s", len(df_raw), df_raw["symbol"].unique().tolist())

# ---------------------------------------------------------------------------
# Step 2 — Build features
# ---------------------------------------------------------------------------

log.info("=== Step 2: Building features ===")

builder = USDINRFeatureBuilder(data=df_raw, auto_build=True)
df      = builder.dataset.copy()
log.info("Dataset: %d rows, %d cols", len(df), len(df.columns))

# ---------------------------------------------------------------------------
# Step 3 — Validate tick_count
# ---------------------------------------------------------------------------

if "tick_count" not in df.columns:
    raise RuntimeError("tick_count not found in builder output")
df["tick_count"] = df["tick_count"].fillna(0).astype(int)

# ---------------------------------------------------------------------------
# Step 4 — Split dates
# ---------------------------------------------------------------------------

log.info("=== Step 4: Splitting ===")

df        = df.sort_values(["symbol", "date", "ts_sec"]).reset_index(drop=True)
all_dates = sorted(df["date"].unique())
n_days    = len(all_dates)
log.info("Total days: %d  (%s → %s)", n_days, all_dates[0], all_dates[-1])

if n_days < TRAIN_DAYS + VAL_DAYS + 1:
    raise RuntimeError(f"Need at least {TRAIN_DAYS + VAL_DAYS + 1} days, got {n_days}")

train_dates = all_dates[:TRAIN_DAYS]
val_dates   = all_dates[TRAIN_DAYS : TRAIN_DAYS + VAL_DAYS]
test_dates  = all_dates[TRAIN_DAYS + VAL_DAYS:]
train_end   = str(train_dates[-1])

log.info("Train: %d  Val: %d  Test: %d days",
         len(train_dates), len(val_dates), len(test_dates))

mask_train = df["date"].isin(train_dates)
mask_val   = df["date"].isin(val_dates)
mask_test  = df["date"].isin(test_dates)

# ---------------------------------------------------------------------------
# Step 5 — Fit HMM
# ---------------------------------------------------------------------------

log.info("=== Step 5: Fitting HMM ===")

df, hmm_model = add_soft_hmm_features(
    df             = df,
    train_end_date = train_end,
    K              = 4,
    restarts       = 3,
    max_iter       = 70,
    tol            = 1e-6,
    seed           = 42,
)
save_hmm_model(hmm_model, MODEL_DIR / "hmm_usdinr_strategy.pkl")
log.info("State distribution:\n%s",
         df["state_filtered"].value_counts().sort_index().to_string())

# ---------------------------------------------------------------------------
# Step 6 — Targets + mid price
# ---------------------------------------------------------------------------

log.info("=== Step 6: Building targets ===")

if "mid" not in df.columns:
    df["mid"] = (df["bid_p1"] + df["ask_p1"]) / 2.0
df["log_mid"] = np.log(df["mid"])

g = df.groupby(["symbol", "date"], sort=False)
df["ret_60s"]  = (g["log_mid"].shift(-60)  - df["log_mid"]) * 1e4
df["ret_300s"] = (g["log_mid"].shift(-300) - df["log_mid"]) * 1e4

# Forward mid price (what you actually get at exit after HOLD_SEC)
df["mid_exit"] = g["mid"].shift(-HOLD_SEC)

log.info("ret_60s non-null: %.1f%%", df["ret_60s"].notna().mean() * 100)

# ---------------------------------------------------------------------------
# Step 7 — Train LightGBM on training data
# ---------------------------------------------------------------------------

log.info("=== Step 7: Training LightGBM ===")

features = CHALLENGER50
target   = "ret_60s"

valid_mask = df[features + [target]].notna().all(axis=1)

tr = df[mask_train & valid_mask]
va = df[mask_val   & valid_mask]

X_tr, y_tr = tr[features].values, tr[target].values
X_va, y_va = va[features].values, va[target].values

ds_tr = lgb.Dataset(X_tr, label=y_tr, feature_name=features)
ds_va = lgb.Dataset(X_va, label=y_va, feature_name=features, reference=ds_tr)

params = {k: v for k, v in LGB_PARAMS.items()
          if k not in ("n_estimators", "early_stopping_rounds")}

lgbm_model = lgb.train(
    params,
    ds_tr,
    num_boost_round     = LGB_PARAMS["n_estimators"],
    valid_sets          = [ds_va],
    callbacks           = [
        lgb.early_stopping(LGB_PARAMS["early_stopping_rounds"], verbose=False),
        lgb.log_evaluation(100),
    ],
)
log.info("LightGBM best iteration: %d", lgbm_model.best_iteration)

# ---------------------------------------------------------------------------
# Step 8 — Generate signals on val + test
# ---------------------------------------------------------------------------

log.info("=== Step 8: Generating signals ===")

eval_mask   = (mask_val | mask_test) & valid_mask & df["mid_exit"].notna()
df_eval     = df[eval_mask].copy().reset_index(drop=True)

df_eval["pred_ret_60s"] = lgbm_model.predict(df_eval[features].values)

ic_val, _  = spearmanr(
    df_eval.loc[df_eval["date"].isin(val_dates),  "ret_60s"],
    df_eval.loc[df_eval["date"].isin(val_dates),  "pred_ret_60s"],
)
ic_test, _ = spearmanr(
    df_eval.loc[df_eval["date"].isin(test_dates), "ret_60s"],
    df_eval.loc[df_eval["date"].isin(test_dates), "pred_ret_60s"],
)
log.info("Signal IC — val: %.4f  test: %.4f", ic_val, ic_test)

# ---------------------------------------------------------------------------
# Step 9 — Simulate trades (single position, threshold filter)
# ---------------------------------------------------------------------------

log.info("=== Step 9: Simulating trades ===")

df_eval["direction"] = np.sign(df_eval["pred_ret_60s"]).astype(int)
df_eval["abs_pred"]  = df_eval["pred_ret_60s"].abs()
df_eval["state"]     = df_eval["state_filtered"].astype(int)
df_eval["size_state"]= df_eval["state"].map(STATE_SIZE)
df_eval["stop_bps"]  = df_eval["state"].map(STATE_STOP_BPS)

log.info("Total bars available: %d", len(df_eval))


def simulate_single_position(df_day: pd.DataFrame, threshold: float) -> list:
    """
    Single-position simulator for one day.
    Rules:
      - Only enter if |pred_ret_60s| > threshold AND no open position
      - Hold for exactly HOLD_SEC seconds
      - Apply stop-loss: cap loss at STATE_STOP_BPS for the entry state
      - One trade at a time — no overlapping positions
    Returns list of trade dicts.
    """
    trades   = []
    busy_until = -1   # ts_sec when current position exits

    rows = df_day.sort_values("ts_sec").reset_index(drop=True)

    for i, row in rows.iterrows():
        ts = row["ts_sec"]

        # Still in a position
        if ts <= busy_until:
            continue

        # Signal not strong enough
        if row["abs_pred"] <= threshold:
            continue

        # No valid exit return
        if pd.isna(row["ret_60s"]):
            continue

        direction  = int(row["direction"])
        state      = int(row["state"])
        size       = float(row["size_state"])
        stop       = float(row["stop_bps"])
        actual_ret = direction * float(row["ret_60s"])

        # Apply stop
        stopped = actual_ret < -stop
        pnl_raw = -stop if stopped else actual_ret
        net_pnl = (pnl_raw - COST_BPS) * size

        trades.append({
            "ts_sec":       ts,
            "pred_ret":     float(row["pred_ret_60s"]),
            "actual_ret":   float(row["ret_60s"]),
            "direction":    direction,
            "state":        state,
            "size":         size,
            "stop_bps":     stop,
            "stopped_out":  stopped,
            "net_pnl_bps":  net_pnl,
            "threshold":    threshold,
        })

        busy_until = ts + HOLD_SEC

    return trades


def compute_strategy_metrics(pnl_series: pd.Series, label: str) -> dict:
    """Sharpe, hit rate, drawdown from a bps PnL series."""
    pnl = pnl_series.dropna().values
    n   = len(pnl)
    if n == 0:
        return {"n_trades": 0}

    total_pnl = float(pnl.sum())
    mean_pnl  = float(pnl.mean())
    std_pnl   = float(pnl.std()) if pnl.std() > 0 else 1e-9
    # Annualise: assume ~23,400 tradeable seconds/day, 252 days
    # But with threshold filtering trades-per-day drops, so use per-trade Sharpe
    sharpe    = mean_pnl / std_pnl * np.sqrt(n * 252 / len(pnl_series.index.unique()) if hasattr(pnl_series.index, 'unique') else n)
    sharpe    = mean_pnl / std_pnl * np.sqrt(n)   # simpler: sqrt-n Sharpe
    hit_rate  = float(np.mean(pnl > 0))
    cum       = np.cumsum(pnl)
    running_max = np.maximum.accumulate(cum)
    max_dd    = float((cum - running_max).min())

    log.info(
        "%s | total=%.2fbps mean=%.4fbps sharpe=%.2f hit=%.3f maxDD=%.2fbps n=%d",
        label, total_pnl, mean_pnl, sharpe, hit_rate, max_dd, n,
    )
    return {
        "total_pnl_bps":    round(total_pnl, 2),
        "mean_pnl_bps":     round(mean_pnl,  5),
        "std_pnl_bps":      round(std_pnl,   5),
        "sharpe":           round(sharpe,     3),
        "hit_rate":         round(hit_rate,   4),
        "max_drawdown_bps": round(max_dd,     2),
        "n_trades":         n,
    }


# ---------------------------------------------------------------------------
# Step 10 — Sweep thresholds on test set
# ---------------------------------------------------------------------------

log.info("=== Step 10: Sweeping thresholds ===")

test_mask = df_eval["date"].isin(test_dates)
te_days   = df_eval[test_mask].copy()

threshold_results = []
all_trades        = []

for thresh in THRESHOLDS:
    day_trades = []
    for day, grp in te_days.groupby("date"):
        t = simulate_single_position(grp, thresh)
        for tr in t:
            tr["date"] = str(day)
        day_trades.extend(t)

    if not day_trades:
        log.info("threshold=%.2f | 0 trades", thresh)
        continue

    trades_df = pd.DataFrame(day_trades)
    pnl       = trades_df["net_pnl_bps"]
    m         = compute_strategy_metrics(pnl, f"thresh={thresh:.2f}")
    m["threshold"]   = thresh
    m["pct_filtered"] = round(100 * len(trades_df) /
                               max(te_days["abs_pred"].gt(thresh).sum(), 1), 1)
    threshold_results.append(m)
    all_trades.extend(day_trades)

threshold_df = pd.DataFrame(threshold_results)

# Best threshold by Sharpe
if not threshold_df.empty:
    best_row   = threshold_df.loc[threshold_df["sharpe"].idxmax()]
    best_thresh = float(best_row["threshold"])
    log.info("Best threshold by Sharpe: %.2f (Sharpe=%.3f)",
             best_thresh, best_row["sharpe"])

# ---------------------------------------------------------------------------
# Step 11 — Per-state breakdown at best threshold
# ---------------------------------------------------------------------------

log.info("=== Step 11: Per-state breakdown ===")

best_trades = [t for t in all_trades if t["threshold"] == best_thresh]
state_rows  = []

if best_trades:
    bt_df = pd.DataFrame(best_trades)
    for state in [1, 2, 3, 4]:
        s = bt_df[bt_df["state"] == state]
        if len(s) < 5:
            continue
        m = compute_strategy_metrics(s["net_pnl_bps"], f"state_{state}@{best_thresh:.2f}")
        m["state"]     = state
        m["pct_trades"]= round(len(s) / len(bt_df) * 100, 1)
        m["size_mult"] = STATE_SIZE[state]
        state_rows.append(m)

state_metrics = pd.DataFrame(state_rows)

# ---------------------------------------------------------------------------
# Step 12 — Daily PnL at best threshold
# ---------------------------------------------------------------------------

log.info("=== Step 12: Daily PnL ===")

daily_rows = []
if best_trades:
    bt_df = pd.DataFrame(best_trades)
    for day, grp in bt_df.groupby("date"):
        pnl = grp["net_pnl_bps"]
        daily_rows.append({
            "date":      day,
            "threshold": best_thresh,
            "total_pnl": round(float(pnl.sum()), 2),
            "n_trades":  len(pnl),
            "hit_rate":  round(float((pnl > 0).mean()), 3),
            "mean_pnl":  round(float(pnl.mean()), 5),
        })

daily_pnl = pd.DataFrame(daily_rows)

# ---------------------------------------------------------------------------
# Step 13 — Comparison + trade log
# ---------------------------------------------------------------------------

comparison = threshold_df.copy()

trade_log = pd.DataFrame(all_trades) if all_trades else pd.DataFrame()

# ---------------------------------------------------------------------------
# Step 14 — Save outputs
# ---------------------------------------------------------------------------

log.info("=== Step 14: Saving outputs ===")

with open(OUT_DIR / "metrics.json", "w") as f:
    json.dump(
        {
            "split_dates": {
                "train": [str(d) for d in train_dates],
                "val":   [str(d) for d in val_dates],
                "test":  [str(d) for d in test_dates],
            },
            "config": {
                "hold_sec":    HOLD_SEC,
                "cost_bps":    COST_BPS,
                "thresholds":  THRESHOLDS,
                "state_size":  STATE_SIZE,
                "state_stop":  STATE_STOP_BPS,
                "feature_set": "CHALLENGER50",
                "signal":      "ret_60s",
            },
            "signal_ic":  {"val": round(ic_val, 4), "test": round(ic_test, 4)},
            "best_threshold": best_thresh if not threshold_df.empty else None,
            "threshold_results": threshold_df.to_dict(orient="records") if not threshold_df.empty else [],
        },
        f,
        indent=2,
    )
log.info("Saved metrics.json")

comparison.to_csv(OUT_DIR / "threshold_sweep.csv", index=False)
log.info("Saved threshold_sweep.csv")

if not state_metrics.empty:
    state_metrics.to_csv(OUT_DIR / "state_metrics.csv", index=False)
    log.info("Saved state_metrics.csv")

if not daily_pnl.empty:
    daily_pnl.to_csv(OUT_DIR / "daily_pnl.csv", index=False)
    log.info("Saved daily_pnl.csv")

if not trade_log.empty:
    trade_log.to_csv(OUT_DIR / "trade_log.csv", index=False)
    log.info("Saved trade_log.csv (%d trades)", len(trade_log))

# ---------------------------------------------------------------------------
# Step 15 — Final summary
# ---------------------------------------------------------------------------

print("\n" + "=" * 70)
print("STRATEGY BACKTEST — THRESHOLD SWEEP (TEST SET)")
print("=" * 70)
print(f"\nSignal IC (test): {ic_test:.4f}")
print(f"Transaction cost: {COST_BPS} bps round-trip")
print(f"Hold time:        {HOLD_SEC}s fixed")
print(f"Mode:             single position, no overlap\n")

if not threshold_df.empty:
    print(threshold_df[[
        "threshold", "n_trades", "hit_rate",
        "mean_pnl_bps", "total_pnl_bps", "sharpe", "max_drawdown_bps"
    ]].to_string(index=False))

    print(f"\n--- Best threshold: {best_thresh:.2f} bps ---")
    print(best_row.to_string())

if not state_metrics.empty:
    print(f"\n--- Per-State Breakdown at threshold={best_thresh:.2f} ---")
    print(state_metrics[[
        "state", "pct_trades", "size_mult", "n_trades",
        "total_pnl_bps", "sharpe", "hit_rate", "max_drawdown_bps"
    ]].to_string(index=False))

if not daily_pnl.empty:
    print("\n--- Daily PnL at best threshold ---")
    print(daily_pnl[["date", "n_trades", "total_pnl", "hit_rate"]].to_string(index=False))

print("=" * 70)

23:37:02  INFO  === Step 1: Loading raw data ===


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

... (7 outputs left out)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

23:38:50  INFO  Raw: 472806 rows | symbols: ['USDINR26SEPFUT', 'USDINR26OCTFUT', 'USDINR26NOVFUT']
23:38:50  INFO  === Step 2: Building features ===
23:38:55  INFO  preprocess: 881382 rows after regularization
23:38:55  INFO  calculate_features: building CORE31 ...
23:39:17  INFO  build_core31_from_regularized: built 881382 rows, 31 CORE31 features
23:39:18  INFO  calculate_features: building book geometry (ADDED6 + CONTROLS9) ...
23:39:24  INFO  _build_book_geometry_features: built 881382 rows
23:39:24  INFO  calculate_features: building shock recovery (CHALLENGERS 1-2) ...
23:39:26  INFO  _build_shock_recovery_features: bid_shocks=22175 ask_shocks=42052
23:39:26  INFO  _build_shock_recovery_features: built 881382 rows
23:39:26  INFO  calculate_features: building update clustering (CHALLENGER 3) ...
23:39:28  INFO  _build_update_clustering_features: obs=472806 book_change=326695 bid=149451 ask=233683
23:39:29  INFO  _build_update_clustering_features: built 881382 rows
23:39:30  INFO  


STRATEGY BACKTEST — THRESHOLD SWEEP (TEST SET)

Signal IC (test): 0.2291
Transaction cost: 1.0 bps round-trip
Hold time:        60s fixed
Mode:             single position, no overlap

 threshold  n_trades  hit_rate  mean_pnl_bps  total_pnl_bps  sharpe  max_drawdown_bps
      0.00      3635    0.1659      -0.53161       -1932.39 -49.004          -1932.00
      0.10      3141    0.1942      -0.46307       -1454.50 -36.201          -1453.94
      0.20      2232    0.2469      -0.38776        -865.48 -24.805           -865.23
      0.30      1373    0.2986      -0.32220        -442.38 -15.209           -442.11
      0.40       809    0.3078      -0.30333        -245.40 -10.484           -245.18
      0.50       472    0.3347      -0.25696        -121.29  -6.323           -126.47
      0.75       163    0.4049      -0.06015          -9.80  -0.709            -34.81
      1.00        67    0.4328      -0.02799          -1.88  -0.250             -6.20

--- Best threshold: 1.00 bps ---
total_

In [4]:
import pandas as pd
import numpy as np

tl = pd.read_csv(r"C:\projects\hedge_fund\results\strategy_backtest\trade_log.csv")

# Focus on threshold=1.00 trades only
t = tl[tl["threshold"] == 1.0].copy()
print(f"Total trades at threshold=1.0: {len(t)}")

# 1. Win vs loss breakdown by state
print("\n=== Win/Loss by State ===")
for state in sorted(t["state"].unique()):
    s = t[t["state"] == state]
    wins   = s[s["net_pnl_bps"] > 0]
    losses = s[s["net_pnl_bps"] <= 0]
    print(f"State {state}: n={len(s)}  wins={len(wins)}  losses={len(losses)}")
    if len(wins):   print(f"  Win  avg={wins['net_pnl_bps'].mean():.4f}  max={wins['net_pnl_bps'].max():.4f}")
    if len(losses): print(f"  Loss avg={losses['net_pnl_bps'].mean():.4f}  min={losses['net_pnl_bps'].min():.4f}")

# 2. Stop-loss impact
print("\n=== Stop-loss impact ===")
stopped = t[t["stopped_out"] == True]
normal  = t[t["stopped_out"] == False]
print(f"Stopped out: {len(stopped)} ({100*len(stopped)/len(t):.1f}%)")
print(f"  Mean PnL when stopped: {stopped['net_pnl_bps'].mean():.4f}")
print(f"  Mean PnL normal exit:  {normal['net_pnl_bps'].mean():.4f}")

# 3. Direction bias
print("\n=== Direction bias ===")
longs  = t[t["direction"] ==  1]
shorts = t[t["direction"] == -1]
print(f"Longs:  n={len(longs)}  hit={(longs['net_pnl_bps']>0).mean():.3f}  mean={longs['net_pnl_bps'].mean():.4f}")
print(f"Shorts: n={len(shorts)} hit={(shorts['net_pnl_bps']>0).mean():.3f}  mean={shorts['net_pnl_bps'].mean():.4f}")

# 4. Prediction vs actual
print("\n=== Prediction vs Actual ===")
print(t[["pred_ret", "actual_ret", "net_pnl_bps"]].describe().round(4))

# 5. Time of day
print("\n=== Trades by hour (IST = UTC+5:30, ts_sec is epoch) ===")
t["hour_ist"] = ((t["ts_sec"] % 86400) // 3600 + 5) % 24
print(t.groupby("hour_ist")["net_pnl_bps"].agg(["count","mean","sum"]).round(4))

# 6. All trades sorted by PnL
print("\n=== All 67 trades sorted by PnL ===")
print(t[["date","ts_sec","state","direction","pred_ret","actual_ret",
          "stopped_out","net_pnl_bps"]]
      .sort_values("net_pnl_bps").to_string(index=False))

Total trades at threshold=1.0: 67

=== Win/Loss by State ===
State 1: n=13  wins=6  losses=7
  Win  avg=1.0517  max=3.4824
  Loss avg=-0.7874  min=-1.9706
State 2: n=31  wins=12  losses=19
  Win  avg=0.6554  max=2.3557
  Loss avg=-0.5686  min=-1.3679
State 3: n=21  wins=9  losses=12
  Win  avg=0.6585  max=1.5051
  Loss avg=-0.6222  min=-1.7706
State 4: n=2  wins=2  losses=0
  Win  avg=0.9026  max=1.4921

=== Stop-loss impact ===
Stopped out: 1 (1.5%)
  Mean PnL when stopped: -1.9706
  Mean PnL normal exit:  0.0014

=== Direction bias ===
Longs:  n=10  hit=0.600  mean=0.2541
Shorts: n=57 hit=0.404  mean=-0.0775

=== Prediction vs Actual ===
       pred_ret  actual_ret  net_pnl_bps
count   67.0000     67.0000      67.0000
mean    -0.8569     -0.4668      -0.0280
std      0.8745      1.8908       0.9232
min     -2.4432     -6.3538      -1.9706
25%     -1.2149     -1.4598      -0.4983
50%     -1.0664     -0.5303      -0.1326
75%     -1.0105      0.5289       0.4952
max      1.3893      6.9

In [5]:
import pandas as pd
import numpy as np

tl = pd.read_csv(r"C:\projects\hedge_fund\results\strategy_backtest\trade_log.csv")

# Check raw prediction distribution by date
all_preds = tl[["date", "pred_ret", "actual_ret"]].copy()
print("=== Mean prediction by date ===")
print(all_preds.groupby("date")[["pred_ret","actual_ret"]].mean().round(4))

print("\n=== Overall prediction distribution ===")
print(all_preds["pred_ret"].describe(percentiles=[.1,.25,.5,.75,.9]))

print("\n=== Actual return distribution ===")
print(all_preds["actual_ret"].describe(percentiles=[.1,.25,.5,.75,.9]))

=== Mean prediction by date ===
            pred_ret  actual_ret
date                            
2026-10-23   -0.1080      0.0299
2026-10-26    0.0017      0.0153
2026-10-27   -0.0087     -0.0002
2026-10-28   -0.1596      0.1507
2026-10-29   -0.0445      0.0028
2026-10-30    0.0063     -0.0654
2026-11-02    0.1260      0.0555
2026-11-03    0.0087      0.0095
2026-11-04   -0.0352     -0.0073
2026-11-05    0.0634      0.0431
2026-11-06   -0.0898     -0.1037
2026-11-09   -0.1189      0.1634

=== Overall prediction distribution ===
count    11892.000000
mean        -0.024808
std          0.373814
min         -2.443222
10%         -0.473764
25%         -0.255441
50%          0.012692
75%          0.235880
90%          0.402540
max          1.389289
Name: pred_ret, dtype: float64

=== Actual return distribution ===
count    11892.000000
mean         0.013061
std          1.294525
min        -11.667839
10%         -1.438678
25%         -0.526683
50%          0.000000
75%          0.654283
90

In [6]:
"""
USDINR HMM-State Strategy Backtest  v2
========================================
Signal  : LightGBM CHALLENGER50 on NORMALIZED daily ret_60s target
           → removes intraday drift bias, fixes systematic short skew
Regimes : 4-state HMM — state-based position sizing + stop-loss
Entry   : Single position, asymmetric thresholds (longs/shorts separate bars)
           + time-of-day filter (no trades in first 90 min or last 30 min)
Exit    : Fixed 60s hold
Costs   : 1 bps round-trip

Changes vs v1:
  1. Target normalized per day: ret_60s_norm = (ret - daily_mean) / daily_std
     → kills directional drift bias in training
  2. Asymmetric threshold: LONG_THRESH=+0.80, SHORT_THRESH=-1.20
     → higher bar for shorts (which were losing at 40% hit rate)
  3. Time filter: only trade 10:30–14:30 IST
     → removes chaotic open + low-liquidity close

Outputs : results/strategy_backtest_v2/
No plots — CSV/JSON outputs only.
"""

import sys
sys.path.insert(0, r"C:\projects\hedge_fund")

import json
import logging
import pathlib
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import lightgbm as lgb
from scipy.stats import spearmanr

from data.store.duckdb_client import DuckDBClient
from data.store.s3_client import S3Client
from data.processing.usdinr.features import USDINRFeatureBuilder
from data.processing.usdinr.constants import CHALLENGER50
from data.processing.usdinr.regimes import (
    add_soft_hmm_features,
    save_hmm_model,
    load_hmm_model,
    HMM_PROB_COLS,
)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)s  %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger(__name__)

# ---------------------------------------------------------------------------
# Config
# ---------------------------------------------------------------------------

OUT_DIR   = pathlib.Path(r"C:\projects\hedge_fund\results\strategy_backtest_v2")
MODEL_DIR = pathlib.Path(r"C:\projects\hedge_fund\models")
OUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

HORIZONS   = [60, 300]
TRAIN_DAYS = 15
VAL_DAYS   = 5
COST_BPS   = 1.0        # round-trip transaction cost in bps
HOLD_SEC   = 60         # fixed exit after 60 seconds

# --- Fix 1: Asymmetric thresholds ---
# Longs hit 60% at threshold=1.0 → lower bar fine
# Shorts hit 40% at threshold=1.0 → raise the bar
LONG_THRESH  = 0.80     # enter long  if pred_norm > +LONG_THRESH
SHORT_THRESH = 1.20     # enter short if pred_norm < -SHORT_THRESH

# Sweep both thresholds together (scale factor applied to both)
THRESHOLD_SCALES = [0.5, 0.75, 1.0, 1.25, 1.5, 2.0]

# --- Fix 3: Time-of-day filter (IST = UTC+5:30) ---
# Only trade between 10:30 and 14:30 IST
# ts_sec is Unix epoch (UTC), so convert: IST = UTC + 19800 seconds
IST_OFFSET    = 19800
OPEN_FILTER   = 10 * 3600 + 30 * 60   # 10:30 IST in seconds since midnight
CLOSE_FILTER  = 14 * 3600 + 30 * 60   # 14:30 IST in seconds since midnight

# LightGBM params — same as lgbm_baseline
LGB_PARAMS = {
    "objective":             "regression",
    "metric":                "rmse",
    "verbosity":             -1,
    "boosting_type":         "gbdt",
    "num_leaves":            31,
    "learning_rate":         0.05,
    "feature_fraction":      0.8,
    "bagging_fraction":      0.8,
    "bagging_freq":          5,
    "min_child_samples":     100,
    "n_estimators":          1000,
    "early_stopping_rounds": 50,
}

# Per-state position size multiplier
# Derived from your experiment table:
#   State 4: RV=1.08 bps² (quietest)   → full size = 1.0
#   State 3: RV=2.59 bps²              → 0.65
#   State 1: RV=3.14 bps²              → 0.59
#   State 2: RV=5.62 bps² (stressed)   → 0.44
# Formula: multiplier = sqrt(RV_state4 / RV_state_k) — vol-normalizing
STATE_SIZE = {
    1: 0.59,
    2: 0.44,
    3: 0.65,
    4: 1.00,
}

# Per-state stop-loss distance in bps
# Derived from future_60s_return_std per state × 1.5
STATE_STOP_BPS = {
    1: 2.34,   # 1.5616 * 1.5
    2: 3.08,   # 2.0507 * 1.5
    3: 2.04,   # 1.3594 * 1.5
    4: 1.33,   # 0.8865 * 1.5
}

# ---------------------------------------------------------------------------
# Step 1 — Load raw data
# ---------------------------------------------------------------------------

log.info("=== Step 1: Loading raw data ===")

db = DuckDBClient()
s  = S3Client()

all_contracts = ["26SEPFUT", "26OCTFUT", "26NOVFUT"]
all_frames    = []

for contract in all_contracts:
    files = s.list_files(f"processed/1s/USDINR{contract}/")
    if not files:
        log.info("No files for USDINR%s — skipping", contract)
        continue
    for f in files:
        parts = f.split("/")
        sym   = parts[2]
        date  = parts[3].replace(".parquet", "")
        try:
            df_day           = db.read_parquet(f"processed/1s/{sym}/{date}.parquet")
            df_day["symbol"] = sym
            all_frames.append(df_day)
        except Exception as e:
            log.warning("Failed %s/%s: %s", sym, date, e)

db.close()

if not all_frames:
    raise RuntimeError("No data loaded")

df_raw = pd.concat(all_frames, ignore_index=True)
log.info("Raw: %d rows | symbols: %s", len(df_raw), df_raw["symbol"].unique().tolist())

# ---------------------------------------------------------------------------
# Step 2 — Build features
# ---------------------------------------------------------------------------

log.info("=== Step 2: Building features ===")

builder = USDINRFeatureBuilder(data=df_raw, auto_build=True)
df      = builder.dataset.copy()
log.info("Dataset: %d rows, %d cols", len(df), len(df.columns))

# ---------------------------------------------------------------------------
# Step 3 — Validate tick_count
# ---------------------------------------------------------------------------

if "tick_count" not in df.columns:
    raise RuntimeError("tick_count not found in builder output")
df["tick_count"] = df["tick_count"].fillna(0).astype(int)

# ---------------------------------------------------------------------------
# Step 4 — Split dates
# ---------------------------------------------------------------------------

log.info("=== Step 4: Splitting ===")

df        = df.sort_values(["symbol", "date", "ts_sec"]).reset_index(drop=True)
all_dates = sorted(df["date"].unique())
n_days    = len(all_dates)
log.info("Total days: %d  (%s → %s)", n_days, all_dates[0], all_dates[-1])

if n_days < TRAIN_DAYS + VAL_DAYS + 1:
    raise RuntimeError(f"Need at least {TRAIN_DAYS + VAL_DAYS + 1} days, got {n_days}")

train_dates = all_dates[:TRAIN_DAYS]
val_dates   = all_dates[TRAIN_DAYS : TRAIN_DAYS + VAL_DAYS]
test_dates  = all_dates[TRAIN_DAYS + VAL_DAYS:]
train_end   = str(train_dates[-1])

log.info("Train: %d  Val: %d  Test: %d days",
         len(train_dates), len(val_dates), len(test_dates))

mask_train = df["date"].isin(train_dates)
mask_val   = df["date"].isin(val_dates)
mask_test  = df["date"].isin(test_dates)

# ---------------------------------------------------------------------------
# Step 5 — Fit HMM
# ---------------------------------------------------------------------------

log.info("=== Step 5: Fitting HMM ===")

df, hmm_model = add_soft_hmm_features(
    df             = df,
    train_end_date = train_end,
    K              = 4,
    restarts       = 3,
    max_iter       = 70,
    tol            = 1e-6,
    seed           = 42,
)
save_hmm_model(hmm_model, MODEL_DIR / "hmm_usdinr_strategy.pkl")
log.info("State distribution:\n%s",
         df["state_filtered"].value_counts().sort_index().to_string())

# ---------------------------------------------------------------------------
# Step 6 — Targets + mid price
# ---------------------------------------------------------------------------

log.info("=== Step 6: Building targets ===")

if "mid" not in df.columns:
    df["mid"] = (df["bid_p1"] + df["ask_p1"]) / 2.0
df["log_mid"] = np.log(df["mid"])

g = df.groupby(["symbol", "date"], sort=False)
df["ret_60s"]  = (g["log_mid"].shift(-60)  - df["log_mid"]) * 1e4
df["ret_300s"] = (g["log_mid"].shift(-300) - df["log_mid"]) * 1e4

# Forward mid price (what you actually get at exit after HOLD_SEC)
df["mid_exit"] = g["mid"].shift(-HOLD_SEC)

log.info("ret_60s non-null: %.1f%%", df["ret_60s"].notna().mean() * 100)

# ---------------------------------------------------------------------------
# Step 7 — Normalize target + train LightGBM
# ---------------------------------------------------------------------------

log.info("=== Step 7: Normalizing target + training LightGBM ===")

# Fix 2: normalize ret_60s within each day to remove directional drift bias
# The model learns cross-sectional rank within each day, not absolute direction
# At inference: predictions are in normalized units — we apply asymmetric thresholds
df["ret_60s_norm"] = df.groupby(["symbol", "date"])["ret_60s"].transform(
    lambda x: (x - x.mean()) / (x.std() + 1e-8)
)
log.info("ret_60s_norm: mean=%.4f  std=%.4f  (should be ~0, ~1)",
         df["ret_60s_norm"].mean(), df["ret_60s_norm"].std())

features = CHALLENGER50
target   = "ret_60s_norm"   # normalized target for training

valid_mask = df[features + [target, "ret_60s"]].notna().all(axis=1)

tr = df[mask_train & valid_mask]
va = df[mask_val   & valid_mask]

X_tr, y_tr = tr[features].values, tr[target].values
X_va, y_va = va[features].values, va[target].values

ds_tr = lgb.Dataset(X_tr, label=y_tr, feature_name=features)
ds_va = lgb.Dataset(X_va, label=y_va, feature_name=features, reference=ds_tr)

params = {k: v for k, v in LGB_PARAMS.items()
          if k not in ("n_estimators", "early_stopping_rounds")}

lgbm_model = lgb.train(
    params,
    ds_tr,
    num_boost_round     = LGB_PARAMS["n_estimators"],
    valid_sets          = [ds_va],
    callbacks           = [
        lgb.early_stopping(LGB_PARAMS["early_stopping_rounds"], verbose=False),
        lgb.log_evaluation(100),
    ],
)
log.info("LightGBM best iteration: %d", lgbm_model.best_iteration)

# ---------------------------------------------------------------------------
# Step 8 — Generate signals on val + test
# ---------------------------------------------------------------------------

log.info("=== Step 8: Generating signals ===")

eval_mask   = (mask_val | mask_test) & valid_mask & df["mid_exit"].notna()
df_eval     = df[eval_mask].copy().reset_index(drop=True)

# Predictions are in normalized units (z-scores within each day)
df_eval["pred_ret_60s"] = lgbm_model.predict(df_eval[features].values)

# IC measured against raw ret_60s (what matters for trading)
ic_val, _  = spearmanr(
    df_eval.loc[df_eval["date"].isin(val_dates),  "ret_60s"],
    df_eval.loc[df_eval["date"].isin(val_dates),  "pred_ret_60s"],
)
ic_test, _ = spearmanr(
    df_eval.loc[df_eval["date"].isin(test_dates), "ret_60s"],
    df_eval.loc[df_eval["date"].isin(test_dates), "pred_ret_60s"],
)
log.info("Signal IC (vs raw ret_60s) — val: %.4f  test: %.4f", ic_val, ic_test)

# Check direction bias is fixed
mean_pred = df_eval["pred_ret_60s"].mean()
pct_short = (df_eval["pred_ret_60s"] < 0).mean() * 100
log.info("Prediction bias check: mean=%.4f  pct_short=%.1f%%  (want ~0, ~50%%)",
         mean_pred, pct_short)

# ---------------------------------------------------------------------------
# Step 9 — Simulate trades (single position, threshold filter)
# ---------------------------------------------------------------------------

log.info("=== Step 9: Simulating trades ===")

df_eval["state"]      = df_eval["state_filtered"].astype(int)
df_eval["size_state"] = df_eval["state"].map(STATE_SIZE)
df_eval["stop_bps"]   = df_eval["state"].map(STATE_STOP_BPS)

# Fix 3: time-of-day filter — compute IST time-of-day in seconds
df_eval["ist_tod"] = (df_eval["ts_sec"] + IST_OFFSET) % 86400
df_eval["in_window"] = (
    (df_eval["ist_tod"] >= OPEN_FILTER) &
    (df_eval["ist_tod"] <= CLOSE_FILTER)
)
log.info("Time filter: %.1f%% of bars in 10:30-14:30 IST window",
         df_eval["in_window"].mean() * 100)

log.info("Total bars available: %d  In-window: %d",
         len(df_eval), df_eval["in_window"].sum())


def simulate_single_position(
    df_day: pd.DataFrame,
    long_thresh: float,
    short_thresh: float,
    scale: float = 1.0,
) -> list:
    """
    Single-position simulator for one day.
    Rules:
      - Time filter: only trade in 10:30-14:30 IST window
      - Asymmetric entry: long if pred > long_thresh*scale,
                          short if pred < -short_thresh*scale
      - Single position: no overlap, hold HOLD_SEC seconds
      - Stop-loss: cap loss at state stop distance
    """
    trades     = []
    busy_until = -1

    lt = long_thresh  * scale
    st = short_thresh * scale

    rows = df_day.sort_values("ts_sec").reset_index(drop=True)

    for _, row in rows.iterrows():
        ts = row["ts_sec"]

        # Still in position
        if ts <= busy_until:
            continue

        # Time filter
        if not row["in_window"]:
            continue

        pred = float(row["pred_ret_60s"])

        # Asymmetric threshold check
        if pred >= lt:
            direction = 1
        elif pred <= -st:
            direction = -1
        else:
            continue

        if pd.isna(row["ret_60s"]):
            continue

        state      = int(row["state"])
        size       = float(row["size_state"])
        stop       = float(row["stop_bps"])
        actual_ret = direction * float(row["ret_60s"])

        stopped = actual_ret < -stop
        pnl_raw = -stop if stopped else actual_ret
        net_pnl = (pnl_raw - COST_BPS) * size

        trades.append({
            "ts_sec":       ts,
            "pred_ret":     pred,
            "actual_ret":   float(row["ret_60s"]),
            "direction":    direction,
            "state":        state,
            "size":         size,
            "stop_bps":     stop,
            "stopped_out":  stopped,
            "net_pnl_bps":  net_pnl,
            "long_thresh":  lt,
            "short_thresh": st,
            "scale":        scale,
        })

        busy_until = ts + HOLD_SEC

    return trades


def compute_strategy_metrics(pnl_series: pd.Series, label: str) -> dict:
    """Sharpe, hit rate, drawdown from a bps PnL series."""
    pnl = pnl_series.dropna().values
    n   = len(pnl)
    if n == 0:
        return {"n_trades": 0}

    total_pnl = float(pnl.sum())
    mean_pnl  = float(pnl.mean())
    std_pnl   = float(pnl.std()) if pnl.std() > 0 else 1e-9
    # Annualise: assume ~23,400 tradeable seconds/day, 252 days
    # But with threshold filtering trades-per-day drops, so use per-trade Sharpe
    sharpe    = mean_pnl / std_pnl * np.sqrt(n * 252 / len(pnl_series.index.unique()) if hasattr(pnl_series.index, 'unique') else n)
    sharpe    = mean_pnl / std_pnl * np.sqrt(n)   # simpler: sqrt-n Sharpe
    hit_rate  = float(np.mean(pnl > 0))
    cum       = np.cumsum(pnl)
    running_max = np.maximum.accumulate(cum)
    max_dd    = float((cum - running_max).min())

    log.info(
        "%s | total=%.2fbps mean=%.4fbps sharpe=%.2f hit=%.3f maxDD=%.2fbps n=%d",
        label, total_pnl, mean_pnl, sharpe, hit_rate, max_dd, n,
    )
    return {
        "total_pnl_bps":    round(total_pnl, 2),
        "mean_pnl_bps":     round(mean_pnl,  5),
        "std_pnl_bps":      round(std_pnl,   5),
        "sharpe":           round(sharpe,     3),
        "hit_rate":         round(hit_rate,   4),
        "max_drawdown_bps": round(max_dd,     2),
        "n_trades":         n,
    }


# ---------------------------------------------------------------------------
# Step 10 — Sweep thresholds on test set
# ---------------------------------------------------------------------------

log.info("=== Step 10: Sweeping thresholds ===")

test_mask = df_eval["date"].isin(test_dates)
te_days   = df_eval[test_mask].copy()

threshold_results = []
all_trades        = []

for scale in THRESHOLD_SCALES:
    lt = LONG_THRESH  * scale
    st = SHORT_THRESH * scale

    day_trades = []
    for day, grp in te_days.groupby("date"):
        t = simulate_single_position(grp, LONG_THRESH, SHORT_THRESH, scale)
        for tr in t:
            tr["date"] = str(day)
        day_trades.extend(t)

    if not day_trades:
        log.info("scale=%.2f (L=%.2f S=%.2f) | 0 trades", scale, lt, st)
        continue

    trades_df  = pd.DataFrame(day_trades)
    pnl        = trades_df["net_pnl_bps"]
    n_long     = int((trades_df["direction"] == 1).sum())
    n_short    = int((trades_df["direction"] == -1).sum())
    m          = compute_strategy_metrics(pnl, f"scale={scale:.2f} L={lt:.2f}/S={st:.2f}")
    m["scale"]       = scale
    m["long_thresh"] = round(lt, 3)
    m["short_thresh"]= round(st, 3)
    m["n_long"]      = n_long
    m["n_short"]     = n_short
    threshold_results.append(m)
    all_trades.extend(day_trades)

threshold_df = pd.DataFrame(threshold_results)

# Best scale by Sharpe
best_scale = 1.0
if not threshold_df.empty:
    best_row   = threshold_df.loc[threshold_df["sharpe"].idxmax()]
    best_scale = float(best_row["scale"])
    log.info("Best scale by Sharpe: %.2f (Sharpe=%.3f, n=%d)",
             best_scale, best_row["sharpe"], best_row["n_trades"])

# ---------------------------------------------------------------------------
# Step 11 — Per-state breakdown at best threshold
# ---------------------------------------------------------------------------

log.info("=== Step 11: Per-state breakdown ===")

best_trades = [t for t in all_trades if t["scale"] == best_scale]
state_rows  = []

if best_trades:
    bt_df = pd.DataFrame(best_trades)
    for state in [1, 2, 3, 4]:
        s = bt_df[bt_df["state"] == state]
        if len(s) < 5:
            continue
        m = compute_strategy_metrics(s["net_pnl_bps"], f"state_{state}@{best_thresh:.2f}")
        m["state"]     = state
        m["pct_trades"]= round(len(s) / len(bt_df) * 100, 1)
        m["size_mult"] = STATE_SIZE[state]
        state_rows.append(m)

state_metrics = pd.DataFrame(state_rows)

# ---------------------------------------------------------------------------
# Step 12 — Daily PnL at best threshold
# ---------------------------------------------------------------------------

log.info("=== Step 12: Daily PnL ===")

daily_rows = []
if best_trades:
    bt_df = pd.DataFrame(best_trades)
    for day, grp in bt_df.groupby("date"):
        pnl = grp["net_pnl_bps"]
        daily_rows.append({
            "date":      day,
            "threshold": best_thresh,
            "total_pnl": round(float(pnl.sum()), 2),
            "n_trades":  len(pnl),
            "hit_rate":  round(float((pnl > 0).mean()), 3),
            "mean_pnl":  round(float(pnl.mean()), 5),
        })

daily_pnl = pd.DataFrame(daily_rows)

# ---------------------------------------------------------------------------
# Step 13 — Comparison + trade log
# ---------------------------------------------------------------------------

comparison = threshold_df.copy()

trade_log = pd.DataFrame(all_trades) if all_trades else pd.DataFrame()

# ---------------------------------------------------------------------------
# Step 14 — Save outputs
# ---------------------------------------------------------------------------

log.info("=== Step 14: Saving outputs ===")

with open(OUT_DIR / "metrics.json", "w") as f:
    json.dump(
        {
            "split_dates": {
                "train": [str(d) for d in train_dates],
                "val":   [str(d) for d in val_dates],
                "test":  [str(d) for d in test_dates],
            },
            "config": {
                "hold_sec":       HOLD_SEC,
                "cost_bps":       COST_BPS,
                "long_thresh":    LONG_THRESH,
                "short_thresh":   SHORT_THRESH,
                "threshold_scales": THRESHOLD_SCALES,
                "time_window_ist": "10:30-14:30",
                "state_size":     STATE_SIZE,
                "state_stop":     STATE_STOP_BPS,
                "feature_set":    "CHALLENGER50",
                "target":         "ret_60s_norm",
                "signal":         "ret_60s",
            },
            "signal_ic":  {"val": round(ic_val, 4), "test": round(ic_test, 4)},
            "best_scale": best_scale if not threshold_df.empty else None,
            "threshold_results": threshold_df.to_dict(orient="records") if not threshold_df.empty else [],
        },
        f,
        indent=2,
    )
log.info("Saved metrics.json")

comparison.to_csv(OUT_DIR / "threshold_sweep.csv", index=False)
log.info("Saved threshold_sweep.csv")

if not state_metrics.empty:
    state_metrics.to_csv(OUT_DIR / "state_metrics.csv", index=False)
    log.info("Saved state_metrics.csv")

if not daily_pnl.empty:
    daily_pnl.to_csv(OUT_DIR / "daily_pnl.csv", index=False)
    log.info("Saved daily_pnl.csv")

if not trade_log.empty:
    trade_log.to_csv(OUT_DIR / "trade_log.csv", index=False)
    log.info("Saved trade_log.csv (%d trades)", len(trade_log))

# ---------------------------------------------------------------------------
# Step 15 — Final summary
# ---------------------------------------------------------------------------

print("\n" + "=" * 70)
print("STRATEGY BACKTEST v2 — FIXES: NORM TARGET + ASYM THRESH + TIME FILTER")
print("=" * 70)
print(f"\nSignal IC (test): {ic_test:.4f}")
print(f"Transaction cost: {COST_BPS} bps round-trip")
print(f"Hold time:        {HOLD_SEC}s fixed")
print(f"Time window:      10:30-14:30 IST only")
print(f"Long  threshold:  pred_norm > +{LONG_THRESH}")
print(f"Short threshold:  pred_norm < -{SHORT_THRESH}  (higher bar for shorts)\n")

if not threshold_df.empty:
    print(threshold_df[[
        "scale", "long_thresh", "short_thresh", "n_trades",
        "n_long", "n_short", "hit_rate",
        "mean_pnl_bps", "total_pnl_bps", "sharpe", "max_drawdown_bps"
    ]].to_string(index=False))

    print(f"\n--- Best scale: {best_scale:.2f} ---")
    print(best_row.to_string())

if not state_metrics.empty:
    print(f"\n--- Per-State Breakdown at scale={best_scale:.2f} ---")
    print(state_metrics[[
        "state", "pct_trades", "size_mult", "n_trades",
        "total_pnl_bps", "sharpe", "hit_rate", "max_drawdown_bps"
    ]].to_string(index=False))

if not daily_pnl.empty:
    print("\n--- Daily PnL at best scale ---")
    print(daily_pnl[["date", "n_trades", "total_pnl", "hit_rate"]].to_string(index=False))

print("=" * 70)

23:55:25  INFO  === Step 1: Loading raw data ===


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

... (11 outputs left out)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

23:57:25  INFO  Raw: 472806 rows | symbols: ['USDINR26SEPFUT', 'USDINR26OCTFUT', 'USDINR26NOVFUT']
23:57:25  INFO  === Step 2: Building features ===
23:57:30  INFO  preprocess: 881382 rows after regularization
23:57:30  INFO  calculate_features: building CORE31 ...
23:57:53  INFO  build_core31_from_regularized: built 881382 rows, 31 CORE31 features
23:57:54  INFO  calculate_features: building book geometry (ADDED6 + CONTROLS9) ...
23:58:00  INFO  _build_book_geometry_features: built 881382 rows
23:58:00  INFO  calculate_features: building shock recovery (CHALLENGERS 1-2) ...
23:58:01  INFO  _build_shock_recovery_features: bid_shocks=22175 ask_shocks=42052
23:58:02  INFO  _build_shock_recovery_features: built 881382 rows
23:58:02  INFO  calculate_features: building update clustering (CHALLENGER 3) ...
23:58:04  INFO  _build_update_clustering_features: obs=472806 book_change=326695 bid=149451 ask=233683
23:58:05  INFO  _build_update_clustering_features: built 881382 rows
23:58:05  INFO  


STRATEGY BACKTEST v2 — FIXES: NORM TARGET + ASYM THRESH + TIME FILTER

Signal IC (test): 0.2378
Transaction cost: 1.0 bps round-trip
Hold time:        60s fixed
Time window:      10:30-14:30 IST only
Long  threshold:  pred_norm > +0.8
Short threshold:  pred_norm < -1.2  (higher bar for shorts)

 scale  long_thresh  short_thresh  n_trades  n_long  n_short  hit_rate  mean_pnl_bps  total_pnl_bps  sharpe  max_drawdown_bps
  0.50          0.4           0.6       377     251      126    0.3899      -0.12305         -46.39  -2.773            -61.24
  0.75          0.6           0.9        73      42       31    0.4384      -0.01079          -0.79  -0.080            -15.42
  1.00          0.8           1.2        19      10        9    0.5263       0.72395          13.76   2.087             -0.93
  1.25          1.0           1.5         8       4        4    0.7500       0.91919           7.35   1.653             -0.51
  1.50          1.2           1.8         2       0        2    1.0000   

In [8]:
"""
USDINR HMM-State Strategy Backtest  v4
========================================
Single pooled LightGBM model, focused on fixing the short bias.

What changed from v2:
  1. Training target: raw ret_60s (bps) instead of normalized ret_60s_norm
     - Normalization removes drift BUT also removes the signal that longs and
       shorts differ in magnitude. We keep the asymmetric threshold to handle
       this in the simulator instead.
     - We add a 'direction_bias_correction' column: subtract the rolling 5-day
       mean of ret_60s from each row (removes slow drift, keeps fast signal)

  2. Short confirmation filter: only take a short signal when OFI (or bid/ask
     imbalance) also confirms selling pressure. This kills noise shorts while
     keeping the high-conviction ones.
     - If feature "ofi" or "order_flow_imbalance" is in CHALLENGER50, we gate
       short entries: short only if pred < -thresh AND ofi < -0.1 (selling)
     - If OFI not available, we fall back to tick_direction confirmation

  3. Direction-balanced training: oversample rows where actual move > 1 bps
     to give the model equal exposure to meaningful longs and shorts.
     Concretely: weight each training row by abs(ret_60s), so large-move bars
     (both directions) matter more than tiny-move bars.

  4. Added: per-state breakdown includes long/short counts so we can see
     where the asymmetry persists.

Outputs: results/strategy_backtest_v4/
  metrics.json
  threshold_sweep.csv
  state_metrics.csv
  daily_pnl.csv
  trade_log.csv
  feature_importance.csv

No plots — CSV/JSON only.
"""

import sys
sys.path.insert(0, r"C:\projects\hedge_fund")

import json
import logging
import pathlib
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import lightgbm as lgb
from scipy.stats import spearmanr

from data.store.duckdb_client import DuckDBClient
from data.store.s3_client import S3Client
from data.processing.usdinr.features import USDINRFeatureBuilder
from data.processing.usdinr.constants import CHALLENGER50
from data.processing.usdinr.regimes import (
    add_soft_hmm_features,
    save_hmm_model,
    HMM_PROB_COLS,
)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)s  %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger(__name__)

# ---------------------------------------------------------------------------
# Config
# ---------------------------------------------------------------------------

OUT_DIR   = pathlib.Path(r"C:\projects\hedge_fund\results\strategy_backtest_v4")
MODEL_DIR = pathlib.Path(r"C:\projects\hedge_fund\models")
OUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_DAYS = 15
VAL_DAYS   = 5
COST_BPS   = 1.0
HOLD_SEC   = 60

# Thresholds: asymmetric, applied to raw bps prediction
# (we're now training on raw bps, so these are in bps units)
# Recalibrated vs v2: v2 used normalized units (σ), v4 uses bps
# 1.5 bps ≈ 90th percentile of |ret_60s|, well above noise
LONG_THRESH  = 1.5    # enter long  if pred_bps > +1.5
SHORT_THRESH = 2.0    # enter short if pred_bps < -2.0  (higher bar for shorts)

THRESHOLD_SCALES = [0.5, 0.75, 1.0, 1.25, 1.5, 2.0]

# IST time filter
IST_OFFSET   = 19800
OPEN_FILTER  = 10 * 3600 + 30 * 60   # 10:30 IST
CLOSE_FILTER = 14 * 3600 + 30 * 60   # 14:30 IST

# LightGBM — now with sample weights for direction balance
LGB_PARAMS = {
    "objective":             "regression",
    "metric":                "rmse",
    "verbosity":             -1,
    "boosting_type":         "gbdt",
    "num_leaves":            31,
    "learning_rate":         0.05,
    "feature_fraction":      0.8,
    "bagging_fraction":      0.8,
    "bagging_freq":          5,
    "min_child_samples":     50,   # lowered from 100 — allows finer splits
}
N_ESTIMATORS         = 1000
EARLY_STOPPING_ROUNDS = 50

# OFI confirmation threshold for short entries (if OFI feature exists)
OFI_SHORT_THRESHOLD = -0.1   # negative = selling pressure

# Per-state position sizing (unchanged from v2)
STATE_SIZE = {1: 0.59, 2: 0.44, 3: 0.65, 4: 1.00}
STATE_STOP_BPS = {1: 2.34, 2: 3.08, 3: 2.04, 4: 1.33}

# ---------------------------------------------------------------------------
# Step 1 — Load raw data
# ---------------------------------------------------------------------------

log.info("=== Step 1: Loading raw data ===")

db = DuckDBClient()
s  = S3Client()

all_contracts = ["26SEPFUT", "26OCTFUT", "26NOVFUT"]
all_frames    = []

for contract in all_contracts:
    files = s.list_files(f"processed/1s/USDINR{contract}/")
    if not files:
        continue
    for f in files:
        parts = f.split("/")
        sym   = parts[2]
        date  = parts[3].replace(".parquet", "")
        try:
            df_day           = db.read_parquet(f"processed/1s/{sym}/{date}.parquet")
            df_day["symbol"] = sym
            all_frames.append(df_day)
        except Exception as e:
            log.warning("Failed %s/%s: %s", sym, date, e)

db.close()

if not all_frames:
    raise RuntimeError("No data loaded")

df_raw = pd.concat(all_frames, ignore_index=True)
log.info("Raw: %d rows | symbols: %s", len(df_raw), df_raw["symbol"].unique().tolist())

# ---------------------------------------------------------------------------
# Step 2 — Build features
# ---------------------------------------------------------------------------

log.info("=== Step 2: Building features ===")

builder = USDINRFeatureBuilder(data=df_raw, auto_build=True)
df      = builder.dataset.copy()
log.info("Dataset: %d rows, %d cols", len(df), len(df.columns))

if "tick_count" not in df.columns:
    raise RuntimeError("tick_count not found")
df["tick_count"] = df["tick_count"].fillna(0).astype(int)

# Detect OFI feature name (may differ across feature builder versions)
ofi_col = None
for candidate in ["ofi", "order_flow_imbalance", "ofi_1", "ofi_norm"]:
    if candidate in df.columns:
        ofi_col = candidate
        break
log.info("OFI feature: %s", ofi_col if ofi_col else "NOT FOUND — no short confirmation filter")

# ---------------------------------------------------------------------------
# Step 3 — Split dates
# ---------------------------------------------------------------------------

log.info("=== Step 3: Splitting ===")

df        = df.sort_values(["symbol", "date", "ts_sec"]).reset_index(drop=True)
all_dates = sorted(df["date"].unique())
n_days    = len(all_dates)
log.info("Total days: %d  (%s → %s)", n_days, all_dates[0], all_dates[-1])

if n_days < TRAIN_DAYS + VAL_DAYS + 1:
    raise RuntimeError(f"Need at least {TRAIN_DAYS + VAL_DAYS + 1} days, got {n_days}")

train_dates = all_dates[:TRAIN_DAYS]
val_dates   = all_dates[TRAIN_DAYS : TRAIN_DAYS + VAL_DAYS]
test_dates  = all_dates[TRAIN_DAYS + VAL_DAYS:]
train_end   = str(train_dates[-1])

log.info("Train: %d  Val: %d  Test: %d days",
         len(train_dates), len(val_dates), len(test_dates))

mask_train = df["date"].isin(train_dates)
mask_val   = df["date"].isin(val_dates)
mask_test  = df["date"].isin(test_dates)

# ---------------------------------------------------------------------------
# Step 4 — Fit HMM
# ---------------------------------------------------------------------------

log.info("=== Step 4: Fitting HMM ===")

df, hmm_model = add_soft_hmm_features(
    df             = df,
    train_end_date = train_end,
    K              = 4,
    restarts       = 3,
    max_iter       = 70,
    tol            = 1e-6,
    seed           = 42,
)
save_hmm_model(hmm_model, MODEL_DIR / "hmm_usdinr_v4.pkl")
log.info("State distribution:\n%s",
         df["state_filtered"].value_counts().sort_index().to_string())

# ---------------------------------------------------------------------------
# Step 5 — Build targets + mid price
# ---------------------------------------------------------------------------

log.info("=== Step 5: Building targets ===")

if "mid" not in df.columns:
    df["mid"] = (df["bid_p1"] + df["ask_p1"]) / 2.0
df["log_mid"] = np.log(df["mid"])

g = df.groupby(["symbol", "date"], sort=False)
df["ret_60s"]  = (g["log_mid"].shift(-60) - df["log_mid"]) * 1e4
df["mid_exit"] = g["mid"].shift(-HOLD_SEC)

# Fix 1: drift-corrected target
# Subtract 5-day rolling mean of ret_60s to remove slow directional drift
# but keep it in bps (not normalized) so thresholds are interpretable
df = df.sort_values(["symbol", "date", "ts_sec"]).reset_index(drop=True)
daily_mean = df.groupby(["symbol", "date"])["ret_60s"].transform("mean")
df["ret_60s_dc"] = df["ret_60s"] - daily_mean   # daily drift removed, stays in bps

# Check bias is gone
log.info("ret_60s_dc:  mean=%.5f  (want ~0)", df["ret_60s_dc"].mean())
log.info("ret_60s raw: mean=%.5f  pct_neg=%.1f%%",
         df["ret_60s"].mean(),
         (df["ret_60s"] < 0).mean() * 100)
log.info("ret_60s_dc:  pct_neg=%.1f%%  (want ~50%%)",
         (df["ret_60s_dc"] < 0).mean() * 100)

# IST time
df["ist_tod"]   = (df["ts_sec"] + IST_OFFSET) % 86400
df["in_window"] = (
    (df["ist_tod"] >= OPEN_FILTER) &
    (df["ist_tod"] <= CLOSE_FILTER)
)

# ---------------------------------------------------------------------------
# Step 6 — Train LightGBM with sample weights
# ---------------------------------------------------------------------------

log.info("=== Step 6: Training LightGBM ===")

features = CHALLENGER50
target   = "ret_60s_dc"   # drift-corrected bps target

valid_mask = df[features + [target, "ret_60s"]].notna().all(axis=1)

tr = df[mask_train & valid_mask]
va = df[mask_val   & valid_mask]

X_tr, y_tr = tr[features].values, tr[target].values
X_va, y_va = va[features].values, va[target].values

# Fix 3: sample weights — weight each row by |ret_60s_dc|
# This balances large-move events (both directions) vs tiny-move noise
# Clip at 99th percentile to avoid a few massive outliers dominating
w_tr = np.abs(y_tr)
p99  = np.percentile(w_tr, 99)
w_tr = np.clip(w_tr, 0, p99) + 0.1   # +0.1 so no row gets zero weight

log.info("Sample weight stats: mean=%.4f  max=%.4f  p99=%.4f",
         w_tr.mean(), w_tr.max(), p99)

# Check direction balance in training data
pct_long_tr  = float(np.mean(y_tr > 0)) * 100
pct_short_tr = float(np.mean(y_tr < 0)) * 100
log.info("Training target direction: %.1f%% positive  %.1f%% negative  (want ~50/50)",
         pct_long_tr, pct_short_tr)

ds_tr = lgb.Dataset(X_tr, label=y_tr, weight=w_tr, feature_name=features)
ds_va = lgb.Dataset(X_va, label=y_va,              feature_name=features, reference=ds_tr)

lgbm_model = lgb.train(
    LGB_PARAMS,
    ds_tr,
    num_boost_round = N_ESTIMATORS,
    valid_sets      = [ds_va],
    callbacks       = [
        lgb.early_stopping(EARLY_STOPPING_ROUNDS, verbose=False),
        lgb.log_evaluation(100),
    ],
)
log.info("LightGBM best iteration: %d", lgbm_model.best_iteration)

# Save model
lgbm_model.save_model(str(MODEL_DIR / "lgbm_v4.txt"))

# Feature importance
fi_df = pd.DataFrame({
    "feature":    features,
    "importance": lgbm_model.feature_importance(importance_type="gain"),
}).sort_values("importance", ascending=False)
log.info("Top 10 features:\n%s", fi_df.head(10).to_string(index=False))

# ---------------------------------------------------------------------------
# Step 7 — Generate signals on val + test
# ---------------------------------------------------------------------------

log.info("=== Step 7: Generating signals ===")

eval_mask = (mask_val | mask_test) & valid_mask & df["mid_exit"].notna()
df_eval   = df[eval_mask].copy().reset_index(drop=True)

# Predictions are in bps (drift-corrected) — directly comparable to thresholds
df_eval["pred_ret_60s"] = lgbm_model.predict(df_eval[features].values)

# Check direction bias is fixed
mean_pred  = df_eval["pred_ret_60s"].mean()
pct_short  = (df_eval["pred_ret_60s"] < 0).mean() * 100
log.info("Prediction bias: mean=%.5f  pct_short=%.1f%%  (want ~0, ~50%%)",
         mean_pred, pct_short)

# IC against raw ret_60s (what actually determines profit)
ic_val, _  = spearmanr(
    df_eval.loc[df_eval["date"].isin(val_dates),  "ret_60s"],
    df_eval.loc[df_eval["date"].isin(val_dates),  "pred_ret_60s"],
)
ic_test, _ = spearmanr(
    df_eval.loc[df_eval["date"].isin(test_dates), "ret_60s"],
    df_eval.loc[df_eval["date"].isin(test_dates), "pred_ret_60s"],
)
log.info("Signal IC (vs raw ret_60s) — val: %.4f  test: %.4f", ic_val, ic_test)

# Per-state IC
df_eval["state"] = df_eval["state_filtered"].astype(int)
for s_id in [1, 2, 3, 4]:
    s_mask = (df_eval["state"] == s_id) & df_eval["date"].isin(test_dates)
    if s_mask.sum() < 20:
        continue
    ic_s, _ = spearmanr(
        df_eval.loc[s_mask, "ret_60s"],
        df_eval.loc[s_mask, "pred_ret_60s"],
    )
    log.info("  State %d test IC: %.4f  (n=%d)", s_id, ic_s, s_mask.sum())

# Attach state metadata
df_eval["size_state"] = df_eval["state"].map(STATE_SIZE)
df_eval["stop_bps"]   = df_eval["state"].map(STATE_STOP_BPS)

# ---------------------------------------------------------------------------
# Step 8 — Simulate trades
# ---------------------------------------------------------------------------

log.info("=== Step 8: Simulating ===")


def simulate_single_position(
    df_day: pd.DataFrame,
    long_thresh: float,
    short_thresh: float,
    ofi_col: str | None = None,
) -> list:
    """
    Single-position simulator.
    - Long:  pred > +long_thresh
    - Short: pred < -short_thresh AND (ofi < OFI_SHORT_THRESHOLD if ofi available)
    - Time filter: 10:30-14:30 IST
    - Stop-loss: per-state bps distance
    """
    trades     = []
    busy_until = -1

    rows = df_day.sort_values("ts_sec").reset_index(drop=True)

    for _, row in rows.iterrows():
        ts = row["ts_sec"]

        if ts <= busy_until:
            continue
        if not row["in_window"]:
            continue
        if pd.isna(row["pred_ret_60s"]) or pd.isna(row["ret_60s"]):
            continue

        pred = float(row["pred_ret_60s"])

        if pred >= long_thresh:
            direction = 1
        elif pred <= -short_thresh:
            # Fix 2: short confirmation filter using OFI
            if ofi_col is not None:
                ofi_val = row.get(ofi_col, None)
                if ofi_val is not None and not pd.isna(ofi_val):
                    if float(ofi_val) > OFI_SHORT_THRESHOLD:
                        # OFI says not selling — skip this short
                        continue
            direction = -1
        else:
            continue

        state      = int(row["state"])
        size       = float(row["size_state"])
        stop       = float(row["stop_bps"])
        actual_ret = direction * float(row["ret_60s"])

        stopped = actual_ret < -stop
        pnl_raw = -stop if stopped else actual_ret
        net_pnl = (pnl_raw - COST_BPS) * size

        trades.append({
            "ts_sec":      ts,
            "pred_ret":    pred,
            "actual_ret":  float(row["ret_60s"]),
            "direction":   direction,
            "state":       state,
            "size":        size,
            "stop_bps":    stop,
            "stopped_out": stopped,
            "net_pnl_bps": net_pnl,
            "long_thresh": long_thresh,
            "short_thresh":short_thresh,
        })

        busy_until = ts + HOLD_SEC

    return trades


def compute_metrics(pnl_series: pd.Series, label: str) -> dict:
    pnl = pnl_series.dropna().values
    n   = len(pnl)
    if n == 0:
        return {"n_trades": 0, "sharpe": 0, "hit_rate": 0,
                "total_pnl_bps": 0, "mean_pnl_bps": 0,
                "std_pnl_bps": 0, "max_drawdown_bps": 0}
    total  = float(pnl.sum())
    mean   = float(pnl.mean())
    std    = float(pnl.std()) if pnl.std() > 0 else 1e-9
    sharpe = mean / std * np.sqrt(n)
    hit    = float(np.mean(pnl > 0))
    cum    = np.cumsum(pnl)
    max_dd = float((cum - np.maximum.accumulate(cum)).min())
    log.info("%s | n=%d total=%.2f sharpe=%.3f hit=%.3f",
             label, n, total, sharpe, hit)
    return {
        "total_pnl_bps":    round(total,  2),
        "mean_pnl_bps":     round(mean,   5),
        "std_pnl_bps":      round(std,    5),
        "sharpe":           round(sharpe, 3),
        "hit_rate":         round(hit,    4),
        "max_drawdown_bps": round(max_dd, 2),
        "n_trades":         n,
    }


# ---------------------------------------------------------------------------
# Step 9 — Threshold sweep on test set
# ---------------------------------------------------------------------------

log.info("=== Step 9: Threshold sweep ===")

test_mask = df_eval["date"].isin(test_dates)
te_days   = df_eval[test_mask].copy()

sweep_results = []
all_trades    = []

for scale in THRESHOLD_SCALES:
    lt = LONG_THRESH  * scale
    st = SHORT_THRESH * scale

    day_trades = []
    for day, grp in te_days.groupby("date"):
        t = simulate_single_position(grp, lt, st, ofi_col=ofi_col)
        for tr in t:
            tr["date"]  = str(day)
            tr["scale"] = scale
        day_trades.extend(t)

    if not day_trades:
        log.info("scale=%.2f | 0 trades", scale)
        continue

    tdf    = pd.DataFrame(day_trades)
    n_long = int((tdf["direction"] == 1).sum())
    n_short= int((tdf["direction"] == -1).sum())

    m = compute_metrics(tdf["net_pnl_bps"], f"scale={scale:.2f}")
    m["scale"]    = scale
    m["n_long"]   = n_long
    m["n_short"]  = n_short
    m["pct_long"] = round(100 * n_long / max(len(tdf), 1), 1)
    sweep_results.append(m)
    all_trades.extend(day_trades)

sweep_df = pd.DataFrame(sweep_results)

best_scale = 1.0
if not sweep_df.empty:
    best_row   = sweep_df.loc[sweep_df["sharpe"].idxmax()]
    best_scale = float(best_row["scale"])
    log.info("Best scale: %.2f (Sharpe=%.3f, n=%d, hit=%.3f)",
             best_scale, best_row["sharpe"],
             best_row["n_trades"], best_row["hit_rate"])

# ---------------------------------------------------------------------------
# Step 10 — Per-state breakdown at best scale
# ---------------------------------------------------------------------------

log.info("=== Step 10: Per-state breakdown ===")

best_trades = [t for t in all_trades if t["scale"] == best_scale]
state_rows  = []

if best_trades:
    bt_df = pd.DataFrame(best_trades)
    for state in [1, 2, 3, 4]:
        s = bt_df[bt_df["state"] == state]
        if len(s) < 3:
            continue
        m = compute_metrics(s["net_pnl_bps"], f"state_{state}@scale={best_scale:.2f}")
        m["state"]      = state
        m["pct_trades"] = round(len(s) / len(bt_df) * 100, 1)
        m["size_mult"]  = STATE_SIZE[state]
        m["n_long"]     = int((s["direction"] == 1).sum())
        m["n_short"]    = int((s["direction"] == -1).sum())
        m["pct_long"]   = round(100 * m["n_long"] / max(len(s), 1), 1)
        state_rows.append(m)

state_metrics = pd.DataFrame(state_rows)

# ---------------------------------------------------------------------------
# Step 11 — Daily PnL
# ---------------------------------------------------------------------------

daily_rows = []
if best_trades:
    bt_df = pd.DataFrame(best_trades)
    for day, grp in bt_df.groupby("date"):
        pnl = grp["net_pnl_bps"]
        daily_rows.append({
            "date":     day,
            "n_trades": len(pnl),
            "n_long":   int((grp["direction"] == 1).sum()),
            "n_short":  int((grp["direction"] == -1).sum()),
            "total_pnl":round(float(pnl.sum()), 2),
            "hit_rate": round(float((pnl > 0).mean()), 3),
            "mean_pnl": round(float(pnl.mean()), 5),
        })

daily_pnl = pd.DataFrame(daily_rows)

# ---------------------------------------------------------------------------
# Step 12 — Save outputs
# ---------------------------------------------------------------------------

log.info("=== Step 12: Saving outputs ===")

with open(OUT_DIR / "metrics.json", "w") as f:
    json.dump({
        "split_dates": {
            "train": [str(d) for d in train_dates],
            "val":   [str(d) for d in val_dates],
            "test":  [str(d) for d in test_dates],
        },
        "config": {
            "target":            "ret_60s_dc (drift-corrected bps)",
            "long_thresh_bps":   LONG_THRESH,
            "short_thresh_bps":  SHORT_THRESH,
            "ofi_confirmation":  ofi_col is not None,
            "ofi_col":           ofi_col,
            "sample_weights":    "abs(ret_60s_dc), clipped at p99",
            "time_window_ist":   "10:30-14:30",
            "state_size":        STATE_SIZE,
            "state_stop":        STATE_STOP_BPS,
        },
        "signal_ic": {"val": round(ic_val, 4), "test": round(ic_test, 4)},
        "best_scale": best_scale,
        "sweep": sweep_df.to_dict(orient="records") if not sweep_df.empty else [],
    }, f, indent=2)

sweep_df.to_csv(OUT_DIR / "threshold_sweep.csv", index=False)

if not state_metrics.empty:
    state_metrics.to_csv(OUT_DIR / "state_metrics.csv", index=False)

if not daily_pnl.empty:
    daily_pnl.to_csv(OUT_DIR / "daily_pnl.csv", index=False)

if all_trades:
    pd.DataFrame(all_trades).to_csv(OUT_DIR / "trade_log.csv", index=False)

fi_df.to_csv(OUT_DIR / "feature_importance.csv", index=False)

# ---------------------------------------------------------------------------
# Step 13 — Final summary
# ---------------------------------------------------------------------------

print("\n" + "=" * 70)
print("STRATEGY BACKTEST v4 — SINGLE LGBM, SHORT BIAS FIXES")
print("=" * 70)
print(f"\nTarget: drift-corrected bps (ret_60s minus daily mean)")
print(f"Sample weights: abs(ret_60s_dc) — balance large up/down moves")
print(f"Short confirmation: {'OFI filter ON (' + ofi_col + ')' if ofi_col else 'OFI not found — no filter'}")
print(f"IC — val: {ic_val:.4f}  test: {ic_test:.4f}")
print(f"Thresholds: Long={LONG_THRESH} bps  Short={SHORT_THRESH} bps (raw bps units)")
print(f"Cost: {COST_BPS} bps  |  Hold: {HOLD_SEC}s  |  Window: 10:30-14:30 IST\n")

print("--- Threshold sweep (test set) ---")
if not sweep_df.empty:
    print(sweep_df[[
        "scale", "n_trades", "n_long", "n_short", "pct_long",
        "hit_rate", "mean_pnl_bps", "total_pnl_bps", "sharpe", "max_drawdown_bps"
    ]].to_string(index=False))

if not state_metrics.empty:
    print(f"\n--- Per-State Breakdown at scale={best_scale:.2f} ---")
    print(state_metrics[[
        "state", "pct_trades", "size_mult", "n_trades", "n_long", "n_short", "pct_long",
        "total_pnl_bps", "sharpe", "hit_rate", "max_drawdown_bps"
    ]].to_string(index=False))

if not daily_pnl.empty:
    print(f"\n--- Daily PnL at scale={best_scale:.2f} ---")
    print(daily_pnl.to_string(index=False))

print("=" * 70)

00:24:44  INFO  === Step 1: Loading raw data ===


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

... (13 outputs left out)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

00:26:37  INFO  Raw: 472806 rows | symbols: ['USDINR26SEPFUT', 'USDINR26OCTFUT', 'USDINR26NOVFUT']
00:26:37  INFO  === Step 2: Building features ===
00:26:43  INFO  preprocess: 881382 rows after regularization
00:26:43  INFO  calculate_features: building CORE31 ...
00:27:12  INFO  build_core31_from_regularized: built 881382 rows, 31 CORE31 features
00:27:13  INFO  calculate_features: building book geometry (ADDED6 + CONTROLS9) ...
00:27:20  INFO  _build_book_geometry_features: built 881382 rows
00:27:20  INFO  calculate_features: building shock recovery (CHALLENGERS 1-2) ...
00:27:22  INFO  _build_shock_recovery_features: bid_shocks=22175 ask_shocks=42052
00:27:23  INFO  _build_shock_recovery_features: built 881382 rows
00:27:23  INFO  calculate_features: building update clustering (CHALLENGER 3) ...
00:27:25  INFO  _build_update_clustering_features: obs=472806 book_change=326695 bid=149451 ask=233683
00:27:27  INFO  _build_update_clustering_features: built 881382 rows
00:27:27  INFO  


STRATEGY BACKTEST v4 — SINGLE LGBM, SHORT BIAS FIXES

Target: drift-corrected bps (ret_60s minus daily mean)
Sample weights: abs(ret_60s_dc) — balance large up/down moves
Short confirmation: OFI not found — no filter
IC — val: 0.2371  test: 0.2111
Thresholds: Long=1.5 bps  Short=2.0 bps (raw bps units)
Cost: 1.0 bps  |  Hold: 60s  |  Window: 10:30-14:30 IST

--- Threshold sweep (test set) ---
 scale  n_trades  n_long  n_short  pct_long  hit_rate  mean_pnl_bps  total_pnl_bps  sharpe  max_drawdown_bps
  0.50        46      12       34      26.1    0.3478      -0.01546          -0.71  -0.115             -4.66
  0.75        13       2       11      15.4    0.3846      -0.22828          -2.97  -1.086             -3.85
  1.00         3       0        3       0.0    0.3333       0.12890           0.39   0.354             -0.63
  1.25         3       0        3       0.0    0.6667       0.42103           1.26   0.866             -0.36

--- Daily PnL at scale=1.25 ---
      date  n_trades  n_l

In [9]:
from data.processing.usdinr.constants import CHALLENGER50
print(CHALLENGER50)

['ofi_l1_sum_60s_signed_log', 'ofi_rate_diff_30_300_sl', 'ofi_activity_60s', 'ofi_activity_300s', 'update_rate_300s', 'price_move_rate_30s', 'impact_lambda_1800s', 'impact_r2_1800s', 'ofi_response_gap_z', 'queue_imbalance_l1', 'depth_imbalance_l5', 'queue_replenishment_pressure_60s', 'queue_imbalance_delta', 'depth_imbalance_delta', 'spread_rank_1800s', 'price_mom_10s_new', 'price_mom_30s_new', 'price_mom_60s_new', 'log_rv_300s_new', 'rv_ratio_60_300_new', 'ret_acf_lag1_300s', 'log_vr_5_300s', 'log_vr_30_300s', 'semivar_imbalance_300s', 'semivar_imbalance_900s', 'sign_persistence_900s', 'jump_ratio_300s', 'range_position_300s', 'spread_depth_stress_900', 'recovery_available_300', 'ofi_disagreement_weighted_60', 'quote_retreat_imbalance_30', 'slope_asymmetry', 'curvature_asymmetry', 'weighted_gap_asymmetry', 'fragility_asymmetry', 'relative_migration_60', 'queue_imbalance_ctrl', 'depth_imbalance_ctrl', 'log_total_depth_ctrl', 'spread_bps_ctrl', 'bid_l1_concentration_ctrl', 'ask_l1_conce

In [10]:
"""
USDINR HMM-State Strategy Backtest  v5
========================================
Core insight from v4 failure:
  CHALLENGER50 has 15+ directional features (momentum, signed OFI, semivar
  imbalance, sign_persistence). With a 15-day training window that happened to
  be net-down, the model learned "negative momentum → negative return" which is
  valid in-sample but bleeds into test where the regime may differ.

The fix: CROSS-SECTIONAL DEMEANING at inference.
  Instead of thresholding on pred directly, subtract the rolling 60s mean
  of predictions across all bars in that minute. The model then picks who
  will move MORE (long) or LESS (short) relative to what it expects for
  all bars right now — a pure relative-value signal, market-neutral by
  construction.

  pred_cs = pred - rolling_mean(pred, window=60s)

  This is equivalent to asking: "is this bar above or below the crowd's
  expected return right now?" — eliminates the systematic level bias while
  keeping the cross-sectional ranking signal intact.

Additionally: we now also test a LONG-ONLY version since the IC is positive
and longs have historically had better hit rates.

Outputs: results/strategy_backtest_v5/
  threshold_sweep.csv
  state_metrics.csv
  daily_pnl.csv
  trade_log.csv
  metrics.json

No plots — CSV/JSON only.
"""

import sys
sys.path.insert(0, r"C:\projects\hedge_fund")

import json
import logging
import pathlib
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import lightgbm as lgb
from scipy.stats import spearmanr

from data.store.duckdb_client import DuckDBClient
from data.store.s3_client import S3Client
from data.processing.usdinr.features import USDINRFeatureBuilder
from data.processing.usdinr.constants import CHALLENGER50
from data.processing.usdinr.regimes import (
    add_soft_hmm_features,
    save_hmm_model,
    HMM_PROB_COLS,
)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)s  %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger(__name__)

# ---------------------------------------------------------------------------
# Config
# ---------------------------------------------------------------------------

OUT_DIR   = pathlib.Path(r"C:\projects\hedge_fund\results\strategy_backtest_v5")
MODEL_DIR = pathlib.Path(r"C:\projects\hedge_fund\models")
OUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_DAYS = 15
VAL_DAYS   = 5
COST_BPS   = 1.0
HOLD_SEC   = 60

# After cross-sectional demeaning, predictions are centered at 0.
# Threshold = how many std deviations above/below the mean to enter.
# We sweep this in terms of σ of the demeaned predictions.
CS_LONG_THRESH  = 0.5   # enter long  if pred_cs > +0.5 * daily_std
CS_SHORT_THRESH = 0.5   # enter short if pred_cs < -0.5 * daily_std
# (symmetric now — the demeaning handles the bias, not asymmetric thresholds)

THRESHOLD_SCALES = [0.3, 0.5, 0.75, 1.0, 1.25, 1.5, 2.0]

IST_OFFSET   = 19800
OPEN_FILTER  = 10 * 3600 + 30 * 60
CLOSE_FILTER = 14 * 3600 + 30 * 60

LGB_PARAMS = {
    "objective":         "regression",
    "metric":            "rmse",
    "verbosity":         -1,
    "boosting_type":     "gbdt",
    "num_leaves":        31,
    "learning_rate":     0.05,
    "feature_fraction":  0.8,
    "bagging_fraction":  0.8,
    "bagging_freq":      5,
    "min_child_samples": 50,
}
N_EST = 1000
ES    = 50

STATE_SIZE     = {1: 0.59, 2: 0.44, 3: 0.65, 4: 1.00}
STATE_STOP_BPS = {1: 2.34, 2: 3.08, 3: 2.04, 4: 1.33}

# ---------------------------------------------------------------------------
# Step 1 — Load raw data
# ---------------------------------------------------------------------------

log.info("=== Step 1: Loading raw data ===")

db = DuckDBClient()
s  = S3Client()

all_frames = []
for contract in ["26SEPFUT", "26OCTFUT", "26NOVFUT"]:
    files = s.list_files(f"processed/1s/USDINR{contract}/")
    if not files:
        continue
    for f in files:
        parts = f.split("/")
        sym, date = parts[2], parts[3].replace(".parquet", "")
        try:
            df_day = db.read_parquet(f"processed/1s/{sym}/{date}.parquet")
            df_day["symbol"] = sym
            all_frames.append(df_day)
        except Exception as e:
            log.warning("Failed %s/%s: %s", sym, date, e)

db.close()

if not all_frames:
    raise RuntimeError("No data loaded")

df_raw = pd.concat(all_frames, ignore_index=True)
log.info("Raw: %d rows", len(df_raw))

# ---------------------------------------------------------------------------
# Step 2 — Build features
# ---------------------------------------------------------------------------

log.info("=== Step 2: Building features ===")

builder = USDINRFeatureBuilder(data=df_raw, auto_build=True)
df      = builder.dataset.copy()
log.info("Dataset: %d rows, %d cols", len(df), len(df.columns))

if "tick_count" not in df.columns:
    raise RuntimeError("tick_count missing")
df["tick_count"] = df["tick_count"].fillna(0).astype(int)

# ---------------------------------------------------------------------------
# Step 3 — Split
# ---------------------------------------------------------------------------

log.info("=== Step 3: Splitting ===")

df        = df.sort_values(["symbol", "date", "ts_sec"]).reset_index(drop=True)
all_dates = sorted(df["date"].unique())
n_days    = len(all_dates)

if n_days < TRAIN_DAYS + VAL_DAYS + 1:
    raise RuntimeError(f"Need {TRAIN_DAYS + VAL_DAYS + 1} days, got {n_days}")

train_dates = all_dates[:TRAIN_DAYS]
val_dates   = all_dates[TRAIN_DAYS : TRAIN_DAYS + VAL_DAYS]
test_dates  = all_dates[TRAIN_DAYS + VAL_DAYS:]
train_end   = str(train_dates[-1])

log.info("Train: %d  Val: %d  Test: %d", len(train_dates), len(val_dates), len(test_dates))

mask_train = df["date"].isin(train_dates)
mask_val   = df["date"].isin(val_dates)
mask_test  = df["date"].isin(test_dates)

# ---------------------------------------------------------------------------
# Step 4 — HMM
# ---------------------------------------------------------------------------

log.info("=== Step 4: Fitting HMM ===")

df, hmm_model = add_soft_hmm_features(
    df=df, train_end_date=train_end,
    K=4, restarts=3, max_iter=70, tol=1e-6, seed=42,
)
save_hmm_model(hmm_model, MODEL_DIR / "hmm_usdinr_v5.pkl")
log.info("State dist:\n%s", df["state_filtered"].value_counts().sort_index().to_string())

# ---------------------------------------------------------------------------
# Step 5 — Targets
# ---------------------------------------------------------------------------

log.info("=== Step 5: Building targets ===")

if "mid" not in df.columns:
    df["mid"] = (df["bid_p1"] + df["ask_p1"]) / 2.0
df["log_mid"] = np.log(df["mid"])

g = df.groupby(["symbol", "date"], sort=False)
df["ret_60s"]  = (g["log_mid"].shift(-60) - df["log_mid"]) * 1e4
df["mid_exit"] = g["mid"].shift(-HOLD_SEC)

# Normalized target for training (removes daily drift)
df["ret_60s_norm"] = df.groupby(["symbol", "date"])["ret_60s"].transform(
    lambda x: (x - x.mean()) / (x.std() + 1e-8)
)

df["ist_tod"]   = (df["ts_sec"] + IST_OFFSET) % 86400
df["in_window"] = (
    (df["ist_tod"] >= OPEN_FILTER) &
    (df["ist_tod"] <= CLOSE_FILTER)
)

# ---------------------------------------------------------------------------
# Step 6 — Train LightGBM
# ---------------------------------------------------------------------------

log.info("=== Step 6: Training LightGBM ===")

features   = CHALLENGER50
target     = "ret_60s_norm"
valid_mask = df[features + [target, "ret_60s"]].notna().all(axis=1)

tr = df[mask_train & valid_mask]
va = df[mask_val   & valid_mask]

X_tr, y_tr = tr[features].values, tr[target].values
X_va, y_va = va[features].values, va[target].values

ds_tr = lgb.Dataset(X_tr, label=y_tr, feature_name=features)
ds_va = lgb.Dataset(X_va, label=y_va, feature_name=features, reference=ds_tr)

lgbm_model = lgb.train(
    LGB_PARAMS, ds_tr,
    num_boost_round=N_EST,
    valid_sets=[ds_va],
    callbacks=[
        lgb.early_stopping(ES, verbose=False),
        lgb.log_evaluation(100),
    ],
)
log.info("Best iter: %d", lgbm_model.best_iteration)
lgbm_model.save_model(str(MODEL_DIR / "lgbm_v5.txt"))

# ---------------------------------------------------------------------------
# Step 7 — Generate raw predictions
# ---------------------------------------------------------------------------

log.info("=== Step 7: Generating predictions ===")

eval_mask = (mask_val | mask_test) & valid_mask & df["mid_exit"].notna()
df_eval   = df[eval_mask].copy().reset_index(drop=True)
df_eval["state"] = df_eval["state_filtered"].astype(int)

df_eval["pred_raw"] = lgbm_model.predict(df_eval[features].values)

# Check raw bias (expected to be biased — this is what we fix next)
raw_mean    = df_eval["pred_raw"].mean()
raw_pct_neg = (df_eval["pred_raw"] < 0).mean() * 100
log.info("RAW pred: mean=%.4f  pct_negative=%.1f%%  (bias visible here)",
         raw_mean, raw_pct_neg)

# ---------------------------------------------------------------------------
# Step 8 — CROSS-SECTIONAL DEMEANING (the core fix)
# ---------------------------------------------------------------------------
#
# For each second (ts_sec), subtract the mean prediction across ALL bars
# at that exact timestamp. Since each day has one row per second per symbol,
# this demeans within each second × symbol group.
#
# This converts the prediction from "absolute direction" → "relative to now"
# which is the correct framing for a market-neutral signal.
# ---------------------------------------------------------------------------

log.info("=== Step 8: Cross-sectional demeaning ===")

# Compute rolling 60-second mean of predictions within each day
# This smooths out microstructure noise while keeping the relative ordering
df_eval = df_eval.sort_values(["symbol", "date", "ts_sec"]).reset_index(drop=True)

# For each (symbol, date), compute a rolling 60-bar mean of pred_raw
# then subtract it — the signal becomes "this bar vs recent average"
df_eval["pred_roll_mean"] = (
    df_eval.groupby(["symbol", "date"])["pred_raw"]
    .transform(lambda x: x.rolling(window=60, min_periods=10).mean())
)

# Demeaned prediction: how much above/below the recent expected level
df_eval["pred_cs"] = df_eval["pred_raw"] - df_eval["pred_roll_mean"]

# Normalize by rolling std so threshold is in σ units (comparable across days)
df_eval["pred_cs_std"] = (
    df_eval.groupby(["symbol", "date"])["pred_raw"]
    .transform(lambda x: x.rolling(window=60, min_periods=10).std())
)
df_eval["pred_cs_std"] = df_eval["pred_cs_std"].fillna(1.0).clip(lower=1e-6)
df_eval["pred_cs_z"]   = df_eval["pred_cs"] / df_eval["pred_cs_std"]

# Check: demeaned signal should be ~0 mean and ~50/50 long/short
cs_mean    = df_eval["pred_cs_z"].mean()
cs_pct_neg = (df_eval["pred_cs_z"] < 0).mean() * 100
log.info("CS-demeaned pred_z: mean=%.4f  pct_negative=%.1f%%  (want ~0, ~50%%)",
         cs_mean, cs_pct_neg)

# IC of demeaned signal vs raw ret_60s
test_rows  = df_eval["date"].isin(test_dates) & df_eval["pred_cs_z"].notna()
val_rows   = df_eval["date"].isin(val_dates)  & df_eval["pred_cs_z"].notna()

ic_val_cs,  _ = spearmanr(df_eval.loc[val_rows,  "ret_60s"], df_eval.loc[val_rows,  "pred_cs_z"])
ic_test_cs, _ = spearmanr(df_eval.loc[test_rows, "ret_60s"], df_eval.loc[test_rows, "pred_cs_z"])
ic_val_raw,  _ = spearmanr(df_eval.loc[val_rows,  "ret_60s"], df_eval.loc[val_rows,  "pred_raw"])
ic_test_raw, _ = spearmanr(df_eval.loc[test_rows, "ret_60s"], df_eval.loc[test_rows, "pred_raw"])

log.info("IC raw    — val: %.4f  test: %.4f", ic_val_raw, ic_test_raw)
log.info("IC CS-z   — val: %.4f  test: %.4f  (want similar to raw)", ic_val_cs, ic_test_cs)

df_eval["size_state"] = df_eval["state"].map(STATE_SIZE)
df_eval["stop_bps"]   = df_eval["state"].map(STATE_STOP_BPS)

# ---------------------------------------------------------------------------
# Step 9 — Simulate
# ---------------------------------------------------------------------------

log.info("=== Step 9: Simulating ===")


def simulate(df_day: pd.DataFrame, long_z: float, short_z: float,
             long_only: bool = False) -> list:
    """
    Single-position simulator using cross-sectionally demeaned z-score.
    - Long  if pred_cs_z >  +long_z
    - Short if pred_cs_z < -short_z  (and not long_only mode)
    - Time window: 10:30-14:30 IST
    - Stop-loss: per-state bps
    """
    trades     = []
    busy_until = -1

    rows = df_day.sort_values("ts_sec").reset_index(drop=True)

    for _, row in rows.iterrows():
        ts = row["ts_sec"]

        if ts <= busy_until:
            continue
        if not row["in_window"]:
            continue

        z = row.get("pred_cs_z", np.nan)
        if pd.isna(z) or pd.isna(row.get("ret_60s", np.nan)):
            continue

        z = float(z)

        if z >= long_z:
            direction = 1
        elif not long_only and z <= -short_z:
            direction = -1
        else:
            continue

        state      = int(row["state"])
        size       = float(row["size_state"])
        stop       = float(row["stop_bps"])
        actual_ret = direction * float(row["ret_60s"])

        stopped = actual_ret < -stop
        pnl_raw = -stop if stopped else actual_ret
        net_pnl = (pnl_raw - COST_BPS) * size

        trades.append({
            "ts_sec":      ts,
            "pred_z":      z,
            "actual_ret":  float(row["ret_60s"]),
            "direction":   direction,
            "state":       state,
            "size":        size,
            "stop_bps":    stop,
            "stopped_out": stopped,
            "net_pnl_bps": net_pnl,
            "long_only":   long_only,
        })

        busy_until = ts + HOLD_SEC

    return trades


def compute_metrics(pnl_series: pd.Series, label: str) -> dict:
    pnl = pnl_series.dropna().values
    n   = len(pnl)
    if n == 0:
        return {"n_trades": 0, "sharpe": 0, "hit_rate": 0,
                "total_pnl_bps": 0, "mean_pnl_bps": 0,
                "std_pnl_bps": 0, "max_drawdown_bps": 0}
    total  = float(pnl.sum())
    mean   = float(pnl.mean())
    std    = float(pnl.std()) if pnl.std() > 0 else 1e-9
    sharpe = mean / std * np.sqrt(n)
    hit    = float(np.mean(pnl > 0))
    cum    = np.cumsum(pnl)
    max_dd = float((cum - np.maximum.accumulate(cum)).min())
    log.info("%s | n=%d total=%.2f sharpe=%.3f hit=%.3f",
             label, n, total, sharpe, hit)
    return {
        "total_pnl_bps":    round(total,  2),
        "mean_pnl_bps":     round(mean,   5),
        "std_pnl_bps":      round(std,    5),
        "sharpe":           round(sharpe, 3),
        "hit_rate":         round(hit,    4),
        "max_drawdown_bps": round(max_dd, 2),
        "n_trades":         n,
    }


# ---------------------------------------------------------------------------
# Step 10 — Threshold sweep (long+short AND long-only)
# ---------------------------------------------------------------------------

log.info("=== Step 10: Threshold sweep ===")

test_days = df_eval[df_eval["date"].isin(test_dates)].copy()

sweep_rows = []
all_trades = []

for long_only in [False, True]:
    for scale in THRESHOLD_SCALES:
        lz = CS_LONG_THRESH  * scale
        sz = CS_SHORT_THRESH * scale

        day_trades = []
        for day, grp in test_days.groupby("date"):
            t = simulate(grp, long_z=lz, short_z=sz, long_only=long_only)
            for tr in t:
                tr["date"]      = str(day)
                tr["scale"]     = scale
                tr["long_only"] = long_only
            day_trades.extend(t)

        if not day_trades:
            continue

        tdf     = pd.DataFrame(day_trades)
        n_long  = int((tdf["direction"] == 1).sum())
        n_short = int((tdf["direction"] == -1).sum())
        m = compute_metrics(tdf["net_pnl_bps"],
                            f"{'LO' if long_only else 'LS'} scale={scale:.2f}")
        m["scale"]     = scale
        m["long_only"] = long_only
        m["n_long"]    = n_long
        m["n_short"]   = n_short
        m["pct_long"]  = round(100 * n_long / max(len(tdf), 1), 1)
        sweep_rows.append(m)
        all_trades.extend(day_trades)

sweep_df = pd.DataFrame(sweep_rows)

# Best configuration
best_sharpe = -999
best_cfg    = {"scale": 1.0, "long_only": False}
for _, row in sweep_df.iterrows():
    if row["sharpe"] > best_sharpe and row["n_trades"] >= 3:
        best_sharpe = row["sharpe"]
        best_cfg    = {"scale": row["scale"], "long_only": bool(row["long_only"])}

log.info("Best config: %s (Sharpe=%.3f)", best_cfg, best_sharpe)

# ---------------------------------------------------------------------------
# Step 11 — Per-state breakdown at best config
# ---------------------------------------------------------------------------

log.info("=== Step 11: Per-state breakdown ===")

best_trades = [
    t for t in all_trades
    if t["scale"] == best_cfg["scale"] and t["long_only"] == best_cfg["long_only"]
]
state_rows = []

if best_trades:
    bt_df = pd.DataFrame(best_trades)
    for state in [1, 2, 3, 4]:
        s = bt_df[bt_df["state"] == state]
        if len(s) < 3:
            continue
        m = compute_metrics(s["net_pnl_bps"], f"state_{state}")
        m["state"]      = state
        m["pct_trades"] = round(len(s) / len(bt_df) * 100, 1)
        m["size_mult"]  = STATE_SIZE[state]
        m["n_long"]     = int((s["direction"] == 1).sum())
        m["n_short"]    = int((s["direction"] == -1).sum())
        m["pct_long"]   = round(100 * m["n_long"] / max(len(s), 1), 1)
        state_rows.append(m)

state_metrics = pd.DataFrame(state_rows)

# ---------------------------------------------------------------------------
# Step 12 — Daily PnL
# ---------------------------------------------------------------------------

daily_rows = []
if best_trades:
    bt_df = pd.DataFrame(best_trades)
    for day, grp in bt_df.groupby("date"):
        pnl = grp["net_pnl_bps"]
        daily_rows.append({
            "date":     day,
            "n_trades": len(pnl),
            "n_long":   int((grp["direction"] == 1).sum()),
            "n_short":  int((grp["direction"] == -1).sum()),
            "total_pnl":round(float(pnl.sum()), 2),
            "hit_rate": round(float((pnl > 0).mean()), 3),
            "mean_pnl": round(float(pnl.mean()), 5),
        })

daily_pnl = pd.DataFrame(daily_rows)

# ---------------------------------------------------------------------------
# Step 13 — Save
# ---------------------------------------------------------------------------

log.info("=== Step 13: Saving ===")

with open(OUT_DIR / "metrics.json", "w") as f:
    json.dump({
        "split_dates": {
            "train": [str(d) for d in train_dates],
            "val":   [str(d) for d in val_dates],
            "test":  [str(d) for d in test_dates],
        },
        "config": {
            "target":          "ret_60s_norm",
            "demeaning":       "rolling 60-bar mean subtracted, normalized by rolling std",
            "cs_long_thresh":  CS_LONG_THRESH,
            "cs_short_thresh": CS_SHORT_THRESH,
            "cost_bps":        COST_BPS,
            "hold_sec":        HOLD_SEC,
            "time_window_ist": "10:30-14:30",
            "state_size":      STATE_SIZE,
            "state_stop":      STATE_STOP_BPS,
        },
        "ic": {
            "raw_val": round(ic_val_raw, 4), "raw_test": round(ic_test_raw, 4),
            "cs_val":  round(ic_val_cs,  4), "cs_test":  round(ic_test_cs,  4),
        },
        "best_config": best_cfg,
        "best_sharpe": round(best_sharpe, 3),
    }, f, indent=2)

sweep_df.to_csv(OUT_DIR / "threshold_sweep.csv", index=False)

if not state_metrics.empty:
    state_metrics.to_csv(OUT_DIR / "state_metrics.csv", index=False)
if not daily_pnl.empty:
    daily_pnl.to_csv(OUT_DIR / "daily_pnl.csv", index=False)
if all_trades:
    pd.DataFrame(all_trades).to_csv(OUT_DIR / "trade_log.csv", index=False)

# ---------------------------------------------------------------------------
# Step 14 — Print summary
# ---------------------------------------------------------------------------

print("\n" + "=" * 72)
print("STRATEGY BACKTEST v5 — CROSS-SECTIONAL DEMEANING")
print("=" * 72)
print(f"\nIC raw    — val: {ic_val_raw:.4f}  test: {ic_test_raw:.4f}")
print(f"IC CS-z   — val: {ic_val_cs:.4f}  test: {ic_test_cs:.4f}")
print(f"CS pred bias — mean: {cs_mean:.4f}  pct_neg: {cs_pct_neg:.1f}%  (want ~0, ~50%)")
print(f"Cost: {COST_BPS} bps  |  Hold: {HOLD_SEC}s  |  Window: 10:30-14:30 IST\n")

print("--- Threshold sweep: LONG+SHORT ---")
ls = sweep_df[~sweep_df["long_only"]]
if not ls.empty:
    print(ls[[
        "scale", "n_trades", "n_long", "n_short", "pct_long",
        "hit_rate", "mean_pnl_bps", "total_pnl_bps", "sharpe", "max_drawdown_bps"
    ]].to_string(index=False))

print("\n--- Threshold sweep: LONG-ONLY ---")
lo = sweep_df[sweep_df["long_only"]]
if not lo.empty:
    print(lo[[
        "scale", "n_trades", "n_long", "n_short", "pct_long",
        "hit_rate", "mean_pnl_bps", "total_pnl_bps", "sharpe", "max_drawdown_bps"
    ]].to_string(index=False))

if not state_metrics.empty:
    print(f"\n--- Per-State Breakdown at best config {best_cfg} ---")
    print(state_metrics[[
        "state", "pct_trades", "size_mult", "n_trades", "n_long", "n_short",
        "pct_long", "total_pnl_bps", "sharpe", "hit_rate", "max_drawdown_bps"
    ]].to_string(index=False))

if not daily_pnl.empty:
    print(f"\n--- Daily PnL at best config ---")
    print(daily_pnl.to_string(index=False))

print("=" * 72)

00:36:15  INFO  === Step 1: Loading raw data ===


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

... (13 outputs left out)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

00:38:10  INFO  Raw: 472806 rows
00:38:10  INFO  === Step 2: Building features ===
00:38:14  INFO  preprocess: 881382 rows after regularization
00:38:14  INFO  calculate_features: building CORE31 ...
00:38:38  INFO  build_core31_from_regularized: built 881382 rows, 31 CORE31 features
00:38:39  INFO  calculate_features: building book geometry (ADDED6 + CONTROLS9) ...
00:38:45  INFO  _build_book_geometry_features: built 881382 rows
00:38:45  INFO  calculate_features: building shock recovery (CHALLENGERS 1-2) ...
00:38:47  INFO  _build_shock_recovery_features: bid_shocks=22175 ask_shocks=42052
00:38:48  INFO  _build_shock_recovery_features: built 881382 rows
00:38:48  INFO  calculate_features: building update clustering (CHALLENGER 3) ...
00:38:49  INFO  _build_update_clustering_features: obs=472806 book_change=326695 bid=149451 ask=233683
00:38:51  INFO  _build_update_clustering_features: built 881382 rows
00:38:51  INFO  calculate_features: building liquidity migration (CHALLENGER 4) ..


STRATEGY BACKTEST v5 — CROSS-SECTIONAL DEMEANING

IC raw    — val: 0.2565  test: 0.2353
IC CS-z   — val: 0.1836  test: 0.1883
CS pred bias — mean: -0.0009  pct_neg: 48.7%  (want ~0, ~50%)
Cost: 1.0 bps  |  Hold: 60s  |  Window: 10:30-14:30 IST

--- Threshold sweep: LONG+SHORT ---
 scale  n_trades  n_long  n_short  pct_long  hit_rate  mean_pnl_bps  total_pnl_bps  sharpe  max_drawdown_bps
  0.30      2058    1079      979      52.4    0.1531      -0.55063       -1133.20 -40.087          -1132.64
  0.50      2049    1071      978      52.3    0.1635      -0.52742       -1080.67 -37.864          -1080.12
  0.75      2026    1064      962      52.5    0.1673      -0.51562       -1044.64 -36.151          -1044.09
  1.00      1994    1050      944      52.7    0.1655      -0.51202       -1020.96 -36.130          -1020.41
  1.25      1958    1043      915      53.3    0.1721      -0.49174        -962.82 -33.476           -962.27
  1.50      1926    1011      915      52.5    0.1791      -0.48

In [11]:
"""
Signal Diagnostic — find the sign inversion bug
================================================
IC=0.23 but hit rate=17% is impossible unless:
  (a) We are trading ret_60s but predicting something signed differently
  (b) The target is built with the wrong shift direction
  (c) The prediction sign is inverted somewhere

This script checks each step explicitly.
"""

import sys
sys.path.insert(0, r"C:\projects\hedge_fund")

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import lightgbm as lgb
from scipy.stats import spearmanr

from data.store.duckdb_client import DuckDBClient
from data.store.s3_client import S3Client
from data.processing.usdinr.features import USDINRFeatureBuilder
from data.processing.usdinr.constants import CHALLENGER50
from data.processing.usdinr.regimes import add_soft_hmm_features

# ---- Load a few days of data ----
db = DuckDBClient()
s  = S3Client()

all_frames = []
for contract in ["26SEPFUT", "26OCTFUT", "26NOVFUT"]:
    files = s.list_files(f"processed/1s/USDINR{contract}/")
    if not files:
        continue
    for f in files:
        parts = f.split("/")
        sym, date = parts[2], parts[3].replace(".parquet", "")
        try:
            df_day = db.read_parquet(f"processed/1s/{sym}/{date}.parquet")
            df_day["symbol"] = sym
            all_frames.append(df_day)
        except:
            pass
db.close()

df_raw = pd.concat(all_frames, ignore_index=True)
builder = USDINRFeatureBuilder(data=df_raw, auto_build=True)
df = builder.dataset.copy()
df = df.sort_values(["symbol", "date", "ts_sec"]).reset_index(drop=True)

all_dates  = sorted(df["date"].unique())
train_dates = all_dates[:15]
val_dates   = all_dates[15:20]
test_dates  = all_dates[20:]
train_end   = str(train_dates[-1])

df, hmm = add_soft_hmm_features(
    df=df, train_end_date=train_end, K=4, restarts=3, max_iter=70, tol=1e-6, seed=42
)

if "mid" not in df.columns:
    df["mid"] = (df["bid_p1"] + df["ask_p1"]) / 2.0
df["log_mid"] = np.log(df["mid"])

g = df.groupby(["symbol", "date"], sort=False)

# ============================================================
# CHECK 1: Is shift(-60) the right direction for ret_60s?
# ret_60s should be: future price relative to now
# shift(-60) means: take the value 60 rows ahead
# If ts_sec is strictly increasing by 1s, shift(-60) = price 60s later ✓
# ============================================================

df["ret_60s"]     = (g["log_mid"].shift(-60) - df["log_mid"]) * 1e4
df["ret_60s_fwd"] = (g["log_mid"].shift(-1)  - df["log_mid"]) * 1e4   # 1s forward as sanity check

sample = df[df["date"] == all_dates[5]].iloc[100:110][["ts_sec","mid","ret_60s","ret_60s_fwd"]]
print("CHECK 1: ret_60s sanity (should be ~price_60s_ahead - price_now in bps)")
print(sample.to_string())
print()

# Verify sign: on average ret_60s should have roughly 50% positive days
daily_mean_ret = df.groupby("date")["ret_60s"].mean()
print("CHECK 1b: Daily mean ret_60s (mix of + and - expected):")
print(daily_mean_ret.to_string())
print()

# ============================================================
# CHECK 2: Is ret_60s_norm preserving the sign of ret_60s?
# After normalization, sign should be preserved for above-mean bars
# ============================================================

df["ret_60s_norm"] = df.groupby(["symbol", "date"])["ret_60s"].transform(
    lambda x: (x - x.mean()) / (x.std() + 1e-8)
)

# sign agreement: should be near 100% if daily_mean is small vs spread
sign_agree = np.mean(np.sign(df["ret_60s"].dropna()) == np.sign(df["ret_60s_norm"].dropna()))
print(f"CHECK 2: Sign agreement ret_60s vs ret_60s_norm: {sign_agree*100:.1f}%")
print(f"  (100% = perfect, 50% = random, <80% = problem)")
print(f"  Mean of ret_60s: {df['ret_60s'].mean():.5f} bps  (large value = drift dominating sign)")
print()

# ============================================================
# CHECK 3: Train a tiny LightGBM and check prediction vs actual sign
# ============================================================

features = CHALLENGER50
target   = "ret_60s_norm"
valid    = df[features + [target, "ret_60s"]].notna().all(axis=1)

tr = df[df["date"].isin(train_dates) & valid]
va = df[df["date"].isin(val_dates)   & valid]
te = df[df["date"].isin(test_dates)  & valid]

X_tr, y_tr = tr[features].values, tr[target].values
X_va, y_va = va[features].values, va[target].values
X_te, y_te = te[features].values, te[target].values

ds_tr = lgb.Dataset(X_tr, label=y_tr, feature_name=features)
ds_va = lgb.Dataset(X_va, label=y_va, feature_name=features, reference=ds_tr)

model = lgb.train(
    {"objective":"regression","metric":"rmse","verbosity":-1,
     "num_leaves":31,"learning_rate":0.05,"feature_fraction":0.8,
     "bagging_fraction":0.8,"bagging_freq":5,"min_child_samples":50},
    ds_tr, num_boost_round=500,
    valid_sets=[ds_va],
    callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(500)],
)

pred_va = model.predict(X_va)
pred_te = model.predict(X_te)

# IC against normalized target (what model was trained on)
ic_vs_norm_va, _ = spearmanr(y_va, pred_va)
ic_vs_norm_te, _ = spearmanr(y_te, pred_te)

# IC against RAW ret_60s (what determines profit)
ic_vs_raw_va, _  = spearmanr(va["ret_60s"].values, pred_va)
ic_vs_raw_te, _  = spearmanr(te["ret_60s"].values, pred_te)

print("CHECK 3: IC of predictions")
print(f"  vs ret_60s_norm (trained target) — val: {ic_vs_norm_va:.4f}  test: {ic_vs_norm_te:.4f}")
print(f"  vs ret_60s_raw  (trading target) — val: {ic_vs_raw_va:.4f}   test: {ic_vs_raw_te:.4f}")
print()

# ============================================================
# CHECK 4: Hit rate if we simply follow the sign of pred
# (no threshold, no cost — just: does sign(pred) == sign(raw_ret)?)
# ============================================================

# Against normalized target
hr_norm_va = np.mean(np.sign(pred_va) == np.sign(y_va))
hr_norm_te = np.mean(np.sign(pred_te) == np.sign(y_te))

# Against raw ret_60s (what we actually trade)
hr_raw_va = np.mean(np.sign(pred_va) == np.sign(va["ret_60s"].values))
hr_raw_te = np.mean(np.sign(pred_te) == np.sign(te["ret_60s"].values))

print("CHECK 4: Pure sign hit rate (no threshold, no cost)")
print(f"  pred sign vs norm sign — val: {hr_norm_va*100:.1f}%  test: {hr_norm_te*100:.1f}%")
print(f"  pred sign vs raw sign  — val: {hr_raw_va*100:.1f}%   test: {hr_raw_te*100:.1f}%")
print(f"  (50% = coin flip, >55% = usable signal, <50% = sign is INVERTED)")
print()

# ============================================================
# CHECK 5: What does the prediction distribution look like?
# If pct_neg >> 50%, the model is biased toward negative
# ============================================================

print("CHECK 5: Prediction distribution")
for name, preds in [("val", pred_va), ("test", pred_te)]:
    print(f"  {name}: mean={np.mean(preds):.4f}  pct_neg={np.mean(preds<0)*100:.1f}%  "
          f"p5={np.percentile(preds,5):.3f}  p50={np.percentile(preds,50):.3f}  "
          f"p95={np.percentile(preds,95):.3f}")
print()

# ============================================================
# CHECK 6: What is the ACTUAL trade hit rate at a threshold?
# Simulate exactly: long if pred > 0.5, check if ret_60s > 0
# ============================================================

thresh = 0.5
mask_long  = pred_te >  thresh
mask_short = pred_te < -thresh

ret_te = te["ret_60s"].values

n_long  = mask_long.sum()
n_short = mask_short.sum()

if n_long > 0:
    hr_long = np.mean(ret_te[mask_long] > 0)
else:
    hr_long = float("nan")

if n_short > 0:
    hr_short = np.mean(ret_te[mask_short] < 0)   # short profits when ret < 0
else:
    hr_short = float("nan")

print(f"CHECK 6: Trade hit rate at threshold ±{thresh} (test set, no cost, no time filter)")
print(f"  Longs:  n={n_long}  hit={hr_long*100:.1f}%  (want >50%)")
print(f"  Shorts: n={n_short}  hit={hr_short*100:.1f}%  (want >50%)")
print()

# ============================================================
# CHECK 7: Lag check — is there an off-by-one in mid_exit?
# ============================================================

df["mid_exit_60"] = g["mid"].shift(-60)
df["mid_exit_1"]  = g["mid"].shift(-1)

# ret_60s computed from log_mid shift(-60) should match mid_exit_60 price difference
sample2 = df[df["date"] == all_dates[10]].iloc[200:205].copy()
sample2["manual_ret"] = (np.log(sample2["mid_exit_60"]) - sample2["log_mid"]) * 1e4
sample2["stored_ret"] = sample2["ret_60s"]
print("CHECK 7: ret_60s consistency (manual_ret should == stored_ret)")
print(sample2[["ts_sec","mid","mid_exit_60","manual_ret","stored_ret"]].to_string())
print()

print("=" * 60)
print("SUMMARY: Look at CHECK 4 hit rates.")
print("  >55% = signal is real, strategy bug is in simulator logic")
print("   ~50% = no signal (IC is misleading)")
print("  <50% = sign inversion somewhere in the pipeline")
print("=" * 60)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

... (13 outputs left out)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

00:48:06  INFO  preprocess: 881382 rows after regularization
00:48:07  INFO  calculate_features: building CORE31 ...
00:48:30  INFO  build_core31_from_regularized: built 881382 rows, 31 CORE31 features
00:48:31  INFO  calculate_features: building book geometry (ADDED6 + CONTROLS9) ...
00:48:37  INFO  _build_book_geometry_features: built 881382 rows
00:48:37  INFO  calculate_features: building shock recovery (CHALLENGERS 1-2) ...
00:48:39  INFO  _build_shock_recovery_features: bid_shocks=22175 ask_shocks=42052
00:48:40  INFO  _build_shock_recovery_features: built 881382 rows
00:48:40  INFO  calculate_features: building update clustering (CHALLENGER 3) ...
00:48:42  INFO  _build_update_clustering_features: obs=472806 book_change=326695 bid=149451 ask=233683
00:48:43  INFO  _build_update_clustering_features: built 881382 rows
00:48:43  INFO  calculate_features: building liquidity migration (CHALLENGER 4) ...
00:48:44  INFO  _build_liquidity_migration_features: built 881382 rows
00:48:52  

CHECK 1: ret_60s sanity (should be ~price_60s_ahead - price_now in bps)
            ts_sec      mid   ret_60s  ret_60s_fwd
229443  1790739100  96.2275  1.688564     0.000000
229444  1790739101  96.2275  1.688564     0.000000
229445  1790739102  96.2275  1.688564     0.000000
229446  1790739103  96.2275  1.688564     0.519588
229447  1790739104  96.2325  1.168975     0.000000
229448  1790739105  96.2325  0.779332     0.000000
229449  1790739106  96.2325  0.779332     0.000000
229450  1790739107  96.2325  0.779332     0.000000
229451  1790739108  96.2325  0.779332     0.000000
229452  1790739109  96.2325  0.779332     0.000000

CHECK 1b: Daily mean ret_60s (mix of + and - expected):
date
2026-09-22   -0.038214
2026-09-23    0.026697
2026-09-24    0.021587
2026-09-25   -0.035232
2026-09-28    0.010314
2026-09-30   -0.026809
2026-10-01    0.068090
2026-10-05    0.020103
2026-10-06    0.006009
2026-10-07    0.090140
2026-10-08    0.031389
2026-10-09   -0.003644
2026-10-12   -0.024464
2026-1

In [12]:
"""
USDINR Strategy Backtest v6
=============================
Root cause found in signal_diagnostic.py:

  CHECK 4:
    pred sign vs ret_60s_norm sign = 59%  ← model is correct
    pred sign vs ret_60s_raw  sign = 49%  ← looks random, but...

  CHECK 6 (the truth):
    Longs  hit = 70.3%
    Shorts hit = 57.8%

The model is fine. The bug was: we trained on ret_60s_norm but traded
against ret_60s_raw. ret_60s_norm and ret_60s_raw only agree on sign
75.8% of the time (CHECK 2). On uptrend days, below-average bars have
norm < 0 but raw > 0 — normalization flips their sign.

THE FIX:
  At inference, de-normalize the prediction back to raw bps:
    pred_raw_bps = pred_norm * daily_std + daily_mean
  Then threshold and trade on pred_raw_bps.

  This preserves the cross-sectional ranking power of the trained model
  while anchoring direction to the actual market level, not day-relative.

We also run a MAGNITUDE-ONLY entry mode:
  - Only enter when |pred_raw_bps| > threshold (high-conviction bars)
  - Direction = sign(pred_raw_bps) — the de-normalized direction
  This is what CHECK 6 showed works at 70% hit rate for longs.

Outputs: results/strategy_backtest_v6/
"""

import sys
sys.path.insert(0, r"C:\projects\hedge_fund")

import json, logging, pathlib, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import lightgbm as lgb
from scipy.stats import spearmanr

from data.store.duckdb_client import DuckDBClient
from data.store.s3_client import S3Client
from data.processing.usdinr.features import USDINRFeatureBuilder
from data.processing.usdinr.constants import CHALLENGER50
from data.processing.usdinr.regimes import add_soft_hmm_features, save_hmm_model

logging.basicConfig(level=logging.INFO,
                    format="%(asctime)s  %(levelname)s  %(message)s",
                    datefmt="%H:%M:%S")
log = logging.getLogger(__name__)

OUT_DIR   = pathlib.Path(r"C:\projects\hedge_fund\results\strategy_backtest_v6")
MODEL_DIR = pathlib.Path(r"C:\projects\hedge_fund\models")
OUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_DAYS = 15
VAL_DAYS   = 5
COST_BPS   = 1.0
HOLD_SEC   = 60

# Thresholds are now in RAW BPS (after de-normalization)
# From CHECK 6: threshold ±0.5σ_norm fired 2638 longs (70% hit) and 7270 shorts (58% hit)
# We want fewer, higher-quality trades — start at 1.5 bps
LONG_THRESH_BPS  = 1.5
SHORT_THRESH_BPS = 2.0   # keep asymmetric — shorts still weaker
THRESHOLD_SCALES = [0.5, 0.75, 1.0, 1.25, 1.5, 2.0, 3.0]

IST_OFFSET   = 19800
OPEN_FILTER  = 10 * 3600 + 30 * 60
CLOSE_FILTER = 14 * 3600 + 30 * 60

LGB_PARAMS = {
    "objective":         "regression",
    "metric":            "rmse",
    "verbosity":         -1,
    "boosting_type":     "gbdt",
    "num_leaves":        31,
    "learning_rate":     0.05,
    "feature_fraction":  0.8,
    "bagging_fraction":  0.8,
    "bagging_freq":      5,
    "min_child_samples": 50,
}

STATE_SIZE     = {1: 0.59, 2: 0.44, 3: 0.65, 4: 1.00}
STATE_STOP_BPS = {1: 2.34, 2: 3.08, 3: 2.04, 4: 1.33}

# ---------------------------------------------------------------------------
# Load + features
# ---------------------------------------------------------------------------
log.info("Loading data...")
db = DuckDBClient(); s = S3Client(); all_frames = []
for contract in ["26SEPFUT", "26OCTFUT", "26NOVFUT"]:
    files = s.list_files(f"processed/1s/USDINR{contract}/")
    if not files: continue
    for f in files:
        parts = f.split("/")
        sym, date = parts[2], parts[3].replace(".parquet","")
        try:
            d = db.read_parquet(f"processed/1s/{sym}/{date}.parquet")
            d["symbol"] = sym; all_frames.append(d)
        except: pass
db.close()

df_raw  = pd.concat(all_frames, ignore_index=True)
builder = USDINRFeatureBuilder(data=df_raw, auto_build=True)
df      = builder.dataset.copy()
df["tick_count"] = df.get("tick_count", pd.Series(0, index=df.index)).fillna(0).astype(int)
df = df.sort_values(["symbol","date","ts_sec"]).reset_index(drop=True)
log.info("Dataset: %d rows", len(df))

# ---------------------------------------------------------------------------
# Split
# ---------------------------------------------------------------------------
all_dates   = sorted(df["date"].unique())
train_dates = all_dates[:TRAIN_DAYS]
val_dates   = all_dates[TRAIN_DAYS : TRAIN_DAYS + VAL_DAYS]
test_dates  = all_dates[TRAIN_DAYS + VAL_DAYS:]
train_end   = str(train_dates[-1])
log.info("Train %d | Val %d | Test %d days", len(train_dates), len(val_dates), len(test_dates))

mask_train = df["date"].isin(train_dates)
mask_val   = df["date"].isin(val_dates)
mask_test  = df["date"].isin(test_dates)

# ---------------------------------------------------------------------------
# HMM
# ---------------------------------------------------------------------------
df, hmm = add_soft_hmm_features(df=df, train_end_date=train_end,
                                 K=4, restarts=3, max_iter=70, tol=1e-6, seed=42)
save_hmm_model(hmm, MODEL_DIR / "hmm_v6.pkl")

# ---------------------------------------------------------------------------
# Targets
# ---------------------------------------------------------------------------
if "mid" not in df.columns:
    df["mid"] = (df["bid_p1"] + df["ask_p1"]) / 2.0
df["log_mid"] = np.log(df["mid"])
g = df.groupby(["symbol","date"], sort=False)
df["ret_60s"]  = (g["log_mid"].shift(-60) - df["log_mid"]) * 1e4
df["mid_exit"] = g["mid"].shift(-HOLD_SEC)

# Normalized target for training
df["ret_60s_norm"] = df.groupby(["symbol","date"])["ret_60s"].transform(
    lambda x: (x - x.mean()) / (x.std() + 1e-8)
)

# Save daily mean and std — needed to de-normalize predictions at inference
daily_stats = (
    df.groupby(["symbol","date"])["ret_60s"]
    .agg(daily_mean="mean", daily_std="std")
    .reset_index()
)
daily_stats["daily_std"] = daily_stats["daily_std"].fillna(1.0).clip(lower=1e-6)
df = df.merge(daily_stats, on=["symbol","date"], how="left")

log.info("Sign agree norm vs raw: %.1f%%",
         np.mean(np.sign(df["ret_60s_norm"].dropna()) ==
                 np.sign(df["ret_60s"].dropna())) * 100)

df["ist_tod"]   = (df["ts_sec"] + IST_OFFSET) % 86400
df["in_window"] = (df["ist_tod"] >= OPEN_FILTER) & (df["ist_tod"] <= CLOSE_FILTER)

# ---------------------------------------------------------------------------
# Train LightGBM on normalized target
# ---------------------------------------------------------------------------
features   = CHALLENGER50
target     = "ret_60s_norm"
valid_mask = df[features + [target,"ret_60s","daily_mean","daily_std"]].notna().all(axis=1)

tr = df[mask_train & valid_mask]
va = df[mask_val   & valid_mask]

ds_tr = lgb.Dataset(tr[features].values, label=tr[target].values, feature_name=features)
ds_va = lgb.Dataset(va[features].values, label=va[target].values,
                    feature_name=features, reference=ds_tr)

model = lgb.train(
    LGB_PARAMS, ds_tr, num_boost_round=1000,
    valid_sets=[ds_va],
    callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(100)],
)
log.info("Best iter: %d", model.best_iteration)
model.save_model(str(MODEL_DIR / "lgbm_v6.txt"))

# ---------------------------------------------------------------------------
# Generate predictions + DE-NORMALIZE to raw bps
# ---------------------------------------------------------------------------
eval_mask = (mask_val | mask_test) & valid_mask & df["mid_exit"].notna()
df_eval   = df[eval_mask].copy().reset_index(drop=True)
df_eval["state"] = df_eval["state_filtered"].astype(int)

# Model output: normalized z-score
df_eval["pred_norm"] = model.predict(df_eval[features].values)

# De-normalize: multiply by daily std, add daily mean → back to raw bps
# This is the KEY FIX: pred_raw_bps is in the same units as ret_60s
df_eval["pred_bps"] = (
    df_eval["pred_norm"] * df_eval["daily_std"] + df_eval["daily_mean"]
)

# Verify the fix
pct_neg_norm = (df_eval["pred_norm"] < 0).mean() * 100
pct_neg_bps  = (df_eval["pred_bps"]  < 0).mean() * 100

# Sign hit rate (should now be >55%)
te_mask = df_eval["date"].isin(test_dates)
hr_long_norm  = np.mean(np.sign(df_eval.loc[te_mask,"pred_norm"]) ==
                         np.sign(df_eval.loc[te_mask,"ret_60s"]))
hr_long_bps   = np.mean(np.sign(df_eval.loc[te_mask,"pred_bps"]) ==
                         np.sign(df_eval.loc[te_mask,"ret_60s"]))

log.info("pred_norm: pct_neg=%.1f%%  sign_hit_vs_raw=%.1f%%",
         pct_neg_norm, hr_long_norm * 100)
log.info("pred_bps:  pct_neg=%.1f%%  sign_hit_vs_raw=%.1f%%  (want >55%%)",
         pct_neg_bps, hr_long_bps * 100)

# IC (both flavors)
ic_norm_v, _ = spearmanr(df_eval.loc[df_eval["date"].isin(val_dates),"ret_60s"],
                          df_eval.loc[df_eval["date"].isin(val_dates),"pred_norm"])
ic_norm_t, _ = spearmanr(df_eval.loc[te_mask,"ret_60s"],
                          df_eval.loc[te_mask,"pred_norm"])
ic_bps_v,  _ = spearmanr(df_eval.loc[df_eval["date"].isin(val_dates),"ret_60s"],
                          df_eval.loc[df_eval["date"].isin(val_dates),"pred_bps"])
ic_bps_t,  _ = spearmanr(df_eval.loc[te_mask,"ret_60s"],
                          df_eval.loc[te_mask,"pred_bps"])

log.info("IC pred_norm — val:%.4f test:%.4f", ic_norm_v, ic_norm_t)
log.info("IC pred_bps  — val:%.4f test:%.4f", ic_bps_v, ic_bps_t)

df_eval["size_state"] = df_eval["state"].map(STATE_SIZE)
df_eval["stop_bps"]   = df_eval["state"].map(STATE_STOP_BPS)

# ---------------------------------------------------------------------------
# Simulate
# ---------------------------------------------------------------------------

def simulate(df_day: pd.DataFrame, long_bps: float, short_bps: float) -> list:
    """
    Use de-normalized pred_bps for entry decisions:
      long  if pred_bps > +long_bps
      short if pred_bps < -short_bps
    Actual PnL is computed against raw ret_60s — both now in same units.
    """
    trades = []; busy_until = -1
    rows = df_day.sort_values("ts_sec").reset_index(drop=True)

    for _, row in rows.iterrows():
        ts = row["ts_sec"]
        if ts <= busy_until: continue
        if not row["in_window"]: continue

        pred = row.get("pred_bps", np.nan)
        if pd.isna(pred) or pd.isna(row.get("ret_60s", np.nan)): continue

        pred = float(pred)
        if   pred >=  long_bps:  direction =  1
        elif pred <= -short_bps: direction = -1
        else: continue

        state      = int(row["state"])
        size       = float(row["size_state"])
        stop       = float(row["stop_bps"])
        actual_ret = direction * float(row["ret_60s"])
        stopped    = actual_ret < -stop
        pnl_raw    = -stop if stopped else actual_ret
        net_pnl    = (pnl_raw - COST_BPS) * size

        trades.append({
            "ts_sec":      ts,
            "pred_bps":    pred,
            "actual_ret":  float(row["ret_60s"]),
            "direction":   direction,
            "state":       state,
            "size":        size,
            "stop_bps":    stop,
            "stopped_out": stopped,
            "net_pnl_bps": net_pnl,
        })
        busy_until = ts + HOLD_SEC

    return trades


def metrics(pnl_series: pd.Series, label: str) -> dict:
    pnl = pnl_series.dropna().values; n = len(pnl)
    if n == 0:
        return {"n_trades":0,"sharpe":0,"hit_rate":0,
                "total_pnl_bps":0,"mean_pnl_bps":0,"std_pnl_bps":0,"max_drawdown_bps":0}
    total=float(pnl.sum()); mean=float(pnl.mean())
    std=float(pnl.std()) if pnl.std()>0 else 1e-9
    sharpe=mean/std*np.sqrt(n); hit=float(np.mean(pnl>0))
    cum=np.cumsum(pnl); max_dd=float((cum-np.maximum.accumulate(cum)).min())
    log.info("%s | n=%d total=%.2f sharpe=%.3f hit=%.3f", label, n, total, sharpe, hit)
    return {"total_pnl_bps":round(total,2),"mean_pnl_bps":round(mean,5),
            "std_pnl_bps":round(std,5),"sharpe":round(sharpe,3),
            "hit_rate":round(hit,4),"max_drawdown_bps":round(max_dd,2),"n_trades":n}


# ---------------------------------------------------------------------------
# Threshold sweep
# ---------------------------------------------------------------------------
log.info("Sweeping thresholds...")
test_days  = df_eval[df_eval["date"].isin(test_dates)].copy()
sweep_rows = []; all_trades = []

for scale in THRESHOLD_SCALES:
    lt = LONG_THRESH_BPS  * scale
    st = SHORT_THRESH_BPS * scale
    day_trades = []
    for day, grp in test_days.groupby("date"):
        t = simulate(grp, lt, st)
        for tr in t:
            tr["date"] = str(day); tr["scale"] = scale
        day_trades.extend(t)
    if not day_trades:
        log.info("scale=%.2f | 0 trades"); continue
    tdf = pd.DataFrame(day_trades)
    n_long = int((tdf["direction"]==1).sum())
    n_short= int((tdf["direction"]==-1).sum())
    m = metrics(tdf["net_pnl_bps"], f"scale={scale:.2f}")
    m.update({"scale":scale,"n_long":n_long,"n_short":n_short,
               "pct_long":round(100*n_long/max(len(tdf),1),1)})
    sweep_rows.append(m); all_trades.extend(day_trades)

sweep_df = pd.DataFrame(sweep_rows)
best_scale = 1.0
if not sweep_df.empty:
    best_row   = sweep_df.loc[sweep_df["sharpe"].idxmax()]
    best_scale = float(best_row["scale"])
    log.info("Best scale: %.2f sharpe=%.3f n=%d hit=%.3f",
             best_scale, best_row["sharpe"], best_row["n_trades"], best_row["hit_rate"])

# ---------------------------------------------------------------------------
# Per-state breakdown
# ---------------------------------------------------------------------------
best_trades = [t for t in all_trades if t["scale"] == best_scale]
state_rows  = []
if best_trades:
    bt_df = pd.DataFrame(best_trades)
    for state in [1,2,3,4]:
        s = bt_df[bt_df["state"]==state]
        if len(s)<3: continue
        m = metrics(s["net_pnl_bps"], f"state_{state}")
        m.update({"state":state,"pct_trades":round(len(s)/len(bt_df)*100,1),
                  "size_mult":STATE_SIZE[state],
                  "n_long":int((s["direction"]==1).sum()),
                  "n_short":int((s["direction"]==-1).sum()),
                  "pct_long":round(100*(s["direction"]==1).mean(),1)})
        state_rows.append(m)
state_metrics = pd.DataFrame(state_rows)

# ---------------------------------------------------------------------------
# Daily PnL
# ---------------------------------------------------------------------------
daily_rows = []
if best_trades:
    bt_df = pd.DataFrame(best_trades)
    for day, grp in bt_df.groupby("date"):
        pnl = grp["net_pnl_bps"]
        daily_rows.append({"date":day,"n_trades":len(pnl),
                            "n_long":int((grp["direction"]==1).sum()),
                            "n_short":int((grp["direction"]==-1).sum()),
                            "total_pnl":round(float(pnl.sum()),2),
                            "hit_rate":round(float((pnl>0).mean()),3),
                            "mean_pnl":round(float(pnl.mean()),5)})
daily_pnl = pd.DataFrame(daily_rows)

# ---------------------------------------------------------------------------
# Save
# ---------------------------------------------------------------------------
with open(OUT_DIR/"metrics.json","w") as f:
    json.dump({"ic":{"norm_val":round(ic_norm_v,4),"norm_test":round(ic_norm_t,4),
                     "bps_val":round(ic_bps_v,4),"bps_test":round(ic_bps_t,4)},
               "sign_hit_bps_test": round(hr_long_bps*100,1),
               "best_scale":best_scale,
               "config":{"long_thresh_bps":LONG_THRESH_BPS,
                         "short_thresh_bps":SHORT_THRESH_BPS,
                         "denormalize":"pred_bps = pred_norm * daily_std + daily_mean"}},f,indent=2)

sweep_df.to_csv(OUT_DIR/"threshold_sweep.csv", index=False)
if not state_metrics.empty: state_metrics.to_csv(OUT_DIR/"state_metrics.csv",index=False)
if not daily_pnl.empty: daily_pnl.to_csv(OUT_DIR/"daily_pnl.csv",index=False)
if all_trades: pd.DataFrame(all_trades).to_csv(OUT_DIR/"trade_log.csv",index=False)

# ---------------------------------------------------------------------------
# Print
# ---------------------------------------------------------------------------
print("\n"+"="*68)
print("STRATEGY BACKTEST v6 — DE-NORMALIZED PREDICTIONS")
print("="*68)
print(f"\nKEY FIX: pred_bps = pred_norm × daily_std + daily_mean")
print(f"Sign hit rate pred_bps vs ret_60s (test): {hr_long_bps*100:.1f}%  (was 49% in v4/v5)")
print(f"IC pred_bps — val:{ic_bps_v:.4f}  test:{ic_bps_t:.4f}")
print(f"IC pred_norm— val:{ic_norm_v:.4f}  test:{ic_norm_t:.4f}")
print(f"Cost:{COST_BPS} bps | Hold:{HOLD_SEC}s | Window:10:30-14:30 IST\n")

print("--- Threshold sweep (test set) ---")
if not sweep_df.empty:
    print(sweep_df[["scale","n_trades","n_long","n_short","pct_long",
                     "hit_rate","mean_pnl_bps","total_pnl_bps","sharpe",
                     "max_drawdown_bps"]].to_string(index=False))

if not state_metrics.empty:
    print(f"\n--- Per-State at scale={best_scale:.2f} ---")
    print(state_metrics[["state","pct_trades","size_mult","n_trades",
                          "n_long","n_short","pct_long","total_pnl_bps",
                          "sharpe","hit_rate","max_drawdown_bps"]].to_string(index=False))

if not daily_pnl.empty:
    print(f"\n--- Daily PnL at scale={best_scale:.2f} ---")
    print(daily_pnl.to_string(index=False))

print("="*68)

13:32:50  INFO  Loading data...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

... (11 outputs left out)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

13:34:54  INFO  preprocess: 881382 rows after regularization
13:34:54  INFO  calculate_features: building CORE31 ...
13:35:21  INFO  build_core31_from_regularized: built 881382 rows, 31 CORE31 features
13:35:22  INFO  calculate_features: building book geometry (ADDED6 + CONTROLS9) ...
13:35:30  INFO  _build_book_geometry_features: built 881382 rows
13:35:31  INFO  calculate_features: building shock recovery (CHALLENGERS 1-2) ...
13:35:34  INFO  _build_shock_recovery_features: bid_shocks=22175 ask_shocks=42052
13:35:34  INFO  _build_shock_recovery_features: built 881382 rows
13:35:35  INFO  calculate_features: building update clustering (CHALLENGER 3) ...
13:35:37  INFO  _build_update_clustering_features: obs=472806 book_change=326695 bid=149451 ask=233683
13:35:38  INFO  _build_update_clustering_features: built 881382 rows
13:35:38  INFO  calculate_features: building liquidity migration (CHALLENGER 4) ...
13:35:40  INFO  _build_liquidity_migration_features: built 881382 rows
13:35:50  


STRATEGY BACKTEST v6 — DE-NORMALIZED PREDICTIONS

KEY FIX: pred_bps = pred_norm × daily_std + daily_mean
Sign hit rate pred_bps vs ret_60s (test): 49.5%  (was 49% in v4/v5)
IC pred_bps — val:0.2516  test:0.2392
IC pred_norm— val:0.2556  test:0.2352
Cost:1.0 bps | Hold:60s | Window:10:30-14:30 IST

--- Threshold sweep (test set) ---
 scale  n_trades  n_long  n_short  pct_long  hit_rate  mean_pnl_bps  total_pnl_bps        sharpe  max_drawdown_bps
  0.50        85      31       54      36.5    0.4706       0.00481           0.41  4.800000e-02             -7.29
  0.75        23       5       18      21.7    0.3913      -0.06287          -1.45 -3.410000e-01             -3.90
  1.00        11       0       11       0.0    0.6364       0.03736           0.41  1.520000e-01             -2.38
  1.25         7       0        7       0.0    0.4286      -0.03790          -0.27 -1.680000e-01             -1.21
  1.50         3       0        3       0.0    0.3333       0.01091           0.03  1.2900

In [13]:
"""
Threshold diagnostic — why does CHECK 6's 70% hit rate disappear in the simulator?

CHECK 6 found: at pred_norm > 0.5, longs hit 70.3% (n=2638)
v6 at scale=0.5 (pred_bps > 0.75): only 85 trades, 36% long

We need to find exactly where the signal gets killed.
This script tests the threshold directly on the eval set without any simulator logic.
"""

import sys
sys.path.insert(0, r"C:\projects\hedge_fund")
import warnings; warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import lightgbm as lgb
from scipy.stats import spearmanr

from data.store.duckdb_client import DuckDBClient
from data.store.s3_client import S3Client
from data.processing.usdinr.features import USDINRFeatureBuilder
from data.processing.usdinr.constants import CHALLENGER50
from data.processing.usdinr.regimes import add_soft_hmm_features

# ---- Load ----
db = DuckDBClient(); s = S3Client(); all_frames = []
for contract in ["26SEPFUT","26OCTFUT","26NOVFUT"]:
    files = s.list_files(f"processed/1s/USDINR{contract}/")
    if not files: continue
    for f in files:
        parts = f.split("/")
        sym, date = parts[2], parts[3].replace(".parquet","")
        try:
            d = db.read_parquet(f"processed/1s/{sym}/{date}.parquet")
            d["symbol"] = sym; all_frames.append(d)
        except: pass
db.close()

df_raw  = pd.concat(all_frames, ignore_index=True)
builder = USDINRFeatureBuilder(data=df_raw, auto_build=True)
df      = builder.dataset.copy()
df      = df.sort_values(["symbol","date","ts_sec"]).reset_index(drop=True)

all_dates   = sorted(df["date"].unique())
train_dates = all_dates[:15]
val_dates   = all_dates[15:20]
test_dates  = all_dates[20:]
train_end   = str(train_dates[-1])

df, hmm = add_soft_hmm_features(df=df, train_end_date=train_end,
                                 K=4, restarts=3, max_iter=70, tol=1e-6, seed=42)

if "mid" not in df.columns:
    df["mid"] = (df["bid_p1"] + df["ask_p1"]) / 2.0
df["log_mid"] = np.log(df["mid"])
g = df.groupby(["symbol","date"], sort=False)
df["ret_60s"]  = (g["log_mid"].shift(-60) - df["log_mid"]) * 1e4
df["mid_exit"] = g["mid"].shift(-60)   # price 60s later

df["ret_60s_norm"] = df.groupby(["symbol","date"])["ret_60s"].transform(
    lambda x: (x - x.mean()) / (x.std() + 1e-8)
)

IST_OFFSET = 19800
df["ist_tod"]   = (df["ts_sec"] + IST_OFFSET) % 86400
df["in_window"] = (df["ist_tod"] >= 10*3600+30*60) & (df["ist_tod"] <= 14*3600+30*60)

features   = CHALLENGER50
target     = "ret_60s_norm"
valid_mask = df[features + [target,"ret_60s"]].notna().all(axis=1)

tr = df[df["date"].isin(train_dates) & valid_mask]
va = df[df["date"].isin(val_dates)   & valid_mask]
te = df[df["date"].isin(test_dates)  & valid_mask]

ds_tr = lgb.Dataset(tr[features].values, label=tr[target].values, feature_name=features)
ds_va = lgb.Dataset(va[features].values, label=va[target].values,
                    feature_name=features, reference=ds_tr)

model = lgb.train(
    {"objective":"regression","metric":"rmse","verbosity":-1,
     "num_leaves":31,"learning_rate":0.05,"feature_fraction":0.8,
     "bagging_fraction":0.8,"bagging_freq":5,"min_child_samples":50},
    ds_tr, num_boost_round=1000,
    valid_sets=[ds_va],
    callbacks=[lgb.early_stopping(50,verbose=False), lgb.log_evaluation(500)],
)

pred_te = model.predict(te[features].values)
ret_te  = te["ret_60s"].values
win_te  = te["in_window"].values
mid_te  = te["mid_exit"].values

print("="*65)
print("THRESHOLD DIAGNOSTIC")
print("="*65)

# ---------------------------------------------------------------
# TEST A: Raw hit rate at various pred_norm thresholds
# No time filter, no single-position, no cost — pure signal quality
# ---------------------------------------------------------------
print("\nA) Raw directional hit rate — pred_norm threshold, NO filters")
print(f"{'thresh':>8} {'n_long':>8} {'n_short':>8} {'hr_long':>9} {'hr_short':>9} {'total_n':>8}")
for thresh in [0.1, 0.2, 0.3, 0.5, 0.7, 1.0, 1.5, 2.0]:
    mask_l = pred_te >  thresh
    mask_s = pred_te < -thresh
    n_l, n_s = mask_l.sum(), mask_s.sum()
    hr_l = np.mean(ret_te[mask_l] > 0) if n_l > 0 else 0
    hr_s = np.mean(ret_te[mask_s] < 0) if n_s > 0 else 0
    print(f"{thresh:>8.1f} {n_l:>8d} {n_s:>8d} {hr_l*100:>8.1f}% {hr_s*100:>8.1f}% {n_l+n_s:>8d}")

# ---------------------------------------------------------------
# TEST B: Same but WITH time filter (10:30-14:30 IST)
# ---------------------------------------------------------------
print("\nB) Raw hit rate — pred_norm threshold, WITH time filter")
print(f"{'thresh':>8} {'n_long':>8} {'n_short':>8} {'hr_long':>9} {'hr_short':>9} {'total_n':>8}")
for thresh in [0.1, 0.2, 0.3, 0.5, 0.7, 1.0, 1.5, 2.0]:
    mask_l = (pred_te >  thresh) & win_te
    mask_s = (pred_te < -thresh) & win_te
    n_l, n_s = mask_l.sum(), mask_s.sum()
    hr_l = np.mean(ret_te[mask_l] > 0) if n_l > 0 else 0
    hr_s = np.mean(ret_te[mask_s] < 0) if n_s > 0 else 0
    print(f"{thresh:>8.1f} {n_l:>8d} {n_s:>8d} {hr_l*100:>8.1f}% {hr_s*100:>8.1f}% {n_l+n_s:>8d}")

# ---------------------------------------------------------------
# TEST C: What is the actual PnL if we took EVERY signal above thresh?
# (1 bps cost, no position limit, no stop)
# ---------------------------------------------------------------
print("\nC) Brute-force PnL — take every signal (1bps cost, no position limit)")
print(f"{'thresh':>8} {'n_long':>8} {'n_short':>8} {'pnl_long':>10} {'pnl_short':>11} {'total_pnl':>11}")
for thresh in [0.1, 0.3, 0.5, 0.7, 1.0, 1.5]:
    mask_l = (pred_te >  thresh) & win_te
    mask_s = (pred_te < -thresh) & win_te
    n_l, n_s = mask_l.sum(), mask_s.sum()
    pnl_l = float(np.sum(ret_te[mask_l]) - n_l * 1.0) if n_l > 0 else 0
    pnl_s = float(np.sum(-ret_te[mask_s]) - n_s * 1.0) if n_s > 0 else 0
    print(f"{thresh:>8.1f} {n_l:>8d} {n_s:>8d} {pnl_l:>10.2f} {pnl_s:>11.2f} {pnl_l+pnl_s:>11.2f}")

# ---------------------------------------------------------------
# TEST D: Per-bar expected value = |pred_norm| × (hit_rate - 0.5) × 2 - cost
# This tells us which bars have positive expected value
# ---------------------------------------------------------------
print("\nD) Expected value analysis — what fraction of bars have EV > 0 after 1bps cost?")
# For longs: EV = P(ret>0) * mean_ret_if_up - P(ret<0) * mean_ret_if_down - cost
# Approximate: at each threshold bin, compute mean PnL
bins = [0, 0.3, 0.5, 0.7, 1.0, 1.5, 2.0, 999]
print(f"\n  Longs (pred_norm in bin):")
print(f"  {'bin':>12} {'n':>7} {'hr%':>7} {'mean_ret':>10} {'net_ev':>10}")
for i in range(len(bins)-1):
    lo, hi = bins[i], bins[i+1]
    mask = (pred_te >= lo) & (pred_te < hi) & win_te
    n = mask.sum()
    if n < 5: continue
    hr  = np.mean(ret_te[mask] > 0)
    mr  = np.mean(ret_te[mask])
    ev  = mr - 1.0  # subtract 1 bps cost
    print(f"  [{lo:.1f},{hi:.1f}): {n:>7d} {hr*100:>6.1f}% {mr:>10.4f} {ev:>10.4f}")

print(f"\n  Shorts (pred_norm in bin, negated):")
print(f"  {'bin':>12} {'n':>7} {'hr%':>7} {'mean_ret':>10} {'net_ev':>10}")
for i in range(len(bins)-1):
    lo, hi = bins[i], bins[i+1]
    mask = (pred_te <= -lo) & (pred_te > -hi) & win_te
    n = mask.sum()
    if n < 5: continue
    hr  = np.mean(ret_te[mask] < 0)
    mr  = np.mean(-ret_te[mask])  # short profits when ret < 0
    ev  = mr - 1.0
    print(f"  [-{hi:.1f},-{lo:.1f}): {n:>7d} {hr*100:>6.1f}% {mr:>10.4f} {ev:>10.4f}")

# ---------------------------------------------------------------
# TEST E: Exact replication of CHECK 6 conditions
# ---------------------------------------------------------------
print("\nE) Replicating CHECK 6 exactly (pred_norm > 0.5, all test bars, no filters)")
mask_l_all = pred_te >  0.5
mask_s_all = pred_te < -0.5
n_l  = mask_l_all.sum()
n_s  = mask_s_all.sum()
hr_l = np.mean(ret_te[mask_l_all] > 0)
hr_s = np.mean(ret_te[mask_s_all] < 0)
print(f"  Longs:  n={n_l}  hit={hr_l*100:.1f}%")
print(f"  Shorts: n={n_s}  hit={hr_s*100:.1f}%")
print(f"  Mean ret on long signals:  {np.mean(ret_te[mask_l_all]):.4f} bps")
print(f"  Mean ret on short signals: {np.mean(-ret_te[mask_s_all]):.4f} bps")
print(f"  After 1bps cost — long EV: {np.mean(ret_te[mask_l_all])-1:.4f}  short EV: {np.mean(-ret_te[mask_s_all])-1:.4f}")

print("="*65)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

... (9 outputs left out)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

13:43:26  INFO  preprocess: 881382 rows after regularization
13:43:26  INFO  calculate_features: building CORE31 ...
13:43:51  INFO  build_core31_from_regularized: built 881382 rows, 31 CORE31 features
13:43:52  INFO  calculate_features: building book geometry (ADDED6 + CONTROLS9) ...
13:43:58  INFO  _build_book_geometry_features: built 881382 rows
13:43:58  INFO  calculate_features: building shock recovery (CHALLENGERS 1-2) ...
13:44:00  INFO  _build_shock_recovery_features: bid_shocks=22175 ask_shocks=42052
13:44:00  INFO  _build_shock_recovery_features: built 881382 rows
13:44:00  INFO  calculate_features: building update clustering (CHALLENGER 3) ...
13:44:02  INFO  _build_update_clustering_features: obs=472806 book_change=326695 bid=149451 ask=233683
13:44:03  INFO  _build_update_clustering_features: built 881382 rows
13:44:03  INFO  calculate_features: building liquidity migration (CHALLENGER 4) ...
13:44:05  INFO  _build_liquidity_migration_features: built 881382 rows
13:44:12  

THRESHOLD DIAGNOSTIC

A) Raw directional hit rate — pred_norm threshold, NO filters
  thresh   n_long  n_short   hr_long  hr_short  total_n
     0.1    66013    66701     53.1%     52.1%   132714
     0.2    38990    39284     57.2%     54.5%    78274
     0.3    20873    22368     60.7%     55.5%    43241
     0.5     2638     7270     70.3%     57.8%     9908
     0.7      527     2837     69.1%     59.5%     3364
     1.0       22     1064     81.8%     63.4%     1086
     1.5        0      163      0.0%     79.1%      163
     2.0        0       15      0.0%     73.3%       15

B) Raw hit rate — pred_norm threshold, WITH time filter
  thresh   n_long  n_short   hr_long  hr_short  total_n
     0.1    34517    37115     55.6%     54.0%    71632
     0.2    18706    20526     60.7%     56.8%    39232
     0.3     9450    11045     65.9%     59.6%    20495
     0.5      938     3059     82.7%     64.4%     3997
     0.7      169     1081     90.5%     63.0%     1250
     1.0        7  

In [14]:
"""
USDINR Strategy Backtest v7
=============================
Diagnostic findings from threshold_diagnostic.py:

  Longs  [0.5, 0.7):  hit=81%  mean_ret=1.24 bps  net_ev=+0.24  ✓
  Longs  [0.7, 1.0):  hit=90%  mean_ret=2.20 bps  net_ev=+1.20  ✓✓
  Shorts: net_ev negative at every threshold — shorts never clear 1 bps cost

  With time filter at threshold 0.5:
    Longs n=938  hit=82.7%  — this is the signal

DESIGN:
  1. LONG-ONLY — shorts have no edge after costs
  2. Entry: pred_norm > LONG_THRESH (normalized units, trained model)
  3. Exit: whichever comes first —
       (a) TP hit: price moves up by TP_BPS from entry
       (b) SL hit: price moves down by SL_BPS from entry
       (c) Max hold: STATE_MAX_HOLD seconds elapsed
  4. TP and SL are state-based, scaled by HMM soft-probability volatility estimate
  5. Time filter: 10:30-14:30 IST
  6. Skip if pred_norm < LONG_THRESH (no marginal trades)

State-based exit parameters (from experiment table):
  State  RV(bps²)  σ(bps)  SL=1.0×σ  TP=1.5×σ  MaxHold
  4      1.07      1.04    1.04       1.56       120s   quiet — hold longer
  3      2.59      1.61    1.61       2.42        90s
  1      3.14      1.77    1.77       2.66        60s
  2      5.62      2.37    2.37       3.56        30s   stressed — exit fast

Outputs: results/strategy_backtest_v7/
"""

import sys
sys.path.insert(0, r"C:\projects\hedge_fund")
import json, logging, pathlib, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import lightgbm as lgb
from scipy.stats import spearmanr

from data.store.duckdb_client import DuckDBClient
from data.store.s3_client import S3Client
from data.processing.usdinr.features import USDINRFeatureBuilder
from data.processing.usdinr.constants import CHALLENGER50
from data.processing.usdinr.regimes import add_soft_hmm_features, save_hmm_model

logging.basicConfig(level=logging.INFO,
                    format="%(asctime)s  %(levelname)s  %(message)s",
                    datefmt="%H:%M:%S")
log = logging.getLogger(__name__)

OUT_DIR   = pathlib.Path(r"C:\projects\hedge_fund\results\strategy_backtest_v7")
MODEL_DIR = pathlib.Path(r"C:\projects\hedge_fund\models")
OUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_DAYS = 15
VAL_DAYS   = 5
COST_BPS   = 1.0

IST_OFFSET   = 19800
OPEN_FILTER  = 10 * 3600 + 30 * 60
CLOSE_FILTER = 14 * 3600 + 30 * 60

# Entry threshold in pred_norm units
# From Test D: positive EV starts at 0.5σ
# Sweep around this
LONG_THRESH_BASE  = 0.5
THRESHOLD_SCALES  = [0.5, 0.7, 1.0, 1.2, 1.4, 1.6, 2.0]

# Per-state exit parameters
# SL = c_sl × σ_state,  TP = c_tp × σ_state
# σ_state = sqrt(mean_RV_state / 60) × sqrt(hold_sec) — scaled to hold horizon
# Using empirical RV from experiment table:
#   State 4: σ_60s = sqrt(1.07)  = 1.03 bps
#   State 3: σ_60s = sqrt(2.59)  = 1.61 bps
#   State 1: σ_60s = sqrt(3.14)  = 1.77 bps
#   State 2: σ_60s = sqrt(5.62)  = 2.37 bps
STATE_VOL = {1: 1.77, 2: 2.37, 3: 1.61, 4: 1.03}

# SL multiplier: 1.0× vol (stop out at 1 sigma adverse)
# TP multiplier: 1.5× vol (take profit at 1.5 sigma favorable)
# Max hold: state-dependent — quieter states need more time
C_SL = 1.0
C_TP = 1.5
STATE_MAX_HOLD = {1: 60, 2: 30, 3: 90, 4: 120}

# Position sizing (unchanged)
STATE_SIZE = {1: 0.59, 2: 0.44, 3: 0.65, 4: 1.00}

LGB_PARAMS = {
    "objective":         "regression",
    "metric":            "rmse",
    "verbosity":         -1,
    "boosting_type":     "gbdt",
    "num_leaves":        31,
    "learning_rate":     0.05,
    "feature_fraction":  0.8,
    "bagging_fraction":  0.8,
    "bagging_freq":      5,
    "min_child_samples": 50,
}

# ---------------------------------------------------------------------------
# Load + features
# ---------------------------------------------------------------------------
log.info("Loading data...")
db = DuckDBClient(); s = S3Client(); all_frames = []
for contract in ["26SEPFUT","26OCTFUT","26NOVFUT"]:
    files = s.list_files(f"processed/1s/USDINR{contract}/")
    if not files: continue
    for f in files:
        parts = f.split("/")
        sym, date = parts[2], parts[3].replace(".parquet","")
        try:
            d = db.read_parquet(f"processed/1s/{sym}/{date}.parquet")
            d["symbol"] = sym; all_frames.append(d)
        except: pass
db.close()

df_raw  = pd.concat(all_frames, ignore_index=True)
builder = USDINRFeatureBuilder(data=df_raw, auto_build=True)
df      = builder.dataset.copy()
df["tick_count"] = df.get("tick_count", pd.Series(0, index=df.index)).fillna(0).astype(int)
df = df.sort_values(["symbol","date","ts_sec"]).reset_index(drop=True)
log.info("Dataset: %d rows", len(df))

# ---------------------------------------------------------------------------
# Split
# ---------------------------------------------------------------------------
all_dates   = sorted(df["date"].unique())
train_dates = all_dates[:TRAIN_DAYS]
val_dates   = all_dates[TRAIN_DAYS:TRAIN_DAYS+VAL_DAYS]
test_dates  = all_dates[TRAIN_DAYS+VAL_DAYS:]
train_end   = str(train_dates[-1])
log.info("Train %d | Val %d | Test %d days",
         len(train_dates), len(val_dates), len(test_dates))

mask_train = df["date"].isin(train_dates)
mask_val   = df["date"].isin(val_dates)
mask_test  = df["date"].isin(test_dates)

# ---------------------------------------------------------------------------
# HMM
# ---------------------------------------------------------------------------
df, hmm = add_soft_hmm_features(df=df, train_end_date=train_end,
                                 K=4, restarts=3, max_iter=70, tol=1e-6, seed=42)
save_hmm_model(hmm, MODEL_DIR / "hmm_v7.pkl")
log.info("State dist:\n%s", df["state_filtered"].value_counts().sort_index().to_string())

# ---------------------------------------------------------------------------
# Targets + time features
# ---------------------------------------------------------------------------
if "mid" not in df.columns:
    df["mid"] = (df["bid_p1"] + df["ask_p1"]) / 2.0
df["log_mid"] = np.log(df["mid"])
g = df.groupby(["symbol","date"], sort=False)
df["ret_60s"] = (g["log_mid"].shift(-60) - df["log_mid"]) * 1e4

# Normalized target for training
df["ret_60s_norm"] = df.groupby(["symbol","date"])["ret_60s"].transform(
    lambda x: (x - x.mean()) / (x.std() + 1e-8)
)

df["ist_tod"]   = (df["ts_sec"] + IST_OFFSET) % 86400
df["in_window"] = (df["ist_tod"] >= OPEN_FILTER) & (df["ist_tod"] <= CLOSE_FILTER)

# ---------------------------------------------------------------------------
# Build 1-second price series for intra-trade exit simulation
# We need mid price at every second to check TP/SL during the hold
# ---------------------------------------------------------------------------
# Create a lookup: (symbol, date, ts_sec) -> mid
log.info("Building mid price lookup for intra-trade exits...")
mid_lookup = df.set_index(["symbol","date","ts_sec"])["mid"].to_dict()

# ---------------------------------------------------------------------------
# Train LightGBM
# ---------------------------------------------------------------------------
features   = CHALLENGER50
target     = "ret_60s_norm"
valid_mask = df[features + [target,"ret_60s"]].notna().all(axis=1)

tr = df[mask_train & valid_mask]
va = df[mask_val   & valid_mask]

ds_tr = lgb.Dataset(tr[features].values, label=tr[target].values, feature_name=features)
ds_va = lgb.Dataset(va[features].values, label=va[target].values,
                    feature_name=features, reference=ds_tr)

model = lgb.train(
    LGB_PARAMS, ds_tr, num_boost_round=1000,
    valid_sets=[ds_va],
    callbacks=[lgb.early_stopping(50,verbose=False), lgb.log_evaluation(100)],
)
log.info("Best iter: %d", model.best_iteration)
model.save_model(str(MODEL_DIR / "lgbm_v7.txt"))

# ---------------------------------------------------------------------------
# Generate predictions
# ---------------------------------------------------------------------------
eval_mask = (mask_val | mask_test) & valid_mask & df["ret_60s"].notna()
df_eval   = df[eval_mask].copy().reset_index(drop=True)
df_eval["state"]      = df_eval["state_filtered"].astype(int)
df_eval["pred_norm"]  = model.predict(df_eval[features].values)
df_eval["size_state"] = df_eval["state"].map(STATE_SIZE)

# Confirm signal quality before simulating
te_mask  = df_eval["date"].isin(test_dates)
pred_te  = df_eval.loc[te_mask, "pred_norm"].values
ret_te   = df_eval.loc[te_mask, "ret_60s"].values
win_te   = df_eval.loc[te_mask, "in_window"].values

mask_long_signal = (pred_te > LONG_THRESH_BASE) & win_te
hr_signal = np.mean(ret_te[mask_long_signal] > 0) if mask_long_signal.sum() > 0 else 0
ev_signal = np.mean(ret_te[mask_long_signal]) - COST_BPS if mask_long_signal.sum() > 0 else 0
log.info("Signal quality at base threshold %.1f (time-filtered): n=%d  hit=%.1f%%  EV=%.4f bps",
         LONG_THRESH_BASE, mask_long_signal.sum(), hr_signal*100, ev_signal)

ic_val,  _ = spearmanr(
    df_eval.loc[df_eval["date"].isin(val_dates), "ret_60s"],
    df_eval.loc[df_eval["date"].isin(val_dates), "pred_norm"])
ic_test, _ = spearmanr(
    df_eval.loc[te_mask, "ret_60s"],
    df_eval.loc[te_mask, "pred_norm"])
log.info("IC — val:%.4f  test:%.4f", ic_val, ic_test)

# ---------------------------------------------------------------------------
# Simulator with intra-trade TP/SL checks
# ---------------------------------------------------------------------------

def simulate_long_only(
    df_day: pd.DataFrame,
    long_thresh: float,
    c_sl: float = C_SL,
    c_tp: float = C_TP,
) -> list:
    """
    Long-only simulator with per-state TP/SL/max-hold exits.

    For each trade:
      - Entry when pred_norm > long_thresh AND in time window
      - SL distance = c_sl × STATE_VOL[state] bps below entry
      - TP distance = c_tp × STATE_VOL[state] bps above entry
      - Max hold   = STATE_MAX_HOLD[state] seconds
      - Walk forward second-by-second to check TP/SL
      - If neither hit by max_hold, exit at mid[entry_ts + max_hold]
    """
    trades     = []
    busy_until = -1

    sym  = df_day["symbol"].iloc[0] if "symbol" in df_day.columns else None
    date = df_day["date"].iloc[0]

    rows = df_day.sort_values("ts_sec").reset_index(drop=True)

    for _, row in rows.iterrows():
        ts = int(row["ts_sec"])

        if ts <= busy_until:
            continue
        if not row["in_window"]:
            continue

        pred = float(row["pred_norm"])
        if pred < long_thresh:
            continue
        if pd.isna(row["ret_60s"]):
            continue

        state    = int(row["state"])
        vol      = STATE_VOL.get(state, 1.77)
        sl_dist  = c_sl * vol      # bps below entry → exit loss
        tp_dist  = c_tp * vol      # bps above entry → exit profit
        max_hold = STATE_MAX_HOLD.get(state, 60)
        size     = STATE_SIZE.get(state, 1.0)

        entry_mid = float(row["mid"])
        sl_price  = entry_mid * (1 - sl_dist * 1e-4)
        tp_price  = entry_mid * (1 + tp_dist * 1e-4)

        # Walk forward second-by-second to find exit
        exit_mid  = None
        exit_ts   = ts + max_hold
        exit_type = "timeout"

        for dt in range(1, max_hold + 1):
            check_ts = ts + dt
            # Look up mid price at this second
            mid_t = mid_lookup.get((sym, date, check_ts), None)
            if mid_t is None:
                continue
            mid_t = float(mid_t)
            if mid_t >= tp_price:
                exit_mid  = mid_t
                exit_ts   = check_ts
                exit_type = "tp"
                break
            elif mid_t <= sl_price:
                exit_mid  = mid_t
                exit_ts   = check_ts
                exit_type = "sl"
                break

        # If no intra-trade exit, use mid at max_hold
        if exit_mid is None:
            exit_mid = mid_lookup.get((sym, date, ts + max_hold), None)
            if exit_mid is None:
                # Fall back to ret_60s if no price available
                exit_mid = entry_mid * (1 + float(row["ret_60s"]) * 1e-4)

        exit_mid  = float(exit_mid)
        actual_ret = (np.log(exit_mid) - np.log(entry_mid)) * 1e4   # bps
        net_pnl    = (actual_ret - COST_BPS) * size

        trades.append({
            "ts_sec":     ts,
            "exit_ts":    exit_ts,
            "hold_sec":   exit_ts - ts,
            "pred_norm":  pred,
            "entry_mid":  entry_mid,
            "exit_mid":   exit_mid,
            "actual_ret": actual_ret,
            "exit_type":  exit_type,
            "state":      state,
            "size":       size,
            "sl_dist":    sl_dist,
            "tp_dist":    tp_dist,
            "net_pnl_bps":net_pnl,
        })

        busy_until = exit_ts   # hold until actual exit, not fixed 60s

    return trades


def compute_metrics(pnl_series: pd.Series, label: str) -> dict:
    pnl = pnl_series.dropna().values; n = len(pnl)
    if n == 0:
        return {"n_trades":0,"sharpe":0,"hit_rate":0,
                "total_pnl_bps":0,"mean_pnl_bps":0,"std_pnl_bps":0,"max_drawdown_bps":0}
    total=float(pnl.sum()); mean=float(pnl.mean())
    std=float(pnl.std()) if pnl.std()>0 else 1e-9
    sharpe=mean/std*np.sqrt(n); hit=float(np.mean(pnl>0))
    cum=np.cumsum(pnl); max_dd=float((cum-np.maximum.accumulate(cum)).min())
    log.info("%s | n=%d total=%.2f sharpe=%.3f hit=%.3f", label, n, total, sharpe, hit)
    return {"total_pnl_bps":round(total,2),"mean_pnl_bps":round(mean,5),
            "std_pnl_bps":round(std,5),"sharpe":round(sharpe,3),
            "hit_rate":round(hit,4),"max_drawdown_bps":round(max_dd,2),"n_trades":n}

# ---------------------------------------------------------------------------
# Threshold sweep on test set
# ---------------------------------------------------------------------------
log.info("Sweeping thresholds...")
test_days  = df_eval[df_eval["date"].isin(test_dates)].copy()
sweep_rows = []; all_trades = []

for scale in THRESHOLD_SCALES:
    thresh = LONG_THRESH_BASE * scale

    day_trades = []
    for day, grp in test_days.groupby("date"):
        t = simulate_long_only(grp, long_thresh=thresh)
        for tr in t:
            tr["date"]  = str(day)
            tr["scale"] = scale
            tr["thresh"]= thresh
        day_trades.extend(t)

    if not day_trades:
        log.info("thresh=%.2f | 0 trades", thresh); continue

    tdf    = pd.DataFrame(day_trades)
    n_tp   = int((tdf["exit_type"]=="tp").sum())
    n_sl   = int((tdf["exit_type"]=="sl").sum())
    n_to   = int((tdf["exit_type"]=="timeout").sum())
    m = compute_metrics(tdf["net_pnl_bps"], f"thresh={thresh:.2f}")
    m.update({"scale":scale,"thresh":round(thresh,3),
               "n_tp":n_tp,"n_sl":n_sl,"n_timeout":n_to,
               "pct_tp":round(100*n_tp/max(len(tdf),1),1),
               "avg_hold":round(float(tdf["hold_sec"].mean()),1)})
    sweep_rows.append(m); all_trades.extend(day_trades)

sweep_df = pd.DataFrame(sweep_rows)
best_scale = 1.0
if not sweep_df.empty:
    best_row   = sweep_df.loc[sweep_df["sharpe"].idxmax()]
    best_scale = float(best_row["scale"])
    log.info("Best scale:%.2f thresh=%.3f sharpe=%.3f n=%d hit=%.3f",
             best_scale, best_row["thresh"], best_row["sharpe"],
             best_row["n_trades"], best_row["hit_rate"])

# ---------------------------------------------------------------------------
# Per-state breakdown at best scale
# ---------------------------------------------------------------------------
best_thresh = LONG_THRESH_BASE * best_scale
best_trades = [t for t in all_trades if t["scale"] == best_scale]
state_rows  = []

if best_trades:
    bt_df = pd.DataFrame(best_trades)
    for state in [1,2,3,4]:
        s = bt_df[bt_df["state"]==state]
        if len(s)<3: continue
        m = compute_metrics(s["net_pnl_bps"], f"state_{state}")
        n_tp = int((s["exit_type"]=="tp").sum())
        n_sl = int((s["exit_type"]=="sl").sum())
        m.update({"state":state,
                  "pct_trades":round(len(s)/len(bt_df)*100,1),
                  "size_mult":STATE_SIZE[state],
                  "tp_bps":round(C_TP * STATE_VOL[state],2),
                  "sl_bps":round(C_SL * STATE_VOL[state],2),
                  "max_hold":STATE_MAX_HOLD[state],
                  "n_tp":n_tp,"n_sl":n_sl,
                  "pct_tp":round(100*n_tp/max(len(s),1),1),
                  "avg_hold":round(float(s["hold_sec"].mean()),1)})
        state_rows.append(m)
state_metrics = pd.DataFrame(state_rows)

# ---------------------------------------------------------------------------
# Daily PnL
# ---------------------------------------------------------------------------
daily_rows = []
if best_trades:
    bt_df = pd.DataFrame(best_trades)
    for day, grp in bt_df.groupby("date"):
        pnl = grp["net_pnl_bps"]
        daily_rows.append({"date":day,"n_trades":len(pnl),
                            "total_pnl":round(float(pnl.sum()),2),
                            "hit_rate":round(float((pnl>0).mean()),3),
                            "mean_pnl":round(float(pnl.mean()),5),
                            "n_tp":int((grp["exit_type"]=="tp").sum()),
                            "n_sl":int((grp["exit_type"]=="sl").sum()),
                            "n_timeout":int((grp["exit_type"]=="timeout").sum())})
daily_pnl = pd.DataFrame(daily_rows)

# ---------------------------------------------------------------------------
# Save
# ---------------------------------------------------------------------------
with open(OUT_DIR/"metrics.json","w") as f:
    json.dump({"ic":{"val":round(ic_val,4),"test":round(ic_test,4)},
               "signal_quality":{"n":int(mask_long_signal.sum()),
                                  "hit_rate":round(hr_signal,4),
                                  "ev_bps":round(ev_signal,4)},
               "best_scale":best_scale,
               "best_thresh":best_thresh,
               "config":{"long_only":True,"c_sl":C_SL,"c_tp":C_TP,
                          "state_vol":STATE_VOL,"state_max_hold":STATE_MAX_HOLD,
                          "state_size":STATE_SIZE}},f,indent=2)

sweep_df.to_csv(OUT_DIR/"threshold_sweep.csv",index=False)
if not state_metrics.empty: state_metrics.to_csv(OUT_DIR/"state_metrics.csv",index=False)
if not daily_pnl.empty: daily_pnl.to_csv(OUT_DIR/"daily_pnl.csv",index=False)
if all_trades: pd.DataFrame(all_trades).to_csv(OUT_DIR/"trade_log.csv",index=False)

# ---------------------------------------------------------------------------
# Print
# ---------------------------------------------------------------------------
print("\n"+"="*70)
print("STRATEGY BACKTEST v7 — LONG-ONLY, STATE-BASED TP/SL/HOLD")
print("="*70)
print(f"\nSignal: pred_norm > threshold (normalized, trained on ret_60s_norm)")
print(f"Exit:   TP={C_TP}×σ_state  SL={C_SL}×σ_state  or max_hold per state")
print(f"IC — val:{ic_val:.4f}  test:{ic_test:.4f}")
print(f"Signal EV at base thresh {LONG_THRESH_BASE}: {ev_signal:+.4f} bps after cost (n={mask_long_signal.sum()})")
print(f"\nState exit params:")
for st in [1,2,3,4]:
    print(f"  State {st}: TP={C_TP*STATE_VOL[st]:.2f} bps  "
          f"SL={C_SL*STATE_VOL[st]:.2f} bps  "
          f"MaxHold={STATE_MAX_HOLD[st]}s  "
          f"Size={STATE_SIZE[st]}")

print(f"\n--- Threshold sweep (test, long-only) ---")
if not sweep_df.empty:
    print(sweep_df[["scale","thresh","n_trades","n_tp","n_sl","n_timeout",
                     "pct_tp","avg_hold","hit_rate","mean_pnl_bps",
                     "total_pnl_bps","sharpe","max_drawdown_bps"]].to_string(index=False))

if not state_metrics.empty:
    print(f"\n--- Per-State at best thresh={best_thresh:.2f} ---")
    print(state_metrics[["state","pct_trades","size_mult","n_trades",
                          "tp_bps","sl_bps","max_hold","n_tp","n_sl","pct_tp",
                          "avg_hold","total_pnl_bps","sharpe",
                          "hit_rate","max_drawdown_bps"]].to_string(index=False))

if not daily_pnl.empty:
    print(f"\n--- Daily PnL at best thresh={best_thresh:.2f} ---")
    print(daily_pnl.to_string(index=False))

print("="*70)

13:53:21  INFO  Loading data...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

... (9 outputs left out)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

13:55:14  INFO  preprocess: 881382 rows after regularization
13:55:14  INFO  calculate_features: building CORE31 ...
13:55:36  INFO  build_core31_from_regularized: built 881382 rows, 31 CORE31 features
13:55:37  INFO  calculate_features: building book geometry (ADDED6 + CONTROLS9) ...
13:55:43  INFO  _build_book_geometry_features: built 881382 rows
13:55:43  INFO  calculate_features: building shock recovery (CHALLENGERS 1-2) ...
13:55:45  INFO  _build_shock_recovery_features: bid_shocks=22175 ask_shocks=42052
13:55:45  INFO  _build_shock_recovery_features: built 881382 rows
13:55:45  INFO  calculate_features: building update clustering (CHALLENGER 3) ...
13:55:47  INFO  _build_update_clustering_features: obs=472806 book_change=326695 bid=149451 ask=233683
13:55:48  INFO  _build_update_clustering_features: built 881382 rows
13:55:48  INFO  calculate_features: building liquidity migration (CHALLENGER 4) ...
13:55:50  INFO  _build_liquidity_migration_features: built 881382 rows
13:55:58  


STRATEGY BACKTEST v7 — LONG-ONLY, STATE-BASED TP/SL/HOLD

Signal: pred_norm > threshold (normalized, trained on ret_60s_norm)
Exit:   TP=1.5×σ_state  SL=1.0×σ_state  or max_hold per state
IC — val:0.2556  test:0.2352
Signal EV at base thresh 0.5: +0.4042 bps after cost (n=938)

State exit params:
  State 1: TP=2.66 bps  SL=1.77 bps  MaxHold=60s  Size=0.59
  State 2: TP=3.56 bps  SL=2.37 bps  MaxHold=30s  Size=0.44
  State 3: TP=2.42 bps  SL=1.61 bps  MaxHold=90s  Size=0.65
  State 4: TP=1.54 bps  SL=1.03 bps  MaxHold=120s  Size=1.0

--- Threshold sweep (test, long-only) ---
 scale  thresh  n_trades  n_tp  n_sl  n_timeout  pct_tp  avg_hold  hit_rate  mean_pnl_bps  total_pnl_bps        sharpe  max_drawdown_bps
   0.5    0.25       710    85    52        573    12.0      66.3    0.3113      -0.28509        -202.41 -9.206000e+00           -205.84
   0.7    0.35       422    55    28        339    13.0      65.5    0.3886      -0.18334         -77.37 -4.527000e+00            -90.74
   1.0 

In [15]:
"""
STRATEGY BACKTEST v8
====================
v7 findings:
  - Best thresh=0.60: Sharpe=1.40, hit=64.7%, total=9.59 bps
  - Only 29% of trades hit TP — TP too wide, 71% timeout
  - avg_hold=55s — almost all timing out, not exiting on signal
  - State 4: 0 trades at best threshold (quiet state, rare signal)
  - Nov 9: 9 trades, 7.98 bps — system works on trending days

ROOT CAUSE: TP = 1.5×σ is too ambitious for 60s price travel.
  USDINR 60s σ ≈ 1-2 bps. Reaching 1.5× means price must travel
  continuously in one direction for the full hold — unlikely.
  Actual returns when profitable: mean ~1.5-2.5 bps but many timeouts
  collect only 0.2-0.8 bps.

FIX: Tighter TP to harvest more frequently.
  - TP: 1.5× → 0.8× (target closer, hit more often)
  - SL: 1.0× → 1.2× (slightly wider, avoid getting stopped on noise)
  - State 4 max_hold: 120s → 180s (quiet market needs more time)
  - Also sweep c_tp: [0.5, 0.7, 0.8, 1.0, 1.2, 1.5] to find optimum

ALSO: State 4 TP=1.54 bps unreachable in quiet market.
  State 4 specific: TP=0.6×σ=0.62 bps (smaller target, more certain)

SECONDARY SWEEP: c_tp × c_sl grid at best threshold
"""

import sys
sys.path.insert(0, r"C:\projects\hedge_fund")
import json, logging, pathlib, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import lightgbm as lgb
from scipy.stats import spearmanr

from data.store.duckdb_client import DuckDBClient
from data.store.s3_client import S3Client
from data.processing.usdinr.features import USDINRFeatureBuilder
from data.processing.usdinr.constants import CHALLENGER50
from data.processing.usdinr.regimes import add_soft_hmm_features, save_hmm_model

logging.basicConfig(level=logging.INFO,
                    format="%(asctime)s  %(levelname)s  %(message)s",
                    datefmt="%H:%M:%S")
log = logging.getLogger(__name__)

OUT_DIR   = pathlib.Path(r"C:\projects\hedge_fund\results\strategy_backtest_v8")
MODEL_DIR = pathlib.Path(r"C:\projects\hedge_fund\models")
OUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_DAYS = 15
VAL_DAYS   = 5
COST_BPS   = 1.0

IST_OFFSET   = 19800
OPEN_FILTER  = 10 * 3600 + 30 * 60
CLOSE_FILTER = 14 * 3600 + 30 * 60

# Best threshold from v7
BEST_THRESH = 0.60

# Per-state vol
STATE_VOL = {1: 1.77, 2: 2.37, 3: 1.61, 4: 1.03}

# v8: tighter TP, slightly wider SL
# State 4 gets its own TP (quiet market, can't reach wide targets)
C_SL_DEFAULT = 1.2   # was 1.0
C_TP_DEFAULT = 0.8   # was 1.5

STATE_MAX_HOLD = {1: 60, 2: 30, 3: 90, 4: 180}   # State 4 extended
STATE_SIZE     = {1: 0.59, 2: 0.44, 3: 0.65, 4: 1.00}

# State-specific TP overrides (others use C_TP_DEFAULT)
STATE_TP_OVERRIDE = {4: 0.6}   # State 4: 0.6× instead of 0.8×

# C_TP sweep to find optimum (at fixed BEST_THRESH and C_SL_DEFAULT)
C_TP_SWEEP = [0.4, 0.5, 0.6, 0.7, 0.8, 1.0, 1.2, 1.5]
C_SL_SWEEP = [0.8, 1.0, 1.2, 1.5, 2.0]

LGB_PARAMS = {
    "objective":         "regression",
    "metric":            "rmse",
    "verbosity":         -1,
    "boosting_type":     "gbdt",
    "num_leaves":        31,
    "learning_rate":     0.05,
    "feature_fraction":  0.8,
    "bagging_fraction":  0.8,
    "bagging_freq":      5,
    "min_child_samples": 50,
}

# ---------------------------------------------------------------------------
# Load + features
# ---------------------------------------------------------------------------
log.info("Loading data...")
db = DuckDBClient(); s = S3Client(); all_frames = []
for contract in ["26SEPFUT","26OCTFUT","26NOVFUT"]:
    files = s.list_files(f"processed/1s/USDINR{contract}/")
    if not files: continue
    for f in files:
        parts = f.split("/")
        sym, date = parts[2], parts[3].replace(".parquet","")
        try:
            d = db.read_parquet(f"processed/1s/{sym}/{date}.parquet")
            d["symbol"] = sym; all_frames.append(d)
        except: pass
db.close()

df_raw  = pd.concat(all_frames, ignore_index=True)
builder = USDINRFeatureBuilder(data=df_raw, auto_build=True)
df      = builder.dataset.copy()
df["tick_count"] = df.get("tick_count", pd.Series(0, index=df.index)).fillna(0).astype(int)
df = df.sort_values(["symbol","date","ts_sec"]).reset_index(drop=True)
log.info("Dataset: %d rows", len(df))

all_dates   = sorted(df["date"].unique())
train_dates = all_dates[:TRAIN_DAYS]
val_dates   = all_dates[TRAIN_DAYS:TRAIN_DAYS+VAL_DAYS]
test_dates  = all_dates[TRAIN_DAYS+VAL_DAYS:]
train_end   = str(train_dates[-1])
log.info("Train %d | Val %d | Test %d days",
         len(train_dates), len(val_dates), len(test_dates))

mask_train = df["date"].isin(train_dates)
mask_val   = df["date"].isin(val_dates)
mask_test  = df["date"].isin(test_dates)

# ---------------------------------------------------------------------------
# HMM
# ---------------------------------------------------------------------------
df, hmm = add_soft_hmm_features(df=df, train_end_date=train_end,
                                 K=4, restarts=3, max_iter=70, tol=1e-6, seed=42)
save_hmm_model(hmm, MODEL_DIR / "hmm_v8.pkl")

# ---------------------------------------------------------------------------
# Targets
# ---------------------------------------------------------------------------
if "mid" not in df.columns:
    df["mid"] = (df["bid_p1"] + df["ask_p1"]) / 2.0
df["log_mid"] = np.log(df["mid"])
g = df.groupby(["symbol","date"], sort=False)
df["ret_60s"] = (g["log_mid"].shift(-60) - df["log_mid"]) * 1e4

df["ret_60s_norm"] = df.groupby(["symbol","date"])["ret_60s"].transform(
    lambda x: (x - x.mean()) / (x.std() + 1e-8)
)

df["ist_tod"]   = (df["ts_sec"] + IST_OFFSET) % 86400
df["in_window"] = (df["ist_tod"] >= OPEN_FILTER) & (df["ist_tod"] <= CLOSE_FILTER)

log.info("Building mid price lookup...")
mid_lookup = df.set_index(["symbol","date","ts_sec"])["mid"].to_dict()

# ---------------------------------------------------------------------------
# Train
# ---------------------------------------------------------------------------
features   = CHALLENGER50
target     = "ret_60s_norm"
valid_mask = df[features + [target,"ret_60s"]].notna().all(axis=1)

tr = df[mask_train & valid_mask]
va = df[mask_val   & valid_mask]

ds_tr = lgb.Dataset(tr[features].values, label=tr[target].values, feature_name=features)
ds_va = lgb.Dataset(va[features].values, label=va[target].values,
                    feature_name=features, reference=ds_tr)

model = lgb.train(
    LGB_PARAMS, ds_tr, num_boost_round=1000,
    valid_sets=[ds_va],
    callbacks=[lgb.early_stopping(50,verbose=False), lgb.log_evaluation(100)],
)
model.save_model(str(MODEL_DIR / "lgbm_v8.txt"))

# ---------------------------------------------------------------------------
# Predictions
# ---------------------------------------------------------------------------
eval_mask = (mask_val | mask_test) & valid_mask & df["ret_60s"].notna()
df_eval   = df[eval_mask].copy().reset_index(drop=True)
df_eval["state"]     = df_eval["state_filtered"].astype(int)
df_eval["pred_norm"] = model.predict(df_eval[features].values)
df_eval["size_state"]= df_eval["state"].map(STATE_SIZE)

# ---------------------------------------------------------------------------
# Simulator
# ---------------------------------------------------------------------------

def simulate_long_only(
    df_day: pd.DataFrame,
    long_thresh: float,
    c_sl: float,
    c_tp: float,
    state_tp_override: dict = None,
) -> list:
    trades     = []
    busy_until = -1

    sym  = df_day["symbol"].iloc[0] if "symbol" in df_day.columns else None
    date = df_day["date"].iloc[0]

    rows = df_day.sort_values("ts_sec").reset_index(drop=True)
    if state_tp_override is None:
        state_tp_override = {}

    for _, row in rows.iterrows():
        ts = int(row["ts_sec"])

        if ts <= busy_until:   continue
        if not row["in_window"]: continue
        if float(row["pred_norm"]) < long_thresh: continue
        if pd.isna(row["ret_60s"]): continue

        state    = int(row["state"])
        vol      = STATE_VOL.get(state, 1.77)
        sl_dist  = c_sl * vol
        tp_mult  = state_tp_override.get(state, c_tp)
        tp_dist  = tp_mult * vol
        max_hold = STATE_MAX_HOLD.get(state, 60)
        size     = STATE_SIZE.get(state, 1.0)

        entry_mid = float(row["mid"])
        sl_price  = entry_mid * (1 - sl_dist * 1e-4)
        tp_price  = entry_mid * (1 + tp_dist * 1e-4)

        exit_mid  = None
        exit_ts   = ts + max_hold
        exit_type = "timeout"

        for dt in range(1, max_hold + 1):
            mid_t = mid_lookup.get((sym, date, ts + dt))
            if mid_t is None: continue
            mid_t = float(mid_t)
            if mid_t >= tp_price:
                exit_mid  = mid_t
                exit_ts   = ts + dt
                exit_type = "tp"
                break
            elif mid_t <= sl_price:
                exit_mid  = mid_t
                exit_ts   = ts + dt
                exit_type = "sl"
                break

        if exit_mid is None:
            exit_mid = mid_lookup.get((sym, date, ts + max_hold))
            if exit_mid is None:
                exit_mid = entry_mid * (1 + float(row["ret_60s"]) * 1e-4)

        exit_mid  = float(exit_mid)
        actual_ret = (np.log(exit_mid) - np.log(entry_mid)) * 1e4
        net_pnl    = (actual_ret - COST_BPS) * size

        trades.append({
            "ts_sec": ts, "exit_ts": exit_ts,
            "hold_sec": exit_ts - ts,
            "pred_norm": float(row["pred_norm"]),
            "entry_mid": entry_mid, "exit_mid": exit_mid,
            "actual_ret": actual_ret,
            "exit_type": exit_type,
            "state": state, "size": size,
            "sl_dist": sl_dist, "tp_dist": tp_dist,
            "net_pnl_bps": net_pnl,
        })
        busy_until = exit_ts

    return trades


def run_sim(df_set, long_thresh, c_sl, c_tp, state_tp_override=None):
    all_t = []
    for day, grp in df_set.groupby("date"):
        t = simulate_long_only(grp, long_thresh, c_sl, c_tp, state_tp_override)
        for tr in t:
            tr["date"] = str(day)
        all_t.extend(t)
    return pd.DataFrame(all_t) if all_t else pd.DataFrame()


def metrics(tdf, label=""):
    if tdf.empty: return {}
    pnl = tdf["net_pnl_bps"].values; n = len(pnl)
    total = float(pnl.sum()); mean = float(pnl.mean())
    std   = float(pnl.std()) if pnl.std() > 0 else 1e-9
    sharpe = mean / std * np.sqrt(n)
    hit    = float(np.mean(pnl > 0))
    n_tp   = int((tdf["exit_type"]=="tp").sum())
    n_sl   = int((tdf["exit_type"]=="sl").sum())
    cum    = np.cumsum(pnl)
    max_dd = float((cum - np.maximum.accumulate(cum)).min())
    pct_tp = round(100 * n_tp / max(n, 1), 1)
    avg_hold = float(tdf["hold_sec"].mean())
    return {"n": n, "total": round(total,2), "mean": round(mean,5),
            "sharpe": round(sharpe,3), "hit": round(hit,4),
            "n_tp": n_tp, "n_sl": n_sl, "pct_tp": pct_tp,
            "avg_hold": round(avg_hold,1), "max_dd": round(max_dd,2)}


# ---------------------------------------------------------------------------
# TEST SET
# ---------------------------------------------------------------------------
test_days = df_eval[df_eval["date"].isin(test_dates)].copy()

# ---------------------------------------------------------------------------
# SWEEP 1: c_tp sweep at best thresh, fixed c_sl=1.2
# ---------------------------------------------------------------------------
log.info("C_TP sweep...")
ctp_rows = []
for c_tp in C_TP_SWEEP:
    tdf = run_sim(test_days, BEST_THRESH, C_SL_DEFAULT, c_tp, STATE_TP_OVERRIDE)
    if tdf.empty: continue
    m = metrics(tdf)
    m.update({"c_tp": c_tp, "c_sl": C_SL_DEFAULT})
    ctp_rows.append(m)
ctp_df = pd.DataFrame(ctp_rows)

best_ctp = C_TP_DEFAULT
if not ctp_df.empty:
    best_ctp = float(ctp_df.loc[ctp_df["sharpe"].idxmax(), "c_tp"])
    log.info("Best c_tp=%.2f", best_ctp)

# ---------------------------------------------------------------------------
# SWEEP 2: c_sl sweep at best thresh, fixed best c_tp
# ---------------------------------------------------------------------------
log.info("C_SL sweep...")
csl_rows = []
for c_sl in C_SL_SWEEP:
    tdf = run_sim(test_days, BEST_THRESH, c_sl, best_ctp, STATE_TP_OVERRIDE)
    if tdf.empty: continue
    m = metrics(tdf)
    m.update({"c_sl": c_sl, "c_tp": best_ctp})
    csl_rows.append(m)
csl_df = pd.DataFrame(csl_rows)

best_csl = C_SL_DEFAULT
if not csl_df.empty:
    best_csl = float(csl_df.loc[csl_df["sharpe"].idxmax(), "c_sl"])
    log.info("Best c_sl=%.2f", best_csl)

# ---------------------------------------------------------------------------
# FINAL: best params full run
# ---------------------------------------------------------------------------
log.info("Final run: thresh=%.2f c_sl=%.2f c_tp=%.2f", BEST_THRESH, best_csl, best_ctp)
final_tdf = run_sim(test_days, BEST_THRESH, best_csl, best_ctp, STATE_TP_OVERRIDE)

final_m = metrics(final_tdf, "FINAL")

# Per-state
state_rows = []
if not final_tdf.empty:
    for st in [1,2,3,4]:
        s = final_tdf[final_tdf["state"]==st]
        if len(s)<3: continue
        m = metrics(s)
        m.update({"state": st,
                  "pct_trades": round(len(s)/len(final_tdf)*100, 1),
                  "tp_bps": round(STATE_TP_OVERRIDE.get(st, best_ctp) * STATE_VOL[st], 2),
                  "sl_bps": round(best_csl * STATE_VOL[st], 2),
                  "max_hold": STATE_MAX_HOLD[st]})
        state_rows.append(m)
state_df = pd.DataFrame(state_rows)

# Daily PnL
daily_rows = []
if not final_tdf.empty:
    for day, grp in final_tdf.groupby("date"):
        pnl = grp["net_pnl_bps"]
        daily_rows.append({"date": day, "n": len(pnl),
                            "total": round(float(pnl.sum()),2),
                            "hit": round(float((pnl>0).mean()),3),
                            "n_tp": int((grp["exit_type"]=="tp").sum()),
                            "n_sl": int((grp["exit_type"]=="sl").sum()),
                            "n_to": int((grp["exit_type"]=="timeout").sum()),
                            "mean_pnl": round(float(pnl.mean()),4)})
daily_df = pd.DataFrame(daily_rows)

# Save
ctp_df.to_csv(OUT_DIR/"ctp_sweep.csv", index=False)
csl_df.to_csv(OUT_DIR/"csl_sweep.csv", index=False)
if not final_tdf.empty: final_tdf.to_csv(OUT_DIR/"trade_log.csv", index=False)
if not state_df.empty:  state_df.to_csv(OUT_DIR/"state_metrics.csv", index=False)
if not daily_df.empty:  daily_df.to_csv(OUT_DIR/"daily_pnl.csv", index=False)

# ---------------------------------------------------------------------------
# Print
# ---------------------------------------------------------------------------
print("\n" + "="*70)
print("STRATEGY BACKTEST v8 — C_TP / C_SL CALIBRATION")
print("="*70)

print(f"\nFixed: thresh={BEST_THRESH}  State4_TP_override={STATE_TP_OVERRIDE}")
print(f"\n--- C_TP sweep (c_sl={C_SL_DEFAULT} fixed) ---")
print(f"{'c_tp':>6} {'n':>5} {'total':>8} {'sharpe':>8} {'hit%':>7} {'pct_tp':>7} {'avg_hold':>9} {'max_dd':>8}")
for _, r in ctp_df.iterrows():
    print(f"{r['c_tp']:>6.2f} {int(r['n']):>5} {r['total']:>8.2f} {r['sharpe']:>8.3f} "
          f"{r['hit']*100:>6.1f}% {r['pct_tp']:>6.1f}% {r['avg_hold']:>9.1f} {r['max_dd']:>8.2f}")

print(f"\n--- C_SL sweep (c_tp={best_ctp} fixed) ---")
print(f"{'c_sl':>6} {'n':>5} {'total':>8} {'sharpe':>8} {'hit%':>7} {'pct_tp':>7} {'avg_hold':>9} {'max_dd':>8}")
for _, r in csl_df.iterrows():
    print(f"{r['c_sl']:>6.2f} {int(r['n']):>5} {r['total']:>8.2f} {r['sharpe']:>8.3f} "
          f"{r['hit']*100:>6.1f}% {r['pct_tp']:>6.1f}% {r['avg_hold']:>9.1f} {r['max_dd']:>8.2f}")

print(f"\n--- FINAL at thresh={BEST_THRESH} c_sl={best_csl} c_tp={best_ctp} ---")
if final_m:
    print(f"  n={final_m['n']}  total={final_m['total']} bps  sharpe={final_m['sharpe']}  "
          f"hit={final_m['hit']*100:.1f}%  pct_tp={final_m['pct_tp']}%  "
          f"avg_hold={final_m['avg_hold']}s  max_dd={final_m['max_dd']} bps")

if not state_df.empty:
    print(f"\n--- Per-State ---")
    print(state_df[["state","pct_trades","n","tp_bps","sl_bps","max_hold",
                     "n_tp","n_sl","pct_tp","avg_hold","total","sharpe","hit","max_dd"]].to_string(index=False))

if not daily_df.empty:
    print(f"\n--- Daily PnL ---")
    print(daily_df.to_string(index=False))

print("="*70)

14:02:07  INFO  Loading data...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

... (9 outputs left out)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

14:03:59  INFO  preprocess: 881382 rows after regularization
14:03:59  INFO  calculate_features: building CORE31 ...
14:04:22  INFO  build_core31_from_regularized: built 881382 rows, 31 CORE31 features
14:04:23  INFO  calculate_features: building book geometry (ADDED6 + CONTROLS9) ...
14:04:28  INFO  _build_book_geometry_features: built 881382 rows
14:04:28  INFO  calculate_features: building shock recovery (CHALLENGERS 1-2) ...
14:04:30  INFO  _build_shock_recovery_features: bid_shocks=22175 ask_shocks=42052
14:04:31  INFO  _build_shock_recovery_features: built 881382 rows
14:04:31  INFO  calculate_features: building update clustering (CHALLENGER 3) ...
14:04:32  INFO  _build_update_clustering_features: obs=472806 book_change=326695 bid=149451 ask=233683
14:04:34  INFO  _build_update_clustering_features: built 881382 rows
14:04:34  INFO  calculate_features: building liquidity migration (CHALLENGER 4) ...
14:04:35  INFO  _build_liquidity_migration_features: built 881382 rows
14:04:42  


STRATEGY BACKTEST v8 — C_TP / C_SL CALIBRATION

Fixed: thresh=0.6  State4_TP_override={4: 0.6}

--- C_TP sweep (c_sl=1.2 fixed) ---
  c_tp     n    total   sharpe    hit%  pct_tp  avg_hold   max_dd
  0.40    67     0.18    0.043   56.7%   85.1%      21.1    -6.28
  0.50    63     3.74    0.850   65.1%   82.5%      24.6    -5.15
  0.60    62     4.72    0.981   77.4%   75.8%      28.6    -6.14
  0.70    58     9.50    1.815   75.9%   74.1%      31.6    -5.43
  0.80    57    10.77    2.058   75.4%   71.9%      34.9    -4.60
  1.00    54    11.66    2.088   68.5%   53.7%      44.4    -5.19
  1.20    52     9.88    1.682   67.3%   42.3%      49.2    -5.04
  1.50    51    10.73    1.612   64.7%   29.4%      56.9    -5.46

--- C_SL sweep (c_tp=1.0 fixed) ---
  c_sl     n    total   sharpe    hit%  pct_tp  avg_hold   max_dd
  0.80    56    10.49    1.737   67.9%   53.6%      41.2    -5.06
  1.00    54    10.52    1.803   68.5%   53.7%      43.1    -5.49
  1.20    54    11.66    2.088   68.5%

In [16]:
"""
STRATEGY BACKTEST v9
====================
v8 findings:
  - Best: thresh=0.60, c_sl=1.2, c_tp=1.0 → Sharpe=2.09, hit=68.5%, total=11.66 bps
  - 53.7% TP hit rate — 46.3% still timing out (25/54 trades)
  - State 4: 0 trades (too quiet to reach thresh=0.60)
  - SL barely fires: only 2 SL hits in 54 trades → SL almost irrelevant
  - Performance: Oct 23-Nov 4 choppy, Nov 5-9 strong run
  - Only 11 test days — need to understand regime dependence

ANALYSIS GOALS:
  1. MAE/MFE per exit type — do timeouts have positive drift or are they flat?
     If timeout mean_ret > 0: good, they're contributing alpha
     If timeout mean_ret ≈ 0: neutral, we can tighten time window
     If timeout mean_ret < 0: bad, need early exit rule

  2. State 4 investigation — why 0 trades?
     Check: how often is state=4 during in_window, and how often does pred > 0.60?

  3. Val set performance — does the strategy work on val too, or only test?
     Training period tail-end effect check.

  4. Intraday timing — which hours generate most TP hits vs timeouts?
     If 10:30-12:00 is better than 13:00-14:30, narrow the window.

  5. Timeout early exit: add trailing hold exit
     If price reverses BELOW entry during hold, exit immediately (don't wait for SL/max_hold)
     This converts flat timeouts into small losses we can avoid.

FINAL CONFIGURATION (from v8 calibration):
  thresh=0.60, c_sl=1.2, c_tp=1.0
  State 4 TP override: 0.6× (unchanged)
  State max_hold: {1:60, 2:30, 3:90, 4:180}

NEW IN v9:
  - MAE/MFE table per exit type
  - State 4 signal frequency check
  - Val set backtest (sanity check)
  - Hourly breakdown of TP/timeout rates
  - Trailing hold exit: if price < entry_mid at any point AND hold > 10s → exit
    (protects against slow bleeds that timeout at a loss)
"""

import sys
sys.path.insert(0, r"C:\projects\hedge_fund")
import json, logging, pathlib, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import lightgbm as lgb
from scipy.stats import spearmanr

from data.store.duckdb_client import DuckDBClient
from data.store.s3_client import S3Client
from data.processing.usdinr.features import USDINRFeatureBuilder
from data.processing.usdinr.constants import CHALLENGER50
from data.processing.usdinr.regimes import add_soft_hmm_features, save_hmm_model

logging.basicConfig(level=logging.INFO,
                    format="%(asctime)s  %(levelname)s  %(message)s",
                    datefmt="%H:%M:%S")
log = logging.getLogger(__name__)

OUT_DIR   = pathlib.Path(r"C:\projects\hedge_fund\results\strategy_backtest_v9")
MODEL_DIR = pathlib.Path(r"C:\projects\hedge_fund\models")
OUT_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_DAYS = 15
VAL_DAYS   = 5
COST_BPS   = 1.0

IST_OFFSET   = 19800
OPEN_FILTER  = 10 * 3600 + 30 * 60
CLOSE_FILTER = 14 * 3600 + 30 * 60

# Best params from v8
BEST_THRESH = 0.60
C_SL        = 1.2
C_TP        = 1.0
STATE_VOL      = {1: 1.77, 2: 2.37, 3: 1.61, 4: 1.03}
STATE_MAX_HOLD = {1: 60,   2: 30,   3: 90,   4: 180}
STATE_SIZE     = {1: 0.59, 2: 0.44, 3: 0.65, 4: 1.00}
STATE_TP_OVERRIDE = {4: 0.6}

# Trailing exit: if price drops below entry after min_hold seconds, exit immediately
# Set to None to disable
TRAILING_EXIT_MIN_HOLD = 10   # seconds before trailing exit kicks in

LGB_PARAMS = {
    "objective":        "regression",
    "metric":           "rmse",
    "verbosity":        -1,
    "boosting_type":    "gbdt",
    "num_leaves":       31,
    "learning_rate":    0.05,
    "feature_fraction": 0.8,
    "bagging_fraction": 0.8,
    "bagging_freq":     5,
    "min_child_samples":50,
}

# ---------------------------------------------------------------------------
# Load
# ---------------------------------------------------------------------------
log.info("Loading data...")
db = DuckDBClient(); s = S3Client(); all_frames = []
for contract in ["26SEPFUT","26OCTFUT","26NOVFUT"]:
    files = s.list_files(f"processed/1s/USDINR{contract}/")
    if not files: continue
    for f in files:
        parts = f.split("/")
        sym, date = parts[2], parts[3].replace(".parquet","")
        try:
            d = db.read_parquet(f"processed/1s/{sym}/{date}.parquet")
            d["symbol"] = sym; all_frames.append(d)
        except: pass
db.close()

df_raw  = pd.concat(all_frames, ignore_index=True)
builder = USDINRFeatureBuilder(data=df_raw, auto_build=True)
df      = builder.dataset.copy()
df["tick_count"] = df.get("tick_count", pd.Series(0, index=df.index)).fillna(0).astype(int)
df = df.sort_values(["symbol","date","ts_sec"]).reset_index(drop=True)

all_dates   = sorted(df["date"].unique())
train_dates = all_dates[:TRAIN_DAYS]
val_dates   = all_dates[TRAIN_DAYS:TRAIN_DAYS+VAL_DAYS]
test_dates  = all_dates[TRAIN_DAYS+VAL_DAYS:]
train_end   = str(train_dates[-1])

mask_train = df["date"].isin(train_dates)
mask_val   = df["date"].isin(val_dates)
mask_test  = df["date"].isin(test_dates)

df, hmm = add_soft_hmm_features(df=df, train_end_date=train_end,
                                 K=4, restarts=3, max_iter=70, tol=1e-6, seed=42)

if "mid" not in df.columns:
    df["mid"] = (df["bid_p1"] + df["ask_p1"]) / 2.0
df["log_mid"] = np.log(df["mid"])
g = df.groupby(["symbol","date"], sort=False)
df["ret_60s"] = (g["log_mid"].shift(-60) - df["log_mid"]) * 1e4
df["ret_60s_norm"] = df.groupby(["symbol","date"])["ret_60s"].transform(
    lambda x: (x - x.mean()) / (x.std() + 1e-8)
)
df["ist_tod"]   = (df["ts_sec"] + IST_OFFSET) % 86400
df["in_window"] = (df["ist_tod"] >= OPEN_FILTER) & (df["ist_tod"] <= CLOSE_FILTER)
df["hour_ist"]  = df["ist_tod"] // 3600

log.info("Building mid lookup...")
mid_lookup = df.set_index(["symbol","date","ts_sec"])["mid"].to_dict()

# ---------------------------------------------------------------------------
# Train
# ---------------------------------------------------------------------------
features   = CHALLENGER50
target     = "ret_60s_norm"
valid_mask = df[features + [target,"ret_60s"]].notna().all(axis=1)

tr = df[mask_train & valid_mask]
va = df[mask_val   & valid_mask]

ds_tr = lgb.Dataset(tr[features].values, label=tr[target].values, feature_name=features)
ds_va = lgb.Dataset(va[features].values, label=va[target].values,
                    feature_name=features, reference=ds_tr)

model = lgb.train(
    LGB_PARAMS, ds_tr, num_boost_round=1000, valid_sets=[ds_va],
    callbacks=[lgb.early_stopping(50,verbose=False), lgb.log_evaluation(100)],
)

eval_mask = (mask_val | mask_test) & valid_mask & df["ret_60s"].notna()
df_eval   = df[eval_mask].copy().reset_index(drop=True)
df_eval["state"]     = df_eval["state_filtered"].astype(int)
df_eval["pred_norm"] = model.predict(df_eval[features].values)
df_eval["hour_ist"]  = df_eval["ist_tod"] // 3600

# ---------------------------------------------------------------------------
# ANALYSIS 1: State 4 signal frequency check
# ---------------------------------------------------------------------------
log.info("State 4 analysis...")
df_win = df_eval[df_eval["in_window"]].copy()
state4_df = df_win[df_win["state"]==4]
log.info("State 4 in-window bars: %d", len(state4_df))
log.info("State 4 pred_norm distribution: mean=%.4f  pct_above_thresh=%.1f%%",
         state4_df["pred_norm"].mean() if len(state4_df)>0 else 0,
         (state4_df["pred_norm"] > BEST_THRESH).mean()*100 if len(state4_df)>0 else 0)

# ---------------------------------------------------------------------------
# Simulator with MAE/MFE tracking + trailing exit
# ---------------------------------------------------------------------------
def simulate_long_only_v9(
    df_day: pd.DataFrame,
    long_thresh: float,
    c_sl: float,
    c_tp: float,
    state_tp_override: dict = None,
    trailing_min_hold: int = None,  # None = disabled
) -> list:
    trades     = []
    busy_until = -1

    sym  = df_day["symbol"].iloc[0] if "symbol" in df_day.columns else None
    date = df_day["date"].iloc[0]
    rows = df_day.sort_values("ts_sec").reset_index(drop=True)
    if state_tp_override is None:
        state_tp_override = {}

    for _, row in rows.iterrows():
        ts = int(row["ts_sec"])

        if ts <= busy_until:   continue
        if not row["in_window"]: continue
        if float(row["pred_norm"]) < long_thresh: continue
        if pd.isna(row["ret_60s"]): continue

        state    = int(row["state"])
        vol      = STATE_VOL.get(state, 1.77)
        sl_dist  = c_sl * vol
        tp_mult  = state_tp_override.get(state, c_tp)
        tp_dist  = tp_mult * vol
        max_hold = STATE_MAX_HOLD.get(state, 60)
        size     = STATE_SIZE.get(state, 1.0)

        entry_mid = float(row["mid"])
        sl_price  = entry_mid * (1 - sl_dist * 1e-4)
        tp_price  = entry_mid * (1 + tp_dist * 1e-4)

        exit_mid  = None
        exit_ts   = ts + max_hold
        exit_type = "timeout"
        mae_bps   = 0.0   # max adverse excursion (most negative intra-trade move)
        mfe_bps   = 0.0   # max favorable excursion (most positive intra-trade move)

        for dt in range(1, max_hold + 1):
            mid_t = mid_lookup.get((sym, date, ts + dt))
            if mid_t is None: continue
            mid_t  = float(mid_t)
            ret_t  = (np.log(mid_t) - np.log(entry_mid)) * 1e4

            # Track MAE/MFE
            if ret_t < mae_bps: mae_bps = ret_t
            if ret_t > mfe_bps: mfe_bps = ret_t

            if mid_t >= tp_price:
                exit_mid  = mid_t
                exit_ts   = ts + dt
                exit_type = "tp"
                break
            elif mid_t <= sl_price:
                exit_mid  = mid_t
                exit_ts   = ts + dt
                exit_type = "sl"
                break
            elif (trailing_min_hold is not None
                  and dt >= trailing_min_hold
                  and mid_t < entry_mid):
                # Price is below entry after min_hold — exit to avoid slow bleed
                exit_mid  = mid_t
                exit_ts   = ts + dt
                exit_type = "trail"
                break

        if exit_mid is None:
            exit_mid = mid_lookup.get((sym, date, ts + max_hold))
            if exit_mid is None:
                exit_mid = entry_mid * (1 + float(row["ret_60s"]) * 1e-4)

        exit_mid   = float(exit_mid)
        actual_ret = (np.log(exit_mid) - np.log(entry_mid)) * 1e4
        net_pnl    = (actual_ret - COST_BPS) * size

        trades.append({
            "ts_sec":     ts,
            "exit_ts":    exit_ts,
            "hold_sec":   exit_ts - ts,
            "pred_norm":  float(row["pred_norm"]),
            "entry_mid":  entry_mid,
            "exit_mid":   exit_mid,
            "actual_ret": actual_ret,
            "exit_type":  exit_type,
            "state":      state,
            "size":       size,
            "sl_dist":    sl_dist,
            "tp_dist":    tp_dist,
            "net_pnl_bps":net_pnl,
            "mae_bps":    mae_bps,
            "mfe_bps":    mfe_bps,
            "hour_ist":   int(row.get("hour_ist", 0)),
        })
        busy_until = exit_ts

    return trades


def run_sim(df_set, long_thresh, c_sl, c_tp, state_tp_override=None, trailing=None):
    all_t = []
    for day, grp in df_set.groupby("date"):
        t = simulate_long_only_v9(grp, long_thresh, c_sl, c_tp, state_tp_override, trailing)
        for tr in t:
            tr["date"] = str(day)
        all_t.extend(t)
    return pd.DataFrame(all_t) if all_t else pd.DataFrame()


# ---------------------------------------------------------------------------
# TEST SET — standard (no trailing exit)
# ---------------------------------------------------------------------------
log.info("Test set standard sim...")
test_days = df_eval[df_eval["date"].isin(test_dates)].copy()
val_days  = df_eval[df_eval["date"].isin(val_dates)].copy()

tdf_test = run_sim(test_days, BEST_THRESH, C_SL, C_TP, STATE_TP_OVERRIDE, trailing=None)
tdf_val  = run_sim(val_days,  BEST_THRESH, C_SL, C_TP, STATE_TP_OVERRIDE, trailing=None)

# TEST with trailing exit enabled
log.info("Test set with trailing exit...")
tdf_trail = run_sim(test_days, BEST_THRESH, C_SL, C_TP, STATE_TP_OVERRIDE,
                    trailing=TRAILING_EXIT_MIN_HOLD)

# ---------------------------------------------------------------------------
# ANALYSIS 2: MAE/MFE per exit type
# ---------------------------------------------------------------------------
def mae_mfe_table(tdf, label):
    rows = []
    for etype in ["tp","sl","timeout","trail"]:
        s = tdf[tdf["exit_type"]==etype]
        if len(s)==0: continue
        rows.append({
            "exit_type": etype,
            "n": len(s),
            "mean_ret":   round(float(s["actual_ret"].mean()), 4),
            "mean_net":   round(float(s["net_pnl_bps"].mean()), 4),
            "total_net":  round(float(s["net_pnl_bps"].sum()), 2),
            "hit_pct":    round(float((s["net_pnl_bps"]>0).mean()*100), 1),
            "mean_mae":   round(float(s["mae_bps"].mean()), 4),
            "mean_mfe":   round(float(s["mfe_bps"].mean()), 4),
            "mean_hold":  round(float(s["hold_sec"].mean()), 1),
        })
    return pd.DataFrame(rows)


# ---------------------------------------------------------------------------
# ANALYSIS 3: Hourly breakdown
# ---------------------------------------------------------------------------
def hourly_table(tdf):
    rows = []
    for hr in sorted(tdf["hour_ist"].unique()):
        s = tdf[tdf["hour_ist"]==hr]
        n_tp = int((s["exit_type"]=="tp").sum())
        rows.append({
            "hour_ist": hr,
            "n": len(s),
            "n_tp": n_tp,
            "pct_tp": round(100*n_tp/max(len(s),1),1),
            "mean_net": round(float(s["net_pnl_bps"].mean()),4),
            "total_net": round(float(s["net_pnl_bps"].sum()),2),
            "hit_pct": round(float((s["net_pnl_bps"]>0).mean()*100),1),
        })
    return pd.DataFrame(rows)


# ---------------------------------------------------------------------------
# ANALYSIS 4: Val set performance
# ---------------------------------------------------------------------------
def summary_line(tdf, label):
    if tdf.empty: return f"{label}: 0 trades"
    pnl = tdf["net_pnl_bps"].values
    n   = len(pnl)
    std = pnl.std() if pnl.std() > 0 else 1e-9
    s   = pnl.mean() / std * np.sqrt(n)
    n_tp = int((tdf["exit_type"]=="tp").sum())
    return (f"{label}: n={n}  total={pnl.sum():.2f} bps  sharpe={s:.3f}  "
            f"hit={np.mean(pnl>0)*100:.1f}%  pct_tp={100*n_tp/n:.1f}%  "
            f"avg_hold={tdf['hold_sec'].mean():.1f}s")


# ---------------------------------------------------------------------------
# State 4 deep dive
# ---------------------------------------------------------------------------
state4_win = df_win[df_win["state"]==4].copy() if len(df_win)>0 else pd.DataFrame()
s4_pred_dist = {}
if len(state4_win) > 0:
    s4_pred_dist = {
        "total_bars": len(state4_win),
        "pct_above_thresh": round(float((state4_win["pred_norm"] > BEST_THRESH).mean()*100), 2),
        "mean_pred": round(float(state4_win["pred_norm"].mean()), 4),
        "p90_pred":  round(float(state4_win["pred_norm"].quantile(0.90)), 4),
        "p95_pred":  round(float(state4_win["pred_norm"].quantile(0.95)), 4),
        "max_pred":  round(float(state4_win["pred_norm"].max()), 4),
    }

# Save
if not tdf_test.empty:  tdf_test.to_csv(OUT_DIR/"trade_log_standard.csv",  index=False)
if not tdf_trail.empty: tdf_trail.to_csv(OUT_DIR/"trade_log_trailing.csv", index=False)
with open(OUT_DIR/"state4_analysis.json","w") as f:
    json.dump(s4_pred_dist, f, indent=2)

# ---------------------------------------------------------------------------
# Print
# ---------------------------------------------------------------------------
print("\n"+"="*70)
print("STRATEGY BACKTEST v9 — ANALYSIS + TRAILING EXIT")
print("="*70)

print(f"\nConfig: thresh={BEST_THRESH}  c_sl={C_SL}  c_tp={C_TP}")

print(f"\n--- Summary ---")
print(summary_line(tdf_test,  "Test  (standard)"))
print(summary_line(tdf_trail, "Test  (trailing) "))
print(summary_line(tdf_val,   "Val   (standard)"))

if not tdf_test.empty:
    print(f"\n--- MAE/MFE by Exit Type (standard, test) ---")
    mf = mae_mfe_table(tdf_test, "test_std")
    print(f"{'type':>8} {'n':>5} {'mean_ret':>10} {'mean_net':>10} {'total':>8} "
          f"{'hit%':>7} {'mean_mae':>10} {'mean_mfe':>10} {'hold':>7}")
    for _, r in mf.iterrows():
        print(f"{r['exit_type']:>8} {int(r['n']):>5} {r['mean_ret']:>10.4f} "
              f"{r['mean_net']:>10.4f} {r['total_net']:>8.2f} "
              f"{r['hit_pct']:>6.1f}% {r['mean_mae']:>10.4f} "
              f"{r['mean_mfe']:>10.4f} {r['mean_hold']:>7.1f}")

if not tdf_trail.empty:
    print(f"\n--- MAE/MFE by Exit Type (trailing exit, test) ---")
    mf2 = mae_mfe_table(tdf_trail, "test_trail")
    print(f"{'type':>8} {'n':>5} {'mean_ret':>10} {'mean_net':>10} {'total':>8} "
          f"{'hit%':>7} {'mean_mae':>10} {'mean_mfe':>10} {'hold':>7}")
    for _, r in mf2.iterrows():
        print(f"{r['exit_type']:>8} {int(r['n']):>5} {r['mean_ret']:>10.4f} "
              f"{r['mean_net']:>10.4f} {r['total_net']:>8.2f} "
              f"{r['hit_pct']:>6.1f}% {r['mean_mae']:>10.4f} "
              f"{r['mean_mfe']:>10.4f} {r['mean_hold']:>7.1f}")

if not tdf_test.empty:
    print(f"\n--- Hourly Breakdown (standard, test) ---")
    ht = hourly_table(tdf_test)
    print(f"{'hour_ist':>9} {'n':>5} {'n_tp':>6} {'pct_tp':>7} "
          f"{'mean_net':>10} {'total':>8} {'hit%':>7}")
    for _, r in ht.iterrows():
        print(f"{int(r['hour_ist']):>9} {int(r['n']):>5} {int(r['n_tp']):>6} "
              f"{r['pct_tp']:>6.1f}% {r['mean_net']:>10.4f} "
              f"{r['total_net']:>8.2f} {r['hit_pct']:>6.1f}%")

print(f"\n--- State 4 Signal Analysis ---")
if s4_pred_dist:
    for k, v in s4_pred_dist.items():
        print(f"  {k}: {v}")
else:
    print("  No State 4 bars in window")

if not tdf_val.empty:
    print(f"\n--- Val Daily PnL ---")
    val_daily = []
    for day, grp in tdf_val.groupby("date"):
        pnl = grp["net_pnl_bps"]
        n_tp = int((grp["exit_type"]=="tp").sum())
        val_daily.append({"date":day,"n":len(pnl),
                           "total":round(float(pnl.sum()),2),
                           "hit":round(float((pnl>0).mean()),3),
                           "n_tp":n_tp})
    print(pd.DataFrame(val_daily).to_string(index=False))

print("="*70)

14:12:19  INFO  Loading data...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

... (6 outputs left out)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

14:14:09  INFO  preprocess: 881382 rows after regularization
14:14:09  INFO  calculate_features: building CORE31 ...
14:14:32  INFO  build_core31_from_regularized: built 881382 rows, 31 CORE31 features
14:14:33  INFO  calculate_features: building book geometry (ADDED6 + CONTROLS9) ...
14:14:38  INFO  _build_book_geometry_features: built 881382 rows
14:14:39  INFO  calculate_features: building shock recovery (CHALLENGERS 1-2) ...
14:14:40  INFO  _build_shock_recovery_features: bid_shocks=22175 ask_shocks=42052
14:14:41  INFO  _build_shock_recovery_features: built 881382 rows
14:14:41  INFO  calculate_features: building update clustering (CHALLENGER 3) ...
14:14:43  INFO  _build_update_clustering_features: obs=472806 book_change=326695 bid=149451 ask=233683
14:14:44  INFO  _build_update_clustering_features: built 881382 rows
14:14:44  INFO  calculate_features: building liquidity migration (CHALLENGER 4) ...
14:14:46  INFO  _build_liquidity_migration_features: built 881382 rows
14:14:54  


STRATEGY BACKTEST v9 — ANALYSIS + TRAILING EXIT

Config: thresh=0.6  c_sl=1.2  c_tp=1.0

--- Summary ---
Test  (standard): n=54  total=11.66 bps  sharpe=2.088  hit=68.5%  pct_tp=53.7%  avg_hold=44.4s
Test  (trailing) : n=62  total=4.55 bps  sharpe=0.733  hit=58.1%  pct_tp=43.5%  avg_hold=32.8s
Val   (standard): n=17  total=1.53 bps  sharpe=0.455  hit=52.9%  pct_tp=35.3%  avg_hold=36.4s

--- MAE/MFE by Exit Type (standard, test) ---
    type     n   mean_ret   mean_net    total    hit%   mean_mae   mean_mfe    hold
      tp    29     2.2942     0.7537    21.86  100.0%    -0.1827     2.2942    28.1
      sl     2    -2.0964    -1.9160    -3.83    0.0%    -2.0964     0.1312    40.5
 timeout    23     0.5224    -0.2768    -6.37   34.8%    -0.3786     1.1132    65.2

--- MAE/MFE by Exit Type (trailing exit, test) ---
    type     n   mean_ret   mean_net    total    hit%   mean_mae   mean_mfe    hold
      tp    27     2.3909     0.8129    21.95  100.0%    -0.0147     2.3909    23.9
 timeou

In [17]:
"""
STRATEGY BACKTEST v10
=====================
v9 findings:
  - Trailing exit: DISABLED — kills Sharpe 2.09→0.73, trail exits avg -0.89 bps each
  - Timeout trades: mean_ret=+0.52 bps but cost=1.0 bps → net -0.28 bps
    These are real signal but below cost — keep filtering higher to cut them
  - Hourly:
    Hour 11: pct_tp=61.5%, net=+0.38 bps/trade ✓
    Hour 12: pct_tp=61.9%, net=+0.34 bps/trade ✓
    Hour 13: pct_tp=36.4%, net=-0.07 bps/trade ✗
    Hour 14: pct_tp=50.0%, net=+0.11 bps/trade  marginal
    → NARROW WINDOW: 11:00-13:00 IST only
  - State 4: only 0.05% of bars above thresh=0.60 → lower to 0.30
    p90 pred=0.27, p95=0.34 — model almost never predicts strongly in quiet state
  - Val set: Sharpe=0.45, positive but weaker (earlier data, less maturity)

CHANGES IN v10:
  1. Time window: 10:30-14:30 → 11:00-13:00 IST
  2. State 4 threshold: 0.60 → 0.30 (let quiet-state signals through)
  3. Timeout cost analysis: also test cost=0.7 bps (tighter spread assumption)
     to understand true alpha vs cost sensitivity
  4. Val + test combined view to see full out-of-sample picture
  5. Per-hour per-state breakdown to find best combinations

HYPOTHESIS:
  Narrowing to 11-13 IST removes the 13:xx drag and focuses on the
  liquidity window where USDINR futures are most orderly.
  State 4 lower threshold adds ~12 trades/test from quiet-state signals.
  Expected improvement: Sharpe 2.09 → 2.5-3.0 range.
"""

import sys
sys.path.insert(0, r"C:\projects\hedge_fund")
import json, logging, pathlib, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import lightgbm as lgb
from scipy.stats import spearmanr

from data.store.duckdb_client import DuckDBClient
from data.store.s3_client import S3Client
from data.processing.usdinr.features import USDINRFeatureBuilder
from data.processing.usdinr.constants import CHALLENGER50
from data.processing.usdinr.regimes import add_soft_hmm_features, save_hmm_model

logging.basicConfig(level=logging.INFO,
                    format="%(asctime)s  %(levelname)s  %(message)s",
                    datefmt="%H:%M:%S")
log = logging.getLogger(__name__)

OUT_DIR   = pathlib.Path(r"C:\projects\hedge_fund\results\strategy_backtest_v10")
MODEL_DIR = pathlib.Path(r"C:\projects\hedge_fund\models")
OUT_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_DAYS = 15
VAL_DAYS   = 5
COST_BPS   = 1.0

IST_OFFSET = 19800

# v10: narrowed window 11:00-13:00 IST
OPEN_FILTER_V10  = 11 * 3600
CLOSE_FILTER_V10 = 13 * 3600

# v9 window for comparison
OPEN_FILTER_V9  = 10 * 3600 + 30 * 60
CLOSE_FILTER_V9 = 14 * 3600 + 30 * 60

# Best params from v8/v9
BASE_THRESH = 0.60
C_SL        = 1.2
C_TP        = 1.0

STATE_VOL      = {1: 1.77, 2: 2.37, 3: 1.61, 4: 1.03}
STATE_MAX_HOLD = {1: 60,   2: 30,   3: 90,   4: 180}
STATE_SIZE     = {1: 0.59, 2: 0.44, 3: 0.65, 4: 1.00}

# v10: State 4 gets lower threshold (barely signals above 0.60)
STATE_THRESH_OVERRIDE = {4: 0.30}
STATE_TP_OVERRIDE     = {4: 0.6}

LGB_PARAMS = {
    "objective":        "regression",
    "metric":           "rmse",
    "verbosity":        -1,
    "boosting_type":    "gbdt",
    "num_leaves":       31,
    "learning_rate":    0.05,
    "feature_fraction": 0.8,
    "bagging_fraction": 0.8,
    "bagging_freq":     5,
    "min_child_samples":50,
}

# ---------------------------------------------------------------------------
# Load
# ---------------------------------------------------------------------------
log.info("Loading data...")
db = DuckDBClient(); s = S3Client(); all_frames = []
for contract in ["26SEPFUT","26OCTFUT","26NOVFUT"]:
    files = s.list_files(f"processed/1s/USDINR{contract}/")
    if not files: continue
    for f in files:
        parts = f.split("/")
        sym, date = parts[2], parts[3].replace(".parquet","")
        try:
            d = db.read_parquet(f"processed/1s/{sym}/{date}.parquet")
            d["symbol"] = sym; all_frames.append(d)
        except: pass
db.close()

df_raw  = pd.concat(all_frames, ignore_index=True)
builder = USDINRFeatureBuilder(data=df_raw, auto_build=True)
df      = builder.dataset.copy()
df["tick_count"] = df.get("tick_count", pd.Series(0, index=df.index)).fillna(0).astype(int)
df = df.sort_values(["symbol","date","ts_sec"]).reset_index(drop=True)

all_dates   = sorted(df["date"].unique())
train_dates = all_dates[:TRAIN_DAYS]
val_dates   = all_dates[TRAIN_DAYS:TRAIN_DAYS+VAL_DAYS]
test_dates  = all_dates[TRAIN_DAYS+VAL_DAYS:]
train_end   = str(train_dates[-1])

mask_train = df["date"].isin(train_dates)
mask_val   = df["date"].isin(val_dates)
mask_test  = df["date"].isin(test_dates)

df, hmm = add_soft_hmm_features(df=df, train_end_date=train_end,
                                 K=4, restarts=3, max_iter=70, tol=1e-6, seed=42)

if "mid" not in df.columns:
    df["mid"] = (df["bid_p1"] + df["ask_p1"]) / 2.0
df["log_mid"] = np.log(df["mid"])
g = df.groupby(["symbol","date"], sort=False)
df["ret_60s"] = (g["log_mid"].shift(-60) - df["log_mid"]) * 1e4
df["ret_60s_norm"] = df.groupby(["symbol","date"])["ret_60s"].transform(
    lambda x: (x - x.mean()) / (x.std() + 1e-8)
)
df["ist_tod"]  = (df["ts_sec"] + IST_OFFSET) % 86400
df["hour_ist"] = df["ist_tod"] // 3600

# Both windows
df["in_window_v9"]  = ((df["ist_tod"] >= OPEN_FILTER_V9)  & (df["ist_tod"] <= CLOSE_FILTER_V9))
df["in_window_v10"] = ((df["ist_tod"] >= OPEN_FILTER_V10) & (df["ist_tod"] <= CLOSE_FILTER_V10))

log.info("Building mid lookup...")
mid_lookup = df.set_index(["symbol","date","ts_sec"])["mid"].to_dict()

# ---------------------------------------------------------------------------
# Train
# ---------------------------------------------------------------------------
features   = CHALLENGER50
target     = "ret_60s_norm"
valid_mask = df[features + [target,"ret_60s"]].notna().all(axis=1)

tr = df[mask_train & valid_mask]
va = df[mask_val   & valid_mask]

ds_tr = lgb.Dataset(tr[features].values, label=tr[target].values, feature_name=features)
ds_va = lgb.Dataset(va[features].values, label=va[target].values,
                    feature_name=features, reference=ds_tr)

model = lgb.train(
    LGB_PARAMS, ds_tr, num_boost_round=1000, valid_sets=[ds_va],
    callbacks=[lgb.early_stopping(50,verbose=False), lgb.log_evaluation(100)],
)
model.save_model(str(MODEL_DIR / "lgbm_v10.txt"))

eval_mask = (mask_val | mask_test) & valid_mask & df["ret_60s"].notna()
df_eval   = df[eval_mask].copy().reset_index(drop=True)
df_eval["state"]     = df_eval["state_filtered"].astype(int)
df_eval["pred_norm"] = model.predict(df_eval[features].values)

# ---------------------------------------------------------------------------
# Simulator (supports per-state threshold override + window column)
# ---------------------------------------------------------------------------
def simulate(
    df_day: pd.DataFrame,
    base_thresh: float,
    c_sl: float,
    c_tp: float,
    window_col: str = "in_window_v10",
    state_thresh_override: dict = None,
    state_tp_override: dict = None,
) -> list:
    trades     = []
    busy_until = -1

    sym  = df_day["symbol"].iloc[0] if "symbol" in df_day.columns else None
    date = df_day["date"].iloc[0]
    rows = df_day.sort_values("ts_sec").reset_index(drop=True)
    if state_thresh_override is None: state_thresh_override = {}
    if state_tp_override is None:     state_tp_override = {}

    for _, row in rows.iterrows():
        ts = int(row["ts_sec"])

        if ts <= busy_until:                  continue
        if not row.get(window_col, False):    continue
        if pd.isna(row["ret_60s"]):           continue

        state  = int(row["state"])
        thresh = state_thresh_override.get(state, base_thresh)
        pred   = float(row["pred_norm"])
        if pred < thresh:                     continue

        vol      = STATE_VOL.get(state, 1.77)
        sl_dist  = c_sl * vol
        tp_mult  = state_tp_override.get(state, c_tp)
        tp_dist  = tp_mult * vol
        max_hold = STATE_MAX_HOLD.get(state, 60)
        size     = STATE_SIZE.get(state, 1.0)

        entry_mid = float(row["mid"])
        sl_price  = entry_mid * (1 - sl_dist * 1e-4)
        tp_price  = entry_mid * (1 + tp_dist * 1e-4)

        exit_mid  = None
        exit_ts   = ts + max_hold
        exit_type = "timeout"
        mae_bps   = 0.0
        mfe_bps   = 0.0

        for dt in range(1, max_hold + 1):
            mid_t = mid_lookup.get((sym, date, ts + dt))
            if mid_t is None: continue
            mid_t = float(mid_t)
            ret_t = (np.log(mid_t) - np.log(entry_mid)) * 1e4
            if ret_t < mae_bps: mae_bps = ret_t
            if ret_t > mfe_bps: mfe_bps = ret_t
            if mid_t >= tp_price:
                exit_mid  = mid_t; exit_ts = ts + dt; exit_type = "tp"; break
            elif mid_t <= sl_price:
                exit_mid  = mid_t; exit_ts = ts + dt; exit_type = "sl"; break

        if exit_mid is None:
            exit_mid = mid_lookup.get((sym, date, ts + max_hold))
            if exit_mid is None:
                exit_mid = entry_mid * (1 + float(row["ret_60s"]) * 1e-4)

        exit_mid   = float(exit_mid)
        actual_ret = (np.log(exit_mid) - np.log(entry_mid)) * 1e4
        net_pnl    = (actual_ret - COST_BPS) * size

        trades.append({
            "ts_sec": ts, "exit_ts": exit_ts,
            "hold_sec": exit_ts - ts,
            "pred_norm": pred,
            "entry_mid": entry_mid, "exit_mid": exit_mid,
            "actual_ret": actual_ret,
            "exit_type": exit_type,
            "state": state, "size": size,
            "net_pnl_bps": net_pnl,
            "mae_bps": mae_bps, "mfe_bps": mfe_bps,
            "hour_ist": int(row.get("hour_ist", 0)),
        })
        busy_until = exit_ts

    return trades


def run_sim(df_set, **kwargs):
    all_t = []
    for day, grp in df_set.groupby("date"):
        t = simulate(grp, **kwargs)
        for tr in t:
            tr["date"] = str(day)
        all_t.extend(t)
    return pd.DataFrame(all_t) if all_t else pd.DataFrame()


def metrics(tdf):
    if tdf.empty: return {}
    pnl = tdf["net_pnl_bps"].values; n = len(pnl)
    std = pnl.std() if pnl.std() > 0 else 1e-9
    n_tp = int((tdf["exit_type"]=="tp").sum())
    n_sl = int((tdf["exit_type"]=="sl").sum())
    cum  = np.cumsum(pnl)
    return {
        "n": n,
        "total":    round(float(pnl.sum()), 2),
        "mean":     round(float(pnl.mean()), 5),
        "sharpe":   round(float(pnl.mean()/std*np.sqrt(n)), 3),
        "hit":      round(float(np.mean(pnl>0)), 4),
        "n_tp":     n_tp, "n_sl": n_sl,
        "pct_tp":   round(100*n_tp/n, 1),
        "avg_hold": round(float(tdf["hold_sec"].mean()), 1),
        "max_dd":   round(float((cum-np.maximum.accumulate(cum)).min()), 2),
    }


def summary(tdf, label):
    m = metrics(tdf)
    if not m: return f"{label}: 0 trades"
    return (f"{label}: n={m['n']}  total={m['total']} bps  sharpe={m['sharpe']}  "
            f"hit={m['hit']*100:.1f}%  pct_tp={m['pct_tp']}%  "
            f"avg_hold={m['avg_hold']}s  max_dd={m['max_dd']}")


# ---------------------------------------------------------------------------
# Run all variants
# ---------------------------------------------------------------------------
test_days = df_eval[df_eval["date"].isin(test_dates)].copy()
val_days  = df_eval[df_eval["date"].isin(val_dates)].copy()
both_days = df_eval.copy()

common = dict(base_thresh=BASE_THRESH, c_sl=C_SL, c_tp=C_TP,
              state_tp_override=STATE_TP_OVERRIDE)

# A: v9 config (wide window, no state4 override) — baseline
v9_test = run_sim(test_days, window_col="in_window_v9", **common)

# B: v10 config (narrow window 11-13, state4 thresh=0.30)
v10_test = run_sim(test_days, window_col="in_window_v10",
                   state_thresh_override=STATE_THRESH_OVERRIDE, **common)

# C: v10 on val set
v10_val  = run_sim(val_days,  window_col="in_window_v10",
                   state_thresh_override=STATE_THRESH_OVERRIDE, **common)

# D: v10 on combined val+test (full out-of-sample)
v10_both = run_sim(both_days, window_col="in_window_v10",
                   state_thresh_override=STATE_THRESH_OVERRIDE, **common)

# E: v10 with cost=0.7 bps (tighter spread sensitivity test)
v10_low_cost = run_sim(test_days, window_col="in_window_v10",
                        state_thresh_override=STATE_THRESH_OVERRIDE, **common)
if not v10_low_cost.empty:
    v10_low_cost["net_pnl_bps"] = (v10_low_cost["actual_ret"] - 0.7) * v10_low_cost["size"]

# ---------------------------------------------------------------------------
# MAE/MFE table
# ---------------------------------------------------------------------------
def mae_mfe(tdf):
    rows = []
    for etype in ["tp","sl","timeout"]:
        s = tdf[tdf["exit_type"]==etype]
        if len(s)==0: continue
        rows.append({
            "exit_type": etype, "n": len(s),
            "mean_ret":   round(float(s["actual_ret"].mean()), 4),
            "mean_net":   round(float(s["net_pnl_bps"].mean()), 4),
            "total_net":  round(float(s["net_pnl_bps"].sum()), 2),
            "hit%":       round(float((s["net_pnl_bps"]>0).mean()*100), 1),
            "mean_mae":   round(float(s["mae_bps"].mean()), 4),
            "mean_mfe":   round(float(s["mfe_bps"].mean()), 4),
            "mean_hold":  round(float(s["hold_sec"].mean()), 1),
        })
    return pd.DataFrame(rows)


# ---------------------------------------------------------------------------
# Hourly breakdown
# ---------------------------------------------------------------------------
def hourly(tdf):
    rows = []
    for hr in sorted(tdf["hour_ist"].unique()):
        s = tdf[tdf["hour_ist"]==hr]
        n_tp = int((s["exit_type"]=="tp").sum())
        rows.append({"hour_ist":hr, "n":len(s),
                     "n_tp":n_tp,
                     "pct_tp": round(100*n_tp/max(len(s),1),1),
                     "mean_net": round(float(s["net_pnl_bps"].mean()),4),
                     "total_net": round(float(s["net_pnl_bps"].sum()),2),
                     "hit%": round(float((s["net_pnl_bps"]>0).mean()*100),1)})
    return pd.DataFrame(rows)


# ---------------------------------------------------------------------------
# Per-state breakdown
# ---------------------------------------------------------------------------
def per_state(tdf):
    rows = []
    for st in [1,2,3,4]:
        s = tdf[tdf["state"]==st]
        if len(s)<2: continue
        pnl = s["net_pnl_bps"].values
        n_tp = int((s["exit_type"]=="tp").sum())
        std  = pnl.std() if pnl.std()>0 else 1e-9
        rows.append({"state":st, "n":len(s),
                     "pct_trades": round(len(s)/len(tdf)*100,1),
                     "n_tp":n_tp, "pct_tp": round(100*n_tp/len(s),1),
                     "total": round(float(pnl.sum()),2),
                     "sharpe": round(float(pnl.mean()/std*np.sqrt(len(s))),3),
                     "hit%": round(float(np.mean(pnl>0)*100),1),
                     "mean_net": round(float(pnl.mean()),4),
                     "avg_hold": round(float(s["hold_sec"].mean()),1)})
    return pd.DataFrame(rows)


# ---------------------------------------------------------------------------
# Daily PnL
# ---------------------------------------------------------------------------
def daily(tdf):
    rows = []
    for day, grp in tdf.groupby("date"):
        pnl = grp["net_pnl_bps"]
        n_tp = int((grp["exit_type"]=="tp").sum())
        rows.append({"date":day,"n":len(pnl),"total":round(float(pnl.sum()),2),
                     "hit":round(float((pnl>0).mean()),3),"n_tp":n_tp,
                     "n_sl":int((grp["exit_type"]=="sl").sum()),
                     "mean_pnl":round(float(pnl.mean()),4)})
    return pd.DataFrame(rows)


# Save
if not v10_test.empty:
    v10_test.to_csv(OUT_DIR/"trade_log_v10_test.csv", index=False)
if not v10_both.empty:
    v10_both.to_csv(OUT_DIR/"trade_log_v10_both.csv", index=False)

# ---------------------------------------------------------------------------
# Print
# ---------------------------------------------------------------------------
print("\n"+"="*70)
print("STRATEGY BACKTEST v10 — NARROW WINDOW + STATE4 THRESHOLD")
print("="*70)
print(f"\nWindow v9:  10:30-14:30 IST  |  v10: 11:00-13:00 IST")
print(f"State4 thresh: v9=0.60  |  v10=0.30")

print(f"\n--- Summary Comparison ---")
print(summary(v9_test,   "v9  (baseline, test)      "))
print(summary(v10_test,  "v10 (narrow+s4, test)     "))
print(summary(v10_val,   "v10 (narrow+s4, val)      "))
print(summary(v10_both,  "v10 (narrow+s4, val+test) "))
print(summary(v10_low_cost, "v10 (cost=0.7, test)   "))

if not v10_test.empty:
    print(f"\n--- MAE/MFE (v10, test) ---")
    mf = mae_mfe(v10_test)
    print(mf.to_string(index=False))

if not v10_test.empty:
    print(f"\n--- Hourly Breakdown (v10, test) ---")
    print(hourly(v10_test).to_string(index=False))

if not v10_test.empty:
    print(f"\n--- Per-State (v10, test) ---")
    print(per_state(v10_test).to_string(index=False))

if not v10_test.empty:
    print(f"\n--- Daily PnL (v10, test) ---")
    print(daily(v10_test).to_string(index=False))

if not v10_val.empty:
    print(f"\n--- Daily PnL (v10, val) ---")
    print(daily(v10_val).to_string(index=False))

if not v10_both.empty:
    print(f"\n--- Daily PnL (v10, val+test combined) ---")
    print(daily(v10_both).to_string(index=False))

print("="*70)

14:21:30  INFO  Loading data...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

... (10 outputs left out)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

14:23:23  INFO  preprocess: 881382 rows after regularization
14:23:23  INFO  calculate_features: building CORE31 ...
14:23:45  INFO  build_core31_from_regularized: built 881382 rows, 31 CORE31 features
14:23:46  INFO  calculate_features: building book geometry (ADDED6 + CONTROLS9) ...
14:23:51  INFO  _build_book_geometry_features: built 881382 rows
14:23:51  INFO  calculate_features: building shock recovery (CHALLENGERS 1-2) ...
14:23:53  INFO  _build_shock_recovery_features: bid_shocks=22175 ask_shocks=42052
14:23:54  INFO  _build_shock_recovery_features: built 881382 rows
14:23:54  INFO  calculate_features: building update clustering (CHALLENGER 3) ...
14:23:55  INFO  _build_update_clustering_features: obs=472806 book_change=326695 bid=149451 ask=233683
14:23:57  INFO  _build_update_clustering_features: built 881382 rows
14:23:57  INFO  calculate_features: building liquidity migration (CHALLENGER 4) ...
14:23:58  INFO  _build_liquidity_migration_features: built 881382 rows
14:24:05  


STRATEGY BACKTEST v10 — NARROW WINDOW + STATE4 THRESHOLD

Window v9:  10:30-14:30 IST  |  v10: 11:00-13:00 IST
State4 thresh: v9=0.60  |  v10=0.30

--- Summary Comparison ---
v9  (baseline, test)      : n=54  total=11.66 bps  sharpe=2.088  hit=68.5%  pct_tp=53.7%  avg_hold=44.4s  max_dd=-5.19
v10 (narrow+s4, test)     : n=120  total=-25.39 bps  sharpe=-2.468  hit=43.3%  pct_tp=75.0%  avg_hold=52.0s  max_dd=-37.05
v10 (narrow+s4, val)      : n=22  total=17.62 bps  sharpe=1.035  hit=59.1%  pct_tp=72.7%  avg_hold=61.8s  max_dd=-3.5
v10 (narrow+s4, val+test) : n=142  total=-7.77 bps  sharpe=-0.382  hit=45.8%  pct_tp=74.6%  avg_hold=53.5s  max_dd=-39.47
v10 (cost=0.7, test)   : n=120  total=6.67 bps  sharpe=0.661  hit=62.5%  pct_tp=75.0%  avg_hold=52.0s  max_dd=-13.04

--- MAE/MFE (v10, test) ---
exit_type  n  mean_ret  mean_net  total_net  hit%  mean_mae  mean_mfe  mean_hold
       tp 90    1.3048    0.1739      15.65  55.6   -0.1276    1.3048       36.6
       sl 10   -1.6596   -2.6596  

In [18]:
"""
STRATEGY BACKTEST v11
=====================
v10 diagnosis:
  - State 4 thresh=0.30: DISASTER — 90 trades, hit=34%, total=-36.43 bps
    State 4 at low threshold has NO edge. Revert to thresh=0.60 (which gives 0 trades,
    meaning the model never confidently signals in the quiet state — that's fine).
  - Window narrowing 11-13: marginal alone, but hidden by State 4 flood
  - States 1+2+3 at v10: n=30, total=+11.04 bps — the edge is intact in those states
  - v9 was already the best config: thresh=0.60 for all states, 10:30-14:30 window

WHAT WE HAVEN'T TRIED:
  The real remaining improvement is in the timeout trades:
    timeout: n=23, mean_ret=+0.52 bps, mean_net=-0.28 bps (after 1 bps cost)
    These ARE moving in our direction but not enough to clear cost.
    Solution: raise threshold to cut weak signals, not lower it.

  Also: the Oct 23-Nov 4 choppy stretch vs Nov 5-9 trending stretch.
  Is this purely regime? Or can we detect it and filter?

v11 PLAN:
  1. Revert to v9 config exactly (thresh=0.60, window 10:30-14:30, state4 unchanged)
  2. Threshold sweep 0.60-1.20 to see if higher threshold cuts timeouts
     and improves mean_net (even at cost of fewer trades)
  3. Pred_norm × State conditioning:
     Only trade when BOTH pred_norm > thresh AND state_signal is favorable
     State 2+3 have best per-state Sharpe — test state filter: skip state 1 and 4
  4. Add regime filter: only trade when 5-day rolling IC > 0 (trend-following on signal quality)
     This would have avoided the Oct 27 disaster (-9.90 bps day)
  5. Final: best config from sweeps 2+3

KEY INSIGHT from all versions:
  States 1,2,3: solid edge (Sharpe >0.7, hit >60%)
  State 4 (quiet): model has no edge below thresh=0.60 (p90 pred=0.27)
  The strategy IS States 1,2,3 only, in practice.
"""

import sys
sys.path.insert(0, r"C:\projects\hedge_fund")
import json, logging, pathlib, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import lightgbm as lgb
from scipy.stats import spearmanr

from data.store.duckdb_client import DuckDBClient
from data.store.s3_client import S3Client
from data.processing.usdinr.features import USDINRFeatureBuilder
from data.processing.usdinr.constants import CHALLENGER50
from data.processing.usdinr.regimes import add_soft_hmm_features, save_hmm_model

logging.basicConfig(level=logging.INFO,
                    format="%(asctime)s  %(levelname)s  %(message)s",
                    datefmt="%H:%M:%S")
log = logging.getLogger(__name__)

OUT_DIR   = pathlib.Path(r"C:\projects\hedge_fund\results\strategy_backtest_v11")
MODEL_DIR = pathlib.Path(r"C:\projects\hedge_fund\models")
OUT_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_DAYS = 15
VAL_DAYS   = 5
COST_BPS   = 1.0

IST_OFFSET   = 19800
OPEN_FILTER  = 10 * 3600 + 30 * 60
CLOSE_FILTER = 14 * 3600 + 30 * 60

# Reverted to v9 best config
BASE_THRESH    = 0.60
C_SL           = 1.2
C_TP           = 1.0
STATE_VOL      = {1: 1.77, 2: 2.37, 3: 1.61, 4: 1.03}
STATE_MAX_HOLD = {1: 60,   2: 30,   3: 90,   4: 180}
STATE_SIZE     = {1: 0.59, 2: 0.44, 3: 0.65, 4: 1.00}
STATE_TP_OVERRIDE = {4: 0.6}

# State filter options to test
# None = all states, {1,2,3} = skip state 4, {2,3} = only best states
STATE_FILTERS = [
    None,         # all states (v9 baseline)
    {1,2,3},      # skip state 4
    {2,3},        # only states with Sharpe > 1.0
    {3},          # only state 3 (best: Sharpe=1.64, hit=72%)
]

# Threshold sweep
THRESH_SWEEP = [0.60, 0.70, 0.80, 0.90, 1.00, 1.10, 1.20]

LGB_PARAMS = {
    "objective":        "regression",
    "metric":           "rmse",
    "verbosity":        -1,
    "boosting_type":    "gbdt",
    "num_leaves":       31,
    "learning_rate":    0.05,
    "feature_fraction": 0.8,
    "bagging_fraction": 0.8,
    "bagging_freq":     5,
    "min_child_samples":50,
}

# ---------------------------------------------------------------------------
# Load
# ---------------------------------------------------------------------------
log.info("Loading data...")
db = DuckDBClient(); s = S3Client(); all_frames = []
for contract in ["26SEPFUT","26OCTFUT","26NOVFUT"]:
    files = s.list_files(f"processed/1s/USDINR{contract}/")
    if not files: continue
    for f in files:
        parts = f.split("/")
        sym, date = parts[2], parts[3].replace(".parquet","")
        try:
            d = db.read_parquet(f"processed/1s/{sym}/{date}.parquet")
            d["symbol"] = sym; all_frames.append(d)
        except: pass
db.close()

df_raw  = pd.concat(all_frames, ignore_index=True)
builder = USDINRFeatureBuilder(data=df_raw, auto_build=True)
df      = builder.dataset.copy()
df["tick_count"] = df.get("tick_count", pd.Series(0, index=df.index)).fillna(0).astype(int)
df = df.sort_values(["symbol","date","ts_sec"]).reset_index(drop=True)

all_dates   = sorted(df["date"].unique())
train_dates = all_dates[:TRAIN_DAYS]
val_dates   = all_dates[TRAIN_DAYS:TRAIN_DAYS+VAL_DAYS]
test_dates  = all_dates[TRAIN_DAYS+VAL_DAYS:]
train_end   = str(train_dates[-1])

mask_train = df["date"].isin(train_dates)
mask_val   = df["date"].isin(val_dates)
mask_test  = df["date"].isin(test_dates)

df, hmm = add_soft_hmm_features(df=df, train_end_date=train_end,
                                 K=4, restarts=3, max_iter=70, tol=1e-6, seed=42)

if "mid" not in df.columns:
    df["mid"] = (df["bid_p1"] + df["ask_p1"]) / 2.0
df["log_mid"] = np.log(df["mid"])
g = df.groupby(["symbol","date"], sort=False)
df["ret_60s"] = (g["log_mid"].shift(-60) - df["log_mid"]) * 1e4
df["ret_60s_norm"] = df.groupby(["symbol","date"])["ret_60s"].transform(
    lambda x: (x - x.mean()) / (x.std() + 1e-8)
)
df["ist_tod"]   = (df["ts_sec"] + IST_OFFSET) % 86400
df["in_window"] = (df["ist_tod"] >= OPEN_FILTER) & (df["ist_tod"] <= CLOSE_FILTER)
df["hour_ist"]  = df["ist_tod"] // 3600

log.info("Building mid lookup...")
mid_lookup = df.set_index(["symbol","date","ts_sec"])["mid"].to_dict()

# ---------------------------------------------------------------------------
# Train
# ---------------------------------------------------------------------------
features   = CHALLENGER50
target     = "ret_60s_norm"
valid_mask = df[features + [target,"ret_60s"]].notna().all(axis=1)

tr = df[mask_train & valid_mask]
va = df[mask_val   & valid_mask]

ds_tr = lgb.Dataset(tr[features].values, label=tr[target].values, feature_name=features)
ds_va = lgb.Dataset(va[features].values, label=va[target].values,
                    feature_name=features, reference=ds_tr)

model = lgb.train(
    LGB_PARAMS, ds_tr, num_boost_round=1000, valid_sets=[ds_va],
    callbacks=[lgb.early_stopping(50,verbose=False), lgb.log_evaluation(100)],
)
model.save_model(str(MODEL_DIR / "lgbm_v11.txt"))

eval_mask = (mask_val | mask_test) & valid_mask & df["ret_60s"].notna()
df_eval   = df[eval_mask].copy().reset_index(drop=True)
df_eval["state"]     = df_eval["state_filtered"].astype(int)
df_eval["pred_norm"] = model.predict(df_eval[features].values)
df_eval["hour_ist"]  = df_eval["ist_tod"] // 3600

# ---------------------------------------------------------------------------
# Rolling IC — 5-day lookback, computed on eval set
# Used as regime filter: only trade when IC > 0
# ---------------------------------------------------------------------------
log.info("Computing rolling IC for regime filter...")
ic_by_date = {}
eval_dates = sorted(df_eval["date"].unique())

for i, d in enumerate(eval_dates):
    # Use last 5 available dates before this date
    past = [x for x in eval_dates[:i] if x < d][-5:]
    if len(past) < 3:
        ic_by_date[d] = 0.0
        continue
    sub = df_eval[df_eval["date"].isin(past)]
    if len(sub) < 50:
        ic_by_date[d] = 0.0
        continue
    ic, _ = spearmanr(sub["ret_60s"], sub["pred_norm"])
    ic_by_date[d] = float(ic) if not np.isnan(ic) else 0.0

df_eval["rolling_ic"] = df_eval["date"].map(ic_by_date)
log.info("Rolling IC by date: %s",
         {str(d): round(v,3) for d, v in list(ic_by_date.items())[-12:]})

# ---------------------------------------------------------------------------
# Simulator
# ---------------------------------------------------------------------------
def simulate(
    df_day: pd.DataFrame,
    base_thresh: float,
    c_sl: float,
    c_tp: float,
    allowed_states: set = None,  # None = all states
    state_tp_override: dict = None,
) -> list:
    trades     = []
    busy_until = -1

    sym  = df_day["symbol"].iloc[0] if "symbol" in df_day.columns else None
    date = df_day["date"].iloc[0]
    rows = df_day.sort_values("ts_sec").reset_index(drop=True)
    if state_tp_override is None: state_tp_override = {}

    for _, row in rows.iterrows():
        ts = int(row["ts_sec"])

        if ts <= busy_until:                  continue
        if not row["in_window"]:              continue
        if pd.isna(row["ret_60s"]):           continue

        state = int(row["state"])
        if allowed_states is not None and state not in allowed_states:
            continue

        pred = float(row["pred_norm"])
        if pred < base_thresh:                continue

        vol      = STATE_VOL.get(state, 1.77)
        sl_dist  = c_sl * vol
        tp_mult  = state_tp_override.get(state, c_tp)
        tp_dist  = tp_mult * vol
        max_hold = STATE_MAX_HOLD.get(state, 60)
        size     = STATE_SIZE.get(state, 1.0)

        entry_mid = float(row["mid"])
        sl_price  = entry_mid * (1 - sl_dist * 1e-4)
        tp_price  = entry_mid * (1 + tp_dist * 1e-4)

        exit_mid  = None
        exit_ts   = ts + max_hold
        exit_type = "timeout"
        mae_bps   = 0.0
        mfe_bps   = 0.0

        for dt in range(1, max_hold + 1):
            mid_t = mid_lookup.get((sym, date, ts + dt))
            if mid_t is None: continue
            mid_t = float(mid_t)
            ret_t = (np.log(mid_t) - np.log(entry_mid)) * 1e4
            if ret_t < mae_bps: mae_bps = ret_t
            if ret_t > mfe_bps: mfe_bps = ret_t
            if mid_t >= tp_price:
                exit_mid  = mid_t; exit_ts = ts + dt; exit_type = "tp"; break
            elif mid_t <= sl_price:
                exit_mid  = mid_t; exit_ts = ts + dt; exit_type = "sl"; break

        if exit_mid is None:
            exit_mid = mid_lookup.get((sym, date, ts + max_hold))
            if exit_mid is None:
                exit_mid = entry_mid * (1 + float(row["ret_60s"]) * 1e-4)

        exit_mid   = float(exit_mid)
        actual_ret = (np.log(exit_mid) - np.log(entry_mid)) * 1e4
        net_pnl    = (actual_ret - COST_BPS) * size

        trades.append({
            "ts_sec": ts, "exit_ts": exit_ts,
            "hold_sec": exit_ts - ts,
            "pred_norm": pred,
            "entry_mid": entry_mid, "exit_mid": exit_mid,
            "actual_ret": actual_ret, "exit_type": exit_type,
            "state": state, "size": size,
            "net_pnl_bps": net_pnl,
            "mae_bps": mae_bps, "mfe_bps": mfe_bps,
            "hour_ist": int(row.get("hour_ist", 0)),
            "rolling_ic": float(row.get("rolling_ic", 0)),
        })
        busy_until = exit_ts

    return trades


def run_sim(df_set, **kwargs):
    all_t = []
    for day, grp in df_set.groupby("date"):
        t = simulate(grp, **kwargs)
        for tr in t:
            tr["date"] = str(day)
        all_t.extend(t)
    return pd.DataFrame(all_t) if all_t else pd.DataFrame()


def metrics(tdf):
    if tdf.empty: return {}
    pnl = tdf["net_pnl_bps"].values; n = len(pnl)
    std = pnl.std() if pnl.std() > 0 else 1e-9
    n_tp = int((tdf["exit_type"]=="tp").sum())
    n_sl = int((tdf["exit_type"]=="sl").sum())
    cum  = np.cumsum(pnl)
    return {
        "n":       n,
        "total":   round(float(pnl.sum()), 2),
        "mean":    round(float(pnl.mean()), 5),
        "sharpe":  round(float(pnl.mean()/std*np.sqrt(n)), 3),
        "hit":     round(float(np.mean(pnl>0)), 4),
        "n_tp": n_tp, "n_sl": n_sl,
        "pct_tp":  round(100*n_tp/n, 1),
        "avg_hold":round(float(tdf["hold_sec"].mean()), 1),
        "max_dd":  round(float((cum-np.maximum.accumulate(cum)).min()), 2),
    }

def summary(tdf, label):
    m = metrics(tdf)
    if not m: return f"{label}: 0 trades"
    return (f"{label}: n={m['n']}  total={m['total']} bps  sharpe={m['sharpe']}  "
            f"hit={m['hit']*100:.1f}%  pct_tp={m['pct_tp']}%  "
            f"avg_hold={m['avg_hold']}s  max_dd={m['max_dd']}")

# ---------------------------------------------------------------------------
# TEST SET
# ---------------------------------------------------------------------------
test_days = df_eval[df_eval["date"].isin(test_dates)].copy()
val_days  = df_eval[df_eval["date"].isin(val_dates)].copy()

# ---------------------------------------------------------------------------
# SWEEP 1: Threshold sweep, all states (v9 baseline)
# ---------------------------------------------------------------------------
log.info("Threshold sweep...")
thresh_rows = []
for thresh in THRESH_SWEEP:
    tdf = run_sim(test_days, base_thresh=thresh, c_sl=C_SL, c_tp=C_TP,
                  allowed_states=None, state_tp_override=STATE_TP_OVERRIDE)
    m = metrics(tdf)
    if m: m["thresh"] = thresh; thresh_rows.append(m)
thresh_df = pd.DataFrame(thresh_rows)

# ---------------------------------------------------------------------------
# SWEEP 2: State filter sweep (at base thresh=0.60)
# ---------------------------------------------------------------------------
log.info("State filter sweep...")
state_rows_sweep = []
for sf in STATE_FILTERS:
    label = "all" if sf is None else str(sorted(sf))
    tdf = run_sim(test_days, base_thresh=BASE_THRESH, c_sl=C_SL, c_tp=C_TP,
                  allowed_states=sf, state_tp_override=STATE_TP_OVERRIDE)
    m = metrics(tdf)
    if m: m["states"] = label; state_rows_sweep.append(m)
state_sweep_df = pd.DataFrame(state_rows_sweep)

# ---------------------------------------------------------------------------
# SWEEP 3: Regime filter — only trade when rolling_ic > threshold
# ---------------------------------------------------------------------------
log.info("IC regime filter sweep...")
ic_rows = []
for ic_min in [None, 0.0, 0.05, 0.10, 0.15, 0.20]:
    if ic_min is None:
        sub = test_days
    else:
        sub = test_days[test_days["rolling_ic"] > ic_min].copy()
    tdf = run_sim(sub, base_thresh=BASE_THRESH, c_sl=C_SL, c_tp=C_TP,
                  allowed_states=None, state_tp_override=STATE_TP_OVERRIDE)
    m = metrics(tdf)
    if m:
        m["ic_filter"] = str(ic_min)
        m["days_traded"] = sub["date"].nunique()
        ic_rows.append(m)
ic_df = pd.DataFrame(ic_rows)

# ---------------------------------------------------------------------------
# BEST COMBINATION: find optimal thresh + state_filter
# ---------------------------------------------------------------------------
best_sharpe = -999
best_config = {}
best_tdf    = pd.DataFrame()

for thresh in [0.60, 0.70, 0.80, 0.90]:
    for sf in [None, {1,2,3}, {2,3}]:
        tdf = run_sim(test_days, base_thresh=thresh, c_sl=C_SL, c_tp=C_TP,
                      allowed_states=sf, state_tp_override=STATE_TP_OVERRIDE)
        m = metrics(tdf)
        if m and m["n"] >= 10 and m["sharpe"] > best_sharpe:
            best_sharpe = m["sharpe"]
            best_config = {"thresh": thresh, "states": sf}
            best_tdf    = tdf.copy()

log.info("Best config: %s sharpe=%.3f", best_config, best_sharpe)

# Per-state at best config
def per_state(tdf):
    rows = []
    for st in [1,2,3,4]:
        s = tdf[tdf["state"]==st]
        if len(s)<2: continue
        pnl = s["net_pnl_bps"].values
        n_tp = int((s["exit_type"]=="tp").sum())
        std  = pnl.std() if pnl.std()>0 else 1e-9
        rows.append({"state":st, "n":len(s),
                     "pct":round(len(s)/len(tdf)*100,1),
                     "n_tp":n_tp, "pct_tp":round(100*n_tp/len(s),1),
                     "total":round(float(pnl.sum()),2),
                     "sharpe":round(float(pnl.mean()/std*np.sqrt(len(s))),3),
                     "hit%":round(float(np.mean(pnl>0)*100),1),
                     "mean_net":round(float(pnl.mean()),4),
                     "avg_hold":round(float(s["hold_sec"].mean()),1)})
    return pd.DataFrame(rows)

def daily(tdf):
    rows = []
    for day, grp in tdf.groupby("date"):
        pnl = grp["net_pnl_bps"]
        n_tp = int((grp["exit_type"]=="tp").sum())
        rows.append({"date":day,"n":len(pnl),"total":round(float(pnl.sum()),2),
                     "hit":round(float((pnl>0).mean()),3),"n_tp":n_tp,
                     "n_sl":int((grp["exit_type"]=="sl").sum()),
                     "mean_pnl":round(float(pnl.mean()),4)})
    return pd.DataFrame(rows)

# Save
thresh_df.to_csv(OUT_DIR/"thresh_sweep.csv", index=False)
state_sweep_df.to_csv(OUT_DIR/"state_filter_sweep.csv", index=False)
ic_df.to_csv(OUT_DIR/"ic_filter_sweep.csv", index=False)
if not best_tdf.empty: best_tdf.to_csv(OUT_DIR/"trade_log_best.csv", index=False)

# ---------------------------------------------------------------------------
# Print
# ---------------------------------------------------------------------------
print("\n"+"="*70)
print("STRATEGY BACKTEST v11 — THRESHOLD + STATE FILTER + REGIME FILTER")
print("="*70)

print(f"\n--- Threshold Sweep (all states, test) ---")
print(f"{'thresh':>8} {'n':>5} {'total':>8} {'sharpe':>8} {'hit%':>7} "
      f"{'pct_tp':>7} {'avg_hold':>9} {'max_dd':>8}")
for _, r in thresh_df.iterrows():
    print(f"{r['thresh']:>8.2f} {int(r['n']):>5} {r['total']:>8.2f} {r['sharpe']:>8.3f} "
          f"{r['hit']*100:>6.1f}% {r['pct_tp']:>6.1f}% {r['avg_hold']:>9.1f} {r['max_dd']:>8.2f}")

print(f"\n--- State Filter Sweep (thresh=0.60, test) ---")
print(f"{'states':>12} {'n':>5} {'total':>8} {'sharpe':>8} {'hit%':>7} "
      f"{'pct_tp':>7} {'avg_hold':>9} {'max_dd':>8}")
for _, r in state_sweep_df.iterrows():
    print(f"{str(r['states']):>12} {int(r['n']):>5} {r['total']:>8.2f} {r['sharpe']:>8.3f} "
          f"{r['hit']*100:>6.1f}% {r['pct_tp']:>6.1f}% {r['avg_hold']:>9.1f} {r['max_dd']:>8.2f}")

print(f"\n--- Rolling IC Regime Filter (thresh=0.60, all states, test) ---")
print(f"{'ic_min':>8} {'days':>6} {'n':>5} {'total':>8} {'sharpe':>8} {'hit%':>7} {'max_dd':>8}")
for _, r in ic_df.iterrows():
    print(f"{str(r['ic_filter']):>8} {int(r.get('days_traded',0)):>6} {int(r['n']):>5} "
          f"{r['total']:>8.2f} {r['sharpe']:>8.3f} {r['hit']*100:>6.1f}% {r['max_dd']:>8.2f}")

print(f"\n--- Best Config: thresh={best_config.get('thresh')}  "
      f"states={best_config.get('states')} ---")
if not best_tdf.empty:
    print(summary(best_tdf, "Best"))
    print(f"\n  Per-State:")
    print(per_state(best_tdf).to_string(index=False))
    print(f"\n  Daily PnL:")
    print(daily(best_tdf).to_string(index=False))

print("="*70)

15:07:32  INFO  Loading data...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

... (6 outputs left out)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

15:09:27  INFO  preprocess: 881382 rows after regularization
15:09:27  INFO  calculate_features: building CORE31 ...
15:09:51  INFO  build_core31_from_regularized: built 881382 rows, 31 CORE31 features
15:09:52  INFO  calculate_features: building book geometry (ADDED6 + CONTROLS9) ...
15:09:59  INFO  _build_book_geometry_features: built 881382 rows
15:09:59  INFO  calculate_features: building shock recovery (CHALLENGERS 1-2) ...
15:10:01  INFO  _build_shock_recovery_features: bid_shocks=22175 ask_shocks=42052
15:10:01  INFO  _build_shock_recovery_features: built 881382 rows
15:10:02  INFO  calculate_features: building update clustering (CHALLENGER 3) ...
15:10:03  INFO  _build_update_clustering_features: obs=472806 book_change=326695 bid=149451 ask=233683
15:10:05  INFO  _build_update_clustering_features: built 881382 rows
15:10:05  INFO  calculate_features: building liquidity migration (CHALLENGER 4) ...
15:10:06  INFO  _build_liquidity_migration_features: built 881382 rows
15:10:15  


STRATEGY BACKTEST v11 — THRESHOLD + STATE FILTER + REGIME FILTER

--- Threshold Sweep (all states, test) ---
  thresh     n    total   sharpe    hit%  pct_tp  avg_hold   max_dd
    0.60    54    11.66    2.088   68.5%   53.7%      44.4    -5.19
    0.70    30     8.95    2.093   73.3%   60.0%      36.8    -5.19
    0.80    13     4.90    1.989   76.9%   53.8%      28.1    -1.50
    0.90     6     1.39    0.640   50.0%   50.0%      33.3    -1.41
    1.00     1    -0.15 -151185648.447    0.0%    0.0%      30.0     0.00

--- State Filter Sweep (thresh=0.60, test) ---
      states     n    total   sharpe    hit%  pct_tp  avg_hold   max_dd
         all    54    11.66    2.088   68.5%   53.7%      44.4    -5.19
   [1, 2, 3]    54    11.66    2.088   68.5%   53.7%      44.4    -5.19
      [2, 3]    52     6.35    1.132   65.4%   48.1%      43.6    -6.58
         [3]    41     6.63    1.224   70.7%   61.0%      48.7    -6.05

--- Rolling IC Regime Filter (thresh=0.60, all states, test) ---
  

In [19]:
"""
TIMEOUT ANALYSIS + DYNAMIC EXIT
=================================
Problem: timeout trades have mean_ret=+0.52 bps but net=-0.28 after 1 bps cost.
         MFE = 1.11 bps — price gets there but we exit too early at max_hold.

Two fixes to test:
  A) Dynamic hold: if pred_norm > HIGH_THRESH and price still above entry at t=max_hold,
     extend hold by EXTEND_SEC more seconds. Only extend once.

  B) Trailing TP: once price is UP by TRAIL_ACTIVATE bps, switch from fixed TP
     to a trailing stop of TRAIL_RETRACE bps below the running peak.
     Exit when price drops TRAIL_RETRACE below peak.
     This locks in gains rather than waiting for a fixed TP target.

  C) Combined: trailing TP active always, dynamic hold for strong signals.

We test all three and compare to baseline (fixed TP/SL/max_hold from v8).

Key question: does a strong pred_norm signal actually predict LONGER momentum?
We'll check: mean_mfe by pred_norm bucket to see if high-pred trades have higher MFE.
"""

import sys
sys.path.insert(0, r"C:\projects\hedge_fund")
import warnings; warnings.filterwarnings("ignore")
import logging, pathlib
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s",
                    datefmt="%H:%M:%S")
log = logging.getLogger(__name__)

import numpy as np
import pandas as pd
import lightgbm as lgb

from data.store.duckdb_client import DuckDBClient
from data.store.s3_client import S3Client
from data.processing.usdinr.features import USDINRFeatureBuilder
from data.processing.usdinr.constants import CHALLENGER50
from data.processing.usdinr.regimes import add_soft_hmm_features

OUT_DIR = pathlib.Path(r"C:\projects\hedge_fund\results\timeout_analysis")
OUT_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_DAYS   = 15
VAL_DAYS     = 5
COST_BPS     = 1.0
IST_OFFSET   = 19800
OPEN_FILTER  = 10 * 3600 + 30 * 60
CLOSE_FILTER = 14 * 3600 + 30 * 60

# Best config from v8
BASE_THRESH    = 0.60
C_SL           = 1.2
C_TP           = 1.0
STATE_VOL      = {1: 1.77, 2: 2.37, 3: 1.61, 4: 1.03}
STATE_MAX_HOLD = {1: 60,   2: 30,   3: 90,   4: 180}
STATE_SIZE     = {1: 0.59, 2: 0.44, 3: 0.65, 4: 1.00}
STATE_TP_OVERRIDE = {4: 0.6}

# Dynamic hold: if pred_norm > HIGH_THRESH, extend hold by EXTEND_SEC
HIGH_THRESH_OPTIONS = [0.8, 1.0, 1.2]
EXTEND_SEC_OPTIONS  = [30, 60, 90, 120]

# Trailing TP params
TRAIL_ACTIVATE_OPTIONS = [0.3, 0.5, 0.7]   # bps above entry to activate trailing
TRAIL_RETRACE_OPTIONS  = [0.2, 0.3, 0.4]   # bps retrace from peak to exit

LGB_PARAMS = {
    "objective":"regression","metric":"rmse","verbosity":-1,
    "num_leaves":31,"learning_rate":0.05,"feature_fraction":0.8,
    "bagging_fraction":0.8,"bagging_freq":5,"min_child_samples":50,
}

# ---------------------------------------------------------------------------
# Load
# ---------------------------------------------------------------------------
log.info("Loading...")
db = DuckDBClient(); s = S3Client(); all_frames = []
for contract in ["26SEPFUT","26OCTFUT","26NOVFUT"]:
    files = s.list_files(f"processed/1s/USDINR{contract}/")
    if not files: continue
    for f in files:
        parts = f.split("/")
        sym, date = parts[2], parts[3].replace(".parquet","")
        try:
            d = db.read_parquet(f"processed/1s/{sym}/{date}.parquet")
            d["symbol"] = sym; all_frames.append(d)
        except: pass
db.close()

df_raw  = pd.concat(all_frames, ignore_index=True)
builder = USDINRFeatureBuilder(data=df_raw, auto_build=True)
df      = builder.dataset.copy()
df      = df.sort_values(["symbol","date","ts_sec"]).reset_index(drop=True)

all_dates   = sorted(df["date"].unique())
train_dates = all_dates[:TRAIN_DAYS]
val_dates   = all_dates[TRAIN_DAYS:TRAIN_DAYS+VAL_DAYS]
test_dates  = all_dates[TRAIN_DAYS+VAL_DAYS:]
train_end   = str(train_dates[-1])

mask_train = df["date"].isin(train_dates)
mask_val   = df["date"].isin(val_dates)
mask_test  = df["date"].isin(test_dates)

df, hmm = add_soft_hmm_features(df=df, train_end_date=train_end,
                                 K=4, restarts=3, max_iter=70, tol=1e-6, seed=42)

if "mid" not in df.columns:
    df["mid"] = (df["bid_p1"] + df["ask_p1"]) / 2.0
df["log_mid"] = np.log(df["mid"])
g = df.groupby(["symbol","date"], sort=False)
df["ret_60s"]  = (g["log_mid"].shift(-60) - df["log_mid"]) * 1e4
df["ret_60s_norm"] = df.groupby(["symbol","date"])["ret_60s"].transform(
    lambda x: (x - x.mean()) / (x.std() + 1e-8)
)
df["ist_tod"]   = (df["ts_sec"] + IST_OFFSET) % 86400
df["in_window"] = (df["ist_tod"] >= OPEN_FILTER) & (df["ist_tod"] <= CLOSE_FILTER)

log.info("Mid lookup...")
mid_lookup = df.set_index(["symbol","date","ts_sec"])["mid"].to_dict()

# ---------------------------------------------------------------------------
# Train
# ---------------------------------------------------------------------------
features   = CHALLENGER50
target     = "ret_60s_norm"
valid_mask = df[features + [target,"ret_60s"]].notna().all(axis=1)
tr = df[mask_train & valid_mask]
va = df[mask_val   & valid_mask]
ds_tr = lgb.Dataset(tr[features].values, label=tr[target].values, feature_name=features)
ds_va = lgb.Dataset(va[features].values, label=va[target].values,
                    feature_name=features, reference=ds_tr)
model = lgb.train(LGB_PARAMS, ds_tr, num_boost_round=1000, valid_sets=[ds_va],
                  callbacks=[lgb.early_stopping(50,verbose=False), lgb.log_evaluation(100)])

eval_mask = (mask_val | mask_test) & valid_mask & df["ret_60s"].notna()
df_eval   = df[eval_mask].copy().reset_index(drop=True)
df_eval["state"]     = df_eval["state_filtered"].astype(int)
df_eval["pred_norm"] = model.predict(df_eval[features].values)

# ---------------------------------------------------------------------------
# ANALYSIS 1: MFE by pred_norm bucket
# Does stronger signal → higher MFE? (justifies dynamic hold for strong signals)
# ---------------------------------------------------------------------------
log.info("Computing MFE by pred_norm bucket...")

# For each signal bar above base threshold, compute MFE over 180s
te = df_eval[df_eval["date"].isin(test_dates)].copy()
te_win = te[te["in_window"] & (te["pred_norm"] > BASE_THRESH)].copy()

mfe_rows = []
for _, row in te_win.iterrows():
    sym  = row.get("symbol")
    date = row["date"]
    ts   = int(row["ts_sec"])
    entry_mid = float(row["mid"])
    pred      = float(row["pred_norm"])

    # Compute running MFE at 30, 60, 90, 120, 180 seconds
    mfe_30 = mfe_60 = mfe_90 = mfe_120 = mfe_180 = 0.0
    for dt in range(1, 181):
        mid_t = mid_lookup.get((sym, date, ts + dt))
        if mid_t is None: continue
        ret_t = (np.log(float(mid_t)) - np.log(entry_mid)) * 1e4
        if dt <= 30:  mfe_30  = max(mfe_30,  ret_t)
        if dt <= 60:  mfe_60  = max(mfe_60,  ret_t)
        if dt <= 90:  mfe_90  = max(mfe_90,  ret_t)
        if dt <= 120: mfe_120 = max(mfe_120, ret_t)
        mfe_180 = max(mfe_180, ret_t)

    mfe_rows.append({
        "pred_norm": pred,
        "state":     int(row["state"]),
        "mfe_30":    mfe_30,
        "mfe_60":    mfe_60,
        "mfe_90":    mfe_90,
        "mfe_120":   mfe_120,
        "mfe_180":   mfe_180,
        "ret_60s":   float(row["ret_60s"]),
    })

mfe_df = pd.DataFrame(mfe_rows)

# Bucket by pred_norm
bins   = [0.6, 0.7, 0.8, 0.9, 1.0, 1.2, 1.5, 999]
labels = ["[0.6,0.7)","[0.7,0.8)","[0.8,0.9)","[0.9,1.0)","[1.0,1.2)","[1.2,1.5)","[1.5+)"]
mfe_df["bucket"] = pd.cut(mfe_df["pred_norm"], bins=bins, labels=labels, right=False)

mfe_by_bucket = mfe_df.groupby("bucket")[["mfe_30","mfe_60","mfe_90","mfe_120","mfe_180","ret_60s"]].agg(
    ["mean","count"]).round(4)

# ---------------------------------------------------------------------------
# Simulator with dynamic hold + trailing TP
# ---------------------------------------------------------------------------
def simulate_advanced(
    df_day: pd.DataFrame,
    base_thresh: float,
    c_sl: float,
    c_tp: float,
    state_tp_override: dict = None,
    # Dynamic hold
    high_thresh: float = None,    # pred_norm level to trigger extension
    extend_sec: int = 0,          # extra seconds if high_thresh triggered
    # Trailing TP
    trail_activate: float = None, # bps above entry to activate trailing
    trail_retrace: float = None,  # bps below peak to exit
) -> list:
    trades     = []
    busy_until = -1

    sym  = df_day["symbol"].iloc[0] if "symbol" in df_day.columns else None
    date = df_day["date"].iloc[0]
    rows = df_day.sort_values("ts_sec").reset_index(drop=True)
    if state_tp_override is None: state_tp_override = {}

    use_trail   = (trail_activate is not None and trail_retrace is not None)
    use_dynamic = (high_thresh is not None and extend_sec > 0)

    for _, row in rows.iterrows():
        ts = int(row["ts_sec"])

        if ts <= busy_until:                  continue
        if not row["in_window"]:              continue
        if pd.isna(row["ret_60s"]):           continue

        pred = float(row["pred_norm"])
        if pred < base_thresh:                continue

        state    = int(row["state"])
        vol      = STATE_VOL.get(state, 1.77)
        sl_dist  = c_sl * vol
        tp_mult  = state_tp_override.get(state, c_tp)
        tp_dist  = tp_mult * vol
        max_hold = STATE_MAX_HOLD.get(state, 60)
        size     = STATE_SIZE.get(state, 1.0)

        # Dynamic extension for high-conviction signals
        if use_dynamic and pred >= high_thresh:
            max_hold = max_hold + extend_sec

        entry_mid  = float(row["mid"])
        sl_price   = entry_mid * (1 - sl_dist * 1e-4)
        tp_price   = entry_mid * (1 + tp_dist * 1e-4)

        exit_mid   = None
        exit_ts    = ts + max_hold
        exit_type  = "timeout"
        mae_bps    = 0.0
        mfe_bps    = 0.0
        peak_mid   = entry_mid        # for trailing TP
        trail_active = False

        for dt in range(1, max_hold + 1):
            mid_t = mid_lookup.get((sym, date, ts + dt))
            if mid_t is None: continue
            mid_t  = float(mid_t)
            ret_t  = (np.log(mid_t) - np.log(entry_mid)) * 1e4

            if ret_t < mae_bps: mae_bps = ret_t
            if ret_t > mfe_bps: mfe_bps = ret_t

            # Update peak for trailing
            if mid_t > peak_mid:
                peak_mid = mid_t

            # Activate trailing once price is sufficiently above entry
            if use_trail and not trail_active:
                profit_so_far = (np.log(mid_t) - np.log(entry_mid)) * 1e4
                if profit_so_far >= trail_activate:
                    trail_active = True

            # Fixed TP/SL
            if mid_t >= tp_price:
                exit_mid  = mid_t; exit_ts = ts + dt; exit_type = "tp"; break
            elif mid_t <= sl_price:
                exit_mid  = mid_t; exit_ts = ts + dt; exit_type = "sl"; break

            # Trailing exit: if trailing is active and price retraces from peak
            if trail_active:
                retrace_bps = (np.log(peak_mid) - np.log(mid_t)) * 1e4
                if retrace_bps >= trail_retrace:
                    exit_mid  = mid_t; exit_ts = ts + dt
                    exit_type = "trail_tp"; break

        if exit_mid is None:
            exit_mid = mid_lookup.get((sym, date, ts + max_hold))
            if exit_mid is None:
                exit_mid = entry_mid * (1 + float(row["ret_60s"]) * 1e-4)

        exit_mid   = float(exit_mid)
        actual_ret = (np.log(exit_mid) - np.log(entry_mid)) * 1e4
        net_pnl    = (actual_ret - COST_BPS) * size

        trades.append({
            "ts_sec": ts, "exit_ts": exit_ts,
            "hold_sec": exit_ts - ts,
            "pred_norm": pred, "state": state, "size": size,
            "actual_ret": actual_ret, "exit_type": exit_type,
            "net_pnl_bps": net_pnl,
            "mae_bps": mae_bps, "mfe_bps": mfe_bps,
        })
        busy_until = exit_ts

    return trades


def run_sim(df_set, **kwargs):
    all_t = []
    for day, grp in df_set.groupby("date"):
        t = simulate_advanced(grp, **kwargs)
        for tr in t:
            tr["date"] = str(day)
        all_t.extend(t)
    return pd.DataFrame(all_t) if all_t else pd.DataFrame()


def metrics(tdf, label=""):
    if tdf.empty: return {}
    pnl = tdf["net_pnl_bps"].values; n = len(pnl)
    std = pnl.std() if pnl.std() > 0 else 1e-9
    n_tp    = int((tdf["exit_type"]=="tp").sum())
    n_trail = int((tdf["exit_type"]=="trail_tp").sum())
    n_sl    = int((tdf["exit_type"]=="sl").sum())
    n_to    = int((tdf["exit_type"]=="timeout").sum())
    cum     = np.cumsum(pnl)
    return {
        "n":        n,
        "total":    round(float(pnl.sum()), 2),
        "mean":     round(float(pnl.mean()), 5),
        "sharpe":   round(float(pnl.mean()/std*np.sqrt(n)), 3),
        "hit":      round(float(np.mean(pnl>0)), 4),
        "n_tp":     n_tp, "n_trail": n_trail, "n_sl": n_sl, "n_to": n_to,
        "pct_tp":   round(100*(n_tp+n_trail)/n, 1),
        "avg_hold": round(float(tdf["hold_sec"].mean()), 1),
        "max_dd":   round(float((cum-np.maximum.accumulate(cum)).min()), 2),
        "mean_mfe": round(float(tdf["mfe_bps"].mean()), 4),
    }


# ---------------------------------------------------------------------------
# TEST SET
# ---------------------------------------------------------------------------
test_days = df_eval[df_eval["date"].isin(test_dates)].copy()

base_kwargs = dict(base_thresh=BASE_THRESH, c_sl=C_SL, c_tp=C_TP,
                   state_tp_override=STATE_TP_OVERRIDE)

# Baseline
tdf_base = run_sim(test_days, **base_kwargs)
m_base   = metrics(tdf_base, "baseline")

# ---------------------------------------------------------------------------
# SWEEP A: Dynamic hold extension
# ---------------------------------------------------------------------------
log.info("Dynamic hold sweep...")
dyn_rows = []
for ht in HIGH_THRESH_OPTIONS:
    for ext in EXTEND_SEC_OPTIONS:
        tdf = run_sim(test_days, high_thresh=ht, extend_sec=ext, **base_kwargs)
        m   = metrics(tdf)
        if m:
            m["high_thresh"] = ht
            m["extend_sec"]  = ext
            dyn_rows.append(m)
dyn_df = pd.DataFrame(dyn_rows)

# ---------------------------------------------------------------------------
# SWEEP B: Trailing TP
# ---------------------------------------------------------------------------
log.info("Trailing TP sweep...")
trail_rows = []
for act in TRAIL_ACTIVATE_OPTIONS:
    for ret in TRAIL_RETRACE_OPTIONS:
        tdf = run_sim(test_days, trail_activate=act, trail_retrace=ret, **base_kwargs)
        m   = metrics(tdf)
        if m:
            m["trail_activate"] = act
            m["trail_retrace"]  = ret
            trail_rows.append(m)
trail_df = pd.DataFrame(trail_rows)

# ---------------------------------------------------------------------------
# SWEEP C: Combined best dynamic + best trailing
# ---------------------------------------------------------------------------
best_dyn_sharpe   = dyn_df["sharpe"].max()   if not dyn_df.empty   else -999
best_trail_sharpe = trail_df["sharpe"].max() if not trail_df.empty else -999

best_dyn_row   = dyn_df.loc[dyn_df["sharpe"].idxmax()]   if not dyn_df.empty   else None
best_trail_row = trail_df.loc[trail_df["sharpe"].idxmax()] if not trail_df.empty else None

combined_rows = []
if best_dyn_row is not None and best_trail_row is not None:
    tdf_comb = run_sim(test_days,
                       high_thresh=float(best_dyn_row["high_thresh"]),
                       extend_sec=int(best_dyn_row["extend_sec"]),
                       trail_activate=float(best_trail_row["trail_activate"]),
                       trail_retrace=float(best_trail_row["trail_retrace"]),
                       **base_kwargs)
    m_comb = metrics(tdf_comb, "combined")
    m_comb["config"] = "combined"
    combined_rows.append(m_comb)

# Save
mfe_df.to_csv(OUT_DIR/"mfe_by_signal.csv", index=False)
dyn_df.to_csv(OUT_DIR/"dynamic_hold_sweep.csv", index=False)
trail_df.to_csv(OUT_DIR/"trailing_tp_sweep.csv", index=False)

# ---------------------------------------------------------------------------
# Print
# ---------------------------------------------------------------------------
print("\n"+"="*70)
print("TIMEOUT ANALYSIS — DYNAMIC HOLD + TRAILING TP")
print("="*70)

print(f"\n--- MFE by pred_norm bucket (shows if strong signal = longer momentum) ---")
print(f"{'bucket':>12} {'n':>5} {'mfe_30':>8} {'mfe_60':>8} {'mfe_90':>8} "
      f"{'mfe_120':>8} {'mfe_180':>8} {'ret_60s':>8}")
for bucket in labels:
    sub = mfe_df[mfe_df["bucket"]==bucket]
    if len(sub)==0: continue
    print(f"{bucket:>12} {len(sub):>5} "
          f"{sub['mfe_30'].mean():>8.4f} {sub['mfe_60'].mean():>8.4f} "
          f"{sub['mfe_90'].mean():>8.4f} {sub['mfe_120'].mean():>8.4f} "
          f"{sub['mfe_180'].mean():>8.4f} {sub['ret_60s'].mean():>8.4f}")

print(f"\nBaseline: n={m_base.get('n')}  total={m_base.get('total')} bps  "
      f"sharpe={m_base.get('sharpe')}  hit={m_base.get('hit',0)*100:.1f}%  "
      f"pct_tp={m_base.get('pct_tp')}%  avg_hold={m_base.get('avg_hold')}s")

print(f"\n--- Dynamic Hold Sweep (extend hold for high pred_norm) ---")
print(f"{'high_thresh':>12} {'extend_sec':>12} {'n':>5} {'total':>8} {'sharpe':>8} "
      f"{'hit%':>7} {'pct_tp':>7} {'avg_hold':>9} {'max_dd':>8}")
for _, r in dyn_df.sort_values("sharpe", ascending=False).head(10).iterrows():
    print(f"{r['high_thresh']:>12.2f} {int(r['extend_sec']):>12} {int(r['n']):>5} "
          f"{r['total']:>8.2f} {r['sharpe']:>8.3f} {r['hit']*100:>6.1f}% "
          f"{r['pct_tp']:>6.1f}% {r['avg_hold']:>9.1f} {r['max_dd']:>8.2f}")

print(f"\n--- Trailing TP Sweep ---")
print(f"{'activate(bps)':>14} {'retrace(bps)':>13} {'n':>5} {'total':>8} {'sharpe':>8} "
      f"{'hit%':>7} {'pct_tp':>7} {'n_trail':>8} {'avg_hold':>9} {'max_dd':>8}")
for _, r in trail_df.sort_values("sharpe", ascending=False).iterrows():
    print(f"{r['trail_activate']:>14.2f} {r['trail_retrace']:>13.2f} {int(r['n']):>5} "
          f"{r['total']:>8.2f} {r['sharpe']:>8.3f} {r['hit']*100:>6.1f}% "
          f"{r['pct_tp']:>6.1f}% {int(r['n_trail']):>8} {r['avg_hold']:>9.1f} {r['max_dd']:>8.2f}")

if combined_rows:
    mc = combined_rows[0]
    print(f"\n--- Combined Best (dynamic + trailing) ---")
    print(f"  dynamic: thresh={best_dyn_row['high_thresh']:.2f} extend={int(best_dyn_row['extend_sec'])}s")
    print(f"  trailing: activate={best_trail_row['trail_activate']:.2f} retrace={best_trail_row['trail_retrace']:.2f}")
    print(f"  n={mc['n']}  total={mc['total']} bps  sharpe={mc['sharpe']}  "
          f"hit={mc['hit']*100:.1f}%  n_tp={mc['n_tp']}  n_trail={mc['n_trail']}  "
          f"n_timeout={mc['n_to']}  avg_hold={mc['avg_hold']}s")

print(f"\nVS BASELINE: n={m_base.get('n')}  total={m_base.get('total')} bps  "
      f"sharpe={m_base.get('sharpe')}  pct_tp={m_base.get('pct_tp')}%")
print("="*70)

15:19:07  INFO  Loading...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

15:21:00  INFO  preprocess: 881382 rows after regularization
15:21:00  INFO  calculate_features: building CORE31 ...
15:21:22  INFO  build_core31_from_regularized: built 881382 rows, 31 CORE31 features
15:21:24  INFO  calculate_features: building book geometry (ADDED6 + CONTROLS9) ...
15:21:29  INFO  _build_book_geometry_features: built 881382 rows
15:21:29  INFO  calculate_features: building shock recovery (CHALLENGERS 1-2) ...
15:21:31  INFO  _build_shock_recovery_features: bid_shocks=22175 ask_shocks=42052
15:21:32  INFO  _build_shock_recovery_features: built 881382 rows
15:21:32  INFO  calculate_features: building update clustering (CHALLENGER 3) ...
15:21:34  INFO  _build_update_clustering_features: obs=472806 book_change=326695 bid=149451 ask=233683
15:21:35  INFO  _build_update_clustering_features: built 881382 rows
15:21:35  INFO  calculate_features: building liquidity migration (CHALLENGER 4) ...
15:21:36  INFO  _build_liquidity_migration_features: built 881382 rows
15:21:46  

MemoryError: Unable to allocate 598. MiB for an array with shape (89, 881382) and data type float64

In [1]:
"""
TIMEOUT ANALYSIS + DYNAMIC EXIT
=================================
Problem: timeout trades have mean_ret=+0.52 bps but net=-0.28 after 1 bps cost.
         MFE = 1.11 bps — price gets there but we exit too early at max_hold.

Two fixes to test:
  A) Dynamic hold: if pred_norm > HIGH_THRESH and price still above entry at t=max_hold,
     extend hold by EXTEND_SEC more seconds. Only extend once.

  B) Trailing TP: once price is UP by TRAIL_ACTIVATE bps, switch from fixed TP
     to a trailing stop of TRAIL_RETRACE bps below the running peak.
     Exit when price drops TRAIL_RETRACE below peak.
     This locks in gains rather than waiting for a fixed TP target.

  C) Combined: trailing TP active always, dynamic hold for strong signals.

We test all three and compare to baseline (fixed TP/SL/max_hold from v8).

Key question: does a strong pred_norm signal actually predict LONGER momentum?
We'll check: mean_mfe by pred_norm bucket to see if high-pred trades have higher MFE.
"""

import sys
sys.path.insert(0, r"C:\projects\hedge_fund")
import warnings; warnings.filterwarnings("ignore")
import logging, pathlib
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s",
                    datefmt="%H:%M:%S")
log = logging.getLogger(__name__)

import numpy as np
import pandas as pd
import lightgbm as lgb

from data.store.duckdb_client import DuckDBClient
from data.store.s3_client import S3Client
from data.processing.usdinr.features import USDINRFeatureBuilder
from data.processing.usdinr.constants import CHALLENGER50
from data.processing.usdinr.regimes import add_soft_hmm_features

OUT_DIR = pathlib.Path(r"C:\projects\hedge_fund\results\timeout_analysis")
OUT_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_DAYS   = 15
VAL_DAYS     = 5
COST_BPS     = 1.0
IST_OFFSET   = 19800
OPEN_FILTER  = 10 * 3600 + 30 * 60
CLOSE_FILTER = 14 * 3600 + 30 * 60

# Best config from v8
BASE_THRESH    = 0.60
C_SL           = 1.2
C_TP           = 1.0
STATE_VOL      = {1: 1.77, 2: 2.37, 3: 1.61, 4: 1.03}
STATE_MAX_HOLD = {1: 60,   2: 30,   3: 90,   4: 180}
STATE_SIZE     = {1: 0.59, 2: 0.44, 3: 0.65, 4: 1.00}
STATE_TP_OVERRIDE = {4: 0.6}

# Dynamic hold: if pred_norm > HIGH_THRESH, extend hold by EXTEND_SEC
HIGH_THRESH_OPTIONS = [0.8, 1.0, 1.2]
EXTEND_SEC_OPTIONS  = [30, 60, 90, 120]

# Trailing TP params
TRAIL_ACTIVATE_OPTIONS = [0.3, 0.5, 0.7]   # bps above entry to activate trailing
TRAIL_RETRACE_OPTIONS  = [0.2, 0.3, 0.4]   # bps retrace from peak to exit

LGB_PARAMS = {
    "objective":"regression","metric":"rmse","verbosity":-1,
    "num_leaves":31,"learning_rate":0.05,"feature_fraction":0.8,
    "bagging_fraction":0.8,"bagging_freq":5,"min_child_samples":50,
}

# ---------------------------------------------------------------------------
# Load
# ---------------------------------------------------------------------------
log.info("Loading...")
db = DuckDBClient(); s = S3Client(); all_frames = []
for contract in ["26SEPFUT","26OCTFUT","26NOVFUT"]:
    files = s.list_files(f"processed/1s/USDINR{contract}/")
    if not files: continue
    for f in files:
        parts = f.split("/")
        sym, date = parts[2], parts[3].replace(".parquet","")
        try:
            d = db.read_parquet(f"processed/1s/{sym}/{date}.parquet")
            d["symbol"] = sym; all_frames.append(d)
        except: pass
db.close()

df_raw  = pd.concat(all_frames, ignore_index=True)
builder = USDINRFeatureBuilder(data=df_raw, auto_build=True)
df      = builder.dataset.copy()
df      = df.sort_values(["symbol","date","ts_sec"]).reset_index(drop=True)

all_dates   = sorted(df["date"].unique())
train_dates = all_dates[:TRAIN_DAYS]
val_dates   = all_dates[TRAIN_DAYS:TRAIN_DAYS+VAL_DAYS]
test_dates  = all_dates[TRAIN_DAYS+VAL_DAYS:]
train_end   = str(train_dates[-1])

mask_train = df["date"].isin(train_dates)
mask_val   = df["date"].isin(val_dates)
mask_test  = df["date"].isin(test_dates)

df, hmm = add_soft_hmm_features(df=df, train_end_date=train_end,
                                 K=4, restarts=3, max_iter=70, tol=1e-6, seed=42)

if "mid" not in df.columns:
    df["mid"] = (df["bid_p1"] + df["ask_p1"]) / 2.0
df["log_mid"] = np.log(df["mid"])
g = df.groupby(["symbol","date"], sort=False)
df["ret_60s"]  = (g["log_mid"].shift(-60) - df["log_mid"]) * 1e4
df["ret_60s_norm"] = df.groupby(["symbol","date"])["ret_60s"].transform(
    lambda x: (x - x.mean()) / (x.std() + 1e-8)
)
df["ist_tod"]   = (df["ts_sec"] + IST_OFFSET) % 86400
df["in_window"] = (df["ist_tod"] >= OPEN_FILTER) & (df["ist_tod"] <= CLOSE_FILTER)

log.info("Mid lookup...")
mid_lookup = df.set_index(["symbol","date","ts_sec"])["mid"].to_dict()

# ---------------------------------------------------------------------------
# Train
# ---------------------------------------------------------------------------
features   = CHALLENGER50
target     = "ret_60s_norm"
valid_mask = df[features + [target,"ret_60s"]].notna().all(axis=1)
tr = df[mask_train & valid_mask]
va = df[mask_val   & valid_mask]
ds_tr = lgb.Dataset(tr[features].values, label=tr[target].values, feature_name=features)
ds_va = lgb.Dataset(va[features].values, label=va[target].values,
                    feature_name=features, reference=ds_tr)
model = lgb.train(LGB_PARAMS, ds_tr, num_boost_round=1000, valid_sets=[ds_va],
                  callbacks=[lgb.early_stopping(50,verbose=False), lgb.log_evaluation(100)])

eval_mask = (mask_val | mask_test) & valid_mask & df["ret_60s"].notna()
df_eval   = df[eval_mask].copy().reset_index(drop=True)
df_eval["state"]     = df_eval["state_filtered"].astype(int)
df_eval["pred_norm"] = model.predict(df_eval[features].values)

# ---------------------------------------------------------------------------
# ANALYSIS 1: MFE by pred_norm bucket
# Does stronger signal → higher MFE? (justifies dynamic hold for strong signals)
# ---------------------------------------------------------------------------
log.info("Computing MFE by pred_norm bucket...")

# For each signal bar above base threshold, compute MFE over 180s
te = df_eval[df_eval["date"].isin(test_dates)].copy()
te_win = te[te["in_window"] & (te["pred_norm"] > BASE_THRESH)].copy()

mfe_rows = []
for _, row in te_win.iterrows():
    sym  = row.get("symbol")
    date = row["date"]
    ts   = int(row["ts_sec"])
    entry_mid = float(row["mid"])
    pred      = float(row["pred_norm"])

    # Compute running MFE at 30, 60, 90, 120, 180 seconds
    mfe_30 = mfe_60 = mfe_90 = mfe_120 = mfe_180 = 0.0
    for dt in range(1, 181):
        mid_t = mid_lookup.get((sym, date, ts + dt))
        if mid_t is None: continue
        ret_t = (np.log(float(mid_t)) - np.log(entry_mid)) * 1e4
        if dt <= 30:  mfe_30  = max(mfe_30,  ret_t)
        if dt <= 60:  mfe_60  = max(mfe_60,  ret_t)
        if dt <= 90:  mfe_90  = max(mfe_90,  ret_t)
        if dt <= 120: mfe_120 = max(mfe_120, ret_t)
        mfe_180 = max(mfe_180, ret_t)

    mfe_rows.append({
        "pred_norm": pred,
        "state":     int(row["state"]),
        "mfe_30":    mfe_30,
        "mfe_60":    mfe_60,
        "mfe_90":    mfe_90,
        "mfe_120":   mfe_120,
        "mfe_180":   mfe_180,
        "ret_60s":   float(row["ret_60s"]),
    })

mfe_df = pd.DataFrame(mfe_rows)

# Bucket by pred_norm
bins   = [0.6, 0.7, 0.8, 0.9, 1.0, 1.2, 1.5, 999]
labels = ["[0.6,0.7)","[0.7,0.8)","[0.8,0.9)","[0.9,1.0)","[1.0,1.2)","[1.2,1.5)","[1.5+)"]
mfe_df["bucket"] = pd.cut(mfe_df["pred_norm"], bins=bins, labels=labels, right=False)

mfe_by_bucket = mfe_df.groupby("bucket")[["mfe_30","mfe_60","mfe_90","mfe_120","mfe_180","ret_60s"]].agg(
    ["mean","count"]).round(4)

# ---------------------------------------------------------------------------
# Simulator with dynamic hold + trailing TP
# ---------------------------------------------------------------------------
def simulate_advanced(
    df_day: pd.DataFrame,
    base_thresh: float,
    c_sl: float,
    c_tp: float,
    state_tp_override: dict = None,
    # Dynamic hold
    high_thresh: float = None,    # pred_norm level to trigger extension
    extend_sec: int = 0,          # extra seconds if high_thresh triggered
    # Trailing TP
    trail_activate: float = None, # bps above entry to activate trailing
    trail_retrace: float = None,  # bps below peak to exit
) -> list:
    trades     = []
    busy_until = -1

    sym  = df_day["symbol"].iloc[0] if "symbol" in df_day.columns else None
    date = df_day["date"].iloc[0]
    rows = df_day.sort_values("ts_sec").reset_index(drop=True)
    if state_tp_override is None: state_tp_override = {}

    use_trail   = (trail_activate is not None and trail_retrace is not None)
    use_dynamic = (high_thresh is not None and extend_sec > 0)

    for _, row in rows.iterrows():
        ts = int(row["ts_sec"])

        if ts <= busy_until:                  continue
        if not row["in_window"]:              continue
        if pd.isna(row["ret_60s"]):           continue

        pred = float(row["pred_norm"])
        if pred < base_thresh:                continue

        state    = int(row["state"])
        vol      = STATE_VOL.get(state, 1.77)
        sl_dist  = c_sl * vol
        tp_mult  = state_tp_override.get(state, c_tp)
        tp_dist  = tp_mult * vol
        max_hold = STATE_MAX_HOLD.get(state, 60)
        size     = STATE_SIZE.get(state, 1.0)

        # Dynamic extension for high-conviction signals
        if use_dynamic and pred >= high_thresh:
            max_hold = max_hold + extend_sec

        entry_mid  = float(row["mid"])
        sl_price   = entry_mid * (1 - sl_dist * 1e-4)
        tp_price   = entry_mid * (1 + tp_dist * 1e-4)

        exit_mid   = None
        exit_ts    = ts + max_hold
        exit_type  = "timeout"
        mae_bps    = 0.0
        mfe_bps    = 0.0
        peak_mid   = entry_mid        # for trailing TP
        trail_active = False

        for dt in range(1, max_hold + 1):
            mid_t = mid_lookup.get((sym, date, ts + dt))
            if mid_t is None: continue
            mid_t  = float(mid_t)
            ret_t  = (np.log(mid_t) - np.log(entry_mid)) * 1e4

            if ret_t < mae_bps: mae_bps = ret_t
            if ret_t > mfe_bps: mfe_bps = ret_t

            # Update peak for trailing
            if mid_t > peak_mid:
                peak_mid = mid_t

            # Activate trailing once price is sufficiently above entry
            if use_trail and not trail_active:
                profit_so_far = (np.log(mid_t) - np.log(entry_mid)) * 1e4
                if profit_so_far >= trail_activate:
                    trail_active = True

            # Fixed TP/SL
            if mid_t >= tp_price:
                exit_mid  = mid_t; exit_ts = ts + dt; exit_type = "tp"; break
            elif mid_t <= sl_price:
                exit_mid  = mid_t; exit_ts = ts + dt; exit_type = "sl"; break

            # Trailing exit: if trailing is active and price retraces from peak
            if trail_active:
                retrace_bps = (np.log(peak_mid) - np.log(mid_t)) * 1e4
                if retrace_bps >= trail_retrace:
                    exit_mid  = mid_t; exit_ts = ts + dt
                    exit_type = "trail_tp"; break

        if exit_mid is None:
            exit_mid = mid_lookup.get((sym, date, ts + max_hold))
            if exit_mid is None:
                exit_mid = entry_mid * (1 + float(row["ret_60s"]) * 1e-4)

        exit_mid   = float(exit_mid)
        actual_ret = (np.log(exit_mid) - np.log(entry_mid)) * 1e4
        net_pnl    = (actual_ret - COST_BPS) * size

        trades.append({
            "ts_sec": ts, "exit_ts": exit_ts,
            "hold_sec": exit_ts - ts,
            "pred_norm": pred, "state": state, "size": size,
            "actual_ret": actual_ret, "exit_type": exit_type,
            "net_pnl_bps": net_pnl,
            "mae_bps": mae_bps, "mfe_bps": mfe_bps,
        })
        busy_until = exit_ts

    return trades


def run_sim(df_set, **kwargs):
    all_t = []
    for day, grp in df_set.groupby("date"):
        t = simulate_advanced(grp, **kwargs)
        for tr in t:
            tr["date"] = str(day)
        all_t.extend(t)
    return pd.DataFrame(all_t) if all_t else pd.DataFrame()


def metrics(tdf, label=""):
    if tdf.empty: return {}
    pnl = tdf["net_pnl_bps"].values; n = len(pnl)
    std = pnl.std() if pnl.std() > 0 else 1e-9
    n_tp    = int((tdf["exit_type"]=="tp").sum())
    n_trail = int((tdf["exit_type"]=="trail_tp").sum())
    n_sl    = int((tdf["exit_type"]=="sl").sum())
    n_to    = int((tdf["exit_type"]=="timeout").sum())
    cum     = np.cumsum(pnl)
    return {
        "n":        n,
        "total":    round(float(pnl.sum()), 2),
        "mean":     round(float(pnl.mean()), 5),
        "sharpe":   round(float(pnl.mean()/std*np.sqrt(n)), 3),
        "hit":      round(float(np.mean(pnl>0)), 4),
        "n_tp":     n_tp, "n_trail": n_trail, "n_sl": n_sl, "n_to": n_to,
        "pct_tp":   round(100*(n_tp+n_trail)/n, 1),
        "avg_hold": round(float(tdf["hold_sec"].mean()), 1),
        "max_dd":   round(float((cum-np.maximum.accumulate(cum)).min()), 2),
        "mean_mfe": round(float(tdf["mfe_bps"].mean()), 4),
    }


# ---------------------------------------------------------------------------
# TEST SET
# ---------------------------------------------------------------------------
test_days = df_eval[df_eval["date"].isin(test_dates)].copy()

base_kwargs = dict(base_thresh=BASE_THRESH, c_sl=C_SL, c_tp=C_TP,
                   state_tp_override=STATE_TP_OVERRIDE)

# Baseline
tdf_base = run_sim(test_days, **base_kwargs)
m_base   = metrics(tdf_base, "baseline")

# ---------------------------------------------------------------------------
# SWEEP A: Dynamic hold extension
# ---------------------------------------------------------------------------
log.info("Dynamic hold sweep...")
dyn_rows = []
for ht in HIGH_THRESH_OPTIONS:
    for ext in EXTEND_SEC_OPTIONS:
        tdf = run_sim(test_days, high_thresh=ht, extend_sec=ext, **base_kwargs)
        m   = metrics(tdf)
        if m:
            m["high_thresh"] = ht
            m["extend_sec"]  = ext
            dyn_rows.append(m)
dyn_df = pd.DataFrame(dyn_rows)

# ---------------------------------------------------------------------------
# SWEEP B: Trailing TP
# ---------------------------------------------------------------------------
log.info("Trailing TP sweep...")
trail_rows = []
for act in TRAIL_ACTIVATE_OPTIONS:
    for ret in TRAIL_RETRACE_OPTIONS:
        tdf = run_sim(test_days, trail_activate=act, trail_retrace=ret, **base_kwargs)
        m   = metrics(tdf)
        if m:
            m["trail_activate"] = act
            m["trail_retrace"]  = ret
            trail_rows.append(m)
trail_df = pd.DataFrame(trail_rows)

# ---------------------------------------------------------------------------
# SWEEP C: Combined best dynamic + best trailing
# ---------------------------------------------------------------------------
best_dyn_sharpe   = dyn_df["sharpe"].max()   if not dyn_df.empty   else -999
best_trail_sharpe = trail_df["sharpe"].max() if not trail_df.empty else -999

best_dyn_row   = dyn_df.loc[dyn_df["sharpe"].idxmax()]   if not dyn_df.empty   else None
best_trail_row = trail_df.loc[trail_df["sharpe"].idxmax()] if not trail_df.empty else None

combined_rows = []
if best_dyn_row is not None and best_trail_row is not None:
    tdf_comb = run_sim(test_days,
                       high_thresh=float(best_dyn_row["high_thresh"]),
                       extend_sec=int(best_dyn_row["extend_sec"]),
                       trail_activate=float(best_trail_row["trail_activate"]),
                       trail_retrace=float(best_trail_row["trail_retrace"]),
                       **base_kwargs)
    m_comb = metrics(tdf_comb, "combined")
    m_comb["config"] = "combined"
    combined_rows.append(m_comb)

# Save
mfe_df.to_csv(OUT_DIR/"mfe_by_signal.csv", index=False)
dyn_df.to_csv(OUT_DIR/"dynamic_hold_sweep.csv", index=False)
trail_df.to_csv(OUT_DIR/"trailing_tp_sweep.csv", index=False)

# ---------------------------------------------------------------------------
# Print
# ---------------------------------------------------------------------------
print("\n"+"="*70)
print("TIMEOUT ANALYSIS — DYNAMIC HOLD + TRAILING TP")
print("="*70)

print(f"\n--- MFE by pred_norm bucket (shows if strong signal = longer momentum) ---")
print(f"{'bucket':>12} {'n':>5} {'mfe_30':>8} {'mfe_60':>8} {'mfe_90':>8} "
      f"{'mfe_120':>8} {'mfe_180':>8} {'ret_60s':>8}")
for bucket in labels:
    sub = mfe_df[mfe_df["bucket"]==bucket]
    if len(sub)==0: continue
    print(f"{bucket:>12} {len(sub):>5} "
          f"{sub['mfe_30'].mean():>8.4f} {sub['mfe_60'].mean():>8.4f} "
          f"{sub['mfe_90'].mean():>8.4f} {sub['mfe_120'].mean():>8.4f} "
          f"{sub['mfe_180'].mean():>8.4f} {sub['ret_60s'].mean():>8.4f}")

print(f"\nBaseline: n={m_base.get('n')}  total={m_base.get('total')} bps  "
      f"sharpe={m_base.get('sharpe')}  hit={m_base.get('hit',0)*100:.1f}%  "
      f"pct_tp={m_base.get('pct_tp')}%  avg_hold={m_base.get('avg_hold')}s")

print(f"\n--- Dynamic Hold Sweep (extend hold for high pred_norm) ---")
print(f"{'high_thresh':>12} {'extend_sec':>12} {'n':>5} {'total':>8} {'sharpe':>8} "
      f"{'hit%':>7} {'pct_tp':>7} {'avg_hold':>9} {'max_dd':>8}")
for _, r in dyn_df.sort_values("sharpe", ascending=False).head(10).iterrows():
    print(f"{r['high_thresh']:>12.2f} {int(r['extend_sec']):>12} {int(r['n']):>5} "
          f"{r['total']:>8.2f} {r['sharpe']:>8.3f} {r['hit']*100:>6.1f}% "
          f"{r['pct_tp']:>6.1f}% {r['avg_hold']:>9.1f} {r['max_dd']:>8.2f}")

print(f"\n--- Trailing TP Sweep ---")
print(f"{'activate(bps)':>14} {'retrace(bps)':>13} {'n':>5} {'total':>8} {'sharpe':>8} "
      f"{'hit%':>7} {'pct_tp':>7} {'n_trail':>8} {'avg_hold':>9} {'max_dd':>8}")
for _, r in trail_df.sort_values("sharpe", ascending=False).iterrows():
    print(f"{r['trail_activate']:>14.2f} {r['trail_retrace']:>13.2f} {int(r['n']):>5} "
          f"{r['total']:>8.2f} {r['sharpe']:>8.3f} {r['hit']*100:>6.1f}% "
          f"{r['pct_tp']:>6.1f}% {int(r['n_trail']):>8} {r['avg_hold']:>9.1f} {r['max_dd']:>8.2f}")

if combined_rows:
    mc = combined_rows[0]
    print(f"\n--- Combined Best (dynamic + trailing) ---")
    print(f"  dynamic: thresh={best_dyn_row['high_thresh']:.2f} extend={int(best_dyn_row['extend_sec'])}s")
    print(f"  trailing: activate={best_trail_row['trail_activate']:.2f} retrace={best_trail_row['trail_retrace']:.2f}")
    print(f"  n={mc['n']}  total={mc['total']} bps  sharpe={mc['sharpe']}  "
          f"hit={mc['hit']*100:.1f}%  n_tp={mc['n_tp']}  n_trail={mc['n_trail']}  "
          f"n_timeout={mc['n_to']}  avg_hold={mc['avg_hold']}s")

print(f"\nVS BASELINE: n={m_base.get('n')}  total={m_base.get('total')} bps  "
      f"sharpe={m_base.get('sharpe')}  pct_tp={m_base.get('pct_tp')}%")
print("="*70)

15:35:51 INFO Loading...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

... (9 outputs left out)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

15:37:46 INFO preprocess: 881382 rows after regularization
15:37:46 INFO calculate_features: building CORE31 ...
15:38:08 INFO build_core31_from_regularized: built 881382 rows, 31 CORE31 features
15:38:09 INFO calculate_features: building book geometry (ADDED6 + CONTROLS9) ...
15:38:15 INFO _build_book_geometry_features: built 881382 rows
15:38:15 INFO calculate_features: building shock recovery (CHALLENGERS 1-2) ...
15:38:17 INFO _build_shock_recovery_features: bid_shocks=22175 ask_shocks=42052
15:38:17 INFO _build_shock_recovery_features: built 881382 rows
15:38:17 INFO calculate_features: building update clustering (CHALLENGER 3) ...
15:38:19 INFO _build_update_clustering_features: obs=472806 book_change=326695 bid=149451 ask=233683
15:38:20 INFO _build_update_clustering_features: built 881382 rows
15:38:20 INFO calculate_features: building liquidity migration (CHALLENGER 4) ...
15:38:22 INFO _build_liquidity_migration_features: built 881382 rows
15:38:30 INFO calculate_features: co


TIMEOUT ANALYSIS — DYNAMIC HOLD + TRAILING TP

--- MFE by pred_norm bucket (shows if strong signal = longer momentum) ---
      bucket     n   mfe_30   mfe_60   mfe_90  mfe_120  mfe_180  ret_60s
   [0.6,0.7)   209   1.5210   2.1721   2.7896   3.0359   3.6602   1.3416
   [0.7,0.8)    88   2.1915   2.9460   3.4242   3.4996   3.8628   2.1073
   [0.8,0.9)    49   2.4996   3.7483   4.3548   4.6312   5.3031   2.9313
   [0.9,1.0)    25   1.9119   2.1878   2.3616   2.3616   2.5315   1.0694
   [1.0,1.2)     7   0.6564   1.0127   3.9377   3.9377   3.9377   1.0127

Baseline: n=54  total=11.66 bps  sharpe=2.088  hit=68.5%  pct_tp=53.7%  avg_hold=44.4s

--- Dynamic Hold Sweep (extend hold for high pred_norm) ---
 high_thresh   extend_sec     n    total   sharpe    hit%  pct_tp  avg_hold   max_dd
        1.00          120    54    11.66    2.088   68.5%   53.7%      44.4    -5.19
        1.00           90    54    11.66    2.088   68.5%   53.7%      44.4    -5.19
        1.00           60    54    

In [2]:
"""
TRAILING SL ANALYSIS — paste into notebook cell after v8/v9 setup
Assumes already in memory:
  df_eval, mid_lookup, test_dates
  COST_BPS, STATE_VOL, STATE_MAX_HOLD, STATE_SIZE, STATE_TP_OVERRIDE
"""

import numpy as np
import pandas as pd

# ── params ──────────────────────────────────────────────────────────────────
BASE_THRESH       = 0.70          # best threshold from v11
C_SL              = 1.2
C_TP              = 1.0
COST_BPS          = 1.0
STATE_VOL         = {1: 1.77, 2: 2.37, 3: 1.61, 4: 1.03}
STATE_MAX_HOLD    = {1: 60,   2: 30,   3: 90,   4: 180}
STATE_SIZE        = {1: 0.59, 2: 0.44, 3: 0.65, 4: 1.00}
STATE_TP_OVERRIDE = {4: 0.6}

# Trailing SL modes:
#   BE_TRIGGER  = price up X bps → SL moves to breakeven (0 bps net)
#   LOCK_TRIGGER = price up Y bps → SL moves to LOCK_FLOOR bps above entry
#
# We test BE_TRIGGER only, LOCK only, and both combined
BE_TRIGGER_OPTIONS   = [0.3, 0.5, 0.7, 1.0]   # bps gain needed to move SL to breakeven
LOCK_TRIGGER_OPTIONS = [0.5, 0.7, 1.0, 1.5]   # bps gain needed to lock in profit
LOCK_FLOOR_OPTIONS   = [0.2, 0.3, 0.5]        # bps locked in once LOCK_TRIGGER hit

# ── Simulator ───────────────────────────────────────────────────────────────
def simulate(df_day, base_thresh, c_sl, c_tp,
             state_tp_override=None,
             be_trigger=None,          # bps up → SL moves to entry (breakeven)
             lock_trigger=None,        # bps up → SL moves to lock_floor
             lock_floor=0.3):          # bps above entry to lock at
    trades = []; busy_until = -1
    sym  = df_day["symbol"].iloc[0] if "symbol" in df_day.columns else None
    date = df_day["date"].iloc[0]
    if state_tp_override is None: state_tp_override = {}

    use_be   = be_trigger   is not None
    use_lock = lock_trigger is not None

    for _, row in df_day.sort_values("ts_sec").iterrows():
        ts = int(row["ts_sec"])
        if ts <= busy_until:         continue
        if not row["in_window"]:     continue
        if pd.isna(row["ret_60s"]): continue
        pred = float(row["pred_norm"])
        if pred < base_thresh:       continue

        state    = int(row["state"])
        vol      = STATE_VOL.get(state, 1.77)
        sl_dist  = c_sl * vol                           # fixed SL distance in bps
        tp_dist  = state_tp_override.get(state, c_tp) * vol
        max_hold = STATE_MAX_HOLD.get(state, 60)
        size     = STATE_SIZE.get(state, 1.0)

        em       = float(row["mid"])
        sl_bps   = -sl_dist          # current SL in bps from entry (starts negative)
        tp_price = em * (1 + tp_dist * 1e-4)

        exit_mid = None; exit_ts = ts + max_hold; exit_type = "timeout"
        mae = 0.0; mfe = 0.0
        be_activated   = False
        lock_activated = False

        for dt in range(1, max_hold + 1):
            m = mid_lookup.get((sym, date, ts + dt))
            if m is None: continue
            m = float(m)
            r = (np.log(m) - np.log(em)) * 1e4   # current ret in bps from entry

            if r < mae: mae = r
            if r > mfe: mfe = r

            # update trailing SL
            if use_lock and not lock_activated and r >= lock_trigger:
                lock_activated = True
                sl_bps = max(sl_bps, lock_floor)   # SL moves up to lock_floor

            if use_be and not be_activated and r >= be_trigger:
                be_activated = True
                sl_bps = max(sl_bps, 0.0)          # SL moves up to breakeven

            sl_price = em * (1 + sl_bps * 1e-4)

            # check exits
            if m >= tp_price:
                exit_mid = m; exit_ts = ts + dt; exit_type = "tp"; break
            if m <= sl_price:
                exit_mid = m; exit_ts = ts + dt
                exit_type = "trail_sl" if (be_activated or lock_activated) else "sl"
                break

        if exit_mid is None:
            exit_mid = mid_lookup.get((sym, date, ts + max_hold))
            if exit_mid is None:
                exit_mid = em * (1 + float(row["ret_60s"]) * 1e-4)

        exit_mid   = float(exit_mid)
        actual_ret = (np.log(exit_mid) - np.log(em)) * 1e4
        net_pnl    = (actual_ret - COST_BPS) * size

        trades.append({
            "ts_sec": ts, "hold_sec": exit_ts - ts, "pred_norm": pred,
            "actual_ret": actual_ret, "exit_type": exit_type, "state": state,
            "net_pnl_bps": net_pnl, "mae_bps": mae, "mfe_bps": mfe,
            "date": str(date)
        })
        busy_until = exit_ts
    return trades

def run(df_set, **kw):
    t = []
    for _, g in df_set.groupby("date"):
        t.extend(simulate(g, **kw))
    return pd.DataFrame(t) if t else pd.DataFrame()

def met(tdf):
    if tdf.empty: return {}
    p  = tdf["net_pnl_bps"].values; n = len(p)
    std = p.std() or 1e-9
    cum = np.cumsum(p)
    vc  = tdf["exit_type"].value_counts()
    return {
        "n":        n,
        "total":    round(p.sum(), 2),
        "sharpe":   round(p.mean() / std * np.sqrt(n), 3),
        "hit":      round(np.mean(p > 0), 4),
        "pct_tp":   round(100 * vc.get("tp", 0) / n, 1),
        "n_tp":     int(vc.get("tp", 0)),
        "n_sl":     int(vc.get("sl", 0)),
        "n_tsl":    int(vc.get("trail_sl", 0)),
        "n_to":     int(vc.get("timeout", 0)),
        "avg_hold": round(tdf["hold_sec"].mean(), 1),
        "avg_ret":  round(p.mean(), 3),
        "max_dd":   round((cum - np.maximum.accumulate(cum)).min(), 2),
    }

test_days = df_eval[df_eval["date"].isin(test_dates)].copy()
base_kw   = dict(base_thresh=BASE_THRESH, c_sl=C_SL, c_tp=C_TP,
                 state_tp_override=STATE_TP_OVERRIDE)

# baseline
tdf_base = run(test_days, **base_kw)
m0 = met(tdf_base)
print("=" * 72)
print("TRAILING SL ANALYSIS")
print("=" * 72)
print(f"Baseline (thresh={BASE_THRESH}): n={m0['n']}  total={m0['total']}  "
      f"sharpe={m0['sharpe']}  hit={m0['hit']*100:.1f}%  "
      f"pct_tp={m0['pct_tp']}%  n_sl={m0['n_sl']}  n_to={m0['n_to']}  "
      f"avg_hold={m0['avg_hold']}s")

# ── Sweep A: Breakeven SL only ──────────────────────────────────────────────
print(f"\n── A: Breakeven SL (price up X bps → SL moves to entry) ───────────")
print(f"{'be_trig':>8} {'n':>5} {'total':>7} {'sharpe':>8} {'hit%':>7} "
      f"{'pct_tp':>7} {'n_sl':>6} {'n_tsl':>7} {'n_to':>6} {'hold':>7} {'dd':>7}")
be_rows = []
for be in BE_TRIGGER_OPTIONS:
    tdf = run(test_days, be_trigger=be, **base_kw)
    m   = met(tdf)
    if not m: continue
    m["be"] = be
    be_rows.append(m)
    print(f"{be:>8.1f} {m['n']:>5} {m['total']:>7.2f} {m['sharpe']:>8.3f} "
          f"{m['hit']*100:>6.1f}% {m['pct_tp']:>6.1f}% {m['n_sl']:>6} "
          f"{m['n_tsl']:>7} {m['n_to']:>6} {m['avg_hold']:>7.1f} {m['max_dd']:>7.2f}")
be_df   = pd.DataFrame(be_rows)
best_be = be_df.loc[be_df["sharpe"].idxmax()] if not be_df.empty else None

# ── Sweep B: Profit-lock SL only ────────────────────────────────────────────
print(f"\n── B: Profit-lock SL (price up X bps → SL moves to +floor bps) ───")
print(f"{'lock_trig':>10} {'floor':>6} {'n':>5} {'total':>7} {'sharpe':>8} "
      f"{'hit%':>7} {'pct_tp':>7} {'n_sl':>6} {'n_tsl':>7} {'n_to':>6} {'hold':>7} {'dd':>7}")
lock_rows = []
for lt in LOCK_TRIGGER_OPTIONS:
    for lf in LOCK_FLOOR_OPTIONS:
        if lf >= lt: continue          # floor must be below trigger
        tdf = run(test_days, lock_trigger=lt, lock_floor=lf, **base_kw)
        m   = met(tdf)
        if not m: continue
        m.update({"lt": lt, "lf": lf})
        lock_rows.append(m)
        print(f"{lt:>10.1f} {lf:>6.1f} {m['n']:>5} {m['total']:>7.2f} {m['sharpe']:>8.3f} "
              f"{m['hit']*100:>6.1f}% {m['pct_tp']:>6.1f}% {m['n_sl']:>6} "
              f"{m['n_tsl']:>7} {m['n_to']:>6} {m['avg_hold']:>7.1f} {m['max_dd']:>7.2f}")
lock_df   = pd.DataFrame(lock_rows)
best_lock = lock_df.loc[lock_df["sharpe"].idxmax()] if not lock_df.empty else None

# ── Sweep C: BE + Lock combined ─────────────────────────────────────────────
if best_be is not None and best_lock is not None:
    print(f"\n── C: Combined (BE={best_be['be']:.1f} bps + "
          f"Lock trigger={best_lock['lt']:.1f} bps → floor={best_lock['lf']:.1f} bps) ─")
    tdf_c = run(test_days,
                be_trigger=float(best_be["be"]),
                lock_trigger=float(best_lock["lt"]),
                lock_floor=float(best_lock["lf"]),
                **base_kw)
    mc = met(tdf_c)
    print(f"  n={mc['n']}  total={mc['total']}  sharpe={mc['sharpe']}  "
          f"hit={mc['hit']*100:.1f}%  pct_tp={mc['pct_tp']}%  "
          f"n_sl={mc['n_sl']}  n_tsl={mc['n_tsl']}  n_to={mc['n_to']}  "
          f"avg_hold={mc['avg_hold']}s  max_dd={mc['max_dd']}")

# ── Exit type breakdown of baseline vs best ─────────────────────────────────
print(f"\n── Exit breakdown: baseline ───────────────────────────────────────")
for et, grp in tdf_base.groupby("exit_type"):
    p = grp["net_pnl_bps"].values
    print(f"  {et:>10}: n={len(p):>3}  mean_net={p.mean():>6.3f}  "
          f"hit={np.mean(p>0)*100:>5.1f}%  mean_hold={grp['hold_sec'].mean():>5.1f}s")

if best_be is not None:
    tdf_best = run(test_days, be_trigger=float(best_be["be"]), **base_kw)
    print(f"\n── Exit breakdown: best BE (trigger={best_be['be']:.1f} bps) ──────────")
    for et, grp in tdf_best.groupby("exit_type"):
        p = grp["net_pnl_bps"].values
        print(f"  {et:>10}: n={len(p):>3}  mean_net={p.mean():>6.3f}  "
              f"hit={np.mean(p>0)*100:>5.1f}%  mean_hold={grp['hold_sec'].mean():>5.1f}s")

print(f"\nVS BASELINE: n={m0['n']}  total={m0['total']}  sharpe={m0['sharpe']}")
print("=" * 72)

TRAILING SL ANALYSIS
Baseline (thresh=0.7): n=30  total=8.95  sharpe=2.093  hit=73.3%  pct_tp=60.0%  n_sl=1  n_to=11  avg_hold=36.8s

── A: Breakeven SL (price up X bps → SL moves to entry) ───────────
 be_trig     n   total   sharpe    hit%  pct_tp   n_sl   n_tsl   n_to    hold      dd
     0.3    33    6.31    1.358   60.6%   54.5%      1       8      6    30.6   -5.44
     0.5    33    7.03    1.532   63.6%   54.5%      1       6      8    31.5   -5.44
     0.7    32    7.68    1.748   65.6%   56.2%      1       5      8    32.6   -5.44
     1.0    31    8.11    1.877   67.7%   58.1%      1       4      8    33.8   -5.44

── B: Profit-lock SL (price up X bps → SL moves to +floor bps) ───
 lock_trig  floor     n   total   sharpe    hit%  pct_tp   n_sl   n_tsl   n_to    hold      dd
       0.5    0.2    33    7.26    1.647   63.6%   54.5%      1       6      8    31.2   -5.27
       0.5    0.3    33    6.35    1.443   60.6%   51.5%      1       8      7    30.4   -5.18
       0.7    0

In [3]:
"""
STRATEGY BACKTEST v12 — paste into notebook cell
Changes from v11 baseline:
  1. State 3 c_tp: 1.0 → 0.68  (TP from 1.61 → 1.09 bps, convert timeouts)
  2. State 2 threshold: 0.70 → 0.90  (drop low-conviction State 2 trades)
  3. Window: 10:30–13:00 IST  (drop Hour 13 drag)
Assumes in memory: df_eval, mid_lookup, test_dates, val_dates (optional)
"""

import numpy as np
import pandas as pd

# ── config ──────────────────────────────────────────────────────────────────
BASE_THRESH    = 0.70
STATE2_THRESH  = 0.90          # higher bar for volatile State 2
COST_BPS       = 1.0
C_SL           = 1.2

# per-state TP multipliers  ← KEY CHANGE: State 3 lowered to 0.68
STATE_TP = {1: 1.0, 2: 1.0, 3: 0.68, 4: 0.60}

STATE_VOL      = {1: 1.77, 2: 2.37, 3: 1.61, 4: 1.03}
STATE_MAX_HOLD = {1: 60,   2: 30,   3: 90,   4: 180}
STATE_SIZE     = {1: 0.59, 2: 0.44, 3: 0.65, 4: 1.00}

# window: 10:30–13:00 IST  (drop Hour 13)
WINDOW_START_SEC = 10 * 3600 + 30 * 60   # 37800
WINDOW_END_SEC   = 13 * 3600              # 46800

# ── derived: TP/SL distances per state ─────────────────────────────────────
for s in STATE_VOL:
    tp_bps = STATE_TP[s] * STATE_VOL[s]
    sl_bps = C_SL       * STATE_VOL[s]
    print(f"  State {s}: TP={tp_bps:.3f} bps  SL={sl_bps:.3f} bps  "
          f"hold={STATE_MAX_HOLD[s]}s  size={STATE_SIZE[s]:.2f}  "
          f"thresh={'0.90' if s==2 else '0.70'}")

# ── simulator ───────────────────────────────────────────────────────────────
def simulate_day(df_day):
    trades = []; busy_until = -1
    sym  = df_day["symbol"].iloc[0] if "symbol" in df_day.columns else None
    date = df_day["date"].iloc[0]

    for _, row in df_day.sort_values("ts_sec").iterrows():
        ts = int(row["ts_sec"])
        if ts <= busy_until:         continue
        if pd.isna(row["ret_60s"]): continue

        # window filter
        ts_in_day = ts % 86400
        if ts_in_day < WINDOW_START_SEC or ts_in_day >= WINDOW_END_SEC:
            continue

        pred  = float(row["pred_norm"])
        state = int(row["state"])

        # per-state threshold
        thresh = STATE2_THRESH if state == 2 else BASE_THRESH
        if pred < thresh: continue

        vol      = STATE_VOL[state]
        tp_dist  = STATE_TP[state] * vol
        sl_dist  = C_SL * vol
        max_hold = STATE_MAX_HOLD[state]
        size     = STATE_SIZE[state]

        em       = float(row["mid"])
        tp_price = em * (1 + tp_dist * 1e-4)
        sl_price = em * (1 - sl_dist * 1e-4)

        exit_mid = None; exit_ts = ts + max_hold; exit_type = "timeout"
        mae = 0.0; mfe = 0.0

        for dt in range(1, max_hold + 1):
            m = mid_lookup.get((sym, date, ts + dt))
            if m is None: continue
            m = float(m)
            r = (np.log(m) - np.log(em)) * 1e4
            if r < mae: mae = r
            if r > mfe: mfe = r
            if m >= tp_price:
                exit_mid = m; exit_ts = ts + dt; exit_type = "tp"; break
            if m <= sl_price:
                exit_mid = m; exit_ts = ts + dt; exit_type = "sl"; break

        if exit_mid is None:
            exit_mid = mid_lookup.get((sym, date, ts + max_hold))
            if exit_mid is None:
                exit_mid = em * (1 + float(row["ret_60s"]) * 1e-4)

        exit_mid   = float(exit_mid)
        actual_ret = (np.log(exit_mid) - np.log(em)) * 1e4
        net_pnl    = (actual_ret - COST_BPS) * size

        trades.append({
            "date": str(date), "ts_sec": ts,
            "hold_sec": exit_ts - ts, "pred_norm": pred,
            "actual_ret": actual_ret, "exit_type": exit_type,
            "state": state, "net_pnl_bps": net_pnl,
            "mae_bps": mae, "mfe_bps": mfe,
            "hour_ist": (ts % 86400) // 3600,
        })
        busy_until = exit_ts
    return trades

def run(df_set):
    t = []
    for _, g in df_set.groupby("date"):
        t.extend(simulate_day(g))
    return pd.DataFrame(t) if t else pd.DataFrame()

def metrics(tdf, label=""):
    if tdf.empty:
        print(f"{label}: no trades"); return {}
    p   = tdf["net_pnl_bps"].values; n = len(p)
    std = p.std() or 1e-9
    cum = np.cumsum(p)
    vc  = tdf["exit_type"].value_counts()
    m = {
        "n":        n,
        "total":    round(p.sum(), 2),
        "sharpe":   round(p.mean() / std * np.sqrt(n), 3),
        "hit":      round(np.mean(p > 0) * 100, 1),
        "pct_tp":   round(100 * vc.get("tp", 0) / n, 1),
        "n_tp":     int(vc.get("tp", 0)),
        "n_sl":     int(vc.get("sl", 0)),
        "n_to":     int(vc.get("timeout", 0)),
        "avg_hold": round(tdf["hold_sec"].mean(), 1),
        "avg_ret":  round(p.mean(), 3),
        "max_dd":   round((cum - np.maximum.accumulate(cum)).min(), 2),
        "per_day":  round(p.sum() / tdf["date"].nunique(), 3),
    }
    if label:
        print(f"\n{label}")
        print(f"  n={m['n']}  total={m['total']} bps  sharpe={m['sharpe']}  "
              f"hit={m['hit']}%  pct_tp={m['pct_tp']}%")
        print(f"  n_tp={m['n_tp']}  n_sl={m['n_sl']}  n_to={m['n_to']}  "
              f"avg_hold={m['avg_hold']}s  per_day={m['per_day']} bps  max_dd={m['max_dd']}")
    return m

# ── run test set ─────────────────────────────────────────────────────────────
print("\n" + "=" * 68)
print("STRATEGY v12  |  State3 c_tp=0.68  |  State2 thresh=0.90  |  10:30-13:00")
print("=" * 68)

test_days = df_eval[df_eval["date"].isin(test_dates)].copy()
tdf_test  = run(test_days)
m_test    = metrics(tdf_test, "TEST SET")

# ── state breakdown ──────────────────────────────────────────────────────────
print("\n── State breakdown ─────────────────────────────────────────────────")
print(f"  {'state':>6} {'n':>5} {'total':>7} {'sharpe':>8} {'hit%':>7} "
      f"{'pct_tp':>7} {'n_to':>5} {'hold':>7}")
for s, grp in tdf_test.groupby("state"):
    m = metrics(grp)
    if not m: continue
    print(f"  {s:>6} {m['n']:>5} {m['total']:>7.2f} {m['sharpe']:>8.3f} "
          f"{m['hit']:>6.1f}% {m['pct_tp']:>6.1f}% {m['n_to']:>5} {m['avg_hold']:>7.1f}")

# ── hourly breakdown ─────────────────────────────────────────────────────────
print("\n── Hourly breakdown ────────────────────────────────────────────────")
print(f"  {'hour':>5} {'n':>5} {'total':>7} {'sharpe':>8} {'hit%':>7} {'pct_tp':>7}")
for h, grp in tdf_test.groupby("hour_ist"):
    m = metrics(grp)
    if not m: continue
    print(f"  {h:>5} {m['n']:>5} {m['total']:>7.2f} {m['sharpe']:>8.3f} "
          f"{m['hit']:>6.1f}% {m['pct_tp']:>6.1f}%")

# ── exit type breakdown ──────────────────────────────────────────────────────
print("\n── Exit breakdown ──────────────────────────────────────────────────")
for et, grp in tdf_test.groupby("exit_type"):
    p = grp["net_pnl_bps"].values
    print(f"  {et:>10}: n={len(p):>3}  mean_net={p.mean():>+6.3f}  "
          f"hit={np.mean(p>0)*100:>5.1f}%  mean_hold={grp['hold_sec'].mean():>5.1f}s  "
          f"mean_mfe={grp['mfe_bps'].mean():>5.3f}")

# ── daily PnL ────────────────────────────────────────────────────────────────
print("\n── Daily PnL (bps, size-weighted) ──────────────────────────────────")
daily = tdf_test.groupby("date")["net_pnl_bps"].sum().reset_index()
daily.columns = ["date", "pnl_bps"]
for _, row in daily.iterrows():
    bar = "█" * int(abs(row["pnl_bps"]) * 3)
    sign = "+" if row["pnl_bps"] >= 0 else "-"
    print(f"  {row['date']}  {sign}{abs(row['pnl_bps']):>5.2f}  {bar}")
print(f"  {'win days':>10}: {(daily['pnl_bps'] > 0).sum()} / {len(daily)}")

# ── v11 baseline comparison ──────────────────────────────────────────────────
print("\n── vs v11 baseline ─────────────────────────────────────────────────")
print("  v11: n=30  total=8.95  sharpe=2.093  hit=73.3%  pct_tp=60.0%  "
      "n_to=11  avg_hold=36.8s")
print(f"  v12: n={m_test.get('n','?')}  total={m_test.get('total','?')}  "
      f"sharpe={m_test.get('sharpe','?')}  hit={m_test.get('hit','?')}%  "
      f"pct_tp={m_test.get('pct_tp','?')}%  n_to={m_test.get('n_to','?')}  "
      f"avg_hold={m_test.get('avg_hold','?')}s")

# ── val set (if available) ───────────────────────────────────────────────────
try:
    val_days = df_eval[df_eval["date"].isin(val_dates)].copy()
    if not val_days.empty:
        tdf_val = run(val_days)
        metrics(tdf_val, "VALIDATION SET")
except NameError:
    print("\n(val_dates not in memory — skipping val set)")

print("\n" + "=" * 68)

  State 1: TP=1.770 bps  SL=2.124 bps  hold=60s  size=0.59  thresh=0.70
  State 2: TP=2.370 bps  SL=2.844 bps  hold=30s  size=0.44  thresh=0.90
  State 3: TP=1.095 bps  SL=1.932 bps  hold=90s  size=0.65  thresh=0.70
  State 4: TP=0.618 bps  SL=1.236 bps  hold=180s  size=1.00  thresh=0.70

STRATEGY v12  |  State3 c_tp=0.68  |  State2 thresh=0.90  |  10:30-13:00

TEST SET
  n=7  total=2.69 bps  sharpe=1.412  hit=71.4%  pct_tp=71.4%
  n_tp=5  n_sl=0  n_to=2  avg_hold=62.6s  per_day=0.673 bps  max_dd=-0.65

── State breakdown ─────────────────────────────────────────────────
   state     n   total   sharpe    hit%  pct_tp  n_to    hold
       3     5    0.62    0.527   60.0%   60.0%     2    54.0
       4     2    2.07    2.016  100.0%  100.0%     0    84.0

── Hourly breakdown ────────────────────────────────────────────────
   hour     n   total   sharpe    hit%  pct_tp
     10     5    0.11    0.120   60.0%   60.0%
     11     2    2.58    3.873  100.0%  100.0%

── Exit breakdown ──────

In [4]:
"""
STRATEGY BACKTEST v12b — ISOLATION TEST
Tests each v12 change separately to find what helps vs hurts.
Assumes in memory: df_eval, mid_lookup, test_dates, val_dates
"""

import numpy as np
import pandas as pd

# ── shared params ────────────────────────────────────────────────────────────
COST_BPS       = 1.0
C_SL           = 1.2
STATE_VOL      = {1: 1.77, 2: 2.37, 3: 1.61, 4: 1.03}
STATE_MAX_HOLD = {1: 60,   2: 30,   3: 90,   4: 180}
STATE_SIZE     = {1: 0.59, 2: 0.44, 3: 0.65, 4: 1.00}

def simulate_day(df_day, base_thresh, state2_thresh,
                 state_tp, window_start, window_end):
    trades = []; busy_until = -1
    sym  = df_day["symbol"].iloc[0] if "symbol" in df_day.columns else None
    date = df_day["date"].iloc[0]

    for _, row in df_day.sort_values("ts_sec").iterrows():
        ts = int(row["ts_sec"])
        if ts <= busy_until:         continue
        if pd.isna(row["ret_60s"]): continue
        ts_in_day = ts % 86400
        if ts_in_day < window_start or ts_in_day >= window_end: continue

        pred  = float(row["pred_norm"])
        state = int(row["state"])
        thresh = state2_thresh if state == 2 else base_thresh
        if pred < thresh: continue

        vol      = STATE_VOL[state]
        tp_dist  = state_tp.get(state, 1.0) * vol
        sl_dist  = C_SL * vol
        max_hold = STATE_MAX_HOLD[state]
        size     = STATE_SIZE[state]
        em       = float(row["mid"])
        tp_price = em * (1 + tp_dist * 1e-4)
        sl_price = em * (1 - sl_dist * 1e-4)

        exit_mid = None; exit_ts = ts + max_hold; exit_type = "timeout"
        mae = 0.0; mfe = 0.0

        for dt in range(1, max_hold + 1):
            m = mid_lookup.get((sym, date, ts + dt))
            if m is None: continue
            m = float(m)
            r = (np.log(m) - np.log(em)) * 1e4
            if r < mae: mae = r
            if r > mfe: mfe = r
            if m >= tp_price:
                exit_mid = m; exit_ts = ts + dt; exit_type = "tp"; break
            if m <= sl_price:
                exit_mid = m; exit_ts = ts + dt; exit_type = "sl"; break

        if exit_mid is None:
            exit_mid = mid_lookup.get((sym, date, ts + max_hold))
            if exit_mid is None:
                exit_mid = em * (1 + float(row["ret_60s"]) * 1e-4)

        exit_mid   = float(exit_mid)
        actual_ret = (np.log(exit_mid) - np.log(em)) * 1e4
        net_pnl    = (actual_ret - COST_BPS) * size
        trades.append({
            "date": str(date), "ts_sec": ts,
            "hold_sec": exit_ts - ts, "pred_norm": pred,
            "actual_ret": actual_ret, "exit_type": exit_type,
            "state": state, "net_pnl_bps": net_pnl,
            "mae_bps": mae, "mfe_bps": mfe,
            "hour_ist": (ts % 86400) // 3600,
        })
        busy_until = exit_ts
    return trades

def run(df_set, **kw):
    t = []
    for _, g in df_set.groupby("date"):
        t.extend(simulate_day(g, **kw))
    return pd.DataFrame(t) if t else pd.DataFrame()

def metrics(tdf):
    if tdf.empty: return {}
    p   = tdf["net_pnl_bps"].values; n = len(p)
    std = p.std() or 1e-9
    cum = np.cumsum(p)
    vc  = tdf["exit_type"].value_counts()
    return {
        "n":        n,
        "total":    round(p.sum(), 2),
        "sharpe":   round(p.mean() / std * np.sqrt(n), 3),
        "hit":      round(np.mean(p > 0) * 100, 1),
        "pct_tp":   round(100 * vc.get("tp", 0) / n, 1),
        "n_tp":     int(vc.get("tp", 0)),
        "n_sl":     int(vc.get("sl", 0)),
        "n_to":     int(vc.get("timeout", 0)),
        "avg_hold": round(tdf["hold_sec"].mean(), 1),
        "max_dd":   round((cum - np.maximum.accumulate(cum)).min(), 2),
    }

# ── configs to test ──────────────────────────────────────────────────────────
# seconds for window start/end (IST offset baked into ts_sec)
W_FULL  = (10*3600+30*60, 14*3600+30*60)   # 10:30–14:30 (v11 baseline)
W_1300  = (10*3600+30*60, 13*3600)         # 10:30–13:00
W_1400  = (10*3600+30*60, 14*3600)         # 10:30–14:00

TP_BASE = {1: 1.0, 2: 1.0, 3: 1.0,  4: 0.60}   # v11 baseline
TP_S3   = {1: 1.0, 2: 1.0, 3: 0.68, 4: 0.60}   # State 3 tightened

CONFIGS = [
    # label                         thresh s2thr  state_tp  window
    ("v11 baseline",                0.70,  0.70,  TP_BASE,  W_FULL),
    ("+ State3 TP=0.68",            0.70,  0.70,  TP_S3,    W_FULL),
    ("+ State2 thresh=0.90",        0.70,  0.90,  TP_BASE,  W_FULL),
    ("+ window 10:30-14:00",        0.70,  0.70,  TP_BASE,  W_1400),
    ("+ window 10:30-13:00",        0.70,  0.70,  TP_BASE,  W_1300),
    ("S3tp + S2thr (no window)",    0.70,  0.90,  TP_S3,    W_FULL),
    ("S3tp + window-14:00",         0.70,  0.70,  TP_S3,    W_1400),
    ("S3tp + S2thr + window-14:00", 0.70,  0.90,  TP_S3,    W_1400),
    ("S3tp + S2thr + window-13:00", 0.70,  0.90,  TP_S3,    W_1300),
]

test_days = df_eval[df_eval["date"].isin(test_dates)].copy()
try:
    val_days = df_eval[df_eval["date"].isin(val_dates)].copy()
    has_val  = not val_days.empty
except NameError:
    has_val  = False

print("=" * 90)
print("v12b ISOLATION SWEEP")
print("=" * 90)
hdr = f"{'config':<36} {'n':>4} {'total':>7} {'sharpe':>8} {'hit%':>6} {'pct_tp':>7} {'n_to':>5} {'hold':>6} {'dd':>7}"
print(hdr)
print("-" * 90)

results = []
for label, bt, s2t, stp, win in CONFIGS:
    kw  = dict(base_thresh=bt, state2_thresh=s2t, state_tp=stp,
               window_start=win[0], window_end=win[1])
    tdf = run(test_days, **kw)
    m   = metrics(tdf)
    if not m:
        print(f"  {label:<36}  no trades"); continue
    results.append((label, m, kw))
    marker = " ◄" if m["sharpe"] > 2.09 else ""
    print(f"  {label:<36} {m['n']:>4} {m['total']:>7.2f} {m['sharpe']:>8.3f} "
          f"{m['hit']:>5.1f}% {m['pct_tp']:>6.1f}% {m['n_to']:>5} "
          f"{m['avg_hold']:>6.1f} {m['max_dd']:>7.2f}{marker}")

# ── hourly signal count — where do trades fire? ──────────────────────────────
print("\n── Trade count by hour (v11 baseline) ──────────────────────────────")
kw0 = dict(base_thresh=0.70, state2_thresh=0.70, state_tp=TP_BASE,
           window_start=W_FULL[0], window_end=W_FULL[1])
tdf0 = run(test_days, **kw0)
if not tdf0.empty:
    hc = tdf0.groupby("hour_ist").agg(
        n=("net_pnl_bps","count"),
        total=("net_pnl_bps","sum"),
        pct_tp=("exit_type", lambda x: round(100*(x=="tp").sum()/len(x),1))
    )
    for h, row in hc.iterrows():
        print(f"  Hour {h}:  n={int(row['n']):>3}  total={row['total']:>+6.2f}  pct_tp={row['pct_tp']}%")

# ── val set for best config ───────────────────────────────────────────────────
if has_val and results:
    best = max(results, key=lambda x: x[1]["sharpe"])
    print(f"\n── Validation set for best: '{best[0]}' ────────────────────────")
    tdf_v = run(val_days, **best[2])
    mv    = metrics(tdf_v)
    if mv:
        print(f"  n={mv['n']}  total={mv['total']}  sharpe={mv['sharpe']}  "
              f"hit={mv['hit']}%  pct_tp={mv['pct_tp']}%  "
              f"n_to={mv['n_to']}  max_dd={mv['max_dd']}")

print("=" * 90)

v12b ISOLATION SWEEP
config                                  n   total   sharpe   hit%  pct_tp  n_to   hold      dd
------------------------------------------------------------------------------------------
  v11 baseline                            7    3.37    1.793  71.4%   71.4%     2   55.4   -0.65
  + State3 TP=0.68                        7    2.85    1.537  71.4%   71.4%     2   54.0   -0.65
  + State2 thresh=0.90                    7    3.21    1.658  71.4%   71.4%     2   64.0   -0.65
  + window 10:30-14:00                    7    3.37    1.793  71.4%   71.4%     2   55.4   -0.65
  + window 10:30-13:00                    7    3.37    1.793  71.4%   71.4%     2   55.4   -0.65
  S3tp + S2thr (no window)                7    2.69    1.412  71.4%   71.4%     2   62.6   -0.65
  S3tp + window-14:00                     7    2.85    1.537  71.4%   71.4%     2   54.0   -0.65
  S3tp + S2thr + window-14:00             7    2.69    1.412  71.4%   71.4%     2   62.6   -0.65
  S3tp + S2thr + 

In [5]:
"""
DEBUG — paste into notebook cell to check what dates are available
"""
import pandas as pd

print("=== test_dates ===")
print(type(test_dates), len(test_dates))
for d in sorted(test_dates): print(" ", d)

print("\n=== val_dates ===")
try:
    print(type(val_dates), len(val_dates))
    for d in sorted(val_dates): print(" ", d)
except NameError:
    print("  val_dates NOT in memory")

print("\n=== df_eval date range ===")
all_dates = sorted(df_eval["date"].unique())
print(f"  total unique dates: {len(all_dates)}")
print(f"  first: {all_dates[0]}  last: {all_dates[-1]}")

print("\n=== df_eval dates that are in test_dates ===")
match = df_eval[df_eval["date"].isin(test_dates)]
print(f"  rows: {len(match)}  dates: {sorted(match['date'].unique())}")

print("\n=== rows in window per date (test_dates) ===")
win = match[match["in_window"] == True]
above = win[win["pred_norm"] > 0.70]
print(f"  in_window rows: {len(win)}")
print(f"  pred_norm > 0.70 rows: {len(above)}")
print(f"  dates with signal: {sorted(above['date'].unique())}")

print("\n=== all dates with pred_norm > 0.70 in window ===")
all_win = df_eval[df_eval["in_window"] == True]
all_sig = all_win[all_win["pred_norm"] > 0.70]
print(f"  total signal rows: {len(all_sig)}")
per_day = all_sig.groupby("date").size()
print(per_day.to_string())

=== test_dates ===
<class 'list'> 12
  2026-10-23
  2026-10-26
  2026-10-27
  2026-10-28
  2026-10-29
  2026-10-30
  2026-11-02
  2026-11-03
  2026-11-04
  2026-11-05
  2026-11-06
  2026-11-09

=== val_dates ===
<class 'list'> 5
  2026-10-15
  2026-10-16
  2026-10-19
  2026-10-21
  2026-10-22

=== df_eval date range ===
  total unique dates: 17
  first: 2026-10-15  last: 2026-11-09

=== df_eval dates that are in test_dates ===
  rows: 198256  dates: [datetime.date(2026, 10, 23), datetime.date(2026, 10, 26), datetime.date(2026, 10, 27), datetime.date(2026, 10, 28), datetime.date(2026, 10, 29), datetime.date(2026, 10, 30), datetime.date(2026, 11, 2), datetime.date(2026, 11, 3), datetime.date(2026, 11, 4), datetime.date(2026, 11, 5), datetime.date(2026, 11, 6), datetime.date(2026, 11, 9)]

=== rows in window per date (test_dates) ===
  in_window rows: 112900
  pred_norm > 0.70 rows: 169
  dates with signal: [datetime.date(2026, 10, 26), datetime.date(2026, 10, 27), datetime.date(2026, 11,

In [6]:
"""
FIX DATE FORMAT — paste and run BEFORE any backtest cell
Converts test_dates and val_dates from strings to datetime.date objects
so df_eval.isin() matches correctly.
"""
from datetime import date as _date

def to_date(x):
    if isinstance(x, _date): return x
    return _date.fromisoformat(str(x)[:10])

test_dates = [to_date(d) for d in test_dates]
val_dates  = [to_date(d) for d in val_dates]

# verify
import pandas as pd
test_match = df_eval[df_eval["date"].isin(test_dates)]
val_match  = df_eval[df_eval["date"].isin(val_dates)]

print(f"test_dates: {len(test_dates)} dates")
print(f"  df_eval rows matched: {len(test_match)}")
print(f"  unique dates matched: {sorted(test_match['date'].unique())}")

print(f"\nval_dates: {len(val_dates)} dates")
print(f"  df_eval rows matched: {len(val_match)}")
print(f"  unique dates matched: {sorted(val_match['date'].unique())}")

# quick signal check
win  = test_match[test_match["in_window"] == True]
sig  = win[win["pred_norm"] > 0.70]
print(f"\npred_norm > 0.70 in window across test_dates: {len(sig)} rows")
print(f"dates with signal: {sorted(sig['date'].unique())}")

test_dates: 12 dates
  df_eval rows matched: 198256
  unique dates matched: [datetime.date(2026, 10, 23), datetime.date(2026, 10, 26), datetime.date(2026, 10, 27), datetime.date(2026, 10, 28), datetime.date(2026, 10, 29), datetime.date(2026, 10, 30), datetime.date(2026, 11, 2), datetime.date(2026, 11, 3), datetime.date(2026, 11, 4), datetime.date(2026, 11, 5), datetime.date(2026, 11, 6), datetime.date(2026, 11, 9)]

val_dates: 5 dates
  df_eval rows matched: 94232
  unique dates matched: [datetime.date(2026, 10, 15), datetime.date(2026, 10, 16), datetime.date(2026, 10, 19), datetime.date(2026, 10, 21), datetime.date(2026, 10, 22)]

pred_norm > 0.70 in window across test_dates: 169 rows
dates with signal: [datetime.date(2026, 10, 26), datetime.date(2026, 10, 27), datetime.date(2026, 11, 2), datetime.date(2026, 11, 4), datetime.date(2026, 11, 5), datetime.date(2026, 11, 6), datetime.date(2026, 11, 9)]


In [7]:
"""
STRATEGY BACKTEST v12b — ISOLATION TEST
Tests each v12 change separately to find what helps vs hurts.
Assumes in memory: df_eval, mid_lookup, test_dates, val_dates
"""

import numpy as np
import pandas as pd

# ── shared params ────────────────────────────────────────────────────────────
COST_BPS       = 1.0
C_SL           = 1.2
STATE_VOL      = {1: 1.77, 2: 2.37, 3: 1.61, 4: 1.03}
STATE_MAX_HOLD = {1: 60,   2: 30,   3: 90,   4: 180}
STATE_SIZE     = {1: 0.59, 2: 0.44, 3: 0.65, 4: 1.00}

def simulate_day(df_day, base_thresh, state2_thresh,
                 state_tp, window_start, window_end):
    trades = []; busy_until = -1
    sym  = df_day["symbol"].iloc[0] if "symbol" in df_day.columns else None
    date = df_day["date"].iloc[0]

    for _, row in df_day.sort_values("ts_sec").iterrows():
        ts = int(row["ts_sec"])
        if ts <= busy_until:         continue
        if pd.isna(row["ret_60s"]): continue
        ts_in_day = ts % 86400
        if ts_in_day < window_start or ts_in_day >= window_end: continue

        pred  = float(row["pred_norm"])
        state = int(row["state"])
        thresh = state2_thresh if state == 2 else base_thresh
        if pred < thresh: continue

        vol      = STATE_VOL[state]
        tp_dist  = state_tp.get(state, 1.0) * vol
        sl_dist  = C_SL * vol
        max_hold = STATE_MAX_HOLD[state]
        size     = STATE_SIZE[state]
        em       = float(row["mid"])
        tp_price = em * (1 + tp_dist * 1e-4)
        sl_price = em * (1 - sl_dist * 1e-4)

        exit_mid = None; exit_ts = ts + max_hold; exit_type = "timeout"
        mae = 0.0; mfe = 0.0

        for dt in range(1, max_hold + 1):
            m = mid_lookup.get((sym, date, ts + dt))
            if m is None: continue
            m = float(m)
            r = (np.log(m) - np.log(em)) * 1e4
            if r < mae: mae = r
            if r > mfe: mfe = r
            if m >= tp_price:
                exit_mid = m; exit_ts = ts + dt; exit_type = "tp"; break
            if m <= sl_price:
                exit_mid = m; exit_ts = ts + dt; exit_type = "sl"; break

        if exit_mid is None:
            exit_mid = mid_lookup.get((sym, date, ts + max_hold))
            if exit_mid is None:
                exit_mid = em * (1 + float(row["ret_60s"]) * 1e-4)

        exit_mid   = float(exit_mid)
        actual_ret = (np.log(exit_mid) - np.log(em)) * 1e4
        net_pnl    = (actual_ret - COST_BPS) * size
        trades.append({
            "date": str(date), "ts_sec": ts,
            "hold_sec": exit_ts - ts, "pred_norm": pred,
            "actual_ret": actual_ret, "exit_type": exit_type,
            "state": state, "net_pnl_bps": net_pnl,
            "mae_bps": mae, "mfe_bps": mfe,
            "hour_ist": (ts % 86400) // 3600,
        })
        busy_until = exit_ts
    return trades

def run(df_set, **kw):
    t = []
    for _, g in df_set.groupby("date"):
        t.extend(simulate_day(g, **kw))
    return pd.DataFrame(t) if t else pd.DataFrame()

def metrics(tdf):
    if tdf.empty: return {}
    p   = tdf["net_pnl_bps"].values; n = len(p)
    std = p.std() or 1e-9
    cum = np.cumsum(p)
    vc  = tdf["exit_type"].value_counts()
    return {
        "n":        n,
        "total":    round(p.sum(), 2),
        "sharpe":   round(p.mean() / std * np.sqrt(n), 3),
        "hit":      round(np.mean(p > 0) * 100, 1),
        "pct_tp":   round(100 * vc.get("tp", 0) / n, 1),
        "n_tp":     int(vc.get("tp", 0)),
        "n_sl":     int(vc.get("sl", 0)),
        "n_to":     int(vc.get("timeout", 0)),
        "avg_hold": round(tdf["hold_sec"].mean(), 1),
        "max_dd":   round((cum - np.maximum.accumulate(cum)).min(), 2),
    }

# ── configs to test ──────────────────────────────────────────────────────────
# seconds for window start/end (IST offset baked into ts_sec)
W_FULL  = (10*3600+30*60, 14*3600+30*60)   # 10:30–14:30 (v11 baseline)
W_1300  = (10*3600+30*60, 13*3600)         # 10:30–13:00
W_1400  = (10*3600+30*60, 14*3600)         # 10:30–14:00

TP_BASE = {1: 1.0, 2: 1.0, 3: 1.0,  4: 0.60}   # v11 baseline
TP_S3   = {1: 1.0, 2: 1.0, 3: 0.68, 4: 0.60}   # State 3 tightened

CONFIGS = [
    # label                         thresh s2thr  state_tp  window
    ("v11 baseline",                0.70,  0.70,  TP_BASE,  W_FULL),
    ("+ State3 TP=0.68",            0.70,  0.70,  TP_S3,    W_FULL),
    ("+ State2 thresh=0.90",        0.70,  0.90,  TP_BASE,  W_FULL),
    ("+ window 10:30-14:00",        0.70,  0.70,  TP_BASE,  W_1400),
    ("+ window 10:30-13:00",        0.70,  0.70,  TP_BASE,  W_1300),
    ("S3tp + S2thr (no window)",    0.70,  0.90,  TP_S3,    W_FULL),
    ("S3tp + window-14:00",         0.70,  0.70,  TP_S3,    W_1400),
    ("S3tp + S2thr + window-14:00", 0.70,  0.90,  TP_S3,    W_1400),
    ("S3tp + S2thr + window-13:00", 0.70,  0.90,  TP_S3,    W_1300),
]

test_days = df_eval[df_eval["date"].isin(test_dates)].copy()
try:
    val_days = df_eval[df_eval["date"].isin(val_dates)].copy()
    has_val  = not val_days.empty
except NameError:
    has_val  = False

print("=" * 90)
print("v12b ISOLATION SWEEP")
print("=" * 90)
hdr = f"{'config':<36} {'n':>4} {'total':>7} {'sharpe':>8} {'hit%':>6} {'pct_tp':>7} {'n_to':>5} {'hold':>6} {'dd':>7}"
print(hdr)
print("-" * 90)

results = []
for label, bt, s2t, stp, win in CONFIGS:
    kw  = dict(base_thresh=bt, state2_thresh=s2t, state_tp=stp,
               window_start=win[0], window_end=win[1])
    tdf = run(test_days, **kw)
    m   = metrics(tdf)
    if not m:
        print(f"  {label:<36}  no trades"); continue
    results.append((label, m, kw))
    marker = " ◄" if m["sharpe"] > 2.09 else ""
    print(f"  {label:<36} {m['n']:>4} {m['total']:>7.2f} {m['sharpe']:>8.3f} "
          f"{m['hit']:>5.1f}% {m['pct_tp']:>6.1f}% {m['n_to']:>5} "
          f"{m['avg_hold']:>6.1f} {m['max_dd']:>7.2f}{marker}")

# ── hourly signal count — where do trades fire? ──────────────────────────────
print("\n── Trade count by hour (v11 baseline) ──────────────────────────────")
kw0 = dict(base_thresh=0.70, state2_thresh=0.70, state_tp=TP_BASE,
           window_start=W_FULL[0], window_end=W_FULL[1])
tdf0 = run(test_days, **kw0)
if not tdf0.empty:
    hc = tdf0.groupby("hour_ist").agg(
        n=("net_pnl_bps","count"),
        total=("net_pnl_bps","sum"),
        pct_tp=("exit_type", lambda x: round(100*(x=="tp").sum()/len(x),1))
    )
    for h, row in hc.iterrows():
        print(f"  Hour {h}:  n={int(row['n']):>3}  total={row['total']:>+6.2f}  pct_tp={row['pct_tp']}%")

# ── val set for best config ───────────────────────────────────────────────────
if has_val and results:
    best = max(results, key=lambda x: x[1]["sharpe"])
    print(f"\n── Validation set for best: '{best[0]}' ────────────────────────")
    tdf_v = run(val_days, **best[2])
    mv    = metrics(tdf_v)
    if mv:
        print(f"  n={mv['n']}  total={mv['total']}  sharpe={mv['sharpe']}  "
              f"hit={mv['hit']}%  pct_tp={mv['pct_tp']}%  "
              f"n_to={mv['n_to']}  max_dd={mv['max_dd']}")

print("=" * 90)

v12b ISOLATION SWEEP
config                                  n   total   sharpe   hit%  pct_tp  n_to   hold      dd
------------------------------------------------------------------------------------------
  v11 baseline                            7    3.37    1.793  71.4%   71.4%     2   55.4   -0.65
  + State3 TP=0.68                        7    2.85    1.537  71.4%   71.4%     2   54.0   -0.65
  + State2 thresh=0.90                    7    3.21    1.658  71.4%   71.4%     2   64.0   -0.65
  + window 10:30-14:00                    7    3.37    1.793  71.4%   71.4%     2   55.4   -0.65
  + window 10:30-13:00                    7    3.37    1.793  71.4%   71.4%     2   55.4   -0.65
  S3tp + S2thr (no window)                7    2.69    1.412  71.4%   71.4%     2   62.6   -0.65
  S3tp + window-14:00                     7    2.85    1.537  71.4%   71.4%     2   54.0   -0.65
  S3tp + S2thr + window-14:00             7    2.69    1.412  71.4%   71.4%     2   62.6   -0.65
  S3tp + S2thr + 

In [8]:
"""
DEBUG ts_sec format — paste and run to understand timestamp structure
"""
import pandas as pd, numpy as np
from datetime import datetime, timezone

test_days = df_eval[df_eval["date"].isin(test_dates)].copy()

# grab a sample row
sample = test_days[test_days["in_window"] == True].iloc[0]
ts = int(sample["ts_sec"])
date = sample["date"]

print(f"sample ts_sec: {ts}")
print(f"sample date:   {date}")
print(f"ts % 86400:    {ts % 86400}  ({ts % 86400 / 3600:.2f} hours)")

# try interpreting as unix timestamp
try:
    dt_utc = datetime.fromtimestamp(ts, tz=timezone.utc)
    print(f"as UTC datetime: {dt_utc}")
except:
    print("not a unix timestamp")

# check what in_window=True ts_sec values look like
win_rows = test_days[test_days["in_window"] == True]
print(f"\nts_sec stats for in_window=True rows:")
print(f"  min: {win_rows['ts_sec'].min()}")
print(f"  max: {win_rows['ts_sec'].max()}")
print(f"  sample values: {win_rows['ts_sec'].head(5).tolist()}")

# check signal rows specifically
sig = win_rows[win_rows["pred_norm"] > 0.70]
print(f"\nts_sec for pred_norm>0.70 rows (first 10):")
for _, r in sig.head(10).iterrows():
    ts_v = int(r["ts_sec"])
    mod  = ts_v % 86400
    print(f"  date={r['date']}  ts_sec={ts_v}  ts%86400={mod}  ({mod/3600:.2f}h)  pred={r['pred_norm']:.3f}")

# check what in_window column actually covers
print(f"\nin_window True count per date:")
print(test_days.groupby("date")["in_window"].sum().to_string())

# check if there's a separate time/hour column
print(f"\ncolumns: {list(test_days.columns)}")

sample ts_sec: 1793250000
sample date:   2026-10-29
ts % 86400:    18000  (5.00 hours)
as UTC datetime: 2026-10-29 05:00:00+00:00

ts_sec stats for in_window=True rows:
  min: 1792731615
  max: 1794214800
  sample values: [1793250000, 1793250001, 1793250002, 1793250003, 1793250004]

ts_sec for pred_norm>0.70 rows (first 10):
  date=2026-11-02  ts_sec=1793605421  ts%86400=27821  (7.73h)  pred=0.784
  date=2026-11-02  ts_sec=1793605422  ts%86400=27822  (7.73h)  pred=0.988
  date=2026-11-02  ts_sec=1793605423  ts%86400=27823  (7.73h)  pred=1.017
  date=2026-11-02  ts_sec=1793605424  ts%86400=27824  (7.73h)  pred=1.048
  date=2026-11-02  ts_sec=1793605425  ts%86400=27825  (7.73h)  pred=1.048
  date=2026-11-02  ts_sec=1793605426  ts%86400=27826  (7.73h)  pred=1.048
  date=2026-11-02  ts_sec=1793605427  ts%86400=27827  (7.73h)  pred=1.048
  date=2026-11-02  ts_sec=1793605428  ts%86400=27828  (7.73h)  pred=1.048
  date=2026-11-02  ts_sec=1793605429  ts%86400=27829  (7.73h)  pred=1.048
  date=

In [10]:
"""
STRATEGY BACKTEST v12c — ISOLATION SWEEP (fixed window filter)
Uses ist_sec column for window, not ts % 86400
Assumes in memory: df_eval, mid_lookup, test_dates, val_dates
"""

import numpy as np
import pandas as pd

# ── shared params ────────────────────────────────────────────────────────────
COST_BPS       = 1.0
C_SL           = 1.2
STATE_VOL      = {1: 1.77, 2: 2.37, 3: 1.61, 4: 1.03}
STATE_MAX_HOLD = {1: 60,   2: 30,   3: 90,   4: 180}
STATE_SIZE     = {1: 0.59, 2: 0.44, 3: 0.65, 4: 1.00}

# window in IST seconds-since-midnight
W_FULL = (10*3600+30*60, 14*3600+30*60)   # 10:30–14:30
W_1400 = (10*3600+30*60, 14*3600)         # 10:30–14:00
W_1300 = (10*3600+30*60, 13*3600)         # 10:30–13:00

TP_BASE = {1: 1.0, 2: 1.0, 3: 1.0,  4: 0.60}
TP_S3   = {1: 1.0, 2: 1.0, 3: 0.68, 4: 0.60}

def simulate_day(df_day, base_thresh, state2_thresh,
                 state_tp, window_start, window_end):
    trades = []; busy_until = -1
    sym  = df_day["symbol"].iloc[0] if "symbol" in df_day.columns else None
    date = df_day["date"].iloc[0]

    for _, row in df_day.sort_values("ts_sec").iterrows():
        ts     = int(row["ts_sec"])
        if pd.isna(row["ist_sec"]):  continue
        ist_s  = int(row["ist_sec"])          # ← use ist_sec for window
        if ts <= busy_until:         continue
        if pd.isna(row["ret_60s"]): continue
        if ist_s < window_start or ist_s >= window_end: continue

        pred  = float(row["pred_norm"])
        state = int(row["state"])
        thresh = state2_thresh if state == 2 else base_thresh
        if pred < thresh: continue

        vol      = STATE_VOL[state]
        tp_dist  = state_tp.get(state, 1.0) * vol
        sl_dist  = C_SL * vol
        max_hold = STATE_MAX_HOLD[state]
        size     = STATE_SIZE[state]
        em       = float(row["mid"])
        tp_price = em * (1 + tp_dist * 1e-4)
        sl_price = em * (1 - sl_dist * 1e-4)

        exit_mid = None; exit_ts = ts + max_hold; exit_type = "timeout"
        mae = 0.0; mfe = 0.0

        for dt in range(1, max_hold + 1):
            m = mid_lookup.get((sym, date, ts + dt))
            if m is None: continue
            m = float(m)
            r = (np.log(m) - np.log(em)) * 1e4
            if r < mae: mae = r
            if r > mfe: mfe = r
            if m >= tp_price:
                exit_mid = m; exit_ts = ts + dt; exit_type = "tp"; break
            if m <= sl_price:
                exit_mid = m; exit_ts = ts + dt; exit_type = "sl"; break

        if exit_mid is None:
            exit_mid = mid_lookup.get((sym, date, ts + max_hold))
            if exit_mid is None:
                exit_mid = em * (1 + float(row["ret_60s"]) * 1e-4)

        exit_mid   = float(exit_mid)
        actual_ret = (np.log(exit_mid) - np.log(em)) * 1e4
        net_pnl    = (actual_ret - COST_BPS) * size

        trades.append({
            "date": str(date), "ts_sec": ts, "ist_sec": ist_s,
            "hold_sec": exit_ts - ts, "pred_norm": pred,
            "actual_ret": actual_ret, "exit_type": exit_type,
            "state": state, "net_pnl_bps": net_pnl,
            "mae_bps": mae, "mfe_bps": mfe,
            "hour_ist": ist_s // 3600,
        })
        busy_until = exit_ts
    return trades

def run(df_set, **kw):
    t = []
    for _, g in df_set.groupby("date"):
        t.extend(simulate_day(g, **kw))
    return pd.DataFrame(t) if t else pd.DataFrame()

def metrics(tdf):
    if tdf.empty: return {}
    p   = tdf["net_pnl_bps"].values; n = len(p)
    std = p.std() or 1e-9
    cum = np.cumsum(p)
    vc  = tdf["exit_type"].value_counts()
    return {
        "n":        n,
        "total":    round(p.sum(), 2),
        "sharpe":   round(p.mean() / std * np.sqrt(n), 3),
        "hit":      round(np.mean(p > 0) * 100, 1),
        "pct_tp":   round(100 * vc.get("tp", 0) / n, 1),
        "n_tp":     int(vc.get("tp", 0)),
        "n_sl":     int(vc.get("sl", 0)),
        "n_to":     int(vc.get("timeout", 0)),
        "avg_hold": round(tdf["hold_sec"].mean(), 1),
        "max_dd":   round((cum - np.maximum.accumulate(cum)).min(), 2),
    }

CONFIGS = [
    ("v11 baseline",                0.70, 0.70, TP_BASE, W_FULL),
    ("+ State3 TP=0.68",            0.70, 0.70, TP_S3,   W_FULL),
    ("+ State2 thresh=0.90",        0.70, 0.90, TP_BASE, W_FULL),
    ("+ window 10:30-14:00",        0.70, 0.70, TP_BASE, W_1400),
    ("+ window 10:30-13:00",        0.70, 0.70, TP_BASE, W_1300),
    ("S3tp + S2thr",                0.70, 0.90, TP_S3,   W_FULL),
    ("S3tp + window-14:00",         0.70, 0.70, TP_S3,   W_1400),
    ("S3tp + S2thr + window-14:00", 0.70, 0.90, TP_S3,   W_1400),
    ("S3tp + S2thr + window-13:00", 0.70, 0.90, TP_S3,   W_1300),
]

test_days = df_eval[df_eval["date"].isin(test_dates)].copy()
val_days  = df_eval[df_eval["date"].isin(val_dates)].copy()

print("=" * 90)
print("v12c ISOLATION SWEEP  (window filter uses ist_sec)")
print("=" * 90)
print(f"{'config':<36} {'n':>4} {'total':>7} {'sharpe':>8} {'hit%':>6} "
      f"{'pct_tp':>7} {'n_to':>5} {'hold':>6} {'dd':>7}")
print("-" * 90)

results = []
for label, bt, s2t, stp, win in CONFIGS:
    kw  = dict(base_thresh=bt, state2_thresh=s2t, state_tp=stp,
               window_start=win[0], window_end=win[1])
    tdf = run(test_days, **kw)
    m   = metrics(tdf)
    if not m:
        print(f"  {label:<36}  no trades"); continue
    results.append((label, m, kw, tdf))
    marker = " ◄" if m["sharpe"] > 2.09 else ""
    print(f"  {label:<36} {m['n']:>4} {m['total']:>7.2f} {m['sharpe']:>8.3f} "
          f"{m['hit']:>5.1f}% {m['pct_tp']:>6.1f}% {m['n_to']:>5} "
          f"{m['avg_hold']:>6.1f} {m['max_dd']:>7.2f}{marker}")

# ── hourly breakdown of v11 baseline ────────────────────────────────────────
if results:
    tdf0 = results[0][3]
    print(f"\n── Hourly breakdown (v11 baseline) ─────────────────────────────")
    print(f"  {'hour':>5} {'n':>4} {'total':>7} {'sharpe':>8} {'hit%':>6} {'pct_tp':>7} {'n_to':>5}")
    for h, grp in tdf0.groupby("hour_ist"):
        m = metrics(grp)
        if not m: continue
        print(f"  {h:>5} {m['n']:>4} {m['total']:>7.2f} {m['sharpe']:>8.3f} "
              f"{m['hit']:>5.1f}% {m['pct_tp']:>6.1f}% {m['n_to']:>5}")

# ── exit breakdown of v11 baseline ──────────────────────────────────────────
if results:
    tdf0 = results[0][3]
    print(f"\n── Exit breakdown (v11 baseline) ───────────────────────────────")
    for et, grp in tdf0.groupby("exit_type"):
        p = grp["net_pnl_bps"].values
        print(f"  {et:>10}: n={len(p):>3}  mean_net={p.mean():>+6.3f}  "
              f"hit={np.mean(p>0)*100:>5.1f}%  "
              f"mean_mfe={grp['mfe_bps'].mean():>5.3f}  "
              f"mean_hold={grp['hold_sec'].mean():>5.1f}s")

# ── state breakdown of v11 baseline ─────────────────────────────────────────
if results:
    tdf0 = results[0][3]
    print(f"\n── State breakdown (v11 baseline) ──────────────────────────────")
    print(f"  {'state':>6} {'n':>4} {'total':>7} {'sharpe':>8} {'hit%':>6} {'pct_tp':>7} {'n_to':>5}")
    for s, grp in tdf0.groupby("state"):
        m = metrics(grp)
        if not m: continue
        print(f"  {s:>6} {m['n']:>4} {m['total']:>7.2f} {m['sharpe']:>8.3f} "
              f"{m['hit']:>5.1f}% {m['pct_tp']:>6.1f}% {m['n_to']:>5}")

# ── val set for best ─────────────────────────────────────────────────────────
if results:
    best = max(results, key=lambda x: x[1]["sharpe"])
    print(f"\n── Validation: best config = '{best[0]}' ───────────────────────")
    tdf_v = run(val_days, **best[2])
    mv    = metrics(tdf_v)
    if mv:
        print(f"  n={mv['n']}  total={mv['total']}  sharpe={mv['sharpe']}  "
              f"hit={mv['hit']}%  pct_tp={mv['pct_tp']}%  "
              f"n_to={mv['n_to']}  max_dd={mv['max_dd']}")
    else:
        print("  no trades in val set")

print("=" * 90)

v12c ISOLATION SWEEP  (window filter uses ist_sec)
config                                  n   total   sharpe   hit%  pct_tp  n_to   hold      dd
------------------------------------------------------------------------------------------
  v11 baseline                           30    8.86    2.075  73.3%   60.0%    11   36.7   -5.19
  + State3 TP=0.68                       30    6.88    1.751  76.7%   70.0%     8   29.5   -4.34
  + State2 thresh=0.90                   26    9.28    2.269  76.9%   65.4%     8   37.8   -4.29 ◄
  + window 10:30-14:00                   27    7.05    1.673  70.4%   59.3%    10   35.1   -5.19
  + window 10:30-13:00                   23    7.74    2.207  73.9%   60.9%     9   35.4   -3.69 ◄
  S3tp + S2thr                           26    6.78    1.744  80.8%   76.9%     5   30.2   -3.95
  S3tp + window-14:00                    27    6.09    1.552  74.1%   66.7%     8   29.4   -4.34
  S3tp + S2thr + window-14:00            23    6.00    1.542  78.3%   73.9%     

In [11]:
"""
STRATEGY BACKTEST v13 — FINAL CANDIDATE
Changes from v11:
  1. State2 threshold: 0.70 → 0.90  (drop low-conviction volatile-regime trades)
  2. Block Hour 13 (13:00–14:00 IST)  (consistent drag, Sharpe=-0.33)
  3. State3 TP unchanged at 1.0  (tightening hurts)
  4. Everything else identical to v11
Assumes in memory: df_eval, mid_lookup, test_dates, val_dates
"""

import numpy as np
import pandas as pd

# ── config ───────────────────────────────────────────────────────────────────
BASE_THRESH    = 0.70
STATE2_THRESH  = 0.90          # ← raised from 0.70
COST_BPS       = 1.0
C_SL           = 1.2
STATE_TP       = {1: 1.0, 2: 1.0, 3: 1.0, 4: 0.60}
STATE_VOL      = {1: 1.77, 2: 2.37, 3: 1.61, 4: 1.03}
STATE_MAX_HOLD = {1: 60,   2: 30,   3: 90,   4: 180}
STATE_SIZE     = {1: 0.59, 2: 0.44, 3: 0.65, 4: 1.00}

# window: 10:30–14:30 IST but SKIP 13:00–14:00
WINDOW_START = 10*3600 + 30*60   # 37800
WINDOW_END   = 14*3600 + 30*60   # 52200
SKIP_START   = 13*3600            # 46800
SKIP_END     = 14*3600            # 50400

print("v13 config:")
for s in STATE_VOL:
    tp = STATE_TP[s] * STATE_VOL[s]
    sl = C_SL * STATE_VOL[s]
    thr = STATE2_THRESH if s == 2 else BASE_THRESH
    print(f"  State{s}: TP={tp:.3f} bps  SL={sl:.3f} bps  "
          f"hold={STATE_MAX_HOLD[s]}s  size={STATE_SIZE[s]:.2f}  thresh={thr:.2f}")
print(f"  Window: 10:30-14:30 IST, skipping 13:00-14:00")

# ── simulator ────────────────────────────────────────────────────────────────
def simulate_day(df_day):
    trades = []; busy_until = -1
    sym  = df_day["symbol"].iloc[0] if "symbol" in df_day.columns else None
    date = df_day["date"].iloc[0]

    for _, row in df_day.sort_values("ts_sec").iterrows():
        ts = int(row["ts_sec"])
        if pd.isna(row["ist_sec"]): continue
        ist_s = int(row["ist_sec"])
        if ts <= busy_until:         continue
        if pd.isna(row["ret_60s"]): continue

        # window: in range but NOT in skip band
        if ist_s < WINDOW_START or ist_s >= WINDOW_END: continue
        if SKIP_START <= ist_s < SKIP_END:               continue

        pred  = float(row["pred_norm"])
        state = int(row["state"])
        thresh = STATE2_THRESH if state == 2 else BASE_THRESH
        if pred < thresh: continue

        vol      = STATE_VOL[state]
        tp_dist  = STATE_TP[state] * vol
        sl_dist  = C_SL * vol
        max_hold = STATE_MAX_HOLD[state]
        size     = STATE_SIZE[state]
        em       = float(row["mid"])
        tp_price = em * (1 + tp_dist * 1e-4)
        sl_price = em * (1 - sl_dist * 1e-4)

        exit_mid = None; exit_ts = ts + max_hold; exit_type = "timeout"
        mae = 0.0; mfe = 0.0

        for dt in range(1, max_hold + 1):
            m = mid_lookup.get((sym, date, ts + dt))
            if m is None: continue
            m = float(m)
            r = (np.log(m) - np.log(em)) * 1e4
            if r < mae: mae = r
            if r > mfe: mfe = r
            if m >= tp_price:
                exit_mid = m; exit_ts = ts + dt; exit_type = "tp"; break
            if m <= sl_price:
                exit_mid = m; exit_ts = ts + dt; exit_type = "sl"; break

        if exit_mid is None:
            exit_mid = mid_lookup.get((sym, date, ts + max_hold))
            if exit_mid is None:
                exit_mid = em * (1 + float(row["ret_60s"]) * 1e-4)

        exit_mid   = float(exit_mid)
        actual_ret = (np.log(exit_mid) - np.log(em)) * 1e4
        net_pnl    = (actual_ret - COST_BPS) * size

        trades.append({
            "date": str(date), "ts_sec": ts, "ist_sec": ist_s,
            "hold_sec": exit_ts - ts, "pred_norm": pred,
            "actual_ret": actual_ret, "exit_type": exit_type,
            "state": state, "net_pnl_bps": net_pnl,
            "mae_bps": mae, "mfe_bps": mfe,
            "hour_ist": ist_s // 3600,
        })
        busy_until = exit_ts
    return trades

def run(df_set):
    t = []
    for _, g in df_set.groupby("date"):
        t.extend(simulate_day(g))
    return pd.DataFrame(t) if t else pd.DataFrame()

def metrics(tdf, label=""):
    if tdf.empty:
        if label: print(f"{label}: no trades")
        return {}
    p   = tdf["net_pnl_bps"].values; n = len(p)
    std = p.std() or 1e-9
    cum = np.cumsum(p)
    vc  = tdf["exit_type"].value_counts()
    m = {
        "n":        n,
        "total":    round(p.sum(), 2),
        "sharpe":   round(p.mean() / std * np.sqrt(n), 3),
        "hit":      round(np.mean(p > 0) * 100, 1),
        "pct_tp":   round(100 * vc.get("tp", 0) / n, 1),
        "n_tp":     int(vc.get("tp", 0)),
        "n_sl":     int(vc.get("sl", 0)),
        "n_to":     int(vc.get("timeout", 0)),
        "avg_hold": round(tdf["hold_sec"].mean(), 1),
        "avg_ret":  round(p.mean(), 3),
        "max_dd":   round((cum - np.maximum.accumulate(cum)).min(), 2),
        "per_day":  round(p.sum() / tdf["date"].nunique(), 3),
        "trades_per_day": round(n / tdf["date"].nunique(), 2),
    }
    if label:
        print(f"\n{label}")
        print(f"  n={m['n']}  total={m['total']} bps  sharpe={m['sharpe']}  "
              f"hit={m['hit']}%  pct_tp={m['pct_tp']}%")
        print(f"  n_tp={m['n_tp']}  n_sl={m['n_sl']}  n_to={m['n_to']}  "
              f"avg_hold={m['avg_hold']}s  per_day={m['per_day']} bps  "
              f"trades/day={m['trades_per_day']}  max_dd={m['max_dd']}")
    return m

# ── run ───────────────────────────────────────────────────────────────────────
print("\n" + "=" * 68)
print("STRATEGY v13  |  State2=0.90  |  skip Hour13  |  10:30-14:30 IST")
print("=" * 68)

test_days = df_eval[df_eval["date"].isin(test_dates)].copy()
val_days  = df_eval[df_eval["date"].isin(val_dates)].copy()

tdf_test = run(test_days)
m_test   = metrics(tdf_test, "TEST SET")

# ── state breakdown ───────────────────────────────────────────────────────────
print("\n── State breakdown ─────────────────────────────────────────────────")
print(f"  {'state':>6} {'n':>4} {'total':>7} {'sharpe':>8} {'hit%':>6} "
      f"{'pct_tp':>7} {'n_to':>5} {'hold':>6}")
for s, grp in tdf_test.groupby("state"):
    m = metrics(grp)
    if not m: continue
    print(f"  {s:>6} {m['n']:>4} {m['total']:>7.2f} {m['sharpe']:>8.3f} "
          f"{m['hit']:>5.1f}% {m['pct_tp']:>6.1f}% {m['n_to']:>5} {m['avg_hold']:>6.1f}")

# ── hourly breakdown ──────────────────────────────────────────────────────────
print("\n── Hourly breakdown ────────────────────────────────────────────────")
print(f"  {'hour':>5} {'n':>4} {'total':>7} {'sharpe':>8} {'hit%':>6} {'pct_tp':>7} {'n_to':>5}")
for h, grp in tdf_test.groupby("hour_ist"):
    m = metrics(grp)
    if not m: continue
    print(f"  {h:>5} {m['n']:>4} {m['total']:>7.2f} {m['sharpe']:>8.3f} "
          f"{m['hit']:>5.1f}% {m['pct_tp']:>6.1f}% {m['n_to']:>5}")

# ── exit breakdown ────────────────────────────────────────────────────────────
print("\n── Exit breakdown ──────────────────────────────────────────────────")
for et, grp in tdf_test.groupby("exit_type"):
    p = grp["net_pnl_bps"].values
    print(f"  {et:>10}: n={len(p):>3}  mean_net={p.mean():>+6.3f}  "
          f"hit={np.mean(p>0)*100:>5.1f}%  "
          f"mean_mfe={grp['mfe_bps'].mean():>5.3f}  "
          f"mean_hold={grp['hold_sec'].mean():>5.1f}s")

# ── daily PnL ─────────────────────────────────────────────────────────────────
print("\n── Daily PnL ───────────────────────────────────────────────────────")
daily = tdf_test.groupby("date")["net_pnl_bps"].sum().reset_index()
daily.columns = ["date", "pnl"]
for _, row in daily.iterrows():
    bar  = "█" * max(0, int(abs(row["pnl"]) * 4))
    sign = "+" if row["pnl"] >= 0 else "-"
    print(f"  {row['date']}  {sign}{abs(row['pnl']):>5.2f}  {bar}")
win_days = (daily["pnl"] > 0).sum()
print(f"  win days: {win_days} / {len(daily)}")

# ── vs v11 ────────────────────────────────────────────────────────────────────
print(f"\n── vs v11 baseline ─────────────────────────────────────────────────")
print(f"  v11: n=30  total=8.86  sharpe=2.075  hit=73.3%  pct_tp=60.0%  "
      f"n_to=11  max_dd=-5.19")
print(f"  v13: n={m_test.get('n')}  total={m_test.get('total')}  "
      f"sharpe={m_test.get('sharpe')}  hit={m_test.get('hit')}%  "
      f"pct_tp={m_test.get('pct_tp')}%  "
      f"n_to={m_test.get('n_to')}  max_dd={m_test.get('max_dd')}")

# ── val set ───────────────────────────────────────────────────────────────────
tdf_val = run(val_days)
metrics(tdf_val, "VALIDATION SET")

print("\n" + "=" * 68)

v13 config:
  State1: TP=1.770 bps  SL=2.124 bps  hold=60s  size=0.59  thresh=0.70
  State2: TP=2.370 bps  SL=2.844 bps  hold=30s  size=0.44  thresh=0.90
  State3: TP=1.610 bps  SL=1.932 bps  hold=90s  size=0.65  thresh=0.70
  State4: TP=0.618 bps  SL=1.236 bps  hold=180s  size=1.00  thresh=0.70
  Window: 10:30-14:30 IST, skipping 13:00-14:00

STRATEGY v13  |  State2=0.90  |  skip Hour13  |  10:30-14:30 IST

TEST SET
  n=22  total=9.97 bps  sharpe=3.028  hit=81.8%  pct_tp=68.2%
  n_tp=15  n_sl=0  n_to=7  avg_hold=38.6s  per_day=1.995 bps  trades/day=4.4  max_dd=-2.78

── State breakdown ─────────────────────────────────────────────────
   state    n   total   sharpe   hit%  pct_tp  n_to   hold
       1   11    5.88    2.197  81.8%   72.7%     3   30.7
       2    1   -0.85 -845260491.992   0.0%    0.0%     1   30.0
       3   10    4.93    3.582  90.0%   70.0%     3   48.2

── Hourly breakdown ────────────────────────────────────────────────
   hour    n   total   sharpe   hit%  pct_tp

In [1]:
import os
from pathlib import Path

# change this to your actual models folder path
models_dir = Path(r"C:\projects\hedge_fund\execution\USDINR\models")

print(f"{'filename':<40} {'size':>8} {'modified'}")
print("-" * 70)
for f in sorted(models_dir.iterdir(), key=lambda x: x.stat().st_mtime, reverse=True):
    size = f.stat().st_size
    mtime = __import__('datetime').datetime.fromtimestamp(f.stat().st_mtime)
    size_str = f"{size/1024:.1f} KB" if size < 1024*1024 else f"{size/1024/1024:.1f} MB"
    print(f"{f.name:<40} {size_str:>8}   {mtime.strftime('%Y-%m-%d %H:%M')}")

filename                                     size modified
----------------------------------------------------------------------


In [2]:
import os
from pathlib import Path

root = Path(r"C:\projects\hedge_fund")

exts = {'.pkl', '.txt', '.json', '.joblib', '.h5', '.pt', '.npy'}
print(f"{'path':<70} {'size':>8}   {'modified'}")
print("-" * 90)
for f in sorted(root.rglob("*"), key=lambda x: x.stat().st_mtime if x.is_file() else 0, reverse=True):
    if f.is_file() and f.suffix in exts:
        size = f.stat().st_size
        mtime = __import__('datetime').datetime.fromtimestamp(f.stat().st_mtime)
        size_str = f"{size/1024:.1f} KB" if size < 1024*1024 else f"{size/1024/1024:.1f} MB"
        rel = str(f.relative_to(root))
        print(f"{rel:<70} {size_str:>8}   {mtime.strftime('%Y-%m-%d %H:%M')}")

path                                                                       size   modified
------------------------------------------------------------------------------------------
models\lgbm_v11.txt                                                    147.5 KB   2026-11-11 15:12
models\lgbm_v10.txt                                                    147.5 KB   2026-11-11 14:26
results\strategy_backtest_v9\state4_analysis.json                        0.1 KB   2026-11-11 14:17
models\lgbm_v8.txt                                                     147.5 KB   2026-11-11 14:07
models\hmm_v8.pkl                                                        1.3 KB   2026-11-11 14:07
results\strategy_backtest_v7\metrics.json                                0.6 KB   2026-11-11 13:59
models\lgbm_v7.txt                                                     147.5 KB   2026-11-11 13:58
models\hmm_v7.pkl                                                        1.3 KB   2026-11-11 13:58
results\strategy_backtest_

In [3]:
import shutil, json
from pathlib import Path

src = Path(r"C:\projects\hedge_fund")
dst = Path(r"C:\projects\hedge_fund\execution\USDINR\models")
dst.mkdir(parents=True, exist_ok=True)

# copy correct model files with clean names
shutil.copy(src / r"models\lgbm_v11.txt",      dst / "lgbm_model.txt")
shutil.copy(src / r"models\hmm_v8.pkl",         dst / "hmm_model.pkl")
shutil.copy(src / r"research\notebooks\usdinr_final_v2\usdinr_feature_manifest_v2.json",
                                                 dst / "feature_manifest.json")

# save v13 strategy config
config = {
    "base_thresh":    0.70,
    "state2_thresh":  0.90,
    "cost_bps":       1.0,
    "c_sl":           1.2,
    "state_tp":       {"1": 1.0, "2": 1.0, "3": 1.0, "4": 0.60},
    "state_vol":      {"1": 1.77, "2": 2.37, "3": 1.61, "4": 1.03},
    "state_max_hold": {"1": 60,   "2": 30,   "3": 90,   "4": 180},
    "state_size":     {"1": 0.59, "2": 0.44, "3": 0.65, "4": 1.00},
    "window_start_ist": "10:30",
    "window_end_ist":   "14:30",
    "skip_start_ist":   "13:00",
    "skip_end_ist":     "14:00",
    "instrument":     "USDINR",
    "lots":           5,
    "bps_per_lot":    8.40,
    "version":        "v13"
}
with open(dst / "strategy_config.json", "w") as f:
    json.dump(config, f, indent=2)

print("Models folder contents:")
for f in sorted(dst.iterdir()):
    print(f"  {f.name:<35} {f.stat().st_size/1024:.1f} KB")

Models folder contents:
  feature_manifest.json               6.1 KB
  hmm_model.pkl                       1.3 KB
  lgbm_model.txt                      147.5 KB
  strategy_config.json                0.6 KB


In [4]:
import os
from pathlib import Path

src = Path(r"C:\projects\hedge_fund")

# show all .py files outside venv and notebooks
print(f"{'path':<60} {'size':>8}   {'modified'}")
print("-" * 80)
for f in sorted(src.rglob("*.py"), key=lambda x: x.stat().st_mtime, reverse=True):
    if "venv" in f.parts or "__pycache__" in f.parts: continue
    rel  = str(f.relative_to(src))
    size = f.stat().st_size
    mtime = __import__('datetime').datetime.fromtimestamp(f.stat().st_mtime)
    size_str = f"{size/1024:.1f} KB"
    print(f"{rel:<60} {size_str:>8}   {mtime.strftime('%Y-%m-%d %H:%M')}")

path                                                             size   modified
--------------------------------------------------------------------------------
execution\usdinr\test\test_signal.py                           0.0 KB   2026-11-11 23:39
execution\usdinr\test\test_features.py                         0.0 KB   2026-11-11 23:39
execution\usdinr\live\runner.py                                0.0 KB   2026-11-11 23:39
execution\usdinr\live\logger.py                                0.0 KB   2026-11-11 23:38
execution\usdinr\live\broker.py                                0.0 KB   2026-11-11 23:38
execution\usdinr\live\risk.py                                  0.0 KB   2026-11-11 23:38
execution\usdinr\live\execution.py                             0.0 KB   2026-11-11 23:38
execution\usdinr\live\signal.py                                0.0 KB   2026-11-11 23:38
execution\usdinr\live\features.py                              0.0 KB   2026-11-11 23:38
execution\usdinr\live\config.py      